<a href="https://colab.research.google.com/github/menagoubran/menagoubran/blob/claude%2Ffix-govcon-dashboard-UWAtP/GovCon_Front_End.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

token = userdata.get('govcon-deploy')
if not os.path.exists('/content/govcon-dashboard'):
    os.system(f'git clone https://{token}@github.com/menagoubran/govcon-dashboard.git /content/govcon-dashboard')
    print("Cloned")
else:
    os.system('cd /content/govcon-dashboard && git pull')
    print("Pulled latest")

Cloned


In [ ]:
%cd /content/govcon-dashboard
!npm install

/content/govcon-dashboard
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 387 packages, and audited 388 packages in 31s
⠹
⠹149 packages are looking for funding
⠹  run `npm fund` for details
⠹
3 vulnerabilities (1 moderate, 2 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues, run:
  npm audit fix --force

Run `npm audit` for details.
⠸

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/(dashboard)/expiring", exist_ok=True)
os.makedirs("app/api/expiring", exist_ok=True)

# API Route
Path("app/api/expiring/route.ts").write_text("""import { NextRequest, NextResponse } from 'next/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

export async function GET(req: NextRequest) {
  const { searchParams } = new URL(req.url)
  const days = parseInt(searchParams.get('days') || '180')
  const naics = searchParams.get('naics') || ''
  const state = searchParams.get('state') || ''
  const page = parseInt(searchParams.get('page') || '1')
  const limit = 50
  const from = (page - 1) * limit
  const to = from + limit - 1

  const today = new Date().toISOString().split('T')[0]
  const future = new Date(Date.now() + days * 24 * 60 * 60 * 1000).toISOString().split('T')[0]

  let query = supabaseAdmin
    .from('contracts')
    .select('*', { count: 'exact' })
    .gte('award_date', today)
    .lte('award_date', future)
    .gt('award_amount', 0)

  if (naics) query = query.eq('naics_code', naics)
  if (state) query = query.eq('place_of_performance_state', state)

  query = query.order('award_date', { ascending: true }).range(from, to)

  const { data, error, count } = await query

  if (error) return NextResponse.json({ error: error.message }, { status: 500 })

  // Compute RPS score for each contract
  const scored = ((data || []) as any[]).map((c: any) => {
    const expiry = new Date(c.award_date)
    const daysLeft = Math.ceil((expiry.getTime() - Date.now()) / (1000 * 60 * 60 * 24))
    let rps = 0
    if (daysLeft <= 30) rps = 95
    else if (daysLeft <= 60) rps = 85
    else if (daysLeft <= 90) rps = 75
    else if (daysLeft <= 120) rps = 60
    else if (daysLeft <= 180) rps = 45
    return { ...c, days_left: daysLeft, rps_score: rps }
  })

  return NextResponse.json({ data: scored, count, page, totalPages: Math.ceil((count || 0) / limit) })
}
""")

# Page Component
Path("app/(dashboard)/expiring/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'
import { useRouter } from 'next/navigation'

type Contract = {
  id: string
  contractor_name: string
  agency: string
  award_amount: number
  award_date: string
  place_of_performance_state: string
  naics_code: string
  days_left: number
  rps_score: number
}

const NAICS_OPTIONS = [
  { value: '', label: 'All NAICS' },
  { value: '238220', label: '238220 — HVAC' },
  { value: '541511', label: '541511 — IT Services' },
  { value: '236220', label: '236220 — Construction' },
  { value: '561730', label: '561730 — Landscaping' },
  { value: '541611', label: '541611 — Management Consulting' },
  { value: '237310', label: '237310 — Highway Construction' },
  { value: '238210', label: '238210 — Electrical' },
  { value: '238160', label: '238160 — Roofing' },
  { value: '541330', label: '541330 — Engineering' },
  { value: '541512', label: '541512 — Computer Systems' },
]

const STATES = ['','AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC']

function RPSBadge({ score }: { score: number }) {
  const color = score >= 85 ? 'bg-red-500' : score >= 70 ? 'bg-orange-500' : score >= 50 ? 'bg-yellow-500' : 'bg-slate-600'
  return (
    <span className={`${color} text-white text-xs font-bold px-2 py-0.5 rounded`}>
      RPS {score}
    </span>
  )
}

function DaysLeftBadge({ days }: { days: number }) {
  const color = days <= 30 ? 'text-red-400' : days <= 60 ? 'text-orange-400' : days <= 90 ? 'text-yellow-400' : 'text-slate-400'
  return <span className={`${color} font-medium text-sm`}>{days}d left</span>
}

export default function ExpiringPage() {
  const router = useRouter()
  const [data, setData] = useState<Contract[]>([])
  const [page, setPage] = useState(1)
  const [total, setTotal] = useState(0)
  const [totalPages, setTotalPages] = useState(0)
  const [loading, setLoading] = useState(false)
  const [days, setDays] = useState('180')
  const [naics, setNaics] = useState('')
  const [state, setState] = useState('')

  const fetchData = async (p = 1) => {
    setLoading(true)
    const params = new URLSearchParams({ days, page: String(p) })
    if (naics) params.set('naics', naics)
    if (state) params.set('state', state)
    const res = await fetch('/api/expiring?' + params.toString())
    const json = await res.json()
    setData(json.data || [])
    setTotal(json.count || 0)
    setTotalPages(json.totalPages || 0)
    setLoading(false)
  }

  useEffect(() => { fetchData(page) }, [page])

  const handleSearch = () => { setPage(1); fetchData(1) }

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen">
      <div className="mb-6">
        <h1 className="text-2xl font-bold text-blue-400">Recompete Radar</h1>
        <p className="text-slate-400 text-sm mt-1">Contracts expiring soon — ranked by Recompete Probability Score</p>
      </div>

      {/* Filters */}
      <div className="bg-slate-900 border border-slate-700 rounded-lg p-4 mb-4 flex flex-wrap gap-3 items-end">
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Expiring Within</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm" value={days} onChange={e => setDays(e.target.value)}>
            <option value="30">30 days</option>
            <option value="60">60 days</option>
            <option value="90">90 days</option>
            <option value="180">180 days</option>
            <option value="365">1 year</option>
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">NAICS</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-48" value={naics} onChange={e => setNaics(e.target.value)}>
            {NAICS_OPTIONS.map(o => <option key={o.value} value={o.value}>{o.label}</option>)}
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">State</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-24" value={state} onChange={e => setState(e.target.value)}>
            {STATES.map(s => <option key={s} value={s}>{s || 'All'}</option>)}
          </select>
        </div>
        <button onClick={handleSearch} className="bg-blue-600 hover:bg-blue-500 text-white px-4 py-1.5 rounded text-sm font-medium">Search</button>
      </div>

      {/* Stats bar */}
      <div className="flex gap-4 mb-4">
        <div className="bg-red-950 border border-red-800 rounded px-4 py-2 text-sm">
          <span className="text-red-400 font-bold">🔴 Critical</span>
          <span className="text-slate-400 ml-2">RPS 85+ (≤60 days)</span>
        </div>
        <div className="bg-orange-950 border border-orange-800 rounded px-4 py-2 text-sm">
          <span className="text-orange-400 font-bold">🟠 High</span>
          <span className="text-slate-400 ml-2">RPS 70+ (≤90 days)</span>
        </div>
        <div className="bg-yellow-950 border border-yellow-800 rounded px-4 py-2 text-sm">
          <span className="text-yellow-400 font-bold">🟡 Medium</span>
          <span className="text-slate-400 ml-2">RPS 50+ (≤120 days)</span>
        </div>
      </div>

      <div className="text-sm text-slate-400 mb-2">
        {loading ? 'Loading...' : `${total.toLocaleString()} contracts — Page ${page} of ${totalPages}`}
      </div>

      <table className="w-full border border-slate-800 text-sm">
        <thead className="bg-slate-800 text-slate-300">
          <tr>
            <th className="p-2 text-left">RPS</th>
            <th className="p-2 text-left">Contractor</th>
            <th className="p-2 text-left">Agency</th>
            <th className="p-2 text-left">Amount</th>
            <th className="p-2 text-left">Expires</th>
            <th className="p-2 text-left">State</th>
            <th className="p-2 text-left">NAICS</th>
          </tr>
        </thead>
        <tbody>
          {data.map((c, i) => (
            <tr
              key={c.id}
              onClick={() => router.push(`/contracts/${c.id}`)}
              className={(i % 2 === 0 ? 'bg-slate-900' : 'bg-slate-800/50') + ' cursor-pointer hover:bg-slate-700 transition-colors'}
            >
              <td className="p-2"><RPSBadge score={c.rps_score} /></td>
              <td className="p-2 font-medium">{c.contractor_name}</td>
              <td className="p-2 text-slate-400">{c.agency}</td>
              <td className="p-2 text-green-400 font-medium">
                {c.award_amount?.toLocaleString('en-US', { style: 'currency', currency: 'USD' })}
              </td>
              <td className="p-2"><DaysLeftBadge days={c.days_left} /></td>
              <td className="p-2">{c.place_of_performance_state}</td>
              <td className="p-2 text-slate-400">{c.naics_code}</td>
            </tr>
          ))}
        </tbody>
      </table>

      <div className="flex items-center gap-3 mt-4">
        <button onClick={() => setPage(p => Math.max(1, p - 1))} disabled={page === 1}
          className="bg-slate-800 hover:bg-slate-700 disabled:opacity-40 px-4 py-1.5 rounded text-sm">Prev</button>
        <span className="text-sm text-slate-400">Page {page} / {totalPages}</span>
        <button onClick={() => setPage(p => Math.min(totalPages, p + 1))} disabled={page === totalPages}
          className="bg-slate-800 hover:bg-slate-700 disabled:opacity-40 px-4 py-1.5 rounded text-sm">Next</button>
      </div>
    </div>
  )
}
""")

print("Files written:")
print("  app/api/expiring/route.ts")
print("  app/(dashboard)/expiring/page.tsx")

/content/govcon-dashboard
Files written:
  app/api/expiring/route.ts
  app/(dashboard)/expiring/page.tsx


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Add Recompete Radar page with RPS scoring" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

Attention: Next.js now collects completely anonymous telemetry regarding usage.
This information is used to shape Next.js' roadmap and prioritize features.
You can learn more, including how to opt-out if you'd not like to participate in this anonymous program, by visiting the following URL:
https://nextjs.org/telemetry

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 23.8s
✓ Finished TypeScript in 17.1s 
✓ Collecting page data using 1 worker in 960.0ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (1/30)  [=   ]  Generating static pages using 1 worker (11/30)  [==  ]  Generating static pages using 1 worker (25/30)  [=== ]✓ Generating static pages using 1 worker (30/30) in 623.6ms
✓

In [ ]:
%cd /content/govcon-dashboard
!git config user.email "menagoubran@gmail.com"
!git config user.name "Mena Goubran"
!git add -A && git commit -m "Add Recompete Radar page with RPS scoring" && git push origin main

/content/govcon-dashboard
[main 03478ad] Add Recompete Radar page with RPS scoring
 2 files changed, 219 insertions(+)
 create mode 100644 app/(dashboard)/expiring/page.tsx
 create mode 100644 app/api/expiring/route.ts
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (9/9), 3.71 KiB | 1.85 MiB/s, done.
Total 9 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/menagoubran/govcon-dashboard.git
   df26dd9..03478ad  main -> main


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

# Add Recompete Radar to navigation
# First find the nav file
import subprocess
result = subprocess.run(['grep', '-r', 'Lobbying Firms', 'components', 'app'], capture_output=True, text=True)
print(result.stdout[:2000])

/content/govcon-dashboard
components/DashboardShell.tsx:  { href: '/lobbying-firms', label: 'Lobbying Firms' },
app/(dashboard)/contractors/[id]/page.tsx: *   Lobbying Firms    — named firms hired (Senate LDA), fees paid, issues, agencies
app/(dashboard)/contractors/[id]/page.tsx:// ── Lobbying Firms Tab ─────────────────────────────────────────────────────
app/(dashboard)/contractors/[id]/page.tsx:          label="Lobbying Firms Hired"
app/(dashboard)/contractors/[id]/page.tsx:    { id: "lobbying" as const, label: "Lobbying Firms" },
app/(dashboard)/contractors/[id]/page.tsx:      {/* ── Lobbying Firms Tab ────────────────────────────────────────────── */}
app/(dashboard)/contracts/[id]/page.tsx:            {/* Lobbying Firms */}
app/(dashboard)/contracts/[id]/page.tsx:                  Lobbying Firms Hired ({influence!.lobbying_firms.length})
app/(dashboard)/lobbying-firms/page.tsx:        <h1 className="text-xl font-semibold text-white">Lobbying Firms</h1>



In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("components/DashboardShell.tsx")
text = path.read_text()

text = text.replace(
    "{ href: '/lobbying-firms', label: 'Lobbying Firms' },",
    "{ href: '/lobbying-firms', label: 'Lobbying Firms' },\n  { href: '/expiring', label: 'Recompete Radar' },"
)

path.write_text(text)
print("Patched")

# Verify
for line in text.split('\n'):
    if 'href' in line and ('lobbying' in line or 'expiring' in line or 'contract' in line):
        print(line.strip())

/content/govcon-dashboard
Patched
{ href: '/contracts', label: 'Contracts' },
{ href: '/lobbying-firms', label: 'Lobbying Firms' },
{ href: '/expiring', label: 'Recompete Radar' },


In [ ]:
%cd /content/govcon-dashboard
!git config user.email "menagoubran@gmail.com"
!git config user.name "Mena Goubran"
!npm run build && git add -A && git commit -m "Add Recompete Radar to navigation" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 23.5s
✓ Finished TypeScript in 17.1s 
✓ Collecting page data using 1 worker in 843.8ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (4/30)  [=   ]  Generating static pages using 1 worker (25/30)  [==  ]✓ Generating static pages using 1 worker (30/30) in 433.3ms
✓ Finalizing page optimization in 8.5ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins

In [ ]:
import requests
from google.colab import userdata

url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_ROLE')
headers = {'apikey': key, 'Authorization': f'Bearer {key}'}

r = requests.get(
    f"{url}/rest/v1/contracts?select=id,contractor_name,award_date,raw_data&limit=3&raw_data=not.is.null",
    headers=headers
)
rows = r.json()
for row in rows:
    print(f"Contractor: {row['contractor_name']}")
    print(f"Award date: {row['award_date']}")
    if row.get('raw_data'):
        import json
        rd = row['raw_data'] if isinstance(row['raw_data'], dict) else json.loads(row['raw_data'])
        # Show keys that contain 'period' or 'end' or 'expir'
        relevant = {k: v for k, v in rd.items() if any(x in k.lower() for x in ['period', 'end', 'expir', 'date'])}
        print(f"Relevant fields: {relevant}")
    print()

Contractor: DALEY TOWER SERVICE, INC.
Award date: 2023-04-10
Relevant fields: {'Start Date': '2023-04-10', 'Last Modified Date': '2024-08-15T14:47:04'}

Contractor: TOTAL ELECTRIC SERVICE OF TAMPA INC
Award date: 2023-03-13
Relevant fields: {'Start Date': '2023-03-13', 'Last Modified Date': '2023-06-20T13:48:08'}

Contractor: KAIVA SERVICES, LLC
Award date: 2023-09-16
Relevant fields: {'Start Date': '2023-09-16', 'Last Modified Date': '2024-09-17T16:55:59'}



In [ ]:
import requests
from google.colab import userdata

url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_ROLE')
headers = {'apikey': key, 'Authorization': f'Bearer {key}'}

r = requests.get(
    f"{url}/rest/v1/contracts?select=raw_data&limit=1&raw_data=not.is.null",
    headers=headers
)
row = r.json()[0]
import json
rd = row['raw_data'] if isinstance(row['raw_data'], dict) else json.loads(row['raw_data'])
print("All raw_data keys:")
for k, v in rd.items():
    print(f"  {k}: {v}")

All raw_data keys:
  Award ID: 1305M323PNWWP0227
  psc_code: N066
  Start Date: 2023-04-10
  naics_code: 238210
  Description: NONPERSONAL SERVICES AND PRODUCT PURCHASE FOR REPLACEMENT ANTENNA/CABLING REPLACEMENT AT BURAS, BATON ROUGE AND JENA, LOUISIANA NWR SITES, IN ACCORDANCE WITH THE INCORPORATED STATEMENT OF WORK.
  agency_slug: department-of-commerce
  internal_id: 277592444
  Award Amount: 48728.2
  Recipient Name: DALEY TOWER SERVICE, INC.
  Awarding Agency: Department of Commerce
  Last Modified Date: 2024-08-15T14:47:04
  awarding_agency_id: 183
  generated_internal_id: CONT_AWD_1305M323PNWWP0227_1330_-NONE-_-NONE-
  Place of Performance State Code: LA


In [ ]:
import requests
import time
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE')
sb = {'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}', 'Content-Type': 'application/json'}

# First check if expiry_date column exists, if not create it
r = requests.get(f"{SUPABASE_URL}/rest/v1/contracts?select=id&limit=1", headers=sb)
sample = r.json()
print("Sample contract keys:", list(sample[0].keys()) if sample else "no data")

Sample contract keys: ['id']


In [ ]:
import requests
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE')
sb = {'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}'}

r = requests.get(
    f"{SUPABASE_URL}/rest/v1/contracts?select=*&limit=1",
    headers=sb
)
sample = r.json()
print("All contract columns:", list(sample[0].keys()) if sample else "no data")

All contract columns: ['id', 'contractor_name', 'contractor_uei', 'agency', 'sub_agency', 'naics_code', 'psc_code', 'award_amount', 'award_date', 'period_of_performance_start', 'period_of_performance_end', 'place_of_performance_city', 'place_of_performance_state', 'description', 'contract_type', 'set_aside_type', 'raw_data', 'fetched_at', 'created_at']


In [ ]:
import requests
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE')
sb = {'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}'}

# Check how many rows have period_of_performance_end populated
r = requests.get(
    f"{SUPABASE_URL}/rest/v1/contracts?select=id&period_of_performance_end=not.is.null&limit=1",
    headers={**sb, 'Prefer': 'count=exact'}
)
print("Rows with end date:", r.headers.get('content-range'))

# Show 3 samples
r2 = requests.get(
    f"{SUPABASE_URL}/rest/v1/contracts?select=contractor_name,award_date,period_of_performance_end&period_of_performance_end=not.is.null&limit=3",
    headers=sb
)
print("Samples:", r2.json())

Rows with end date: None
Samples: [{'contractor_name': 'XATOR LLC', 'award_date': '2019-12-03', 'period_of_performance_end': '2027-01-05'}, {'contractor_name': 'XATOR LLC', 'award_date': '2013-05-23', 'period_of_performance_end': '2020-01-14'}, {'contractor_name': 'GENERAL DYNAMICS ONE SOURCE LLC', 'award_date': '2013-07-26', 'period_of_performance_end': '2026-09-29'}]


In [ ]:
import requests
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE')
sb = {'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}'}

r = requests.get(
    f"{SUPABASE_URL}/rest/v1/contracts?select=period_of_performance_end&period_of_performance_end=not.is.null&limit=1",
    headers={**sb, 'Prefer': 'count=exact'}
)
print("Content-Range:", r.headers.get('content-range'))

r2 = requests.get(
    f"{SUPABASE_URL}/rest/v1/contracts?select=period_of_performance_end&period_of_performance_end=is.null&limit=1",
    headers={**sb, 'Prefer': 'count=exact'}
)
print("Null count:", r2.headers.get('content-range'))

Content-Range: 0-0/80434
Null count: 0-0/126163


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

route = Path("app/api/expiring/route.ts")
text = route.read_text()

# Replace award_date with period_of_performance_end
text = text.replace(
    "const today = new Date().toISOString().split('T')[0]\n  const future = new Date(Date.now() + days * 24 * 60 * 60 * 1000).toISOString().split('T')[0]\n\n  let query = supabaseAdmin\n    .from('contracts')\n    .select('*', { count: 'exact' })\n    .gte('award_date', today)\n    .lte('award_date', future)\n    .gt('award_amount', 0)",
    "const today = new Date().toISOString().split('T')[0]\n  const future = new Date(Date.now() + days * 24 * 60 * 60 * 1000).toISOString().split('T')[0]\n\n  let query = supabaseAdmin\n    .from('contracts')\n    .select('*', { count: 'exact' })\n    .gte('period_of_performance_end', today)\n    .lte('period_of_performance_end', future)\n    .gt('award_amount', 0)\n    .not('period_of_performance_end', 'is', null)"
)

# Fix the days_left calculation to use period_of_performance_end
text = text.replace(
    "const expiry = new Date(c.award_date)",
    "const expiry = new Date(c.period_of_performance_end)"
)

route.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Fix Recompete Radar to use period_of_performance_end" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 21.5s
✓ Finished TypeScript in 19.3s 
✓ Collecting page data using 1 worker in 1240.1ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (1/30)  [=   ]  Generating static pages using 1 worker (12/30)  [==  ]  Generating static pages using 1 worker (26/30)  [=== ]✓ Generating static pages using 1 worker (30/30) in 618.4ms
✓ Finalizing page optimization in 16.0ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lob

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

route = Path("app/api/contracts/[id]/route.ts")
text = route.read_text()
print(text[1200:1800])

/content/govcon-dashboard
client_matches: [] }

  if ((contract as any).contractor_name) {
    const coreName = stripLegalSuffix((contract as any).contractor_name)
    const searchTerm = coreName.split(' ').slice(0, 3).join(' ')

    // Find matching lobbying clients
    const { data: clients } = await supabaseAdmin
      .from('lobbying_clients')
      .select('name, firm_id, general_description')
      .ilike('general_description', '%' + (contract as any).contractor_name + '%')
      .limit(10)

    if (clients && clients.length > 0) {
      const firmIds = [...new Set((clients as any[]).map((c: any) => c.firm_id).fi


In [ ]:
import requests
from google.colab import userdata

url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_ROLE')
headers = {'apikey': key, 'Authorization': f'Bearer {key}'}

r = requests.get(
    f"{url}/rest/v1/lobbying_clients?general_description=ilike.*CHEMONICS*&select=id,name,firm_id,general_description",
    headers=headers
)
print(r.json())

[{'id': 'f80cbc4f-ddef-440e-9d63-2ce304be9a28', 'name': 'CHEMONICS INTERNATIONAL, INC.', 'firm_id': '52c09850-6c9d-4a0f-ae44-b11a4b9133ed', 'general_description': 'Matched from contracts: CHEMONICS INTERNATIONAL, INC.'}, {'id': '60d7adc3-7458-4d5b-b1ac-052436ca57a7', 'name': 'CHEMONICS INTERNATIONAL', 'firm_id': None, 'general_description': 'Matched from contracts: CHEMONICS INTERNATIONAL, INC.'}]


In [ ]:
import requests
from google.colab import userdata

url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_ROLE')
headers = {'apikey': key, 'Authorization': f'Bearer {key}'}

r = requests.get(
    f"{url}/rest/v1/lobbying_firms?id=eq.52c09850-6c9d-4a0f-ae44-b11a4b9133ed&select=id,name",
    headers=headers
)
print("Firm:", r.json())

Firm: [{'id': '52c09850-6c9d-4a0f-ae44-b11a4b9133ed', 'name': 'BGR GOVERNMENT AFFAIRS'}]


In [ ]:
import requests
from google.colab import userdata

url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_ROLE')
headers = {'apikey': key, 'Authorization': f'Bearer {key}'}

# Simulate exactly what the API route does
contractor_name = "CHEMONICS INTERNATIONAL, INC."

r = requests.get(
    f"{url}/rest/v1/lobbying_clients?general_description=ilike.*{contractor_name}*&select=name,firm_id,general_description&limit=10",
    headers=headers
)
print("Direct match:", r.json())

Direct match: [{'name': 'CHEMONICS INTERNATIONAL, INC.', 'firm_id': '52c09850-6c9d-4a0f-ae44-b11a4b9133ed', 'general_description': 'Matched from contracts: CHEMONICS INTERNATIONAL, INC.'}, {'name': 'CHEMONICS INTERNATIONAL', 'firm_id': None, 'general_description': 'Matched from contracts: CHEMONICS INTERNATIONAL, INC.'}]


In [ ]:
%cd /content/govcon-dashboard
!cat app/api/contracts/\[id\]/route.ts

/content/govcon-dashboard
import { NextRequest, NextResponse } from 'next/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

function stripLegalSuffix(name: string): string {
  return name
    .replace(/[,.]?(\s+)?(INC|LLC|CORP|CO|LTD|LP|LLP|PLC|INCORPORATED|CORPORATION|COMPANY|LIMITED|GROUP|ASSOCIATES|SERVICES|SOLUTIONS|SYSTEMS|INTERNATIONAL|TECHNOLOGIES|TECHNOLOGY|MANAGEMENT|CONSULTING|ENTERPRISES|PARTNERS|HOLDINGS|FEDERAL|GOVERNMENT|GLOBAL|NATIONAL|DIVISION|INDUSTRIES|RESOURCES)(\.)?$/gi, '')
    .trim()
}

export async function GET(
  req: NextRequest,
  { params }: { params: Promise<{ id: string }> }
) {
  const { id } = await params

  // Fetch the contract
  const { data: contract, error } = await supabaseAdmin
    .from('contracts')
    .select('*')
    .eq('id', id)
    .single()

  if (error || !contract) {
    return NextResponse.json({ error: error?.message || 'Not found' }, { status: 404 })
  }

  // Attempt influence lookup
  let influence: {
    lobbying_firms

In [ ]:
import requests
from google.colab import userdata

url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_ROLE')
headers = {'apikey': key, 'Authorization': f'Bearer {key}'}

# Check if BGR Government Affairs has any politician links
firm_id = '52c09850-6c9d-4a0f-ae44-b11a4b9133ed'
r = requests.get(
    f"{url}/rest/v1/lobbying_firm_politician_links?firm_id=eq.{firm_id}&select=firm_id,politician_id&limit=5",
    headers=headers
)
print("BGR politician links:", r.json())

# Also check if the firms table has total_lobbying_income
r2 = requests.get(
    f"{url}/rest/v1/lobbying_firms?id=eq.{firm_id}&select=*",
    headers=headers
)
print("BGR firm full row:", r2.json())

BGR politician links: []
BGR firm full row: [{'id': '52c09850-6c9d-4a0f-ae44-b11a4b9133ed', 'lda_registrant_id': 5357, 'name': 'BGR GOVERNMENT AFFAIRS', 'display_name': None, 'description': None, 'city': None, 'state': None, 'country': 'US', 'client_count': 0, 'total_income': 0.0, 'active_client_ueis': [], 'active_client_names': [], 'top_issues': [], 'has_revolving_door': False, 'revolving_door_notes': None, 'fetched_at': '2026-03-23T08:17:38.83623+00:00', 'expires_at': '2026-04-22T08:17:38.83623+00:00', 'normalized_name': None, 'dedupe_key': None}]


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

route = Path("app/api/contracts/[id]/route.ts")
text = route.read_text()

text = text.replace(
    ".select('id, name, total_lobbying_income')",
    ".select('id, name, total_income')"
)
text = text.replace(
    "total_income: f.total_lobbying_income || 0",
    "total_income: f.total_income || 0"
)

route.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Fix lobbying_firms column name total_income" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 21.3s
✓ Finished TypeScript in 17.0s 
✓ Collecting page data using 1 worker in 1427.9ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (1/30)  [=   ]  Generating static pages using 1 worker (6/30)  [==  ]  Generating static pages using 1 worker (25/30)  [=== ]✓ Generating static pages using 1 worker (30/30) in 705.9ms
✓ Finalizing page optimization in 13.2ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobb

In [ ]:
%cd /content/govcon-dashboard
!git log --oneline -10

/content/govcon-dashboard
a00134f (HEAD -> main, origin/main, origin/HEAD) Fix lobbying_firms column name total_income
4b91aef Fix Recompete Radar to use period_of_performance_end
ad45e1d Add Recompete Radar to navigation
03478ad Add Recompete Radar page with RPS scoring
df26dd9 Fix contractor_name type cast in influence lookup
4ceb26f Fix lobbying_clients column name to name
60a3ccc Fix params Promise in contract detail page
1fd4f4a Add influence intelligence to contract detail page
b378976 Add agency filter, consolidate sort dropdown
dd99217 Fix filter param names and total count


In [ ]:
%cd /content/govcon-dashboard
!cat app/\(dashboard\)/lobbying-firms/\[id\]/page.tsx | head -50

/content/govcon-dashboard
"use client";

import { useEffect, useState, useMemo } from "react";
import { useParams, useSearchParams, useRouter } from "next/navigation";
import Link from "next/link";
import { BARNACLE_DISCLAIMER } from "@/lib/lda";

const ISSUE_LABELS: Record<string, string> = {
  DEF: "Defense", APP: "Appropriations", GOV: "Government Procurement",
  SCI: "Science & Technology", HCR: "Health Care", FIN: "Finance",
  TAX: "Taxation", TRA: "Transportation", ENV: "Environment",
  LAB: "Labor/Employment", INT: "Intelligence", MIL: "Military Affairs",
  VET: "Veterans Affairs", SMB: "Small Business",
  BUD: "Budget",
};

const STATUS_LABELS: Record<string, string> = {
  raw_only: "Raw Only",
  candidate_only: "Candidate Only",
  reviewed: "Reviewed",
  enriched: "Enriched",
  scored: "Scored",
};

const statusBadgeClass = (status?: string) => {
  switch (status) {
    case "scored":
      return "bg-green-900/30 text-green-300 border-green-700/40";
    case "enriched":
     

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("app/(dashboard)/contractors/profile/page.tsx")
text = path.read_text()
print(text[:2000])

/content/govcon-dashboard
'use client'

import { useEffect, useState } from 'react'
import { useSearchParams } from 'next/navigation'

export default function ContractorProfilePage() {
  const params = useSearchParams()
  const name = params.get('name')

  const [data, setData] = useState<any>(null)

  useEffect(() => {
    if (!name) return

    const fetchData = async () => {
      const res = await fetch(`/api/contractors/profile?name=${encodeURIComponent(name)}`)
      const json = await res.json()
      setData(json)
    }

    fetchData()
  }, [name])

  if (!data) {
    return <div className="p-6 bg-slate-950 text-slate-200">Loading...</div>
  }

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen space-y-6">

      <h1 className="text-2xl font-bold text-blue-400">
        {data.contractor_name}
      </h1>

      {/* Stats */}
      <div className="grid grid-cols-4 gap-4">
        <div className="bg-slate-900 p-4 rounded border border-slate-800">
       

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("app/(dashboard)/contractors/profile/page.tsx")
text = path.read_text()

# Add Link import if not present
if "import Link" not in text:
    text = text.replace(
        "import { useSearchParams } from 'next/navigation'",
        "import { useSearchParams } from 'next/navigation'\nimport Link from 'next/link'"
    )

# Add View All Contracts button after the h1
text = text.replace(
    """      <h1 className="text-2xl font-bold text-blue-400">
        {data.contractor_name}
      </h1>""",
    """      <div className="flex items-center justify-between">
        <h1 className="text-2xl font-bold text-blue-400">
          {data.contractor_name}
        </h1>
        <div className="flex gap-3">
          <Link
            href={`/contracts?keyword=${encodeURIComponent(name || '')}`}
            className="bg-blue-600 hover:bg-blue-500 text-white px-4 py-2 rounded text-sm font-medium"
          >
            View All Contracts →
          </Link>
          <Link
            href={`/expiring?keyword=${encodeURIComponent(name || '')}`}
            className="bg-orange-600 hover:bg-orange-500 text-white px-4 py-2 rounded text-sm font-medium"
          >
            Expiring Contracts →
          </Link>
        </div>
      </div>"""
)

path.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Add View All Contracts and Expiring Contracts buttons to contractor profile" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 27.2s
✓ Finished TypeScript in 17.5s 
✓ Collecting page data using 1 worker in 882.0ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (4/30)  [=   ]  Generating static pages using 1 worker (25/30)  [==  ]✓ Generating static pages using 1 worker (30/30) in 461.3ms
✓ Finalizing page optimization in 11.0ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pin

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

# Add keyword filter to expiring page
path = Path("app/(dashboard)/expiring/page.tsx")
text = path.read_text()

# Add keyword state
text = text.replace(
    "  const [state, setState] = useState('')",
    "  const [state, setState] = useState('')\n  const [keyword, setKeyword] = useState('')"
)

# Add keyword to params
text = text.replace(
    "    if (naics) params.set('naics', naics)\n    if (state) params.set('state', state)",
    "    if (naics) params.set('naics', naics)\n    if (state) params.set('state', state)\n    if (keyword) params.set('keyword', keyword)"
)

# Add keyword input to filter bar after the h1/subtitle
text = text.replace(
    "      {/* Filters */}\n      <div className=\"bg-slate-900 border border-slate-700 rounded-lg p-4 mb-4 flex flex-wrap gap-3 items-end\">",
    "      {/* Filters */}\n      <div className=\"bg-slate-900 border border-slate-700 rounded-lg p-4 mb-4 flex flex-wrap gap-3 items-end\">\n        <div className=\"flex flex-col gap-1\">\n          <label className=\"text-xs text-slate-400\">Keyword</label>\n          <input\n            className=\"bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-48\"\n            placeholder=\"Contractor name...\"\n            value={keyword}\n            onChange={e => setKeyword(e.target.value)}\n            onKeyDown={e => e.key === 'Enter' && handleSearch()}\n          />\n        </div>"
)

path.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

route = Path("app/api/expiring/route.ts")
text = route.read_text()

text = text.replace(
    "  const state = searchParams.get('state') || ''",
    "  const state = searchParams.get('state') || ''\n  const keyword = searchParams.get('keyword') || ''"
)

text = text.replace(
    "  if (naics) query = query.eq('naics_code', naics)\n  if (state) query = query.eq('place_of_performance_state', state)",
    "  if (naics) query = query.eq('naics_code', naics)\n  if (state) query = query.eq('place_of_performance_state', state)\n  if (keyword) query = query.ilike('contractor_name', '%' + keyword + '%')"
)

route.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Add keyword filter to Recompete Radar" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 22.1s
✓ Finished TypeScript in 19.1s 
✓ Collecting page data using 1 worker in 898.5ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (3/30)  [=   ]  Generating static pages using 1 worker (25/30)  [==  ]✓ Generating static pages using 1 worker (30/30) in 463.5ms
✓ Finalizing page optimization in 18.1ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pin

In [ ]:
%cd /content/govcon-dashboard

# Check what the lobbying firm API returns
import subprocess
result = subprocess.run(
    ['grep', '-n', 'clients\|contracts\|tab', 'app/(dashboard)/lobbying-firms/[id]/page.tsx'],
    capture_output=True, text=True
)
print(result.stdout[:3000])

/content/govcon-dashboard
145:  const { firm, clients = [], linkedPoliticians = [] } = data;
211:          <h2 className="text-lg font-semibold mb-4">Clients ({clients.length})</h2>
213:            {clients.map((c: any) => (
228:            {clients.length === 0 && <p className="text-sm text-slate-500">No clients on record.</p>}



<>:6: SyntaxWarning: invalid escape sequence '\|'
<>:6: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_366/150407739.py:6: SyntaxWarning: invalid escape sequence '\|'
  ['grep', '-n', 'clients\|contracts\|tab', 'app/(dashboard)/lobbying-firms/[id]/page.tsx'],


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("app/(dashboard)/lobbying-firms/[id]/page.tsx")
text = path.read_text()

# Find the clients section
start = text.find('clients.map')
print(text[start:start+500])

/content/govcon-dashboard
clients.map((c: any) => (
              <div key={c.id} className="p-3 bg-slate-800/50 rounded border border-slate-700/50 flex flex-col gap-1">
                <div className="flex justify-between items-start">
                  <span className="text-sm font-medium text-slate-200">{c.name}</span>
                  {(c.state || c.country) && (
                    <span className="text-xs text-slate-500">
                      {[c.state, c.country].filter(Boolean).join(", ")}
                    <


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("app/(dashboard)/lobbying-firms/[id]/page.tsx")
text = path.read_text()

# Add Link import if not present
if "import Link" not in text:
    text = text.replace(
        '"use client";',
        '"use client";\nimport Link from "next/link";'
    )

# Add View Contracts link to each client card
text = text.replace(
    '                  <span className="text-sm font-medium text-slate-200">{c.name}</span>',
    '                  <Link href={`/contracts?keyword=${encodeURIComponent(c.name)}`} className="text-sm font-medium text-blue-400 hover:text-blue-300">{c.name}</Link>'
)

path.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Add contract search links to lobbying firm client list" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 20.2s
✓ Finished TypeScript in 19.9s 
✓ Collecting page data using 1 worker in 878.1ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (2/30)  [=   ]  Generating static pages using 1 worker (25/30)  [==  ]✓ Generating static pages using 1 worker (30/30) in 477.0ms
✓ Finalizing page optimization in 8.6ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/api/contracts", exist_ok=True)
os.makedirs("app/(dashboard)/contracts", exist_ok=True)

# ── API ROUTE ─────────────────────────────────────────────────────────────────
Path("app/api/contracts/route.ts").write_text("""import { NextRequest, NextResponse } from 'next/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

function computeRPS(daysLeft: number): number {
  if (daysLeft <= 30) return 95
  if (daysLeft <= 60) return 85
  if (daysLeft <= 90) return 75
  if (daysLeft <= 120) return 60
  if (daysLeft <= 180) return 45
  if (daysLeft <= 365) return 30
  return 0
}

export async function GET(req: NextRequest) {
  const { searchParams } = new URL(req.url)

  const page = parseInt(searchParams.get('page') || '1')
  const limit = 50
  const from = (page - 1) * limit
  const to = from + limit - 1

  const keyword = searchParams.get('keyword') || ''
  const naics = searchParams.get('naics') || ''
  const state = searchParams.get('state') || ''
  const agency = searchParams.get('agency') || ''
  const sort = searchParams.get('sort') || 'award_date'
  const order = searchParams.get('order') || 'desc'
  const minAmount = searchParams.get('min_amount') || ''
  const maxAmount = searchParams.get('max_amount') || ''
  const expiringWithin = searchParams.get('expiring_within') || ''
  const minRps = parseInt(searchParams.get('min_rps') || '0')

  const today = new Date().toISOString().split('T')[0]

  let query = supabaseAdmin
    .from('contracts')
    .select('id, contractor_name, agency, naics_code, psc_code, award_amount, award_date, period_of_performance_end, place_of_performance_state, description', { count: 'exact' })

  if (keyword) query = query.or(`contractor_name.ilike.%${keyword}%,description.ilike.%${keyword}%`)
  if (naics) query = query.eq('naics_code', naics)
  if (state) query = query.eq('place_of_performance_state', state)
  if (agency) query = query.eq('agency', agency)
  if (minAmount) query = query.gte('award_amount', Number(minAmount))
  if (maxAmount) query = query.lte('award_amount', Number(maxAmount))

  if (expiringWithin) {
    const days = parseInt(expiringWithin)
    const future = new Date(Date.now() + days * 24 * 60 * 60 * 1000).toISOString().split('T')[0]
    query = query
      .not('period_of_performance_end', 'is', null)
      .gte('period_of_performance_end', today)
      .lte('period_of_performance_end', future)
  }

  if (sort === 'rps_score' || sort === 'days_left') {
    query = query.order('period_of_performance_end', { ascending: order === 'asc', nullsFirst: false })
  } else {
    query = query.order(sort, { ascending: order === 'asc' })
  }

  query = query.range(from, to)

  const { data, error, count } = await query

  if (error) return NextResponse.json({ error: error.message }, { status: 500 })

  const today_ms = Date.now()
  const scored = ((data || []) as any[]).map((c: any) => {
    let days_left: number | null = null
    let rps_score = 0
    if (c.period_of_performance_end) {
      const expiry = new Date(c.period_of_performance_end)
      days_left = Math.ceil((expiry.getTime() - today_ms) / (1000 * 60 * 60 * 24))
      if (days_left > 0) rps_score = computeRPS(days_left)
    }
    return { ...c, days_left, rps_score }
  })

  const filtered = minRps > 0 ? scored.filter((c: any) => c.rps_score >= minRps) : scored

  return NextResponse.json({
    data: filtered,
    count,
    page,
    totalPages: Math.ceil((count || 0) / limit)
  })
}
""")

# ── PAGE COMPONENT ────────────────────────────────────────────────────────────
Path("app/(dashboard)/contracts/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'
import { useRouter } from 'next/navigation'

type Contract = {
  id: string
  contractor_name: string
  agency: string
  award_amount: number
  award_date: string
  period_of_performance_end: string | null
  place_of_performance_state: string
  naics_code: string
  days_left: number | null
  rps_score: number
}

const NAICS_OPTIONS = [
  { value: '', label: 'All NAICS' },
  { value: '238220', label: '238220 — HVAC' },
  { value: '541511', label: '541511 — IT Services' },
  { value: '236220', label: '236220 — Construction' },
  { value: '561730', label: '561730 — Landscaping' },
  { value: '541611', label: '541611 — Management Consulting' },
  { value: '237310', label: '237310 — Highway Construction' },
  { value: '238210', label: '238210 — Electrical' },
  { value: '238160', label: '238160 — Roofing' },
  { value: '541330', label: '541330 — Engineering' },
  { value: '541512', label: '541512 — Computer Systems' },
]

const STATES = ['','AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC']

const AGENCIES = [
  { value: '', label: 'All Agencies' },
  { value: 'Department of Defense', label: 'Dept. of Defense' },
  { value: 'Department of Veterans Affairs', label: 'Dept. of Veterans Affairs' },
  { value: 'Department of Agriculture', label: 'Dept. of Agriculture' },
  { value: 'Department of Transportation', label: 'Dept. of Transportation' },
  { value: 'Department of the Interior', label: 'Dept. of the Interior' },
  { value: 'Department of Health and Human Services', label: 'Dept. of HHS' },
  { value: 'Department of Homeland Security', label: 'Dept. of Homeland Security' },
  { value: 'Department of State', label: 'Dept. of State' },
  { value: 'Department of Commerce', label: 'Dept. of Commerce' },
  { value: 'Department of Justice', label: 'Dept. of Justice' },
  { value: 'Department of Energy', label: 'Dept. of Energy' },
  { value: 'General Services Administration', label: 'GSA' },
  { value: 'National Aeronautics and Space Administration', label: 'NASA' },
]

function RPSBadge({ score }: { score: number }) {
  if (!score) return <span className="text-slate-600 text-xs">—</span>
  const color = score >= 85 ? 'bg-red-600' : score >= 70 ? 'bg-orange-500' : score >= 50 ? 'bg-yellow-500' : 'bg-slate-600'
  return <span className={`${color} text-white text-xs font-bold px-1.5 py-0.5 rounded`}>RPS {score}</span>
}

function ExpiryCell({ date, daysLeft }: { date: string | null, daysLeft: number | null }) {
  if (!date) return <span className="text-slate-600">—</span>
  const color = daysLeft !== null && daysLeft <= 30 ? 'text-red-400' : daysLeft !== null && daysLeft <= 60 ? 'text-orange-400' : daysLeft !== null && daysLeft <= 90 ? 'text-yellow-400' : 'text-slate-400'
  return (
    <div>
      <div className="text-xs text-slate-400">{date}</div>
      {daysLeft !== null && daysLeft > 0 && <div className={`text-xs font-medium ${color}`}>{daysLeft}d left</div>}
      {daysLeft !== null && daysLeft <= 0 && <div className="text-xs text-slate-600">expired</div>}
    </div>
  )
}

export default function ContractsPage() {
  const router = useRouter()
  const [data, setData] = useState<Contract[]>([])
  const [page, setPage] = useState(1)
  const [total, setTotal] = useState(0)
  const [totalPages, setTotalPages] = useState(0)
  const [loading, setLoading] = useState(false)
  const [keyword, setKeyword] = useState('')
  const [naics, setNaics] = useState('')
  const [state, setState] = useState('')
  const [agency, setAgency] = useState('')
  const [minAmount, setMinAmount] = useState('')
  const [maxAmount, setMaxAmount] = useState('')
  const [expiringWithin, setExpiringWithin] = useState('')
  const [sortBy, setSortBy] = useState('award_date')
  const [sortDir, setSortDir] = useState('desc')

  const fetchData = async (p = 1) => {
    setLoading(true)
    const params = new URLSearchParams()
    params.set('page', String(p))
    params.set('sort', sortBy)
    params.set('order', sortDir)
    if (keyword) params.set('keyword', keyword)
    if (naics) params.set('naics', naics)
    if (state) params.set('state', state)
    if (agency) params.set('agency', agency)
    if (minAmount) params.set('min_amount', minAmount)
    if (maxAmount) params.set('max_amount', maxAmount)
    if (expiringWithin) params.set('expiring_within', expiringWithin)
    const res = await fetch('/api/contracts?' + params.toString())
    const json = await res.json()
    setData(json.data || [])
    setTotal(json.count || 0)
    setTotalPages(json.totalPages || 0)
    setLoading(false)
  }

  useEffect(() => { fetchData(page) }, [page, sortBy, sortDir])

  const handleSearch = () => { setPage(1); fetchData(1) }

  const handleClear = () => {
    setKeyword(''); setNaics(''); setState(''); setAgency('')
    setMinAmount(''); setMaxAmount(''); setExpiringWithin('')
    setPage(1)
    setTimeout(() => fetchData(1), 0)
  }

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen">
      <h1 className="text-2xl font-bold mb-4 text-blue-400">Federal Contracts</h1>

      {/* Filters */}
      <div className="sticky top-0 z-10 bg-slate-900 border border-slate-700 rounded-lg p-4 mb-4 flex flex-wrap gap-3 items-end">
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Keyword</label>
          <input className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-44"
            placeholder="Contractor or description..."
            value={keyword} onChange={e => setKeyword(e.target.value)}
            onKeyDown={e => e.key === 'Enter' && handleSearch()} />
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">NAICS</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-44"
            value={naics} onChange={e => setNaics(e.target.value)}>
            {NAICS_OPTIONS.map(o => <option key={o.value} value={o.value}>{o.label}</option>)}
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">State</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-20"
            value={state} onChange={e => setState(e.target.value)}>
            {STATES.map(s => <option key={s} value={s}>{s || 'All'}</option>)}
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Agency</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-44"
            value={agency} onChange={e => setAgency(e.target.value)}>
            {AGENCIES.map(a => <option key={a.value} value={a.value}>{a.label}</option>)}
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Min $</label>
          <input className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-28"
            placeholder="0" value={minAmount} onChange={e => setMinAmount(e.target.value)} />
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Max $</label>
          <input className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-28"
            placeholder="Any" value={maxAmount} onChange={e => setMaxAmount(e.target.value)} />
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Expiring Within</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-32"
            value={expiringWithin} onChange={e => setExpiringWithin(e.target.value)}>
            <option value="">Any</option>
            <option value="30">30 days</option>
            <option value="60">60 days</option>
            <option value="90">90 days</option>
            <option value="180">180 days</option>
            <option value="365">1 year</option>
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Sort</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-44"
            value={sortBy + '__' + sortDir}
            onChange={e => {
              const [s, d] = e.target.value.split('__')
              setSortBy(s); setSortDir(d)
            }}>
            <option value="award_date__desc">Newest First</option>
            <option value="award_date__asc">Oldest First</option>
            <option value="award_amount__desc">Largest Value First</option>
            <option value="award_amount__asc">Smallest Value First</option>
            <option value="rps_score__asc">Expiring Soonest</option>
          </select>
        </div>
        <button onClick={handleSearch}
          className="bg-blue-600 hover:bg-blue-500 text-white px-4 py-1.5 rounded text-sm font-medium">Search</button>
        <button onClick={handleClear}
          className="bg-slate-700 hover:bg-slate-600 text-slate-200 px-4 py-1.5 rounded text-sm">Clear</button>
      </div>

      <div className="text-sm text-slate-400 mb-2">
        {loading ? 'Loading...' : `${total.toLocaleString()} contracts — Page ${page} of ${totalPages.toLocaleString()}`}
      </div>

      <table className="w-full border border-slate-800 text-sm">
        <thead className="bg-slate-800 text-slate-300">
          <tr>
            <th className="p-2 text-left">RPS</th>
            <th className="p-2 text-left">Contractor</th>
            <th className="p-2 text-left">Agency</th>
            <th className="p-2 text-left">Amount</th>
            <th className="p-2 text-left">Awarded</th>
            <th className="p-2 text-left">Expires</th>
            <th className="p-2 text-left">State</th>
            <th className="p-2 text-left">NAICS</th>
          </tr>
        </thead>
        <tbody>
          {data.map((c, i) => (
            <tr key={c.id}
              onClick={() => router.push(`/contracts/${c.id}`)}
              className={(i % 2 === 0 ? 'bg-slate-900' : 'bg-slate-800/50') + ' cursor-pointer hover:bg-slate-700 transition-colors'}>
              <td className="p-2"><RPSBadge score={c.rps_score} /></td>
              <td className="p-2 font-medium">{c.contractor_name}</td>
              <td className="p-2 text-slate-400 text-xs">{c.agency}</td>
              <td className="p-2 text-green-400 font-medium">
                {c.award_amount?.toLocaleString('en-US', { style: 'currency', currency: 'USD' })}
              </td>
              <td className="p-2 text-slate-400 text-xs">{c.award_date}</td>
              <td className="p-2"><ExpiryCell date={c.period_of_performance_end} daysLeft={c.days_left} /></td>
              <td className="p-2">{c.place_of_performance_state}</td>
              <td className="p-2 text-slate-400">{c.naics_code}</td>
            </tr>
          ))}
        </tbody>
      </table>

      <div className="flex items-center gap-3 mt-4">
        <button onClick={() => setPage(p => Math.max(1, p - 1))} disabled={page === 1}
          className="bg-slate-800 hover:bg-slate-700 disabled:opacity-40 px-4 py-1.5 rounded text-sm">Prev</button>
        <span className="text-sm text-slate-400">Page {page} / {totalPages.toLocaleString()}</span>
        <button onClick={() => setPage(p => Math.min(totalPages, p + 1))} disabled={page === totalPages}
          className="bg-slate-800 hover:bg-slate-700 disabled:opacity-40 px-4 py-1.5 rounded text-sm">Next</button>
      </div>
    </div>
  )
}
""")

print("Files written:")
print("  app/api/contracts/route.ts")
print("  app/(dashboard)/contracts/page.tsx")

/content/govcon-dashboard
Files written:
  app/api/contracts/route.ts
  app/(dashboard)/contracts/page.tsx


In [ ]:
%cd /content/govcon-dashboard
!git config user.email "menagoubran@gmail.com"
!git config user.name "Mena Goubran"
!npm run build && git add -A && git commit -m "Merge Recompete Radar into contracts page with RPS scoring and expiry dates" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 22.0s
✓ Finished TypeScript in 18.1s 
✓ Collecting page data using 1 worker in 1438.9ms 
  Generating static pages using 1 worker (0/30)  [    ]  Generating static pages using 1 worker (1/30)  [=   ]  Generating static pages using 1 worker (6/30)  [==  ]  Generating static pages using 1 worker (25/30)  [=== ]✓ Generating static pages using 1 worker (30/30) in 704.0ms
✓ Finalizing page optimization in 7.0ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobby

In [ ]:
Good. Let's build the personalized dashboard home page. This is the first screen a subscriber sees after login — it should show their contracts expiring soon based on their profile.

First, check what the current home page looks like:

```python
%cd /content/govcon-dashboard
!cat app/\(dashboard\)/page.tsx 2>/dev/null || echo "No dashboard page found"
!ls app/\(dashboard\)/
```

Paste the output.

SyntaxError: unterminated string literal (detected at line 1) (530950815.py, line 1)

In [ ]:
%cd /content/govcon-dashboard
import subprocess
result = subprocess.run(['ls', 'app/(dashboard)/'], capture_output=True, text=True)
print(result.stdout)

/content/govcon-dashboard
contractors
contracts
expiring
layout.tsx
lobbying-firms
map
politicians
search
status



In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/(dashboard)", exist_ok=True)
os.makedirs("app/api/dashboard", exist_ok=True)

# API Route for dashboard summary
Path("app/api/dashboard/route.ts").write_text("""import { NextRequest, NextResponse } from 'next/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

export async function GET(req: NextRequest) {
  const today = new Date().toISOString().split('T')[0]
  const in90 = new Date(Date.now() + 90 * 24 * 60 * 60 * 1000).toISOString().split('T')[0]
  const in30 = new Date(Date.now() + 30 * 24 * 60 * 60 * 1000).toISOString().split('T')[0]

  // Critical contracts expiring in 30 days
  const { data: critical, count: criticalCount } = await supabaseAdmin
    .from('contracts')
    .select('id, contractor_name, agency, award_amount, period_of_performance_end, place_of_performance_state, naics_code', { count: 'exact' })
    .not('period_of_performance_end', 'is', null)
    .gte('period_of_performance_end', today)
    .lte('period_of_performance_end', in30)
    .gt('award_amount', 0)
    .order('period_of_performance_end', { ascending: true })
    .limit(5)

  // High priority expiring in 90 days
  const { count: highCount } = await supabaseAdmin
    .from('contracts')
    .select('id', { count: 'exact' })
    .not('period_of_performance_end', 'is', null)
    .gte('period_of_performance_end', today)
    .lte('period_of_performance_end', in90)
    .gt('award_amount', 0)

  // Total contracts
  const { count: totalContracts } = await supabaseAdmin
    .from('contracts')
    .select('id', { count: 'exact' })

  // Total lobbying firms
  const { count: totalFirms } = await supabaseAdmin
    .from('lobbying_firms')
    .select('id', { count: 'exact' })

  // Recent high-value contracts
  const { data: recent } = await supabaseAdmin
    .from('contracts')
    .select('id, contractor_name, agency, award_amount, award_date, place_of_performance_state, naics_code')
    .order('award_amount', { ascending: false })
    .limit(5)

  return NextResponse.json({
    stats: {
      totalContracts,
      totalFirms,
      criticalCount,
      highCount,
    },
    critical: (critical || []).map((c: any) => {
      const expiry = new Date(c.period_of_performance_end)
      const daysLeft = Math.ceil((expiry.getTime() - Date.now()) / (1000 * 60 * 60 * 24))
      return { ...c, days_left: daysLeft }
    }),
    recent: recent || []
  })
}
""")

# Dashboard home page
Path("app/(dashboard)/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'
import { useRouter } from 'next/navigation'
import Link from 'next/link'

type DashboardData = {
  stats: {
    totalContracts: number
    totalFirms: number
    criticalCount: number
    highCount: number
  }
  critical: any[]
  recent: any[]
}

function StatCard({ label, value, sub, color }: { label: string, value: string | number, sub?: string, color?: string }) {
  return (
    <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
      <p className="text-slate-400 text-sm mb-1">{label}</p>
      <p className={`text-2xl font-bold ${color || 'text-slate-100'}`}>{value?.toLocaleString()}</p>
      {sub && <p className="text-slate-500 text-xs mt-1">{sub}</p>}
    </div>
  )
}

function RPSBadge({ days }: { days: number }) {
  const score = days <= 30 ? 95 : days <= 60 ? 85 : 75
  const color = days <= 30 ? 'bg-red-600' : days <= 60 ? 'bg-orange-500' : 'bg-yellow-500'
  return <span className={`${color} text-white text-xs font-bold px-1.5 py-0.5 rounded`}>RPS {score}</span>
}

export default function DashboardHome() {
  const router = useRouter()
  const [data, setData] = useState<DashboardData | null>(null)
  const [loading, setLoading] = useState(true)

  useEffect(() => {
    fetch('/api/dashboard')
      .then(r => r.json())
      .then(json => { setData(json); setLoading(false) })
  }, [])

  if (loading) return (
    <div className="p-8 bg-slate-950 min-h-screen flex items-center justify-center">
      <div className="text-slate-400">Loading intelligence...</div>
    </div>
  )

  if (!data) return null

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen">
      {/* Header */}
      <div className="mb-8">
        <h1 className="text-3xl font-bold text-blue-400">GovCon Intelligence Terminal</h1>
        <p className="text-slate-400 mt-1">Federal contracting intelligence — updated daily</p>
      </div>

      {/* Stats */}
      <div className="grid grid-cols-2 md:grid-cols-4 gap-4 mb-8">
        <StatCard
          label="Federal Contracts"
          value={data.stats.totalContracts || 0}
          sub="Across 10 NAICS codes"
          color="text-blue-400"
        />
        <StatCard
          label="Expiring in 30 Days"
          value={data.stats.criticalCount || 0}
          sub="Critical recompete opportunities"
          color="text-red-400"
        />
        <StatCard
          label="Expiring in 90 Days"
          value={data.stats.highCount || 0}
          sub="High priority opportunities"
          color="text-orange-400"
        />
        <StatCard
          label="Lobbying Firms Tracked"
          value={data.stats.totalFirms || 0}
          sub="With client & politician links"
          color="text-green-400"
        />
      </div>

      <div className="grid grid-cols-1 lg:grid-cols-2 gap-6">
        {/* Critical Expiring */}
        <div className="bg-slate-900 border border-slate-700 rounded-lg overflow-hidden">
          <div className="bg-red-950 border-b border-red-800 px-4 py-3 flex items-center justify-between">
            <h2 className="font-semibold text-red-300">🔴 Critical — Expiring in 30 Days</h2>
            <Link href="/contracts?expiring_within=30" className="text-xs text-red-400 hover:text-red-300">View all →</Link>
          </div>
          <div className="divide-y divide-slate-800">
            {data.critical.length === 0 && (
              <div className="p-4 text-slate-500 text-sm">No critical contracts found</div>
            )}
            {data.critical.map((c: any) => (
              <div key={c.id}
                onClick={() => router.push(`/contracts/${c.id}`)}
                className="p-4 hover:bg-slate-800 cursor-pointer transition-colors">
                <div className="flex items-start justify-between gap-2">
                  <div className="flex-1 min-w-0">
                    <p className="font-medium text-sm truncate">{c.contractor_name}</p>
                    <p className="text-slate-400 text-xs truncate">{c.agency}</p>
                  </div>
                  <div className="text-right shrink-0">
                    <p className="text-green-400 font-medium text-sm">
                      {c.award_amount?.toLocaleString('en-US', { style: 'currency', currency: 'USD', maximumFractionDigits: 0 })}
                    </p>
                    <RPSBadge days={c.days_left} />
                  </div>
                </div>
                <div className="flex items-center gap-3 mt-1">
                  <span className="text-xs text-red-400 font-medium">{c.days_left}d left</span>
                  <span className="text-xs text-slate-500">{c.place_of_performance_state} · {c.naics_code}</span>
                </div>
              </div>
            ))}
          </div>
        </div>

        {/* Largest Recent Contracts */}
        <div className="bg-slate-900 border border-slate-700 rounded-lg overflow-hidden">
          <div className="bg-slate-800 border-b border-slate-700 px-4 py-3 flex items-center justify-between">
            <h2 className="font-semibold text-slate-200">💰 Largest Recent Awards</h2>
            <Link href="/contracts?sort=award_amount__desc" className="text-xs text-blue-400 hover:text-blue-300">View all →</Link>
          </div>
          <div className="divide-y divide-slate-800">
            {data.recent.map((c: any) => (
              <div key={c.id}
                onClick={() => router.push(`/contracts/${c.id}`)}
                className="p-4 hover:bg-slate-800 cursor-pointer transition-colors">
                <div className="flex items-start justify-between gap-2">
                  <div className="flex-1 min-w-0">
                    <p className="font-medium text-sm truncate">{c.contractor_name}</p>
                    <p className="text-slate-400 text-xs truncate">{c.agency}</p>
                  </div>
                  <div className="text-right shrink-0">
                    <p className="text-green-400 font-medium text-sm">
                      {c.award_amount?.toLocaleString('en-US', { style: 'currency', currency: 'USD', maximumFractionDigits: 0 })}
                    </p>
                    <p className="text-slate-500 text-xs">{c.award_date}</p>
                  </div>
                </div>
                <div className="flex items-center gap-3 mt-1">
                  <span className="text-xs text-slate-500">{c.place_of_performance_state} · {c.naics_code}</span>
                </div>
              </div>
            ))}
          </div>
        </div>
      </div>

      {/* Quick Actions */}
      <div className="mt-6 grid grid-cols-2 md:grid-cols-4 gap-3">
        <Link href="/contracts?expiring_within=90"
          className="bg-slate-800 hover:bg-slate-700 border border-slate-700 rounded-lg p-4 text-center transition-colors">
          <div className="text-2xl mb-1">📡</div>
          <div className="text-sm font-medium">Recompete Radar</div>
          <div className="text-xs text-slate-400">Expiring in 90 days</div>
        </Link>
        <Link href="/contracts"
          className="bg-slate-800 hover:bg-slate-700 border border-slate-700 rounded-lg p-4 text-center transition-colors">
          <div className="text-2xl mb-1">🔍</div>
          <div className="text-sm font-medium">Contract Search</div>
          <div className="text-xs text-slate-400">207K+ federal contracts</div>
        </Link>
        <Link href="/lobbying-firms"
          className="bg-slate-800 hover:bg-slate-700 border border-slate-700 rounded-lg p-4 text-center transition-colors">
          <div className="text-2xl mb-1">🏛️</div>
          <div className="text-sm font-medium">Lobbying Intelligence</div>
          <div className="text-xs text-slate-400">177 firms, 3,351 clients</div>
        </Link>
        <Link href="/politicians"
          className="bg-slate-800 hover:bg-slate-700 border border-slate-700 rounded-lg p-4 text-center transition-colors">
          <div className="text-2xl mb-1">🗳️</div>
          <div className="text-sm font-medium">Politicians</div>
          <div className="text-xs text-slate-400">538 Congress members</div>
        </Link>
      </div>
    </div>
  )
}
""")

print("Files written:")
print("  app/api/dashboard/route.ts")
print("  app/(dashboard)/page.tsx")

/content/govcon-dashboard
Files written:
  app/api/dashboard/route.ts
  app/(dashboard)/page.tsx


In [ ]:
%cd /content/govcon-dashboard
!git config user.email "menagoubran@gmail.com"
!git config user.name "Mena Goubran"
!npm run build && git add -A && git commit -m "Add dashboard home page with stats and critical expiring contracts" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 20.7s
✓ Finished TypeScript in 17.9s 
✓ Collecting page data using 1 worker in 978.8ms 
  Generating static pages using 1 worker (0/31)  [    ]  Generating static pages using 1 worker (4/31)  [=   ]  Generating static pages using 1 worker (27/31)  [==  ]✓ Generating static pages using 1 worker (31/31) in 482.1ms
✓ Finalizing page optimization in 8.0ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id

In [ ]:
%cd /content/govcon-dashboard
!cat app/\(dashboard\)/layout.tsx | head -40

/content/govcon-dashboard
import { redirect } from "next/navigation";
import { DashboardShell } from "@/components/DashboardShell";
import { getSupabaseServerClient } from "@/lib/supabase/server";

export default async function DashboardLayout({ children }: { children: React.ReactNode }) {
  const supabase = await getSupabaseServerClient();
  const {
    data: { user },
    error,
  } = await supabase.auth.getUser();

  if (error || !user) {
    redirect("/auth/signin"); // ← was "/login" (Bug #1 fix)
  }

  return <DashboardShell user={user}>{children}</DashboardShell>;
}


In [ ]:
%cd /content/govcon-dashboard
!grep -n "redirect\|push\|contracts" components/DashboardShell.tsx | head -20

/content/govcon-dashboard
15:  { href: '/contracts', label: 'Contracts' },
35:    router.push('/auth/signin');


In [ ]:
%cd /content/govcon-dashboard
!cat app/auth/callback/route.ts

/content/govcon-dashboard
cat: app/auth/callback/route.ts: No such file or directory


In [ ]:
%cd /content/govcon-dashboard
!find app -name "*.ts" -path "*/auth/*" | head -10
!find app -name "*.tsx" -path "*/auth/*" | head -10

/content/govcon-dashboard
app/auth/callback/page.tsx
app/auth/signup/page.tsx
app/auth/signin/page.tsx


In [ ]:
%cd /content/govcon-dashboard
!cat app/auth/callback/page.tsx

/content/govcon-dashboard
"use client";

import { useRouter } from "next/navigation";
import { useEffect, useState } from "react";
import { createBrowserClient } from "@supabase/ssr";

export default function AuthCallbackPage() {
  const router = useRouter();
  const [message, setMessage] = useState("Signing you in…");

  useEffect(() => {
    let active = true;

    async function finalizeAuth() {
      const supabase = createBrowserClient(
        process.env.NEXT_PUBLIC_SUPABASE_URL!,
        process.env.NEXT_PUBLIC_SUPABASE_ANON_KEY!
      );

      // Parse the URL hash — Supabase magic links arrive as:
      // /auth/callback#access_token=XXX&refresh_token=YYY&type=magiclink
      const hash = window.location.hash.substring(1); // strip leading #
      const params = new URLSearchParams(hash);
      const accessToken = params.get("access_token");
      const refreshToken = params.get("refresh_token");

      if (accessToken && refreshToken) {
        // Explicitly set the session

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("app/auth/callback/page.tsx")
text = path.read_text()
text = text.replace('router.replace("/contracts")', 'router.replace("/")')
path.write_text(text)
print("Patched")
print(f"Remaining /contracts refs: {text.count('/contracts')}")

/content/govcon-dashboard
Patched
Remaining /contracts refs: 0


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Redirect to dashboard home after login" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 22.8s
✓ Finished TypeScript in 17.6s 
✓ Collecting page data using 1 worker in 867.6ms 
  Generating static pages using 1 worker (0/31)  [    ]  Generating static pages using 1 worker (4/31)  [=   ]  Generating static pages using 1 worker (27/31)  [==  ]✓ Generating static pages using 1 worker (31/31) in 424.3ms
✓ Finalizing page optimization in 14.4ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[i

In [ ]:
%cd /content/govcon-dashboard
!cat middleware.ts 2>/dev/null || !cat src/middleware.ts 2>/dev/null || echo "not found"
!find . -name "middleware.ts" -not -path "*/node_modules/*"

/content/govcon-dashboard
import { createServerClient, type CookieOptions } from "@supabase/ssr";
import { NextResponse, type NextRequest } from "next/server";

export async function middleware(request: NextRequest) {
  let response = NextResponse.next({
    request: {
      headers: request.headers,
    },
  });

  const supabase = createServerClient(
    process.env.NEXT_PUBLIC_SUPABASE_URL!,
    process.env.NEXT_PUBLIC_SUPABASE_ANON_KEY!,
    {
      cookies: {
        get(name: string) {
          return request.cookies.get(name)?.value;
        },
        set(name: string, value: string, options: CookieOptions) {
          response.cookies.set({ name, value, ...options });
        },
        remove(name: string, options: CookieOptions) {
          response.cookies.set({ name, value: "", ...options });
        },
      },
    }
  );

  const { data: { user } } = await supabase.auth.getUser();

  // Protect dashboard routes - redirect to sign-in if not authenticated
  if (request.ne

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("middleware.ts")
text = path.read_text()

# Fix 1: redirect auth pages to / instead of /contracts for logged in users
text = text.replace(
    'return NextResponse.redirect(new URL("/contracts", request.url));',
    'return NextResponse.redirect(new URL("/", request.url));'
)

# Fix 2: redirect root / to dashboard if authenticated
old = '  // Allow admin seed endpoint without auth'
new = '''  // Redirect root to dashboard if authenticated
  if (request.nextUrl.pathname === "/" && user) {
    return NextResponse.redirect(new URL("/dashboard", request.url));
  }

  // Allow admin seed endpoint without auth'''

text = text.replace(old, new)

path.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("middleware.ts")
text = path.read_text()
# The dashboard home is at / (app/(dashboard)/page.tsx)
# But / is the public landing page for unauthenticated users
# We need authenticated users hitting / to see the dashboard
# The layout.tsx already handles auth — just need to NOT redirect / away
# Actually the issue is the public landing page IS at /
# We need to move the dashboard to /home or keep / as the landing page

print(text)

/content/govcon-dashboard
import { createServerClient, type CookieOptions } from "@supabase/ssr";
import { NextResponse, type NextRequest } from "next/server";

export async function middleware(request: NextRequest) {
  let response = NextResponse.next({
    request: {
      headers: request.headers,
    },
  });

  const supabase = createServerClient(
    process.env.NEXT_PUBLIC_SUPABASE_URL!,
    process.env.NEXT_PUBLIC_SUPABASE_ANON_KEY!,
    {
      cookies: {
        get(name: string) {
          return request.cookies.get(name)?.value;
        },
        set(name: string, value: string, options: CookieOptions) {
          response.cookies.set({ name, value, ...options });
        },
        remove(name: string, options: CookieOptions) {
          response.cookies.set({ name, value: "", ...options });
        },
      },
    }
  );

  const { data: { user } } = await supabase.auth.getUser();

  // Protect dashboard routes - redirect to sign-in if not authenticated
  if (request.ne

In [ ]:
%cd /content/govcon-dashboard
!ls app/*.tsx app/*.ts 2>/dev/null

/content/govcon-dashboard
 app/header.tsx   app/layout.tsx   app/page.tsx


In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("middleware.ts")
text = path.read_text()
text = text.replace(
    'return NextResponse.redirect(new URL("/dashboard", request.url));',
    'return NextResponse.redirect(new URL("/home", request.url));'
)
path.write_text(text)

# Rename the dashboard home page to /home
import os
os.makedirs("app/(dashboard)/home", exist_ok=True)

# Move page.tsx to home
import shutil
shutil.copy("app/(dashboard)/page.tsx", "app/(dashboard)/home/page.tsx")

print("Done")

/content/govcon-dashboard
Done


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Add /home dashboard route, fix middleware redirect" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 23.5s
✓ Finished TypeScript in 17.8s 
✓ Collecting page data using 1 worker in 866.6ms 
  Generating static pages using 1 worker (0/32)  [    ]  Generating static pages using 1 worker (3/32)  [=   ]  Generating static pages using 1 worker (28/32)  [==  ]✓ Generating static pages using 1 worker (32/32) in 462.5ms
✓ Finalizing page optimization in 9.4ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path

path = Path("components/DashboardShell.tsx")
text = path.read_text()

text = text.replace(
    "{ href: '/contracts', label: 'Contracts' },",
    "{ href: '/home', label: 'Home' },\n  { href: '/contracts', label: 'Contracts' },"
)

path.write_text(text)
print("Patched")

/content/govcon-dashboard
Patched


In [ ]:
%cd /content/govcon-dashboard
!npm run build && git add -A && git commit -m "Add Home to navigation" && git push origin main

/content/govcon-dashboard

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 20.9s
✓ Finished TypeScript in 17.3s 
✓ Collecting page data using 1 worker in 1528.6ms 
  Generating static pages using 1 worker (0/32)  [    ]  Generating static pages using 1 worker (0/32)  [=   ]  Generating static pages using 1 worker (8/32)  [==  ]✓ Generating static pages using 1 worker (32/32) in 586.0ms
✓ Finalizing page optimization in 13.4ms 

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[i

In [ ]:
%cd /content/govcon-dashboard
!git log --oneline -15

/content/govcon-dashboard
d98eb47 (HEAD -> main, origin/main, origin/HEAD) Add Home to navigation
97849b6 Add /home dashboard route, fix middleware redirect
060b8cc Redirect to dashboard home after login
446587f Add dashboard home page with stats and critical expiring contracts
7773fff Merge Recompete Radar into contracts page with RPS scoring and expiry dates
2d84cd2 Add contract search links to lobbying firm client list
21d088f Add keyword filter to Recompete Radar
39c96c7 Add View All Contracts and Expiring Contracts buttons to contractor profile
a00134f Fix lobbying_firms column name total_income
4b91aef Fix Recompete Radar to use period_of_performance_end
ad45e1d Add Recompete Radar to navigation
03478ad Add Recompete Radar page with RPS scoring
df26dd9 Fix contractor_name type cast in influence lookup
4ceb26f Fix lobbying_clients column name to name
60a3ccc Fix params Promise in contract detail page


In [ ]:
%cd /content/govcon-dashboard
!cat app/api/stripe/checkout/route.ts
!echo "---"
!cat app/api/stripe/webhook/route.ts

/content/govcon-dashboard
/**
 * POST /api/stripe/checkout
 *
 * Creates a Stripe Checkout Session and returns the URL.
 * Called by <UpgradeButton> on the dashboard.
 *
 * Body: { plan: "pro_monthly" | "pro_annual" }
 * Returns: { url: string }
 *
 * Security:
 *  - Requires authenticated Supabase session (cookie)
 *  - Idempotency key = userId + plan + 5-min window (prevents double-clicks)
 *  - trial_period_days = 30 for V1 beta (remove after beta ends)
 *
 * Env vars required:
 *  STRIPE_SECRET_KEY
 *  STRIPE_PRICE_PRO_MONTHLY
 *  STRIPE_PRICE_PRO_ANNUAL
 *  NEXT_PUBLIC_SITE_URL
 *  NEXT_PUBLIC_SUPABASE_URL      (already set)
 *  SUPABASE_SERVICE_ROLE_KEY     (already set)
 */

import { NextResponse } from 'next/server';
import { getSupabaseServerClient } from '@/lib/supabase/server';
import { supabaseAdmin } from '@/lib/supabase/admin';
import {
  getStripe,
  PRICES,
  PLAN_LABELS,
  checkoutIdempotencyKey,
  type PlanKey,
} from '@/lib/stripe';

const SITE_URL =
  process.env.NE

In [ ]:
%cd /content/govcon-dashboard
!cat lib/stripe.ts

/content/govcon-dashboard
/**
 * lib/stripe.ts
 * Stripe client singleton + price/product constants for GovCon Terminal V1.
 *
 * SETUP (one-time, before going live):
 * 1. Create products + prices in Stripe Dashboard (or CLI):
 *    - Contractor Pro Monthly: $49/mo recurring
 *    - Contractor Pro Annual:  $470/yr recurring
 * 2. Copy the price IDs (price_xxxxx) into STRIPE_PRICE_PRO_MONTHLY
 *    and STRIPE_PRICE_PRO_ANNUAL in Vercel env vars.
 * 3. Add STRIPE_SECRET_KEY and STRIPE_WEBHOOK_SECRET to Vercel.
 *
 * Required env vars (all environments):
 *   STRIPE_SECRET_KEY          sk_live_... or sk_test_...
 *   STRIPE_WEBHOOK_SECRET      whsec_...
 *   STRIPE_PRICE_PRO_MONTHLY   price_...
 *   STRIPE_PRICE_PRO_ANNUAL    price_...
 *   NEXT_PUBLIC_SITE_URL       https://govcon-dashboard-eta.vercel.app
 */

import Stripe from 'stripe';

// ─── Singleton ────────────────────────────────────────────────────────────────

let _stripe: Stripe | null = null;

export function getStripe(): S

In [ ]:
%cd /content/govcon-dashboard
import subprocess
result = subprocess.run(['grep', '-rn', 'UpgradeButton\|upgrade\|stripe', 'app/', 'components/'],
    capture_output=True, text=True)
print(result.stdout[:3000])

/content/govcon-dashboard
app/api/status/route.ts:45:        stripe: process.env.STRIPE_SECRET_KEY ? "configured" : "missing",
app/api/political/route.ts:39:        upgrade_cta: "Unlock lobbying + 50 states: $99/mo",
app/api/fec/route.ts:41:        upgrade_cta: "Unlock political network + lobbying: $99/mo",
app/api/stripe/checkout/route.ts:2: * POST /api/stripe/checkout
app/api/stripe/checkout/route.ts:5: * Called by <UpgradeButton> on the dashboard.
app/api/stripe/checkout/route.ts:33:} from '@/lib/stripe';
app/api/stripe/checkout/route.ts:75:  const stripe = getStripe();
app/api/stripe/checkout/route.ts:76:  let stripeCustomerId: string | null = null;
app/api/stripe/checkout/route.ts:81:    .select('stripe_customer_id')
app/api/stripe/checkout/route.ts:83:    .maybeSingle() as { data: { stripe_customer_id: string | null } | null };
app/api/stripe/checkout/route.ts:85:  if (profile?.stripe_customer_id) {
app/api/stripe/checkout/route.ts:86:    stripeCustomerId = profile.stripe_custome

<>:3: SyntaxWarning: invalid escape sequence '\|'
<>:3: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_366/3071880579.py:3: SyntaxWarning: invalid escape sequence '\|'
  result = subprocess.run(['grep', '-rn', 'UpgradeButton\|upgrade\|stripe', 'app/', 'components/'],


In [ ]:
Let's build it. First let me see the current contract detail page so I know exactly where to add the button:

```python
%cd /content/govcon-dashboard
!cat app/\(dashboard\)/contracts/\[id\]/page.tsx
```

SyntaxError: unterminated string literal (detected at line 1) (2038078316.py, line 1)

In [ ]:
```python
%cd /content/govcon-dashboard
import subprocess
result = subprocess.run(['cat', 'app/(dashboard)/contracts/[id]/page.tsx'], capture_output=True, text=True)
print(result.stdout)
```

SyntaxError: invalid syntax (14624476.py, line 1)

In [ ]:
import subprocess
result = subprocess.run(['cat', 'app/(dashboard)/contracts/[id]/page.tsx'], capture_output=True, text=True)
print(result.stdout)

'use client'

import { useEffect, useState } from 'react'
import { useRouter } from 'next/navigation'
import Link from 'next/link'

type Contract = {
  id: string
  contractor_name: string
  agency: string
  award_amount: number
  award_date: string
  naics_code: string
  psc_code: string
  place_of_performance_state: string
  description: string
}

type InfluenceData = {
  lobbying_firms: { id: string; name: string; total_income: number }[]
  politicians: { id: string; name: string; party: string; state: string; firm_name: string }[]
  client_matches: string[]
}

export default function ContractDetailPage({ params }: { params: any }) {
  const router = useRouter()
  const [contract, setContract] = useState<Contract | null>(null)
  const [influence, setInfluence] = useState<InfluenceData | null>(null)
  const [loading, setLoading] = useState(true)
  const [contractId, setContractId] = useState<string>('')

  useEffect(() => {
    Promise.resolve(params).then(p => setContractId(p.id))
 

In [ ]:
from pathlib import Path

Path("app/(dashboard)/contracts/[id]/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'
import { useRouter } from 'next/navigation'
import Link from 'next/link'

type Contract = {
  id: string
  contractor_name: string
  agency: string
  award_amount: number
  award_date: string
  naics_code: string
  psc_code: string
  place_of_performance_state: string
  description: string
  period_of_performance_end: string | null
}

type InfluenceData = {
  lobbying_firms: { id: string; name: string; total_income: number }[]
  politicians: { id: string; name: string; party: string; state: string; firm_name: string }[]
  client_matches: string[]
}

export default function ContractDetailPage({ params }: { params: any }) {
  const router = useRouter()
  const [contract, setContract] = useState<Contract | null>(null)
  const [influence, setInfluence] = useState<InfluenceData | null>(null)
  const [loading, setLoading] = useState(true)
  const [contractId, setContractId] = useState<string>('')
  const [bidBrief, setBidBrief] = useState<string>('')
  const [bidLoading, setBidLoading] = useState(false)
  const [bidError, setBidError] = useState<string>('')
  const [showBid, setShowBid] = useState(false)

  useEffect(() => {
    Promise.resolve(params).then(p => setContractId(p.id))
  }, [params])

  useEffect(() => {
    if (!contractId) return
    fetch(`/api/contracts/${contractId}`)
      .then(r => r.json())
      .then(json => {
        setContract(json.data)
        setInfluence(json.influence)
        setLoading(false)
      })
  }, [contractId])

  const generateBidBrief = async () => {
    if (!contract) return
    setBidLoading(true)
    setBidBrief('')
    setBidError('')
    setShowBid(true)

    try {
      const res = await fetch('/api/bid-brief', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ contract, influence })
      })
      if (!res.ok) throw new Error('Failed to generate brief')
      const json = await res.json()
      setBidBrief(json.brief)
    } catch (e: any) {
      setBidError('Failed to generate bid brief. Please try again.')
    } finally {
      setBidLoading(false)
    }
  }

  if (loading) return <div className="p-8 bg-slate-950 min-h-screen text-slate-400">Loading...</div>
  if (!contract) return <div className="p-8 bg-slate-950 min-h-screen text-slate-400">Contract not found.</div>

  const hasInfluence = influence && (influence.lobbying_firms.length > 0 || influence.politicians.length > 0)

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen max-w-5xl mx-auto">
      <button onClick={() => router.back()} className="text-blue-400 hover:text-blue-300 text-sm mb-4 flex items-center gap-1">
        ← Back to Contracts
      </button>

      {/* Header */}
      <div className="flex items-start justify-between mb-6">
        <div>
          <h1 className="text-2xl font-bold text-blue-400 mb-1">{contract.contractor_name}</h1>
          <p className="text-4xl font-bold text-green-400 mb-1">
            {contract.award_amount?.toLocaleString('en-US', { style: 'currency', currency: 'USD' })}
          </p>
          <p className="text-slate-400">{contract.agency}</p>
        </div>
        <button
          onClick={generateBidBrief}
          disabled={bidLoading}
          className="bg-blue-600 hover:bg-blue-500 disabled:opacity-50 text-white px-5 py-2.5 rounded-lg text-sm font-semibold flex items-center gap-2 shrink-0 ml-4"
        >
          {bidLoading ? (
            <>
              <span className="animate-spin">⟳</span> Generating...
            </>
          ) : (
            <>📋 Prepare Bid Brief</>
          )}
        </button>
      </div>

      {/* Key Details */}
      <div className="grid grid-cols-2 md:grid-cols-3 gap-4 bg-slate-900 border border-slate-700 rounded-lg p-4 mb-6">
        <div><p className="text-xs text-slate-500 mb-1">Contract ID</p><p className="text-sm font-mono">{contract.id}</p></div>
        <div><p className="text-xs text-slate-500 mb-1">Award Date</p><p className="text-sm">{contract.award_date}</p></div>
        <div><p className="text-xs text-slate-500 mb-1">NAICS</p><p className="text-sm">{contract.naics_code}</p></div>
        <div><p className="text-xs text-slate-500 mb-1">PSC Code</p><p className="text-sm">{contract.psc_code}</p></div>
        <div><p className="text-xs text-slate-500 mb-1">State</p><p className="text-sm">{contract.place_of_performance_state}</p></div>
        {contract.period_of_performance_end && (
          <div><p className="text-xs text-slate-500 mb-1">Expires</p><p className="text-sm text-orange-400">{contract.period_of_performance_end}</p></div>
        )}
      </div>

      {/* Description */}
      {contract.description && (
        <div className="mb-6">
          <h2 className="text-blue-400 font-semibold mb-2">Description</h2>
          <p className="text-slate-300 text-sm leading-relaxed bg-slate-900 border border-slate-700 rounded-lg p-4">{contract.description}</p>
        </div>
      )}

      {/* Bid Brief Panel */}
      {showBid && (
        <div className="mb-6 border border-blue-700 rounded-lg overflow-hidden">
          <div className="bg-blue-950 px-4 py-3 flex items-center justify-between">
            <h2 className="font-semibold text-blue-300">📋 Competitive Bid Brief</h2>
            <button onClick={() => setShowBid(false)} className="text-slate-500 hover:text-slate-300 text-xs">✕ Close</button>
          </div>
          <div className="p-5 bg-slate-900">
            {bidLoading && (
              <div className="flex items-center gap-3 text-slate-400">
                <span className="animate-spin text-blue-400">⟳</span>
                Analyzing contract data and generating competitive intelligence...
              </div>
            )}
            {bidError && <p className="text-red-400 text-sm">{bidError}</p>}
            {bidBrief && (
              <div className="prose prose-invert prose-sm max-w-none">
                <pre className="whitespace-pre-wrap text-slate-200 text-sm leading-relaxed font-sans">{bidBrief}</pre>
                <div className="mt-4 flex gap-3">
                  <button
                    onClick={() => navigator.clipboard.writeText(bidBrief)}
                    className="bg-slate-700 hover:bg-slate-600 text-slate-200 px-4 py-2 rounded text-xs"
                  >
                    📋 Copy to Clipboard
                  </button>
                </div>
              </div>
            )}
          </div>
        </div>
      )}

      {/* Influence Section */}
      <div className="border border-slate-700 rounded-lg overflow-hidden">
        <div className="bg-slate-800 px-4 py-3 flex items-center justify-between">
          <h2 className="font-semibold text-slate-200">Influence Intelligence</h2>
          {!hasInfluence && <span className="text-xs text-slate-500">No lobbying connections found in database</span>}
        </div>

        {hasInfluence ? (
          <div className="p-4 space-y-6">
            {influence!.client_matches.length > 0 && (
              <div className="text-xs text-slate-500">
                Matched via lobbying records: {influence!.client_matches.slice(0, 3).join(', ')}
              </div>
            )}
            {influence!.lobbying_firms.length > 0 && (
              <div>
                <h3 className="text-sm font-semibold text-slate-300 mb-3 uppercase tracking-wide">
                  Lobbying Firms Hired ({influence!.lobbying_firms.length})
                </h3>
                <div className="space-y-2">
                  {influence!.lobbying_firms.map(firm => (
                    <Link key={firm.id} href={`/lobbying-firms/${firm.id}`}
                      className="flex items-center justify-between bg-slate-900 hover:bg-slate-800 border border-slate-700 rounded px-4 py-3 transition-colors">
                      <span className="text-blue-400 font-medium text-sm">{firm.name}</span>
                      {firm.total_income > 0 && (
                        <span className="text-green-400 text-sm font-medium">
                          {firm.total_income.toLocaleString('en-US', { style: 'currency', currency: 'USD', maximumFractionDigits: 0 })} total income
                        </span>
                      )}
                    </Link>
                  ))}
                </div>
              </div>
            )}
            {influence!.politicians.length > 0 && (
              <div>
                <h3 className="text-sm font-semibold text-slate-300 mb-3 uppercase tracking-wide">
                  Connected Politicians ({influence!.politicians.length})
                </h3>
                <div className="space-y-2">
                  {influence!.politicians.map(pol => (
                    <Link key={pol.id} href={`/politicians/${pol.id}`}
                      className="flex items-center justify-between bg-slate-900 hover:bg-slate-800 border border-slate-700 rounded px-4 py-3 transition-colors">
                      <div>
                        <span className="text-blue-400 font-medium text-sm">{pol.name}</span>
                        <span className="text-slate-500 text-xs ml-2">{pol.party} · {pol.state}</span>
                      </div>
                      <span className="text-slate-500 text-xs">via {pol.firm_name}</span>
                    </Link>
                  ))}
                </div>
              </div>
            )}
          </div>
        ) : (
          <div className="p-6 text-center text-slate-600 text-sm">
            This contractor has no matching lobbying activity in the current database.
          </div>
        )}
      </div>
    </div>
  )
}
""")
print("Written")

Written


In [ ]:
import os
os.makedirs("app/api/bid-brief", exist_ok=True)

Path("app/api/bid-brief/route.ts").write_text("""import { NextRequest, NextResponse } from 'next/server'

export async function POST(req: NextRequest) {
  const { contract, influence } = await req.json()

  if (!contract) {
    return NextResponse.json({ error: 'No contract data provided' }, { status: 400 })
  }

  const incumbentName = contract.contractor_name
  const agency = contract.agency
  const amount = contract.award_amount?.toLocaleString('en-US', { style: 'currency', currency: 'USD' })
  const naics = contract.naics_code
  const state = contract.place_of_performance_state
  const description = contract.description || 'Not specified'
  const expiry = contract.period_of_performance_end || 'Not specified'

  const lobbyingContext = influence?.lobbying_firms?.length > 0
    ? `The incumbent has hired the following lobbying firms: ${influence.lobbying_firms.map((f: any) => f.name).join(', ')}. Total lobbying spend: ${influence.lobbying_firms.reduce((sum: number, f: any) => sum + (f.total_income || 0), 0).toLocaleString('en-US', { style: 'currency', currency: 'USD', maximumFractionDigits: 0 })}.`
    : 'No lobbying activity found for this incumbent.'

  const prompt = `You are a federal contracting expert helping a small business prepare a competitive bid brief.

CONTRACT INTELLIGENCE:
- Incumbent Contractor: ${incumbentName}
- Awarding Agency: ${agency}
- Contract Value: ${amount}
- NAICS Code: ${naics}
- State: ${state}
- Contract Expiry: ${expiry}
- Scope: ${description}
- Lobbying Intelligence: ${lobbyingContext}

Generate a competitive bid brief for a small business looking to compete for this contract recompete. Structure it as follows:

## EXECUTIVE SUMMARY
One paragraph summarizing the opportunity and why a small business should pursue it.

## INCUMBENT ANALYSIS
Analyze the incumbent's position — how long they've held it, their likely pricing strategy, where they may be vulnerable (complacency, price bloat, scope creep).

## COMPETITIVE POSITIONING
How should a challenger position against this incumbent? What advantages does a smaller, leaner competitor have?

## RECOMMENDED BID PRICE STRATEGY
Based on the contract value of ${amount}, recommend a pricing approach to come in competitively while remaining profitable.

## KEY WIN THEMES
5 bullet points — the key themes the bid should emphasize to differentiate from the incumbent.

## NEXT STEPS
Concrete action items: who to contact at the agency, what past performance to highlight, what teaming arrangements to consider.

Keep the tone professional but direct. This is actionable intelligence, not marketing copy.`

  try {
    const response = await fetch('https://api.anthropic.com/v1/messages', {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json',
        'x-api-key': process.env.ANTHROPIC_API_KEY || '',
        'anthropic-version': '2023-06-01'
      },
      body: JSON.stringify({
        model: 'claude-sonnet-4-20250514',
        max_tokens: 1500,
        messages: [{ role: 'user', content: prompt }]
      })
    })

    const data = await response.json()
    const brief = data.content?.[0]?.text || 'Unable to generate brief'

    return NextResponse.json({ brief })
  } catch (error) {
    return NextResponse.json({ error: 'Failed to call Claude API' }, { status: 500 })
  }
}
""")
print("Written")

Written


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add Prepare Bid Brief feature with Claude AI" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-1000:])


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 24.1s
  Running TypeScript ...
  Generating static pages using 1 worker (0/33) ...
  Generating static pages using 1 worker (8/33) 
  Generating static pages using 1 worker (16/33) 
  Generating static pages using 1 worker (24/33) 
✓ Generating static pages using 1 worker (33/33) in 690.8ms
  Finalizing page optimization ...

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/bid-brief
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/status

In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()

# Add better error logging
text = text.replace(
    "  try {\n    const response = await fetch('https://api.anthropic.com/v1/messages',",
    """  const apiKey = process.env.ANTHROPIC_API_KEY
  if (!apiKey) {
    return NextResponse.json({ error: 'ANTHROPIC_API_KEY not configured', brief: 'API key missing — check Vercel env vars' }, { status: 200 })
  }

  try {
    const response = await fetch('https://api.anthropic.com/v1/messages',"""
)

text = text.replace(
    "      'x-api-key': process.env.ANTHROPIC_API_KEY || '',",
    "      'x-api-key': apiKey,"
)

text = text.replace(
    "    const brief = data.content?.[0]?.text || 'Unable to generate brief'",
    """    if (data.error) {
      return NextResponse.json({ brief: 'Claude API error: ' + data.error.message }, { status: 200 })
    }
    const brief = data.content?.[0]?.text || 'No content returned from Claude'"""
)

route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add error diagnostics to bid brief API" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 25.1s
  Running TypeScript ...
  Generating static pages using 1 worker (0/33) ...
  Generating static pages using 1 worker (8/33) 
  Generating static pages using 1 worker (16/33) 
  Generating static pages using 1 worker (24/33) 
✓ Generating static pages using 1 worker (33/33) in 466.2ms
  Finalizing page optimization ...

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/bid-brief
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/status

In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()

# Replace Anthropic call with Gemini
old = """  try {
    const response = await fetch('https://api.anthropic.com/v1/messages', {
      method: 'POST',
      headers: {
        'Content-Type': 'application/json',
        'x-api-key': apiKey,
        'anthropic-version': '2023-06-01'
      },
      body: JSON.stringify({
        model: 'claude-sonnet-4-20250514',
        max_tokens: 1500,
        messages: [{ role: 'user', content: prompt }]
      })
    })

    const data = await response.json()
    if (data.error) {
      return NextResponse.json({ brief: 'Claude API error: ' + data.error.message }, { status: 200 })
    }
    const brief = data.content?.[0]?.text || 'No content returned from Claude'

    return NextResponse.json({ brief })
  } catch (error) {
    return NextResponse.json({ error: 'Failed to call Claude API' }, { status: 500 })
  }"""

new = """  try {
    const response = await fetch(
      `https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key=${apiKey}`,
      {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({
          contents: [{ parts: [{ text: prompt }] }],
          generationConfig: { maxOutputTokens: 1500, temperature: 0.7 }
        })
      }
    )

    const data = await response.json()
    if (data.error) {
      return NextResponse.json({ brief: 'Gemini API error: ' + data.error.message }, { status: 200 })
    }
    const brief = data.candidates?.[0]?.content?.parts?.[0]?.text || 'No content returned'

    return NextResponse.json({ brief })
  } catch (error) {
    return NextResponse.json({ error: 'Failed to call Gemini API' }, { status: 500 })
  }"""

text = text.replace(old, new)

# Also rename the key reference
text = text.replace(
    "const apiKey = process.env.ANTHROPIC_API_KEY",
    "const apiKey = process.env.GEMINI_API_KEY"
).replace(
    "'API key missing — check Vercel env vars'",
    "'GEMINI_API_KEY not configured — check Vercel env vars'"
)

route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Switch bid brief to Gemini free tier" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 22.2s
  Running TypeScript ...
  Generating static pages using 1 worker (0/33) ...
  Generating static pages using 1 worker (8/33) 
  Generating static pages using 1 worker (16/33) 
  Generating static pages using 1 worker (24/33) 
✓ Generating static pages using 1 worker (33/33) in 495.1ms
  Finalizing page optimization ...

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/bid-brief
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/status

In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()
text = text.replace(
    "gemini-1.5-flash:generateContent",
    "gemini-2.0-flash:generateContent"
)
route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Fix Gemini model name" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-1000:])

tors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/status
├ ƒ /api/stripe/checkout
├ ƒ /api/stripe/webhook
├ ƒ /api/votes/analysis
├ ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main 1f1f538] Fix Gemini model name
 1 file changed, 1 insertion(+), 1 deletion(-)


In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()
text = text.replace(
    "gemini-2.0-flash:generateContent",
    "gemini-1.5-pro:generateContent"
)
route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Fix Gemini model to gemini-1.5-pro" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-500:])

 ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main 1ba8f5a] Fix Gemini model to gemini-1.5-pro
 1 file changed, 1 insertion(+), 1 deletion(-)



In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()

# Switch to v1 endpoint with correct model name
text = text.replace(
    "`https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-pro:generateContent?key=${apiKey}`",
    "`https://generativelanguage.googleapis.com/v1/models/gemini-1.5-pro-latest:generateContent?key=${apiKey}`"
)

route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Fix Gemini API endpoint to v1" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-500:])

sis
├ ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main e191268] Fix Gemini API endpoint to v1
 1 file changed, 1 insertion(+), 1 deletion(-)



In [ ]:
import requests
from google.colab import userdata

# List available Gemini models
api_key = userdata.get('GEMINI_API_KEY')
r = requests.get(f"https://generativelanguage.googleapis.com/v1/models?key={api_key}")
models = r.json()
for m in models.get('models', []):
    if 'generateContent' in m.get('supportedGenerationMethods', []):
        print(m['name'])

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-lite


In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()

text = text.replace(
    "`https://generativelanguage.googleapis.com/v1/models/gemini-1.5-pro-latest:generateContent?key=${apiKey}`",
    "`https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key=${apiKey}`"
)

route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Fix Gemini model to gemini-2.0-flash v1beta" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-500:])

atchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main fdd40d5] Fix Gemini model to gemini-2.0-flash v1beta
 1 file changed, 1 insertion(+), 1 deletion(-)



In [ ]:
import subprocess
result = subprocess.run(['grep', 'generativelanguage', 'app/api/bid-brief/route.ts'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

      `https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key=${apiKey}`,



In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()
text = text.replace(
    "gemini-2.0-flash:generateContent",
    "gemini-2.5-flash:generateContent"
)
route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Switch to gemini-2.5-flash" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-300:])

irms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main 257b925] Switch to gemini-2.5-flash
 1 file changed, 1 insertion(+), 1 deletion(-)



In [ ]:
from pathlib import Path

route = Path("app/api/bid-brief/route.ts")
text = route.read_text()
text = text.replace(
    "maxOutputTokens: 1500",
    "maxOutputTokens: 3000"
)
route.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Increase bid brief output length" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-300:])

 ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main 2a4842d] Increase bid brief output length
 1 file changed, 1 insertion(+), 1 deletion(-)



In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Increase bid brief output length" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-300:])

obbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean



In [ ]:
import subprocess
result = subprocess.run(['git', 'log', '--oneline', '-10'], capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

2a4842d Increase bid brief output length
257b925 Switch to gemini-2.5-flash
fdd40d5 Fix Gemini model to gemini-2.0-flash v1beta
e191268 Fix Gemini API endpoint to v1
1ba8f5a Fix Gemini model to gemini-1.5-pro
1f1f538 Fix Gemini model name
d2f5d4d Switch bid brief to Gemini free tier
7548901 Add error diagnostics to bid brief API
7dd4414 Add Prepare Bid Brief feature with Claude AI
d98eb47 Add Home to navigation



In [ ]:
import subprocess
result = subprocess.run(['grep', '-r', 'resend\|RESEND', 'app/', 'lib/'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout[:2000])

app/api/radar/route.ts: *  RESEND_API_KEY                 (add to Vercel — all environments)
app/api/radar/route.ts:  const apiKey = process.env.RESEND_API_KEY;
app/api/radar/route.ts:  if (!apiKey) throw new Error('RESEND_API_KEY not set');
app/api/radar/route.ts:  const res = await fetch('https://api.resend.com/emails', {



<>:2: SyntaxWarning: invalid escape sequence '\|'
<>:2: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_366/1267070290.py:2: SyntaxWarning: invalid escape sequence '\|'
  result = subprocess.run(['grep', '-r', 'resend\|RESEND', 'app/', 'lib/'],


In [ ]:
import subprocess
result = subprocess.run(['cat', 'app/api/radar/route.ts'], capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

/**
 * Recompete Radar — Weekly Digest API
 * POST /api/radar          → generate + send for a single user (manual trigger / cron)
 * GET  /api/radar?preview=1 → returns the HTML for the calling authenticated user (preview)
 *
 * Auth strategy:
 *  - Cron calls: Bearer CRON_SECRET header
 *  - Manual preview: authenticated Supabase session (cookie)
 *
 * Env vars required:
 *  NEXT_PUBLIC_SUPABASE_URL       (already set)
 *  SUPABASE_SERVICE_ROLE_KEY      (already set)
 *  RESEND_API_KEY                 (add to Vercel — all environments)
 *  CRON_SECRET                    (add to Vercel — all environments)
 *  NEXT_PUBLIC_SITE_URL           (add to Vercel — e.g. https://govcon-dashboard-eta.vercel.app)
 */

import { NextResponse } from 'next/server';
import { supabaseAdmin } from '@/lib/supabase/admin';
import { getSupabaseServerClient } from '@/lib/supabase/server';
import { buildRadarEmail } from '@/lib/email/radar-template';

// ─── Types ────────────────────────────────────────────

In [ ]:
import subprocess
result = subprocess.run(['cat', 'lib/email/radar-template.ts'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout[:2000] if result.stdout else "NOT FOUND")

/**
 * Recompete Radar — HTML Email Template
 * Barnacle Doctrine compliant:
 *  - Every data point sourced from USAspending.gov
 *  - RPS disclaimer on every send
 *  - AI output disclosure where applicable
 *  - "Report Error" link on every contract row
 *  - No investment advice language
 *  - No speculative claims about incumbents
 */

interface RadarContract {
  id: string;
  contractor_name: string;
  agency: string;
  naics_code: string | null;
  award_amount: number | null;
  expires_at: string | null;
  description: string | null;
  rps_total: number | null;
  days_to_expiry: number | null;
}

interface UserProfile {
  user_id: string;
  email: string;
  naics_codes: string[];
  target_states: string[];
  company_name: string | null;
}

// ─── Formatters ───────────────────────────────────────────────────────────────

function fmtCurrency(n: number | null): string {
  if (n == null) return 'N/A';
  if (n >= 1_000_000) return `$${(n / 1_000_000).toFixed(1)}M`;
  if (n >= 1_000)

In [ ]:
from pathlib import Path
import json

vercel_json = Path("/content/govcon-dashboard/vercel.json")
if vercel_json.exists():
    config = json.loads(vercel_json.read_text())
else:
    config = {}

config["crons"] = [
    {
        "path": "/api/radar",
        "schedule": "0 9 * * 1"
    }
]

vercel_json.write_text(json.dumps(config, indent=2))
print("Written")
print(json.dumps(config, indent=2))

Written
{
  "crons": [
    {
      "path": "/api/radar",
      "schedule": "0 9 * * 1"
    }
  ]
}


In [ ]:
CREATE TABLE IF NOT EXISTS contractor_profiles (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  user_id uuid UNIQUE NOT NULL,
  company_name text,
  naics_codes text[] DEFAULT '{}',
  target_states text[] DEFAULT '{}',
  stripe_customer_id text,
  created_at timestamptz DEFAULT now(),
  updated_at timestamptz DEFAULT now()
);

SyntaxError: invalid syntax (387838178.py, line 1)

In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add weekly radar cron job" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-500:])

atchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main b296201] Add weekly radar cron job
 1 file changed, 8 insertions(+)
 create mode 100644 vercel.json



In [ ]:
import subprocess
result = subprocess.run(['cat', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout[:3000])

"use client";

/**
 * /politicians/[id]
 *
 * Full politician profile. BioGuide ID as [id].
 *
 * Tabs:
 *   Overview        — bio, committee assignments
 *   Votes           — roll call votes from GovTrack (govtrack.us)
 *   Campaign Finance — ALL PAC contributions received (FEC real data)
 *   GovCon Money   — subset of PAC donors flagged as gov-contractor-related
 *
 * Source attribution + Report Error on every tab.
 */

import { useEffect, useState } from "react";
import { useParams, useRouter } from "next/navigation";
import { CONGRESS_DISCLAIMER } from "@/lib/congress";
import { FEC_DISCLAIMER } from "@/lib/fec";

// ── Types ──────────────────────────────────────────────────────────────────

interface Committee {
  code: string;
  name: string;
  title: string;
  side: string | null;
  is_defense_relevant: boolean;
}

interface Member {
  id: string;
  full_name: string;
  first_name: string;
  last_name: string;
  party: string;
  state: string;
  district: string | null;
  cha

In [ ]:
import subprocess
result = subprocess.run(['wc', '-l', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

# Show the last 100 lines to see how it ends
result2 = subprocess.run(['tail', '-100', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result2.stdout)

1010 app/(dashboard)/politicians/[id]/page.tsx


                    {/* AI Analysis — 3 insights */}
                    <div className="space-y-3 border-t border-slate-700 pt-4">
                      <div className="text-xs text-slate-400 uppercase tracking-wider">AI Analysis</div>
                      {err ? (
                        <p className="text-xs text-red-400">⚠ {err}</p>
                      ) : analysis ? (
                        <div className="space-y-3">
                          {analysis.split(/\n(?=\d+\.)/).filter(Boolean).map((insight, i) => (
                            <div key={i} className="text-xs text-slate-300 leading-relaxed bg-slate-800/40 rounded-lg p-3">
                              {insight.trim()}
                            </div>
                          ))}
                          <div className="flex items-center justify-between pt-1">
                            <div className="text-xs text-slate-600">
                              {va.cac

In [ ]:
import subprocess
result = subprocess.run(['cat', 'app/api/politicians/[id]/vote-insight/route.ts'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout[:2000])

/**
 * POST /api/politicians/[id]/vote-insight
 *
 * AI analysis of PAC donor → vote correlation.
 * Model: Google Gemini 2.5 Flash (free tier, 1000 req/day, no billing)
 */

import { NextResponse } from "next/server";
import { supabaseAdmin } from "@/lib/supabase/admin";

const GEMINI_KEY = process.env.GEMINI_API_KEY;
const GEMINI_MODEL = "gemini-2.5-flash";
const ENABLE_POLITICAL_V2 = process.env.ENABLE_POLITICAL_V2 === "true";

export async function POST(
  request: Request,
  { params }: { params: Promise<{ id: string }> }
) {
  if (!ENABLE_POLITICAL_V2) {
    return NextResponse.json({ gated: true }, { status: 402 });
  }
  if (!GEMINI_KEY) {
    return NextResponse.json({ error: "AI not configured" }, { status: 503 });
  }

  const { id: bioguideId } = await params;
  const body = await request.json();
  const { question, position, result, date, is_defense, bill_title } = body;

  // eslint-disable-next-line @typescript-eslint/no-explicit-any
  const { data: politician } = await 

In [ ]:
from pathlib import Path
import os

os.makedirs("app/api/politicians/[id]/brief", exist_ok=True)

Path("app/api/politicians/[id]/brief/route.ts").write_text("""import { NextResponse } from 'next/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

const GEMINI_KEY = process.env.GEMINI_API_KEY

export async function POST(
  request: Request,
  { params }: { params: Promise<{ id: string }> }
) {
  if (!GEMINI_KEY) {
    return NextResponse.json({ error: 'AI not configured' }, { status: 503 })
  }

  const { id: bioguideId } = await params

  // Load politician
  const { data: politician } = await (supabaseAdmin as any)
    .from('politicians')
    .select('full_name, party, state, chamber, committees, votes_with_party_pct, missed_votes_pct')
    .eq('id', bioguideId)
    .single()

  if (!politician) {
    return NextResponse.json({ error: 'Politician not found' }, { status: 404 })
  }

  // Load top GovCon PAC donors
  const { data: govconDonors } = await (supabaseAdmin as any)
    .from('pac_donors')
    .select('contributor_name, contribution_receipt_amount')
    .eq('politician_id', bioguideId)
    .eq('is_govcon_related', true)
    .order('contribution_receipt_amount', { ascending: false })
    .limit(20)

  // Load lobbying firm connections
  const { data: lobbyLinks } = await (supabaseAdmin as any)
    .from('lobbying_firm_politician_links')
    .select('firm_id, confidence_score')
    .eq('politician_id', bioguideId)
    .limit(10)

  const firmIds = (lobbyLinks || []).map((l: any) => l.firm_id)
  let lobbyFirms: any[] = []
  if (firmIds.length > 0) {
    const { data: firms } = await (supabaseAdmin as any)
      .from('lobbying_firms')
      .select('name, total_income, top_issues, client_count')
      .in('id', firmIds)
      .limit(10)
    lobbyFirms = firms || []
  }

  // Load recent key votes
  const { data: votes } = await (supabaseAdmin as any)
    .from('politician_votes')
    .select('question, position, result, date, is_defense_appropriations, bill_title, description')
    .eq('politician_id', bioguideId)
    .order('date', { ascending: false })
    .limit(20)

  // Build context
  const donorTotal = (govconDonors || []).reduce((sum: number, d: any) => sum + Number(d.contribution_receipt_amount || 0), 0)
  const topDonors = (govconDonors || []).slice(0, 10).map((d: any) =>
    `${d.contributor_name}: $${Number(d.contribution_receipt_amount).toLocaleString()}`
  ).join('\\n')

  const lobbyContext = lobbyFirms.length > 0
    ? lobbyFirms.map((f: any) => `${f.name} (${f.client_count} clients, issues: ${(f.top_issues || []).slice(0, 3).join(', ')})`).join('\\n')
    : 'No direct lobbying firm connections found in database'

  const voteContext = (votes || []).slice(0, 10).map((v: any) =>
    `${v.date}: ${v.question} — voted ${v.position} (${v.result})${v.is_defense_appropriations ? ' [DEFENSE]' : ''}`
  ).join('\\n')

  const prompt = `You are a federal contracting intelligence analyst. Generate a comprehensive Political Intelligence Brief for a small business contractor evaluating this politician's influence relationships.

POLITICIAN: ${politician.full_name} (${politician.party === 'R' ? 'Republican' : politician.party === 'D' ? 'Democrat' : 'Independent'})
ROLE: ${politician.chamber === 'senate' ? 'Senator' : 'Representative'}, ${politician.state}
PARTY LINE VOTES: ${politician.votes_with_party_pct}% with party
MISSED VOTES: ${politician.missed_votes_pct}%

GOVCON PAC MONEY RECEIVED (total: $${donorTotal.toLocaleString()}):
${topDonors || 'No GovCon PAC donations on record'}

LOBBYING FIRMS CONNECTED TO THIS POLITICIAN:
${lobbyContext}

RECENT KEY VOTES:
${voteContext || 'No vote records available'}

Generate a Political Intelligence Brief with these sections:

## POLITICAL INTELLIGENCE BRIEF: ${politician.full_name}

## EXECUTIVE SUMMARY
2-3 sentences: who this politician is, their influence on federal contracting, and what a small contractor needs to know about them.

## MONEY & INFLUENCE ANALYSIS
Analyze the GovCon PAC donations. Which industries dominate? What does the lobbying firm network tell us about who has access to this politician? Connect the dots between money sources and likely policy priorities.

## VOTE PATTERN ANALYSIS
Based on the vote record, what are this politician's consistent positions on defense, appropriations, and government spending? Are there patterns that suggest favorable or unfavorable positions toward small business contractors?

## CONTRACTOR OPPORTUNITY ASSESSMENT
Given their committee assignments, funding relationships, and vote history — what types of contracts, agencies, or procurement programs is this politician most likely to support or oppose? What should a small contractor know before pursuing opportunities in their jurisdiction?

## INFLUENCE RISK FLAGS
Any red flags? Heavy reliance on large prime contractor donations that might crowd out small business interests? Voting patterns inconsistent with stated positions? Lobbying relationships that could affect competitive fairness?

## STRATEGIC RECOMMENDATION
One paragraph: should a small contractor actively engage with this politician's office, monitor their legislative activity, or simply note their influence on relevant appropriations? What's the actionable takeaway?

Keep the tone professional, factual, and actionable. Base all claims on the data provided. Do not speculate beyond what the data supports.`

  try {
    const response = await fetch(
      `https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key=${GEMINI_KEY}`,
      {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({
          contents: [{ parts: [{ text: prompt }] }],
          generationConfig: { maxOutputTokens: 2500, temperature: 0.7 }
        })
      }
    )

    const data = await response.json()
    if (data.error) {
      return NextResponse.json({ error: data.error.message }, { status: 500 })
    }

    const brief = data.candidates?.[0]?.content?.parts?.[0]?.text || 'Unable to generate brief'
    return NextResponse.json({ brief, politician: politician.full_name })
  } catch (error) {
    return NextResponse.json({ error: 'Failed to generate brief' }, { status: 500 })
  }
}
""")
print("API route written")

API route written


In [ ]:
import subprocess
result = subprocess.run(['grep', '-n', 'activeTab\|tabPanel\|Overview\|return (',
    'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout[:2000])

9: *   Overview        — bio, committee assignments
130:  return (
148:  return (
174:  return (
214:  const [activeTab, setActiveTab] = useState<"overview" | "votes" | "finance" | "govcon">(
270:    { id: "overview" as const, label: "Overview" },
292:    return (
306:    return (
331:  return (
419:              activeTab === tab.id
429:      {/* ── Overview Tab ─────────────────────────────────────────────────── */}
430:      {activeTab === "overview" && (
511:      {activeTab === "votes" && (
609:      {activeTab === "finance" && (
688:      {activeTab === "govcon" && (
833:                return (
861:                      return (
986:              `BioGuide ID: ${bioguideId}\nName: ${member.full_name}\nTab: ${activeTab}\n\nDescribe the error:\n`



<>:2: SyntaxWarning: invalid escape sequence '\|'
<>:2: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_366/2204676200.py:2: SyntaxWarning: invalid escape sequence '\|'
  result = subprocess.run(['grep', '-n', 'activeTab\|tabPanel\|Overview\|return (',


In [ ]:
import subprocess
result = subprocess.run(['grep', '-n', 'confidence\|politician\|lobbying_firm_politician',
    'app/api/contracts/[id]/route.ts'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

30:    politicians: { id: string; name: string; party: string; state: string; firm_name: string }[]
32:  } = { lobbying_firms: [], politicians: [], client_matches: [] }
65:        // Get politician links
67:          .from('lobbying_firm_politician_links')
68:          .select('firm_id, politician_id')
73:          const politicianIds = [...new Set((links as any[]).map((l: any) => l.politician_id).filter(Boolean))]
75:          if (politicianIds.length > 0) {
76:            const { data: politicians } = await supabaseAdmin
77:              .from('politicians')
79:              .in('id', politicianIds)
82:            if (politicians) {
83:              influence.politicians = (politicians as any[]).map((p: any) => {
84:                const link = (links as any[]).find((l: any) => l.politician_id === p.id)



<>:2: SyntaxWarning: invalid escape sequence '\|'
<>:2: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_366/2185705670.py:2: SyntaxWarning: invalid escape sequence '\|'
  result = subprocess.run(['grep', '-n', 'confidence\|politician\|lobbying_firm_politician',


In [ ]:
from pathlib import Path

# Fix 1: Contract detail API — add confidence_score and source_basis
route = Path("app/api/contracts/[id]/route.ts")
text = route.read_text()

text = text.replace(
    "politicians: { id: string; name: string; party: string; state: string; firm_name: string }[]",
    "politicians: { id: string; name: string; party: string; state: string; firm_name: string; confidence_score: number; source_basis: string }[]"
)

text = text.replace(
    ".from('lobbying_firm_politician_links')\n          .select('firm_id, politician_id')",
    ".from('lobbying_firm_politician_links')\n          .select('firm_id, politician_id, confidence_score, source_basis')"
)

text = text.replace(
    """                return {
                  id: p.id,
                  name: p.name,
                  party: p.party,
                  state: p.state,
                  firm_name: firm?.name || ''
                }""",
    """                return {
                  id: p.id,
                  name: p.name,
                  party: p.party,
                  state: p.state,
                  firm_name: firm?.name || '',
                  confidence_score: Math.round((link?.confidence_score || 0) * 100),
                  source_basis: link?.source_basis || ''
                }"""
)

route.write_text(text)
print("Contract API patched")

Contract API patched


In [ ]:
page = Path("app/(dashboard)/contracts/[id]/page.tsx")
text = page.read_text()

text = text.replace(
    "politicians: { id: string; name: string; party: string; state: string; firm_name: string }[]",
    "politicians: { id: string; name: string; party: string; state: string; firm_name: string; confidence_score: number; source_basis: string }[]"
)

text = text.replace(
    """                    <div>
                        <span className="text-blue-400 font-medium text-sm">{pol.name}</span>
                        <span className="text-slate-500 text-xs ml-2">{pol.party} · {pol.state}</span>
                      </div>
                      <span className="text-slate-500 text-xs">via {pol.firm_name}</span>""",
    """                    <div>
                        <span className="text-blue-400 font-medium text-sm">{pol.name}</span>
                        <span className="text-slate-500 text-xs ml-2">{pol.party} · {pol.state}</span>
                      </div>
                      <div className="flex items-center gap-2">
                        {pol.confidence_score > 0 && (
                          <span className={`text-xs font-bold px-1.5 py-0.5 rounded ${pol.confidence_score >= 80 ? 'bg-red-600 text-white' : pol.confidence_score >= 60 ? 'bg-orange-500 text-white' : 'bg-slate-600 text-slate-300'}`}>
                            {pol.confidence_score}% match
                          </span>
                        )}
                        <span className="text-slate-500 text-xs">via {pol.firm_name}</span>
                      </div>"""
)

page.write_text(text)
print("Contract detail page patched")

Contract detail page patched


In [ ]:
import subprocess

# Check what's in the overview tab around committee assignments
result = subprocess.run(['grep', '-n', 'committee\|Committee\|Overview\|overview',
    'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout[:2000])

9: *   Overview        — bio, committee assignments
24:interface Committee {
51:  committees: Committee[];
214:  const [activeTab, setActiveTab] = useState<"overview" | "votes" | "finance" | "govcon">(
215:    "overview"
270:    { id: "overview" as const, label: "Overview" },
429:      {/* ── Overview Tab ─────────────────────────────────────────────────── */}
430:      {activeTab === "overview" && (
468:          {/* Committee assignments */}
471:              Committee Assignments
473:            {member.committees.length === 0 ? (
474:              <p className="text-slate-500 text-sm">No committee data available.</p>
477:                {member.committees.map((c, i) => (
696:              <strong className="text-white">{member.full_name}</strong> from committees



<>:4: SyntaxWarning: invalid escape sequence '\|'
<>:4: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_366/2357560547.py:4: SyntaxWarning: invalid escape sequence '\|'
  result = subprocess.run(['grep', '-n', 'committee\|Committee\|Overview\|overview',


In [ ]:
import subprocess
result = subprocess.run(['sed', '-n', '480,515p', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

                    className={`flex items-center justify-between rounded px-3 py-2 ${
                      c.is_defense_relevant
                        ? "bg-amber-900/20 border border-amber-800/30"
                        : "bg-slate-800"
                    }`}
                  >
                    <div className="min-w-0">
                      <span className="text-sm text-slate-200">{c.name}</span>
                      {c.title && c.title !== "Member" && (
                        <span className="ml-2 text-xs text-slate-500">{c.title}</span>
                      )}
                    </div>
                    <div className="flex items-center gap-2 ml-2 shrink-0">
                      <span className="text-xs text-slate-500 font-mono">{c.code}</span>
                      {c.is_defense_relevant && (
                        <span className="text-xs text-amber-500">⚠ Defense</span>
                      )}
                    </div>
                  </div>
               

In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/politicians/[id]/page.tsx")
text = page.read_text()

# Add state variables for brief after existing state declarations
text = text.replace(
    '  const [activeTab, setActiveTab] = useState<"overview" | "votes" | "finance" | "govcon">(\n    "overview"\n  )',
    '  const [activeTab, setActiveTab] = useState<"overview" | "votes" | "finance" | "govcon">(\n    "overview"\n  )\n  const [brief, setBrief] = useState<string>("")\n  const [briefLoading, setBriefLoading] = useState(false)\n  const [showBrief, setShowBrief] = useState(false)'
)

# Add the lobbying firms + brief section before the closing of overview tab
old_overview_end = '''            <Attribution
              label="Congress.gov"
              url={`https://www.congress.gov/member/${member.id}`}
            />
          </div>
        </div>
      )}

      {/* ── Votes Tab'''

new_overview_end = '''            <Attribution
              label="Congress.gov"
              url={`https://www.congress.gov/member/${member.id}`}
            />
          </div>
        </div>

        {/* ── Political Intelligence Brief ──────────────────────────────── */}
        <div className="mt-4">
          <button
            onClick={async () => {
              if (brief) { setShowBrief(!showBrief); return; }
              setBriefLoading(true); setShowBrief(true);
              try {
                const res = await fetch(`/api/politicians/${bioguideId}/brief`, { method: 'POST' });
                const json = await res.json();
                setBrief(json.brief || json.error || 'Unable to generate brief');
              } catch { setBrief('Failed to generate brief'); }
              finally { setBriefLoading(false); }
            }}
            disabled={briefLoading}
            className="bg-violet-700 hover:bg-violet-600 disabled:opacity-50 text-white px-5 py-2.5 rounded-lg text-sm font-semibold flex items-center gap-2"
          >
            {briefLoading ? <><span className="animate-spin">⟳</span> Generating...</> : showBrief ? '✕ Close Brief' : '🔍 Generate Political Intelligence Brief'}
          </button>

          {showBrief && (
            <div className="mt-3 border border-violet-700 rounded-lg overflow-hidden">
              <div className="bg-violet-950 px-4 py-3 flex items-center justify-between">
                <h3 className="font-semibold text-violet-300 text-sm">🔍 Political Intelligence Brief</h3>
                <span className="text-xs text-violet-500">Powered by Gemini · Based on FEC + LDA public records</span>
              </div>
              <div className="p-5 bg-slate-900">
                {briefLoading && (
                  <div className="flex items-center gap-3 text-slate-400 text-sm">
                    <span className="animate-spin text-violet-400">⟳</span>
                    Analyzing influence relationships, PAC donations, and vote patterns...
                  </div>
                )}
                {brief && !briefLoading && (
                  <div>
                    <pre className="whitespace-pre-wrap text-slate-200 text-sm leading-relaxed font-sans">{brief}</pre>
                    <div className="mt-4 flex gap-3">
                      <button onClick={() => navigator.clipboard.writeText(brief)}
                        className="bg-slate-700 hover:bg-slate-600 text-slate-200 px-4 py-2 rounded text-xs">
                        📋 Copy to Clipboard
                      </button>
                      <button onClick={() => { setBrief(''); setBriefLoading(false); }}
                        className="bg-slate-700 hover:bg-slate-600 text-slate-200 px-4 py-2 rounded text-xs">
                        🔄 Regenerate
                      </button>
                    </div>
                    <p className="text-xs text-slate-600 mt-3">All data sourced from FEC Schedule A public records, Senate LDA filings, and GovTrack.us. This is not investment or legal advice.</p>
                  </div>
                )}
              </div>
            </div>
          )}
        </div>
      )}

      {/* ── Votes Tab'''

text = text.replace(old_overview_end, new_overview_end)
page.write_text(text)
print("Politician page patched")

Politician page patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add Political Intelligence Brief and influence scores to politician and contract pages" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-500:] if result.stderr else "")


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...

22m 512 |         {/* ── Political Intelligence Brief ──────────────────────────────── */}
      |         ^
  513 |         <div className="mt-4">
  514 |           <button
  515 |             onClick={async () => {

Expected '</', got '{'


    at <unknown> (./app/(dashboard)/politicians/[id]/page.tsx:512:9)



In [ ]:
import subprocess
result = subprocess.run(['sed', '-n', '500,520p', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

                    </div>
                  </div>
                ))}
              </div>
            )}
            <Attribution
              label="Congress.gov"
              url={`https://www.congress.gov/member/${member.id}`}
            />
          </div>
        </div>

        {/* ── Political Intelligence Brief ──────────────────────────────── */}
        <div className="mt-4">
          <button
            onClick={async () => {
              if (brief) { setShowBrief(!showBrief); return; }
              setBriefLoading(true); setShowBrief(true);
              try {
                const res = await fetch(`/api/politicians/${bioguideId}/brief`, { method: 'POST' });
                const json = await res.json();



In [ ]:
import subprocess
result = subprocess.run(['sed', '-n', '428,435p', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

          </button>
        ))}
      </div>

      {/* ── Overview Tab ─────────────────────────────────────────────────── */}
      {activeTab === "overview" && (
        <div className="space-y-4">
          {/* Stats */}



In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/politicians/[id]/page.tsx")
text = page.read_text()

# The brief section needs to be inside the overview tab closing
# Find the misplaced location and fix it
text = text.replace(
    '''          </div>
        </div>

        {/* ── Political Intelligence Brief ──────────────────────────────── */}''',
    '''          </div>
        </div>

        {/* ── Political Intelligence Brief ──────────────────────────────── */}
        <div className="space-y-0">'''
)

# Actually just wrap the whole thing properly - find the exact broken spot
# The issue is the overview tab closing )} is missing before the brief div
text = text.replace(
    '''        {/* ── Political Intelligence Brief ──────────────────────────────── */}
        <div className="mt-4">''',
    '''      )}

      {/* ── Political Intelligence Brief ──────────────────────────────── */}
      {activeTab === "overview" && (
        <div className="mt-4">'''
)

# Fix the closing of the brief section
text = text.replace(
    '''      {/* ── Votes Tab''',
    '''      )}

      {/* ── Votes Tab'''
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build 2>&1 | tail -20'],
    capture_output=True, text=True
)
print(result.stdout)


⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...

> Build error occurred
Error: Turbopack build failed with 1 errors:
./app/(dashboard)/politicians/[id]/page.tsx:512:9
Parsing ecmascript source code failed
  510 |         </div>
  511 |
> 512 |         {/* ── Political Intelligence Brief ──────────────────────────────── */}
      |         ^
  513 |         <div className="space-y-0">
  514 |         <div className="mt-4">
  515 |           <button

Expected '</', got '{'


    at <unknown> (./app/(dashboard)/politicians/[id]/page.tsx:512:9)



In [ ]:
import subprocess
result = subprocess.run(['sed', '-n', '505,530p', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

            <Attribution
              label="Congress.gov"
              url={`https://www.congress.gov/member/${member.id}`}
            />
          </div>
        </div>

        {/* ── Political Intelligence Brief ──────────────────────────────── */}
        <div className="space-y-0">
        <div className="mt-4">
          <button
            onClick={async () => {
              if (brief) { setShowBrief(!showBrief); return; }
              setBriefLoading(true); setShowBrief(true);
              try {
                const res = await fetch(`/api/politicians/${bioguideId}/brief`, { method: 'POST' });
                const json = await res.json();
                setBrief(json.brief || json.error || 'Unable to generate brief');
              } catch { setBrief('Failed to generate brief'); }
              finally { setBriefLoading(false); }
            }}
            disabled={briefLoading}
            className="bg-violet-700 hover:bg-violet-600 disabled:opacity-50 text-white p

In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/politicians/[id]/page.tsx")
text = page.read_text()

text = text.replace(
    '''          </div>
        </div>

        {/* ── Political Intelligence Brief ──────────────────────────────── */}
        <div className="space-y-0">
        <div className="mt-4">''',
    '''          </div>
        </div>
      )}

      {/* ── Political Intelligence Brief ──────────────────────────────── */}
      {activeTab === "overview" && (
        <div className="mt-4">'''
)

# Remove the extra space-y-0 div we added
text = text.replace(
    '      {/* ── Votes Tab',
    '      )}\n\n      {/* ── Votes Tab'
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build 2>&1 | tail -20'],
    capture_output=True, text=True
)
print(result.stdout)

  571 |

Unexpected token. Did you mean `{'}'}` or `&rbrace;`?


./app/(dashboard)/politicians/[id]/page.tsx:570:8
Parsing ecmascript source code failed
  568 |       )}
  569 |
> 570 |       )}
      |        ^
  571 |
  572 |       {/* ── Votes Tab ─────────────────────────────────────────────────────── */}
  573 |       {activeTab === "votes" && (

Unexpected token. Did you mean `{'}'}` or `&rbrace;`?


    at <unknown> (./app/(dashboard)/politicians/[id]/page.tsx:568:8)
    at <unknown> (./app/(dashboard)/politicians/[id]/page.tsx:570:8)



In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/politicians/[id]/page.tsx")
text = page.read_text()

# Remove the duplicate )}
text = text.replace(
    '      )}\n\n      )}\n\n      {/* ── Votes Tab',
    '      )}\n\n      {/* ── Votes Tab'
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build 2>&1 | tail -20'],
    capture_output=True, text=True
)
print(result.stdout)


⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...

> Build error occurred
Error: Turbopack build failed with 1 errors:
./app/(dashboard)/politicians/[id]/page.tsx:568:8
Parsing ecmascript source code failed
  566 |       )}
  567 |
> 568 |       )}
      |        ^
  569 |
  570 |       {/* ── Votes Tab ─────────────────────────────────────────────────────── */}
  571 |       {activeTab === "votes" && (

Unexpected token. Did you mean `{'}'}` or `&rbrace;`?


    at <unknown> (./app/(dashboard)/politicians/[id]/page.tsx:568:8)



In [ ]:
import subprocess
result = subprocess.run(['sed', '-n', '555,575p', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

                        className="bg-slate-700 hover:bg-slate-600 text-slate-200 px-4 py-2 rounded text-xs">
                        🔄 Regenerate
                      </button>
                    </div>
                    <p className="text-xs text-slate-600 mt-3">All data sourced from FEC Schedule A public records, Senate LDA filings, and GovTrack.us. This is not investment or legal advice.</p>
                  </div>
                )}
              </div>
            </div>
          )}
        </div>
      )}

      )}

      {/* ── Votes Tab ─────────────────────────────────────────────────────── */}
      {activeTab === "votes" && (
        <div className="space-y-4">
          <div className="flex items-center justify-between gap-4 flex-wrap">
            <span className="text-sm text-slate-400">
              {defenseVotesOnly



In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/politicians/[id]/page.tsx")
text = page.read_text()

text = text.replace(
    '      )}\n\n      )}\n\n      {/* ── Votes Tab',
    '      )}\n\n      {/* ── Votes Tab'
)

page.write_text(text)

# Verify
import subprocess
result = subprocess.run(['sed', '-n', '562,575p', 'app/(dashboard)/politicians/[id]/page.tsx'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

              </div>
            </div>
          )}
        </div>
      )}

      {/* ── Votes Tab ─────────────────────────────────────────────────────── */}
      {activeTab === "votes" && (
        <div className="space-y-4">
          <div className="flex items-center justify-between gap-4 flex-wrap">
            <span className="text-sm text-slate-400">
              {defenseVotesOnly
                ? `${data?.votes.defense_total ?? 0} defense/appropriations votes`
                : `${data?.votes.total ?? 0} total votes`}



In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build 2>&1 | tail -25'],
    capture_output=True, text=True
)
print(result.stdout)

├ ƒ /api/votes/analysis
├ ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand




In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && git add -A && git commit -m "Add Political Intelligence Brief to politician pages with influence scores" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout)

[main 2c31487] Add Political Intelligence Brief to politician pages with influence scores
 4 files changed, 212 insertions(+), 6 deletions(-)
 create mode 100644 app/api/politicians/[id]/brief/route.ts



In [ ]:
from pathlib import Path

# Fix 1: Increase token limit in the API
route = Path("app/api/politicians/[id]/brief/route.ts")
text = route.read_text()
text = text.replace(
    "maxOutputTokens: 2500",
    "maxOutputTokens: 4000"
)
route.write_text(text)
print("API patched")

# Fix 2: Remove any height constraints on the brief panel in the page
page = Path("app/(dashboard)/politicians/[id]/page.tsx")
text = page.read_text()

# Make the pre tag scrollable and not truncated
text = text.replace(
    '<pre className="whitespace-pre-wrap text-slate-200 text-sm leading-relaxed font-sans">{brief}</pre>',
    '<pre className="whitespace-pre-wrap text-slate-200 text-sm leading-relaxed font-sans overflow-visible">{brief}</pre>'
)
page.write_text(text)
print("Page patched")

API patched
Page patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Increase brief token limit and fix display truncation" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-500:])

/auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main f0d3986] Increase brief token limit and fix display truncation
 2 files changed, 2 insertions(+), 2 deletions(-)



In [ ]:
import subprocess
result = subprocess.run(['git', 'log', '--oneline', '-20'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

f0d3986 Increase brief token limit and fix display truncation
2c31487 Add Political Intelligence Brief to politician pages with influence scores
b296201 Add weekly radar cron job
2a4842d Increase bid brief output length
257b925 Switch to gemini-2.5-flash
fdd40d5 Fix Gemini model to gemini-2.0-flash v1beta
e191268 Fix Gemini API endpoint to v1
1ba8f5a Fix Gemini model to gemini-1.5-pro
1f1f538 Fix Gemini model name
d2f5d4d Switch bid brief to Gemini free tier
7548901 Add error diagnostics to bid brief API
7dd4414 Add Prepare Bid Brief feature with Claude AI
d98eb47 Add Home to navigation
97849b6 Add /home dashboard route, fix middleware redirect
060b8cc Redirect to dashboard home after login
446587f Add dashboard home page with stats and critical expiring contracts
7773fff Merge Recompete Radar into contracts page with RPS scoring and expiry dates
2d84cd2 Add contract search links to lobbying firm client list
21d088f Add keyword filter to Recompete Radar
39c96c7 Add View All Contracts a

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/api/profile", exist_ok=True)
os.makedirs("app/(dashboard)/profile", exist_ok=True)

print("=== PASTE THIS SQL IN SUPABASE SQL EDITOR ===")
print("""
CREATE TABLE IF NOT EXISTS user_profiles (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  user_id uuid UNIQUE NOT NULL REFERENCES auth.users(id),
  business_name text,
  naics_codes text[] DEFAULT '{}',
  target_states text[] DEFAULT '{}',
  business_description text,
  annual_revenue text,
  sam_registered boolean DEFAULT false,
  uei text,
  certifications text[] DEFAULT '{}',
  created_at timestamptz DEFAULT now(),
  updated_at timestamptz DEFAULT now()
);

ALTER TABLE user_profiles ENABLE ROW LEVEL SECURITY;

CREATE POLICY "Users can read own profile" ON user_profiles FOR SELECT USING (auth.uid() = user_id);
CREATE POLICY "Users can insert own profile" ON user_profiles FOR INSERT WITH CHECK (auth.uid() = user_id);
CREATE POLICY "Users can update own profile" ON user_profiles FOR UPDATE USING (auth.uid() = user_id);
""")
print("=== END SQL ===")

Path("app/api/profile/route.ts").write_text("""import { NextRequest, NextResponse } from 'next/server'
import { getSupabaseServerClient } from '@/lib/supabase/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

export async function GET() {
  const supabase = await getSupabaseServerClient()
  const { data: { user }, error: authError } = await supabase.auth.getUser()
  if (authError || !user) return NextResponse.json({ error: 'Unauthorized' }, { status: 401 })

  const { data, error } = await (supabaseAdmin as any)
    .from('user_profiles')
    .select('*')
    .eq('user_id', user.id)
    .maybeSingle()

  if (error) return NextResponse.json({ error: error.message }, { status: 500 })
  return NextResponse.json({ profile: data })
}

export async function POST(req: NextRequest) {
  const supabase = await getSupabaseServerClient()
  const { data: { user }, error: authError } = await supabase.auth.getUser()
  if (authError || !user) return NextResponse.json({ error: 'Unauthorized' }, { status: 401 })

  const body = await req.json()

  const { error } = await (supabaseAdmin as any)
    .from('user_profiles')
    .upsert({
      user_id: user.id,
      business_name: body.business_name || null,
      naics_codes: body.naics_codes || [],
      target_states: body.target_states || [],
      business_description: body.business_description || null,
      annual_revenue: body.annual_revenue || null,
      sam_registered: body.sam_registered || false,
      uei: body.uei || null,
      certifications: body.certifications || [],
      updated_at: new Date().toISOString()
    }, { onConflict: 'user_id' })

  if (error) return NextResponse.json({ error: error.message }, { status: 500 })
  return NextResponse.json({ success: true })
}
""")

Path("app/(dashboard)/profile/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'

const NAICS_OPTIONS = [
  { value: '238220', label: '238220 — HVAC' },
  { value: '541511', label: '541511 — IT Services' },
  { value: '236220', label: '236220 — Construction' },
  { value: '561730', label: '561730 — Landscaping' },
  { value: '541611', label: '541611 — Management Consulting' },
  { value: '237310', label: '237310 — Highway Construction' },
  { value: '238210', label: '238210 — Electrical' },
  { value: '238160', label: '238160 — Roofing' },
  { value: '541330', label: '541330 — Engineering' },
  { value: '541512', label: '541512 — Computer Systems' },
]

const STATES = ['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC']

const CERTIFICATIONS = ['8(a)', 'HUBZone', 'WOSB', 'SDVOSB', 'SDB', 'None']

const REVENUE_OPTIONS = [
  'Under $100K', '$100K-$500K', '$500K-$1M', '$1M-$5M', '$5M-$10M', 'Over $10M'
]

export default function ProfilePage() {
  const [form, setForm] = useState({
    business_name: '',
    naics_codes: [] as string[],
    target_states: [] as string[],
    business_description: '',
    annual_revenue: '',
    sam_registered: false,
    uei: '',
    certifications: [] as string[],
  })
  const [loading, setLoading] = useState(true)
  const [saving, setSaving] = useState(false)
  const [saved, setSaved] = useState(false)
  const [error, setError] = useState('')

  useEffect(() => {
    fetch('/api/profile')
      .then(r => r.json())
      .then(json => {
        if (json.profile) {
          setForm({
            business_name: json.profile.business_name || '',
            naics_codes: json.profile.naics_codes || [],
            target_states: json.profile.target_states || [],
            business_description: json.profile.business_description || '',
            annual_revenue: json.profile.annual_revenue || '',
            sam_registered: json.profile.sam_registered || false,
            uei: json.profile.uei || '',
            certifications: json.profile.certifications || [],
          })
        }
        setLoading(false)
      })
  }, [])

  const toggleArr = (arr: string[], val: string) =>
    arr.includes(val) ? arr.filter(v => v !== val) : [...arr, val]

  const handleSubmit = async () => {
    setSaving(true)
    setSaved(false)
    setError('')
    try {
      const res = await fetch('/api/profile', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(form)
      })
      const json = await res.json()
      if (json.success) { setSaved(true); setTimeout(() => setSaved(false), 4000) }
      else setError(json.error || 'Failed to save')
    } catch { setError('Network error') }
    finally { setSaving(false) }
  }

  if (loading) return <div className="p-8 bg-slate-950 min-h-screen text-slate-400">Loading profile...</div>

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen max-w-3xl mx-auto">
      <div className="mb-8">
        <h1 className="text-2xl font-bold text-blue-400">Business Profile</h1>
        <p className="text-slate-400 text-sm mt-1">Your profile powers personalized contract matching and weekly radar alerts.</p>
      </div>

      {saved && (
        <div className="bg-green-900/40 border border-green-700 text-green-300 rounded-lg px-4 py-3 mb-6 font-medium">
          ✅ Profile saved successfully
        </div>
      )}
      {error && (
        <div className="bg-red-900/40 border border-red-700 text-red-300 rounded-lg px-4 py-3 mb-6">
          ⚠ {error}
        </div>
      )}

      <div className="space-y-6">
        {/* Business Name */}
        <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
          <label className="block text-sm font-medium text-slate-300 mb-2">Business Name</label>
          <input
            className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm"
            placeholder="Your company name"
            value={form.business_name}
            onChange={e => setForm(f => ({ ...f, business_name: e.target.value }))}
          />
        </div>

        {/* NAICS Codes */}
        <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
          <label className="block text-sm font-medium text-slate-300 mb-3">Primary NAICS Codes</label>
          <p className="text-xs text-slate-500 mb-3">Select all that apply to your business</p>
          <div className="grid grid-cols-1 md:grid-cols-2 gap-2">
            {NAICS_OPTIONS.map(opt => (
              <label key={opt.value} className="flex items-center gap-3 bg-slate-800 hover:bg-slate-700 rounded px-3 py-2 cursor-pointer">
                <input type="checkbox" className="accent-blue-500"
                  checked={form.naics_codes.includes(opt.value)}
                  onChange={() => setForm(f => ({ ...f, naics_codes: toggleArr(f.naics_codes, opt.value) }))}
                />
                <span className="text-sm">{opt.label}</span>
              </label>
            ))}
          </div>
        </div>

        {/* Target States */}
        <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
          <label className="block text-sm font-medium text-slate-300 mb-3">Target States</label>
          <p className="text-xs text-slate-500 mb-3">States where you pursue federal contracts</p>
          <div className="grid grid-cols-6 md:grid-cols-10 gap-1.5">
            {STATES.map(s => (
              <label key={s} className={`flex items-center justify-center rounded px-2 py-1.5 cursor-pointer text-xs font-medium border transition-colors ${form.target_states.includes(s) ? 'bg-blue-600 border-blue-500 text-white' : 'bg-slate-800 border-slate-700 text-slate-400 hover:border-slate-500'}`}>
                <input type="checkbox" className="hidden"
                  checked={form.target_states.includes(s)}
                  onChange={() => setForm(f => ({ ...f, target_states: toggleArr(f.target_states, s) }))}
                />
                {s}
              </label>
            ))}
          </div>
        </div>

        {/* Business Description */}
        <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
          <label className="block text-sm font-medium text-slate-300 mb-2">Business Description</label>
          <textarea
            className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm h-24 resize-none"
            placeholder="Briefly describe your business and the services you provide to government agencies..."
            value={form.business_description}
            onChange={e => setForm(f => ({ ...f, business_description: e.target.value }))}
          />
        </div>

        {/* Annual Revenue + SAM */}
        <div className="grid grid-cols-1 md:grid-cols-2 gap-4">
          <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
            <label className="block text-sm font-medium text-slate-300 mb-2">Annual Revenue</label>
            <select
              className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm"
              value={form.annual_revenue}
              onChange={e => setForm(f => ({ ...f, annual_revenue: e.target.value }))}
            >
              <option value="">Select range</option>
              {REVENUE_OPTIONS.map(r => <option key={r} value={r}>{r}</option>)}
            </select>
          </div>
          <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
            <label className="block text-sm font-medium text-slate-300 mb-2">SAM.gov Registered</label>
            <div className="flex items-center gap-4 mt-1">
              {['Yes', 'No'].map(v => (
                <label key={v} className="flex items-center gap-2 cursor-pointer">
                  <input type="radio" className="accent-blue-500"
                    checked={form.sam_registered === (v === 'Yes')}
                    onChange={() => setForm(f => ({ ...f, sam_registered: v === 'Yes' }))}
                  />
                  <span className="text-sm">{v}</span>
                </label>
              ))}
            </div>
            {form.sam_registered && (
              <input
                className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm mt-3"
                placeholder="UEI number"
                value={form.uei}
                onChange={e => setForm(f => ({ ...f, uei: e.target.value }))}
              />
            )}
          </div>
        </div>

        {/* Certifications */}
        <div className="bg-slate-900 border border-slate-700 rounded-lg p-5">
          <label className="block text-sm font-medium text-slate-300 mb-3">Business Certifications</label>
          <div className="flex flex-wrap gap-2">
            {CERTIFICATIONS.map(cert => (
              <label key={cert} className={`flex items-center gap-2 rounded-full px-4 py-1.5 cursor-pointer text-sm border transition-colors ${form.certifications.includes(cert) ? 'bg-green-700 border-green-600 text-white' : 'bg-slate-800 border-slate-700 text-slate-400 hover:border-slate-500'}`}>
                <input type="checkbox" className="hidden"
                  checked={form.certifications.includes(cert)}
                  onChange={() => setForm(f => ({ ...f, certifications: toggleArr(f.certifications, cert) }))}
                />
                {cert}
              </label>
            ))}
          </div>
        </div>

        {/* Save Button */}
        <button
          onClick={handleSubmit}
          disabled={saving}
          className="w-full bg-blue-600 hover:bg-blue-500 disabled:opacity-50 text-white py-3 rounded-lg font-semibold text-sm"
        >
          {saving ? 'Saving...' : 'Save Business Profile'}
        </button>
      </div>
    </div>
  )
}
""")

# Add Profile to navigation
nav_path = Path("components/DashboardShell.tsx")
nav_text = nav_path.read_text()
if "Profile" not in nav_text:
    nav_text = nav_text.replace(
        "{ href: '/expiring', label: 'Recompete Radar' },",
        "{ href: '/expiring', label: 'Recompete Radar' },\n  { href: '/profile', label: 'Profile' },"
    )
    nav_path.write_text(nav_text)
    print("Nav updated")
else:
    print("Nav already has Profile")

print("\nFiles written:")
print("  app/api/profile/route.ts")
print("  app/(dashboard)/profile/page.tsx")
print("  components/DashboardShell.tsx (nav updated)")
print("\nNow run the SQL in Supabase, then build and push.")

/content/govcon-dashboard
=== PASTE THIS SQL IN SUPABASE SQL EDITOR ===

CREATE TABLE IF NOT EXISTS user_profiles (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  user_id uuid UNIQUE NOT NULL REFERENCES auth.users(id),
  business_name text,
  naics_codes text[] DEFAULT '{}',
  target_states text[] DEFAULT '{}',
  business_description text,
  annual_revenue text,
  sam_registered boolean DEFAULT false,
  uei text,
  certifications text[] DEFAULT '{}',
  created_at timestamptz DEFAULT now(),
  updated_at timestamptz DEFAULT now()
);

ALTER TABLE user_profiles ENABLE ROW LEVEL SECURITY;

CREATE POLICY "Users can read own profile" ON user_profiles FOR SELECT USING (auth.uid() = user_id);
CREATE POLICY "Users can insert own profile" ON user_profiles FOR INSERT WITH CHECK (auth.uid() = user_id);
CREATE POLICY "Users can update own profile" ON user_profiles FOR UPDATE USING (auth.uid() = user_id);

=== END SQL ===
Nav updated

Files written:
  app/api/profile/route.ts
  app/(dashboard)/pr

In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add business profile page with NAICS, states, certifications" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-500:] if result.stderr else "")


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 26.9s
  Running TypeScript ...
  Generating static pages using 1 worker (0/35) ...
  Generating static pages using 1 worker (8/35) 
  Generating static pages using 1 worker (17/35) 
  Generating static pages using 1 worker (26/35) 
✓ Generating static pages using 1 worker (35/35) in 501.9ms
  Finalizing page optimization ...

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/bid-brief
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/brief
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/profile
├ ƒ 

In [ ]:
import requests
from google.colab import userdata

url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_SERVICE_ROLE')
headers = {'apikey': key, 'Authorization': f'Bearer {key}'}

r = requests.get(
    f"{url}/rest/v1/user_profiles?select=*&limit=5",
    headers=headers
)
print(r.json())

[{'id': 'aaa20fb5-1642-491e-ad50-3c68f9b0f496', 'user_id': '9b86e789-0749-431d-a338-f511d919d30e', 'business_name': 'Evergreen Enterprises LLC', 'naics_codes': ['541512'], 'target_states': ['NJ'], 'business_description': 'I provide Data Analysis and GenAI consulting to small businesses', 'annual_revenue': 'Under $100K', 'sam_registered': True, 'uei': '123455', 'certifications': ['WOSB', 'SDB'], 'created_at': '2026-03-23T18:29:12.019123+00:00', 'updated_at': '2026-03-23T18:29:11.934+00:00'}]


In [ ]:
from pathlib import Path

route = Path("app/api/dashboard/route.ts")
text = route.read_text()

# Add user profile fetch at the top of the GET handler
text = text.replace(
    "export async function GET(req: NextRequest) {",
    """export async function GET(req: NextRequest) {
  // Try to load user profile for personalization
  let userProfile: any = null
  try {
    const supabase = await (await import('@/lib/supabase/server')).getSupabaseServerClient()
    const { data: { user } } = await supabase.auth.getUser()
    if (user) {
      const { data: profile } = await supabaseAdmin
        .from('user_profiles')
        .select('business_name, naics_codes, target_states')
        .eq('user_id', user.id)
        .maybeSingle()
      userProfile = profile
    }
  } catch {}"""
)

# Add profile to response
text = text.replace(
    "  return NextResponse.json({",
    "  return NextResponse.json({\n    userProfile,"
)

# Personalize the critical contracts query if profile exists
text = text.replace(
    "  const { data: critical, count: criticalCount } = await supabaseAdmin\n    .from('contracts')\n    .select('id, contractor_name, agency, award_amount, period_of_performance_end, place_of_performance_state, naics_code', { count: 'exact' })\n    .not('period_of_performance_end', 'is', null)\n    .gte('period_of_performance_end', today)\n    .lte('period_of_performance_end', in30)\n    .gt('award_amount', 0)\n    .order('period_of_performance_end', { ascending: true })\n    .limit(5)",
    """  let criticalQuery = supabaseAdmin
    .from('contracts')
    .select('id, contractor_name, agency, award_amount, period_of_performance_end, place_of_performance_state, naics_code', { count: 'exact' })
    .not('period_of_performance_end', 'is', null)
    .gte('period_of_performance_end', today)
    .lte('period_of_performance_end', in30)
    .gt('award_amount', 0)
    .order('period_of_performance_end', { ascending: true })
    .limit(5)

  if (userProfile?.naics_codes?.length > 0) {
    criticalQuery = criticalQuery.in('naics_code', userProfile.naics_codes)
  }
  if (userProfile?.target_states?.length > 0) {
    criticalQuery = criticalQuery.in('place_of_performance_state', userProfile.target_states)
  }

  const { data: critical, count: criticalCount } = await criticalQuery"""
)

route.write_text(text)
print("Patched")

Patched


In [ ]:
page = Path("app/(dashboard)/home/page.tsx")
text = page.read_text()

# Add userProfile to the DashboardData type
text = text.replace(
    "type DashboardData = {",
    "type DashboardData = {\n  userProfile?: { business_name: string; naics_codes: string[]; target_states: string[] } | null"
)

# Personalize the header
text = text.replace(
    '      <div className="mb-8">\n        <h1 className="text-3xl font-bold text-blue-400">GovCon Intelligence Terminal</h1>\n        <p className="text-slate-400 mt-1">Federal contracting intelligence — updated daily</p>\n      </div>',
    '''      <div className="mb-8 flex items-start justify-between">
        <div>
          <h1 className="text-3xl font-bold text-blue-400">
            {data.userProfile?.business_name ? `Welcome, ${data.userProfile.business_name}` : 'GovCon Intelligence Terminal'}
          </h1>
          <p className="text-slate-400 mt-1">
            {data.userProfile?.naics_codes?.length > 0
              ? `Showing opportunities for NAICS ${data.userProfile.naics_codes.join(', ')} · ${data.userProfile.target_states?.join(', ') || 'All states'}`
              : 'Federal contracting intelligence — updated daily'}
          </p>
        </div>
        {!data.userProfile && (
          <a href="/profile" className="bg-blue-600 hover:bg-blue-500 text-white px-4 py-2 rounded text-sm font-medium shrink-0">
            Set Up Profile →
          </a>
        )}
      </div>'''
)

# Fix the Critical section title to say "Matching Your Profile" when personalized
text = text.replace(
    '<h2 className="font-semibold text-red-300">🔴 Critical — Expiring in 30 Days</h2>',
    '{data.userProfile?.naics_codes?.length > 0 ? <h2 className="font-semibold text-red-300">🔴 Critical — Matching Your Profile</h2> : <h2 className="font-semibold text-red-300">🔴 Critical — Expiring in 30 Days</h2>}'
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Personalize dashboard home with user profile" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-300:] if result.stderr else "")


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 25.8s
  Running TypeScript ...

userProfile.target_states?.join(', ') || 'All states'}`
  65 |               : 'Federal contracting intelligence — updated daily'}
  66 |           </p>
Next.js build worker exited with code: 1 and signal: null



In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/home/page.tsx")
text = page.read_text()

text = text.replace(
    "              ? `Showing opportunities for NAICS ${data.userProfile.naics_codes.join(', ')} · ${data.userProfile.target_states?.join(', ') || 'All states'}`",
    "              ? `Showing opportunities for NAICS ${(data.userProfile?.naics_codes || []).join(', ')} · ${(data.userProfile?.target_states || []).join(', ') || 'All states'}`"
)

text = text.replace(
    "          {data.userProfile?.naics_codes?.length > 0",
    "          {(data.userProfile?.naics_codes?.length ?? 0) > 0"
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Fix TypeScript in personalized dashboard" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-1000:])

i/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/brief
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/profile
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/status
├ ƒ /api/stripe/checkout
├ ƒ /api/stripe/webhook
├ ƒ /api/votes/analysis
├ ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /profile
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main c6f6bf2] Fix TypeScript in personalized dashboard
 2 files changed, 43 insertions(+), 5 deletions(-)


In [ ]:
import subprocess
result = subprocess.run(['git', 'log', '--oneline', '-5'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
print(result.stdout)

c6f6bf2 Fix TypeScript in personalized dashboard
89db7c7 Add business profile page with NAICS, states, certifications
f0d3986 Increase brief token limit and fix display truncation
2c31487 Add Political Intelligence Brief to politician pages with influence scores
b296201 Add weekly radar cron job



In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/api/foia", exist_ok=True)
os.makedirs("app/(dashboard)/foia", exist_ok=True)

print("=== PASTE THIS SQL IN SUPABASE SQL EDITOR ===")
print("""
CREATE TABLE IF NOT EXISTS foia_requests (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  user_id uuid NOT NULL REFERENCES auth.users(id),
  status text DEFAULT 'submitted',
  target_agency text NOT NULL,
  subject text NOT NULL,
  description text NOT NULL,
  purpose text,
  priority text DEFAULT 'standard',
  admin_notes text,
  result_summary text,
  created_at timestamptz DEFAULT now(),
  updated_at timestamptz DEFAULT now()
);

ALTER TABLE foia_requests ENABLE ROW LEVEL SECURITY;

CREATE POLICY "Users can read own requests" ON foia_requests FOR SELECT USING (auth.uid() = user_id);
CREATE POLICY "Users can insert requests" ON foia_requests FOR INSERT WITH CHECK (auth.uid() = user_id);
""")
print("=== END SQL ===")

Path("app/api/foia/route.ts").write_text("""import { NextRequest, NextResponse } from 'next/server'
import { getSupabaseServerClient } from '@/lib/supabase/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

export async function GET() {
  const supabase = await getSupabaseServerClient()
  const { data: { user }, error: authError } = await supabase.auth.getUser()
  if (authError || !user) return NextResponse.json({ error: 'Unauthorized' }, { status: 401 })

  const { data, error } = await (supabaseAdmin as any)
    .from('foia_requests')
    .select('*')
    .eq('user_id', user.id)
    .order('created_at', { ascending: false })

  if (error) return NextResponse.json({ error: error.message }, { status: 500 })
  return NextResponse.json({ requests: data || [] })
}

export async function POST(req: NextRequest) {
  const supabase = await getSupabaseServerClient()
  const { data: { user }, error: authError } = await supabase.auth.getUser()
  if (authError || !user) return NextResponse.json({ error: 'Unauthorized' }, { status: 401 })

  const body = await req.json()
  const { target_agency, subject, description, purpose, priority } = body

  if (!target_agency || !subject || !description) {
    return NextResponse.json({ error: 'target_agency, subject, and description are required' }, { status: 400 })
  }

  const { data, error } = await (supabaseAdmin as any)
    .from('foia_requests')
    .insert({
      user_id: user.id,
      target_agency,
      subject,
      description,
      purpose: purpose || null,
      priority: priority || 'standard',
      status: 'submitted'
    })
    .select()
    .single()

  if (error) return NextResponse.json({ error: error.message }, { status: 500 })
  return NextResponse.json({ request: data })
}
""")

Path("app/(dashboard)/foia/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'

const AGENCIES = [
  'Department of Defense',
  'Department of Veterans Affairs',
  'Department of Agriculture',
  'Department of Transportation',
  'Department of the Interior',
  'Department of Health and Human Services',
  'Department of Homeland Security',
  'Department of State',
  'Department of Commerce',
  'Department of Justice',
  'Department of Energy',
  'General Services Administration',
  'NASA',
  'Environmental Protection Agency',
  'Small Business Administration',
  'Other',
]

const STATUS_STYLES: Record<string, string> = {
  submitted: 'bg-blue-700 text-blue-200',
  in_progress: 'bg-yellow-700 text-yellow-200',
  filed: 'bg-orange-700 text-orange-200',
  completed: 'bg-green-700 text-green-200',
  denied: 'bg-red-700 text-red-200',
}

const EXAMPLE_PROMPTS = [
  {
    title: '📋 Incumbent Performance',
    desc: 'Request contract performance evaluations and past performance reports for a specific contractor at a specific agency',
    subject: 'Contract Performance Reports — Incumbent Contractor',
    description: 'I am requesting all contract performance evaluations, CPARS reports, and past performance documentation for the current incumbent contractor on contracts within my target NAICS code. I am preparing a competitive bid and need to understand the incumbent\'s performance history.',
  },
  {
    title: '💰 Pricing Intelligence',
    desc: 'Request pricing proposals or cost breakdowns from previously awarded contracts in your NAICS code',
    subject: 'Pricing Proposals and Cost Breakdowns — Awarded Contracts',
    description: 'I am requesting pricing proposals, cost breakdowns, and any non-proprietary cost or pricing data from recently awarded contracts in my NAICS code. I am seeking to understand market pricing to develop a competitive bid strategy.',
  },
  {
    title: '⚖️ Protest Outcomes',
    desc: 'Request bid protest decisions and corrective action reports for contracts you plan to bid on',
    subject: 'Bid Protest Decisions and Corrective Action Reports',
    description: 'I am requesting all bid protest decisions, GAO rulings, and corrective action reports associated with a specific contract or procurement. I am evaluating whether to pursue a bid protest or preparing a bid and need to understand prior procurement history.',
  },
]

type FOIARequest = {
  id: string
  target_agency: string
  subject: string
  status: string
  priority: string
  created_at: string
  description: string
  purpose: string | null
  result_summary: string | null
  admin_notes: string | null
}

export default function FOIAPage() {
  const [form, setForm] = useState({
    target_agency: '',
    other_agency: '',
    subject: '',
    description: '',
    purpose: '',
    priority: 'standard',
  })
  const [requests, setRequests] = useState<FOIARequest[]>([])
  const [loading, setLoading] = useState(true)
  const [submitting, setSubmitting] = useState(false)
  const [submitted, setSubmitted] = useState(false)
  const [error, setError] = useState('')
  const [expanded, setExpanded] = useState<string | null>(null)

  useEffect(() => {
    fetch('/api/foia')
      .then(r => r.json())
      .then(json => { setRequests(json.requests || []); setLoading(false) })
  }, [])

  const handleSubmit = async () => {
    if (!form.target_agency || !form.subject || !form.description) {
      setError('Please fill in all required fields'); return
    }
    setSubmitting(true); setError(''); setSubmitted(false)
    const agency = form.target_agency === 'Other' ? form.other_agency : form.target_agency
    try {
      const res = await fetch('/api/foia', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ ...form, target_agency: agency })
      })
      const json = await res.json()
      if (json.request) {
        setRequests(r => [json.request, ...r])
        setForm({ target_agency: '', other_agency: '', subject: '', description: '', purpose: '', priority: 'standard' })
        setSubmitted(true)
        setTimeout(() => setSubmitted(false), 5000)
      } else setError(json.error || 'Failed to submit')
    } catch { setError('Network error') }
    finally { setSubmitting(false) }
  }

  const applyPrompt = (p: typeof EXAMPLE_PROMPTS[0]) => {
    setForm(f => ({ ...f, subject: p.subject, description: p.description }))
    window.scrollTo({ top: 0, behavior: 'smooth' })
  }

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen max-w-4xl mx-auto">
      {/* Premium Header */}
      <div className="mb-8 border border-slate-700 rounded-xl p-6 bg-gradient-to-br from-slate-900 to-slate-800">
        <div className="flex items-start gap-4">
          <div className="text-4xl">🏛️</div>
          <div>
            <h1 className="text-2xl font-bold text-blue-400">FOIA Intelligence Service</h1>
            <p className="text-slate-300 mt-2 leading-relaxed">
              We file Freedom of Information Act requests on your behalf and deliver the results.
              Get the same government documents that billion-dollar contractors use to gain competitive advantage.
            </p>
            <div className="flex flex-wrap gap-4 mt-4 text-sm">
              <span className="flex items-center gap-1.5 text-green-400"><span>✓</span> Incumbent performance reports</span>
              <span className="flex items-center gap-1.5 text-green-400"><span>✓</span> Historical pricing data</span>
              <span className="flex items-center gap-1.5 text-green-400"><span>✓</span> Protest outcomes</span>
              <span className="flex items-center gap-1.5 text-green-400"><span>✓</span> Agency procurement history</span>
            </div>
          </div>
        </div>
      </div>

      {submitted && (
        <div className="bg-green-900/40 border border-green-700 text-green-300 rounded-lg px-4 py-3 mb-6 font-medium">
          ✅ FOIA request submitted successfully. We will begin processing within 1 business day.
        </div>
      )}
      {error && (
        <div className="bg-red-900/40 border border-red-700 text-red-300 rounded-lg px-4 py-3 mb-6">
          ⚠ {error}
        </div>
      )}

      {/* New Request Form */}
      <div className="bg-slate-900 border border-slate-700 rounded-xl p-6 mb-6">
        <h2 className="text-lg font-semibold text-slate-200 mb-5">Submit a New FOIA Request</h2>
        <div className="space-y-4">
          <div>
            <label className="block text-sm font-medium text-slate-300 mb-1">Target Agency <span className="text-red-400">*</span></label>
            <select className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm"
              value={form.target_agency}
              onChange={e => setForm(f => ({ ...f, target_agency: e.target.value }))}>
              <option value="">Select agency...</option>
              {AGENCIES.map(a => <option key={a} value={a}>{a}</option>)}
            </select>
            {form.target_agency === 'Other' && (
              <input className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm mt-2"
                placeholder="Enter agency name"
                value={form.other_agency}
                onChange={e => setForm(f => ({ ...f, other_agency: e.target.value }))} />
            )}
          </div>

          <div>
            <label className="block text-sm font-medium text-slate-300 mb-1">Subject <span className="text-red-400">*</span></label>
            <input className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm"
              placeholder="e.g., Contract performance reports for HVAC maintenance at Fort Bragg"
              value={form.subject}
              onChange={e => setForm(f => ({ ...f, subject: e.target.value }))} />
          </div>

          <div>
            <label className="block text-sm font-medium text-slate-300 mb-1">What do you want to know? <span className="text-red-400">*</span></label>
            <textarea className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm h-28 resize-none"
              placeholder="Describe the specific documents or information you're looking for. The more specific, the faster and more useful the response."
              value={form.description}
              onChange={e => setForm(f => ({ ...f, description: e.target.value }))} />
          </div>

          <div>
            <label className="block text-sm font-medium text-slate-300 mb-1">Why do you need this? <span className="text-slate-500">(optional)</span></label>
            <textarea className="w-full bg-slate-800 border border-slate-600 rounded px-3 py-2 text-sm h-20 resize-none"
              placeholder="Helps us frame the request effectively — e.g., preparing a bid, researching an incumbent, understanding agency requirements"
              value={form.purpose}
              onChange={e => setForm(f => ({ ...f, purpose: e.target.value }))} />
          </div>

          <div>
            <label className="block text-sm font-medium text-slate-300 mb-2">Priority</label>
            <div className="flex gap-4">
              {[
                { value: 'standard', label: 'Standard', sub: '2-4 weeks' },
                { value: 'expedited', label: 'Expedited', sub: '1-2 weeks, fee may apply' },
              ].map(p => (
                <label key={p.value} className={`flex-1 flex items-center gap-3 rounded-lg border px-4 py-3 cursor-pointer transition-colors ${form.priority === p.value ? 'border-blue-500 bg-blue-900/20' : 'border-slate-700 bg-slate-800 hover:border-slate-500'}`}>
                  <input type="radio" className="accent-blue-500" checked={form.priority === p.value}
                    onChange={() => setForm(f => ({ ...f, priority: p.value }))} />
                  <div>
                    <div className="text-sm font-medium">{p.label}</div>
                    <div className="text-xs text-slate-500">{p.sub}</div>
                  </div>
                </label>
              ))}
            </div>
          </div>

          <button onClick={handleSubmit} disabled={submitting}
            className="w-full bg-blue-600 hover:bg-blue-500 disabled:opacity-50 text-white py-3 rounded-lg font-semibold text-sm">
            {submitting ? 'Submitting...' : 'Submit FOIA Request'}
          </button>

          <p className="text-xs text-slate-500 text-center">
            Included with Terminal tier ($499/month). Pro tier subscribers can purchase individual requests for $299 each.
          </p>
        </div>
      </div>

      {/* Example Prompts */}
      <div className="mb-6">
        <h3 className="text-sm font-medium text-slate-400 uppercase tracking-wider mb-3">Common Request Templates — Click to Auto-Fill</h3>
        <div className="grid grid-cols-1 md:grid-cols-3 gap-3">
          {EXAMPLE_PROMPTS.map((p, i) => (
            <button key={i} onClick={() => applyPrompt(p)}
              className="bg-slate-900 hover:bg-slate-800 border border-slate-700 hover:border-blue-600 rounded-lg p-4 text-left transition-colors">
              <div className="font-medium text-sm mb-1">{p.title}</div>
              <div className="text-xs text-slate-400 leading-relaxed">{p.desc}</div>
            </button>
          ))}
        </div>
      </div>

      {/* My Requests */}
      <div className="bg-slate-900 border border-slate-700 rounded-xl overflow-hidden">
        <div className="bg-slate-800 px-5 py-3 border-b border-slate-700">
          <h2 className="font-semibold text-slate-200">My FOIA Requests</h2>
        </div>
        {loading ? (
          <div className="p-6 text-slate-500 text-sm">Loading...</div>
        ) : requests.length === 0 ? (
          <div className="p-8 text-center text-slate-500 text-sm">
            No requests yet. Submit your first FOIA request above.
          </div>
        ) : (
          <div className="divide-y divide-slate-800">
            {requests.map(r => (
              <div key={r.id} className="p-4">
                <div className="flex items-start justify-between gap-3">
                  <div className="flex-1 min-w-0">
                    <div className="flex items-center gap-2 flex-wrap">
                      <span className={`text-xs font-bold px-2 py-0.5 rounded ${STATUS_STYLES[r.status] || 'bg-slate-700 text-slate-300'}`}>
                        {r.status.replace('_', ' ').toUpperCase()}
                      </span>
                      {r.priority === 'expedited' && (
                        <span className="text-xs bg-orange-900/40 text-orange-300 border border-orange-700 px-2 py-0.5 rounded">EXPEDITED</span>
                      )}
                    </div>
                    <p className="font-medium text-sm mt-1">{r.subject}</p>
                    <p className="text-xs text-slate-400">{r.target_agency} · {new Date(r.created_at).toLocaleDateString()}</p>
                  </div>
                  <button onClick={() => setExpanded(expanded === r.id ? null : r.id)}
                    className="text-xs text-blue-400 hover:text-blue-300 shrink-0">
                    {expanded === r.id ? 'Hide' : 'Details'}
                  </button>
                </div>
                {expanded === r.id && (
                  <div className="mt-3 space-y-2 text-xs text-slate-400 bg-slate-800/50 rounded p-3">
                    <p><span className="text-slate-300 font-medium">Request:</span> {r.description}</p>
                    {r.purpose && <p><span className="text-slate-300 font-medium">Purpose:</span> {r.purpose}</p>}
                    {r.admin_notes && <p><span className="text-slate-300 font-medium">Admin Notes:</span> {r.admin_notes}</p>}
                    {r.result_summary && <p><span className="text-slate-300 font-medium">Results:</span> {r.result_summary}</p>}
                  </div>
                )}
              </div>
            ))}
          </div>
        )}
      </div>
    </div>
  )
}
""")

# Add FOIA Service to navigation
nav_path = Path("components/DashboardShell.tsx")
nav_text = nav_path.read_text()
if 'FOIA' not in nav_text:
    nav_text = nav_text.replace(
        "{ href: '/profile', label: 'Profile' },",
        "{ href: '/profile', label: 'Profile' },\n  { href: '/foia', label: 'FOIA Service' },"
    )
    nav_path.write_text(nav_text)
    print("Nav updated")
else:
    print("Nav already has FOIA")

print("\nFiles written:")
print("  app/api/foia/route.ts")
print("  app/(dashboard)/foia/page.tsx")
print("  components/DashboardShell.tsx (nav updated)")
print("\nRun the SQL in Supabase, then build and push.")

/content/govcon-dashboard
=== PASTE THIS SQL IN SUPABASE SQL EDITOR ===

CREATE TABLE IF NOT EXISTS foia_requests (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  user_id uuid NOT NULL REFERENCES auth.users(id),
  status text DEFAULT 'submitted',
  target_agency text NOT NULL,
  subject text NOT NULL,
  description text NOT NULL,
  purpose text,
  priority text DEFAULT 'standard',
  admin_notes text,
  result_summary text,
  created_at timestamptz DEFAULT now(),
  updated_at timestamptz DEFAULT now()
);

ALTER TABLE foia_requests ENABLE ROW LEVEL SECURITY;

CREATE POLICY "Users can read own requests" ON foia_requests FOR SELECT USING (auth.uid() = user_id);
CREATE POLICY "Users can insert requests" ON foia_requests FOR INSERT WITH CHECK (auth.uid() = user_id);

=== END SQL ===
Nav updated

Files written:
  app/api/foia/route.ts
  app/(dashboard)/foia/page.tsx
  components/DashboardShell.tsx (nav updated)

Run the SQL in Supabase, then build and push.


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add FOIA Intelligence Service page" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-300:] if result.stderr else "")


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...

                                                   ^
  38 |   },
  39 |   {
  40 |     title: '💰 Pricing Intelligence',

Expected ',', got 'ident'


    at <unknown> (./app/(dashboard)/foia/page.tsx:37:272)



In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/foia/page.tsx")
text = page.read_text()

text = text.replace(
    "I am preparing a competitive bid and need to understand the incumbent\\'s performance history.",
    "I am preparing a competitive bid and need to understand the incumbent performance history."
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/foia/page.tsx")
text = page.read_text()

text = text.replace(
    "I am preparing a competitive bid and need to understand the incumbent\\'s performance history.",
    "I am preparing a competitive bid and need to understand the incumbent performance history."
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add FOIA Intelligence Service page" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-1000:])


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...



In [ ]:
from pathlib import Path
page = Path("app/(dashboard)/foia/page.tsx")
text = page.read_text()
# Find the problematic line
for i, line in enumerate(text.split('\n')):
    if 'incumbent' in line.lower():
        print(f"Line {i+1}: {repr(line)}")

Line 34: "    title: '📋 Incumbent Performance',"
Line 36: "    subject: 'Contract Performance Reports — Incumbent Contractor',"
Line 37: "    description: 'I am requesting all contract performance evaluations, CPARS reports, and past performance documentation for the current incumbent contractor on contracts within my target NAICS code. I am preparing a competitive bid and need to understand the incumbent's performance history.',"
Line 129: '              <span className="flex items-center gap-1.5 text-green-400"><span>✓</span> Incumbent performance reports</span>'
Line 188: '              placeholder="Helps us frame the request effectively — e.g., preparing a bid, researching an incumbent, understanding agency requirements"'


In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/foia/page.tsx")
text = page.read_text()

# Replace the exact line
old = "    description: 'I am requesting all contract performance evaluations, CPARS reports, and past performance documentation for the current incumbent contractor on contracts within my target NAICS code. I am preparing a competitive bid and need to understand the incumbent's performance history.',"
new = "    description: 'I am requesting all contract performance evaluations, CPARS reports, and past performance documentation for the current incumbent contractor on contracts within my target NAICS code. I am preparing a competitive bid and need to understand the incumbent performance history.',"

text = text.replace(old, new)
page.write_text(text)

# Verify
for i, line in enumerate(text.split('\n')):
    if 'incumbent' in line.lower() and "'" in line[line.lower().find('incumbent'):]:
        print(f"Line {i+1}: {repr(line)}")
print("Done")

Line 34: "    title: '📋 Incumbent Performance',"
Line 36: "    subject: 'Contract Performance Reports — Incumbent Contractor',"
Line 37: "    description: 'I am requesting all contract performance evaluations, CPARS reports, and past performance documentation for the current incumbent contractor on contracts within my target NAICS code. I am preparing a competitive bid and need to understand the incumbent performance history.',"
Done


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add FOIA Intelligence Service page" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-300:] if result.stderr else "")


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 20.3s
  Running TypeScript ...
  Generating static pages using 1 worker (0/37) ...
  Generating static pages using 1 worker (9/37) 
  Generating static pages using 1 worker (18/37) 
  Generating static pages using 1 worker (27/37) 
✓ Generating static pages using 1 worker (37/37) in 469.6ms
  Finalizing page optimization ...

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/bid-brief
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/foia
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/brief
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /ap

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/(dashboard)/guide", exist_ok=True)

Path("app/(dashboard)/guide/page.tsx").write_text("""'use client'

import { useState } from 'react'
import Link from 'next/link'

const CATEGORIES = [
  {
    id: 'getting-started',
    title: '🚀 Getting Started',
    color: 'blue',
    cta: { text: '→ Your Profile already maps your NAICS codes to matching contracts.', label: 'Go to Profile', href: '/profile' },
    topics: [
      {
        title: 'What is federal procurement?',
        summary: 'The federal government buys over $700 billion in goods and services every year from private businesses.',
        full: 'The federal government buys over $700 billion in goods and services every year from private businesses. The process is governed by rules designed to ensure fair competition — including rules specifically designed to help small businesses win. These rules are codified in the Federal Acquisition Regulation (FAR), a comprehensive set of guidelines that every federal agency must follow when buying goods and services.'
      },
      {
        title: 'Who can bid?',
        summary: 'Any registered business can bid on most federal contracts.',
        full: 'Any registered business can bid on most federal contracts. Registration through SAM.gov is free and typically takes 7-10 business days. You need a Unique Entity ID (UEI) which is also free. There are no fees to register, no fees to bid, and no fees to receive a contract award. The system is designed to be accessible to any legitimate business.'
      },
      {
        title: 'What is a NAICS code?',
        summary: 'A 6-digit code that classifies what your business does.',
        full: 'A 6-digit code that classifies what your business does. The government uses these to match contracts to qualified vendors. Choosing the right codes is one of the most important steps in your federal contracting strategy. You can register for multiple NAICS codes that reflect your capabilities. The SBA also uses NAICS codes to determine whether your business qualifies as "small" for a given contract — size standards vary by industry.'
      },
      {
        title: 'What does SAM registration involve?',
        summary: 'SAM.gov is the System for Award Management. Registration is free and required for all federal contractors.',
        full: 'SAM.gov is the System for Award Management. Registration is free, required for all federal contractors, and must be renewed annually. You will need your EIN, banking information for electronic payments, and your NAICS codes. The process takes about an hour to complete and 7-10 business days to activate. Beware of third-party services that charge fees for SAM registration — the official process at sam.gov is completely free.'
      },
    ]
  },
  {
    id: 'small-business',
    title: '🏆 Small Business Advantages',
    color: 'green',
    cta: { text: '→ Your Profile tracks your certifications. Update them to see which set-asides you qualify for.', label: 'Update Profile', href: '/profile' },
    topics: [
      {
        title: 'Government goals for small business',
        summary: 'The federal government has a statutory goal of awarding at least 23% of all contract dollars to small businesses.',
        full: 'The federal government has a statutory goal of awarding at least 23% of all contract dollars to small businesses. In FY2024, the actual figure exceeded this target. Agencies are measured on meeting these goals — contracting officers have genuine incentives to find qualified small businesses. This is not charity; it is a legal requirement backed by agency performance reviews.'
      },
      {
        title: 'Set-aside contracts',
        summary: "Many contracts are 'set aside' exclusively for small businesses.",
        full: "Many contracts are 'set aside' exclusively for small businesses. This means large contractors cannot compete — the playing field is limited to businesses like yours. Set-asides are required when there is a reasonable expectation that at least two qualified small businesses will submit offers at a fair price. In practice, this covers a significant portion of federal contract spending each year."
      },
      {
        title: 'The 8(a) Business Development Program',
        summary: 'SBA program for small disadvantaged businesses with sole-source authority up to $4.5M.',
        full: 'SBA program for small disadvantaged businesses. Participants can receive sole-source contracts up to $4.5M (services) or $7M (manufacturing) without competition. The program provides a nine-year developmental track with business development assistance, mentoring, and access to set-aside and sole-source contracts. Eligibility requires social and economic disadvantage, good character, and potential for success.'
      },
      {
        title: 'HUBZone Program',
        summary: 'Businesses in Historically Underutilized Business Zones receive a 10% price evaluation preference.',
        full: 'Businesses in Historically Underutilized Business Zones receive a 10% price evaluation preference and access to set-aside contracts. To qualify, your principal office must be in a HUBZone and at least 35% of your employees must live in a HUBZone. The SBA maintains a map of qualified areas — urban and rural communities that have been economically distressed.'
      },
      {
        title: 'Women-Owned Small Business (WOSB)',
        summary: 'Contracts in certain industries are set aside for women-owned businesses.',
        full: 'Contracts in certain industries are set aside for women-owned businesses. Self-certification is available through SBA. To qualify, the business must be at least 51% owned and controlled by women who are U.S. citizens, and the women must manage day-to-day operations. The WOSB program covers dozens of NAICS codes where women-owned businesses are underrepresented in federal contracting.'
      },
      {
        title: 'Service-Disabled Veteran-Owned (SDVOSB)',
        summary: 'Dedicated set-asides and sole-source authority for service-disabled veteran-owned businesses.',
        full: 'Dedicated set-asides and sole-source authority for service-disabled veteran-owned businesses. Sole-source awards up to $4.5M (services) or $7M (manufacturing) are available without competition. The business must be at least 51% owned by one or more service-disabled veterans, and a service-disabled veteran must control the management and daily operations.'
      },
      {
        title: 'Mentor-Protégé Program',
        summary: 'Pairs experienced contractors with newer small businesses.',
        full: 'Pairs experienced contractors with newer small businesses. Protégés can joint venture with mentors and still qualify as small — even if the combined entity would otherwise exceed size standards. Mentors provide technical and management assistance, financial support, and business development help. This is one of the most powerful paths for new small businesses to build past performance quickly.'
      },
    ]
  },
  {
    id: 'awards',
    title: '📋 How Contracts Are Awarded',
    color: 'yellow',
    cta: { text: '→ Search real federal contract awards right now — filter by NAICS, state, agency, and dollar amount.', label: 'Search Contracts', href: '/contracts' },
    topics: [
      {
        title: 'Lowest price vs. best value',
        summary: 'Not every contract goes to the cheapest bidder.',
        full: "Not every contract goes to the cheapest bidder. 'Best value' evaluations consider technical capability, past performance, and price together. A higher-priced bid can win if it demonstrates superior capability. The solicitation will always specify the evaluation methodology — read it carefully. Lowest Price Technically Acceptable (LPTA) evaluations do go to the cheapest qualified bidder, but many contracts use a trade-off process where quality can outweigh price."
      },
      {
        title: 'The evaluation process',
        summary: 'Proposals are reviewed by evaluation teams using published criteria.',
        full: 'Proposals are reviewed by evaluation teams using published criteria. The criteria are always listed in the solicitation — there are no hidden factors. Evaluation teams typically include technical experts, contracting officers, and program managers. Each evaluator scores proposals independently before consensus scoring. Understanding the evaluation criteria and writing directly to each factor is the single most important thing you can do to improve your win rate.'
      },
      {
        title: 'Past performance',
        summary: 'Agencies review your track record on previous contracts.',
        full: 'Agencies review your track record on previous contracts. New businesses can reference commercial work, subcontracting experience, or key personnel experience. Past performance is evaluated on relevance (similar scope, size, and complexity) and quality (ratings from previous customers). Actively managing your CPARS ratings — the official government performance evaluation system — is essential for long-term success.'
      },
      {
        title: 'Simplified acquisition procedures',
        summary: 'Contracts under $250,000 have streamlined rules.',
        full: 'Contracts under $250,000 have streamlined rules. This is often the best entry point for new contractors. The micro-purchase threshold ($10,000) requires no competition at all — agencies can buy directly. The simplified acquisition threshold ($250,000) has reduced documentation requirements and faster award timelines. Winning smaller contracts builds past performance and agency relationships that support larger bids.'
      },
    ]
  },
  {
    id: 'incumbents',
    title: '⚔️ Competing Against Incumbents',
    color: 'orange',
    cta: { text: '→ The Recompete Radar shows contracts expiring soon with RPS scoring. Every contract detail page shows the incumbent and their lobbying connections.', label: 'View Contracts', href: '/contracts?expiring_within=90' },
    topics: [
      {
        title: 'Recompete basics',
        summary: 'Most federal contracts have defined performance periods. When a contract ends, the agency must resolicit.',
        full: 'Most federal contracts have defined performance periods. When a contract ends, the agency must resolicit unless specific exceptions apply. This creates regular opportunities for new competitors. Performance periods typically run 1-5 years with options. Tracking expiration dates — which is exactly what the Recompete Radar does — gives you lead time to prepare a competitive bid before the solicitation is even released.'
      },
      {
        title: 'Full and open competition',
        summary: 'Federal law generally requires agencies to seek competition.',
        full: 'Federal law generally requires agencies to seek competition. The Competition in Contracting Act (CICA) establishes that contracts should be awarded through full and open competition unless specific exceptions are justified. Sole-source awards (no competition) require written justification and approval. If you believe a contract is being improperly sole-sourced to an incumbent, you have the right to question the justification.'
      },
      {
        title: "Reading the incumbent's history",
        summary: "Contract award data and performance information are publicly available.",
        full: "Contract award data, modification history, and performance information are often publicly available on USAspending.gov and FPDS. Understanding the incumbent's contract value, modifications (which can signal scope changes or performance issues), and award history helps you bid smarter. A contract with many modifications may indicate dissatisfied agency customers — an opening for a well-positioned challenger."
      },
      {
        title: 'Price as your advantage',
        summary: 'In the $100K-$10M range, price is often the decisive factor.',
        full: 'In the $100K-$10M range, price is often the decisive factor when technical scores are close. A qualified small business offering competitive pricing has genuine advantages over a large incumbent with higher overhead rates. Large contractors typically carry G&A rates of 15-25% — a lean small business can often underbid them while maintaining healthy margins. Know your cost structure and know theirs.'
      },
    ]
  },
  {
    id: 'subcontracting',
    title: '🤝 Subcontracting: Working With Primes',
    color: 'purple',
    cta: { text: "→ Click 'Prepare Bid Brief' on any contract to generate a competitive intelligence document including subcontracting strategy.", label: 'Search Contracts', href: '/contracts' },
    topics: [
      {
        title: 'FAR 19.7 explained',
        summary: 'Prime contractors with federal contracts over $750,000 must submit small business subcontracting plans.',
        full: 'Prime contractors with federal contracts over $750,000 ($1.5M for construction) must submit small business subcontracting plans. They need small business partners — this is a requirement, not a favor. These plans set specific dollar and percentage goals for subcontracting to small businesses, 8(a) firms, HUBZone businesses, WOSBs, and SDVOSBs. Prime contractors who fail to meet these goals face contract penalties. You have genuine leverage as a qualified small business.'
      },
      {
        title: 'How to approach primes',
        summary: 'Prime contractors actively seek qualified small businesses to meet their subcontracting goals.',
        full: 'Prime contractors actively seek qualified small businesses to meet their subcontracting goals. Your pitch should emphasize your capabilities, certifications, and geographic relevance. Register in the SBA SubNet database and the Dynamic Small Business Search (DSBS) — these are the databases prime contractors use to find subcontractors. A one-page capability statement is the standard business development tool in this world.'
      },
      {
        title: 'Subcontracting vs. prime contracting',
        summary: 'Subcontracting builds past performance, relationships, and cash flow with lower risk.',
        full: 'Subcontracting builds past performance, relationships, and cash flow with lower risk. Many successful contractors started as subcontractors, built their CPARS ratings and agency relationships, then transitioned to prime contracting. The trade-off: margins are lower and you depend on the prime for payment. The benefit: you learn the agency, build relationships, and get real past performance without the full burden of proposal management and contract administration.'
      },
    ]
  },
  {
    id: 'protests',
    title: '⚖️ Bid Protests: Your Right to Fair Process',
    color: 'red',
    cta: { text: '→ Use our FOIA Service to request protest decisions and corrective action reports for contracts you plan to bid on.', label: 'FOIA Service', href: '/search' },
    topics: [
      {
        title: 'What is a bid protest?',
        summary: 'If you believe a contract was awarded unfairly, you have the right to file a protest with the GAO.',
        full: 'If you believe a contract was awarded unfairly, you have the right to file a protest with the Government Accountability Office (GAO). This is a standard part of the procurement system, not an adversarial act. Bid protests are a legal mechanism designed to keep the system honest. They are filed by contractors of all sizes — including large defense contractors — when they believe evaluation criteria were not followed.'
      },
      {
        title: 'When to consider a protest',
        summary: 'Protests are appropriate when evaluation criteria were not followed or when a set-aside was improperly waived.',
        full: "Protests are appropriate when evaluation criteria weren't followed, when a set-aside was improperly waived, or when the award decision contains clear errors. Request a debriefing first — agencies are required to provide one. Debriefings often reveal evaluation information that clarifies whether a protest has merit. Most experienced contractors request debriefings on every significant loss, not just ones they plan to protest."
      },
      {
        title: 'GAO protest basics',
        summary: 'Must be filed within 10 days of award. GAO decides within 100 days. Filing is free.',
        full: 'Must be filed within 10 days of award or knowledge of the issue. GAO decides within 100 days. Filing is free. GAO sustains about 15-20% of protests it decides on the merits — a meaningful rate that shows the system works. When GAO sustains a protest, it can recommend re-evaluation, a new solicitation, or termination of the existing contract. The agency is not legally required to follow GAO recommendations, but they comply in the vast majority of cases.'
      },
    ]
  },
  {
    id: 'resources',
    title: '📚 Free Resources',
    color: 'teal',
    cta: { text: "→ GovCon Intelligence Terminal combines all of these free data sources into one searchable platform with AI-powered analysis. You're already here.", label: 'Go to Dashboard', href: '/home' },
    topics: [
      {
        title: 'APEX Accelerators (formerly PTACs)',
        summary: 'Free counseling, training, and hands-on assistance for businesses pursuing government contracts.',
        full: 'Free counseling, training, and hands-on assistance for businesses pursuing government contracts. Funded by the Department of Defense. APEX Accelerators help with SAM registration, proposal reviews, market research, and connecting you with prime contractors and agency buyers. Find your local office at apexaccelerators.us. This is probably the most underutilized free resource in federal contracting.'
      },
      {
        title: 'SBA District Offices',
        summary: 'Free one-on-one counseling for small business contracting.',
        full: 'Free one-on-one counseling for small business contracting. Every state has multiple offices. SBA staff can help with certification applications (8(a), HUBZone, WOSB), size standard determinations, and general contracting strategy. SBA also administers the Procurement Center Representatives (PCRs) program — federal employees whose job is to increase small business opportunities at major buying agencies.'
      },
      {
        title: 'SCORE Mentorship',
        summary: 'Free mentoring from experienced business professionals including former government contractors.',
        full: 'Free mentoring from experienced business professionals including former government contractors. SCORE has over 10,000 volunteers across more than 250 chapters nationwide. Mentors can help with business planning, financial projections, proposal strategy, and connecting you with their networks. Mentoring is ongoing — not just a one-time meeting.'
      },
      {
        title: 'SAM.gov',
        summary: 'The official government system for vendor registration, opportunity search, and entity validation.',
        full: 'The official government system for vendor registration, opportunity search, and entity validation. Everything is free. SAM.gov also hosts the Federal Business Opportunities (formerly FedBizOpps) database — every contract opportunity over $25,000 must be posted here. Set up saved searches and email alerts so you never miss a relevant solicitation.'
      },
      {
        title: 'USAspending.gov',
        summary: 'Search every federal contract award. See who won, how much they were paid, and where the work was performed.',
        full: 'Search every federal contract award. See who won, how much they were paid, and where the work was performed. USAspending.gov is the primary data source for GovCon Intelligence Terminal. Use it to research incumbents, understand agency buying patterns, find potential teaming partners, and validate market size before investing in a bid.'
      },
    ]
  },
]

const COLOR_MAP: Record<string, { border: string; header: string; badge: string; cta: string }> = {
  blue:   { border: 'border-blue-800',   header: 'bg-blue-950',   badge: 'bg-blue-700',   cta: 'bg-blue-700 hover:bg-blue-600' },
  green:  { border: 'border-green-800',  header: 'bg-green-950',  badge: 'bg-green-700',  cta: 'bg-green-700 hover:bg-green-600' },
  yellow: { border: 'border-yellow-800', header: 'bg-yellow-950', badge: 'bg-yellow-700', cta: 'bg-yellow-700 hover:bg-yellow-600' },
  orange: { border: 'border-orange-800', header: 'bg-orange-950', badge: 'bg-orange-700', cta: 'bg-orange-700 hover:bg-orange-600' },
  purple: { border: 'border-purple-800', header: 'bg-purple-950', badge: 'bg-purple-700', cta: 'bg-purple-700 hover:bg-purple-600' },
  red:    { border: 'border-red-800',    header: 'bg-red-950',    badge: 'bg-red-700',    cta: 'bg-red-700 hover:bg-red-600' },
  teal:   { border: 'border-teal-800',   header: 'bg-teal-950',   badge: 'bg-teal-700',   cta: 'bg-teal-700 hover:bg-teal-600' },
}

function TopicCard({ topic }: { topic: { title: string; summary: string; full: string } }) {
  const [open, setOpen] = useState(false)
  return (
    <div className="border border-slate-700 rounded-lg overflow-hidden">
      <button
        onClick={() => setOpen(o => !o)}
        className="w-full flex items-start justify-between gap-3 p-4 text-left hover:bg-slate-800 transition-colors"
      >
        <div>
          <p className="font-medium text-slate-200 text-sm">{topic.title}</p>
          {!open && <p className="text-slate-400 text-xs mt-1">{topic.summary}</p>}
        </div>
        <span className="text-slate-500 shrink-0 mt-0.5">{open ? '▲' : '▼'}</span>
      </button>
      {open && (
        <div className="px-4 pb-4">
          <p className="text-slate-300 text-sm leading-relaxed">{topic.full}</p>
        </div>
      )}
    </div>
  )
}

export default function GuidePage() {
  const [openCategories, setOpenCategories] = useState<string[]>(['getting-started'])

  const toggleCategory = (id: string) => {
    setOpenCategories(prev =>
      prev.includes(id) ? prev.filter(c => c !== id) : [...prev, id]
    )
  }

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen max-w-4xl mx-auto">
      <div className="mb-8">
        <h1 className="text-3xl font-bold text-blue-400">Procurement Guide</h1>
        <p className="text-slate-400 mt-2 text-sm leading-relaxed max-w-2xl">
          Federal contracting rules explained in plain English. The same knowledge that large contractors pay legal teams to maintain — organized for you.
        </p>
      </div>

      <div className="space-y-4">
        {CATEGORIES.map(cat => {
          const colors = COLOR_MAP[cat.color] || COLOR_MAP.blue
          const isOpen = openCategories.includes(cat.id)
          return (
            <div key={cat.id} className={`border ${colors.border} rounded-lg overflow-hidden`}>
              <button
                onClick={() => toggleCategory(cat.id)}
                className={`w-full flex items-center justify-between px-5 py-4 ${colors.header} hover:opacity-90 transition-opacity`}
              >
                <h2 className="font-semibold text-slate-100">{cat.title}</h2>
                <div className="flex items-center gap-3">
                  <span className={`${colors.badge} text-white text-xs px-2 py-0.5 rounded-full`}>
                    {cat.topics.length} topics
                  </span>
                  <span className="text-slate-400">{isOpen ? '▲' : '▼'}</span>
                </div>
              </button>
              {isOpen && (
                <div className="p-4 space-y-2 bg-slate-900">
                  {cat.topics.map(topic => (
                    <TopicCard key={topic.title} topic={topic} />
                  ))}
                  {/* CTA at bottom of each category */}
                  <div className="mt-4 pt-4 border-t border-slate-700 flex items-center justify-between gap-4 flex-wrap">
                    <p className="text-slate-400 text-xs flex-1">{cat.cta.text}</p>
                    <Link
                      href={cat.cta.href}
                      className={`${colors.cta} text-white text-xs font-semibold px-4 py-2 rounded-lg shrink-0 transition-colors`}
                    >
                      {cat.cta.label} →
                    </Link>
                  </div>
                </div>
              )}
            </div>
          )
        })}
      </div>

      <div className="mt-10 pt-6 border-t border-slate-800">
        <p className="text-slate-500 text-xs leading-relaxed">
          This guide provides general information about federal procurement. It is not legal advice. Specific situations may vary. Consult with a procurement attorney or your local APEX Accelerator for guidance specific to your business.
        </p>
      </div>
    </div>
  )
}
""")

# Add Guide to nav if not already there
nav_path = Path("components/DashboardShell.tsx")
nav_text = nav_path.read_text()
if '/guide' not in nav_text:
    nav_text = nav_text.replace(
        "{ href: '/profile', label: 'Profile' },",
        "{ href: '/profile', label: 'Profile' },\n  { href: '/guide', label: 'Guide' },"
    )
    nav_path.write_text(nav_text)
    print("Nav updated")
else:
    print("Nav already has Guide")

print("\nFiles written:")
print("  app/(dashboard)/guide/page.tsx")
print("  components/DashboardShell.tsx")

/content/govcon-dashboard
Nav updated

Files written:
  app/(dashboard)/guide/page.tsx
  components/DashboardShell.tsx


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add Procurement Guide with product cross-links" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-1500:])

enerating static pages using 1 worker (9/38) 
  Generating static pages using 1 worker (18/38) 
  Generating static pages using 1 worker (28/38) 
✓ Generating static pages using 1 worker (38/38) in 494.9ms
  Finalizing page optimization ...

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/bid-brief
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/foia
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/brief
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/profile
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/status
├ ƒ /api/stripe/checkout
├ ƒ /api/stripe/webhook
├ ƒ /api/votes/analysis
├ ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractor

In [ ]:
%cd /content/govcon-dashboard

# Generate all topic slugs from the guide
topics = [
    ("getting-started", "what-is-federal-procurement", "What is Federal Procurement?"),
    ("getting-started", "who-can-bid", "Who Can Bid on Federal Contracts?"),
    ("getting-started", "naics-codes", "What is a NAICS Code?"),
    ("getting-started", "sam-registration", "SAM.gov Registration Guide"),
    ("small-business", "government-goals", "Government Small Business Goals"),
    ("small-business", "set-aside-contracts", "Set-Aside Contracts Explained"),
    ("small-business", "8a-program", "The 8(a) Business Development Program"),
    ("small-business", "hubzone", "HUBZone Program"),
    ("small-business", "wosb", "Women-Owned Small Business (WOSB)"),
    ("small-business", "sdvosb", "Service-Disabled Veteran-Owned (SDVOSB)"),
    ("small-business", "mentor-protege", "Mentor-Protégé Program"),
    ("awards", "lowest-price-vs-best-value", "Lowest Price vs. Best Value"),
    ("awards", "evaluation-process", "The Evaluation Process"),
    ("awards", "past-performance", "Past Performance"),
    ("awards", "simplified-acquisition", "Simplified Acquisition Procedures"),
    ("incumbents", "recompete-basics", "Recompete Basics"),
    ("incumbents", "full-open-competition", "Full and Open Competition"),
    ("incumbents", "reading-incumbent-history", "Reading the Incumbent's History"),
    ("incumbents", "price-advantage", "Price as Your Advantage"),
    ("subcontracting", "far-19-7", "FAR 19.7 Small Business Subcontracting"),
    ("subcontracting", "approaching-primes", "How to Approach Prime Contractors"),
    ("subcontracting", "sub-vs-prime", "Subcontracting vs. Prime Contracting"),
    ("protests", "what-is-bid-protest", "What is a Bid Protest?"),
    ("protests", "when-to-protest", "When to Consider a Protest"),
    ("protests", "gao-protest-basics", "GAO Protest Basics"),
    ("resources", "apex-accelerators", "APEX Accelerators"),
    ("resources", "sba-district-offices", "SBA District Offices"),
    ("resources", "score-mentorship", "SCORE Mentorship"),
    ("resources", "sam-gov", "SAM.gov Guide"),
    ("resources", "usaspending", "USAspending.gov Guide"),
]

print(f"Total topics: {len(topics)}")
for cat, slug, title in topics:
    print(f"  /guide/{cat}/{slug} — {title}")

/content/govcon-dashboard
Total topics: 30
  /guide/getting-started/what-is-federal-procurement — What is Federal Procurement?
  /guide/getting-started/who-can-bid — Who Can Bid on Federal Contracts?
  /guide/getting-started/naics-codes — What is a NAICS Code?
  /guide/getting-started/sam-registration — SAM.gov Registration Guide
  /guide/small-business/government-goals — Government Small Business Goals
  /guide/small-business/set-aside-contracts — Set-Aside Contracts Explained
  /guide/small-business/8a-program — The 8(a) Business Development Program
  /guide/small-business/hubzone — HUBZone Program
  /guide/small-business/wosb — Women-Owned Small Business (WOSB)
  /guide/small-business/sdvosb — Service-Disabled Veteran-Owned (SDVOSB)
  /guide/small-business/mentor-protege — Mentor-Protégé Program
  /guide/awards/lowest-price-vs-best-value — Lowest Price vs. Best Value
  /guide/awards/evaluation-process — The Evaluation Process
  /guide/awards/past-performance — Past Performance
  /gu

In [ ]:
%cd /content/govcon-dashboard

import requests
import json
import os
import time
from pathlib import Path
from google.colab import userdata

GEMINI_KEY = userdata.get('GEMINI_API_KEY')

def gemini(prompt):
    r = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={GEMINI_KEY}",
        headers={"Content-Type": "application/json"},
        json={"contents": [{"parts": [{"text": prompt}]}],
              "generationConfig": {"maxOutputTokens": 4000, "temperature": 0.4}}
    )
    data = r.json()
    return data.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text", "")

TOPICS = [
    ("getting-started", "what-is-federal-procurement", "What is Federal Procurement?"),
    ("getting-started", "who-can-bid", "Who Can Bid on Federal Contracts?"),
    ("getting-started", "naics-codes", "What is a NAICS Code?"),
    ("getting-started", "sam-registration", "SAM.gov Registration Guide"),
    ("small-business", "government-goals", "Government Small Business Goals"),
    ("small-business", "set-aside-contracts", "Set-Aside Contracts Explained"),
    ("small-business", "8a-program", "The 8(a) Business Development Program"),
    ("small-business", "hubzone", "HUBZone Program"),
    ("small-business", "wosb", "Women-Owned Small Business (WOSB)"),
    ("small-business", "sdvosb", "Service-Disabled Veteran-Owned (SDVOSB)"),
    ("small-business", "mentor-protege", "Mentor-Protégé Program"),
    ("awards", "lowest-price-vs-best-value", "Lowest Price vs. Best Value"),
    ("awards", "evaluation-process", "The Evaluation Process"),
    ("awards", "past-performance", "Past Performance"),
    ("awards", "simplified-acquisition", "Simplified Acquisition Procedures"),
    ("incumbents", "recompete-basics", "Recompete Basics"),
    ("incumbents", "full-open-competition", "Full and Open Competition"),
    ("incumbents", "reading-incumbent-history", "Reading the Incumbent's History"),
    ("incumbents", "price-advantage", "Price as Your Advantage"),
    ("subcontracting", "far-19-7", "FAR 19.7 Small Business Subcontracting"),
    ("subcontracting", "approaching-primes", "How to Approach Prime Contractors"),
    ("subcontracting", "sub-vs-prime", "Subcontracting vs. Prime Contracting"),
    ("protests", "what-is-bid-protest", "What is a Bid Protest?"),
    ("protests", "when-to-protest", "When to Consider a Protest"),
    ("protests", "gao-protest-basics", "GAO Protest Basics"),
    ("resources", "apex-accelerators", "APEX Accelerators"),
    ("resources", "sba-district-offices", "SBA District Offices"),
    ("resources", "score-mentorship", "SCORE Mentorship"),
    ("resources", "sam-gov", "SAM.gov Guide"),
    ("resources", "usaspending", "USAspending.gov Guide"),
]

PROMPT_TEMPLATE = """You are writing content for GovCon Intelligence Terminal, a federal contracting intelligence platform for small businesses.

Write a comprehensive, deep-dive guide page on: "{title}"

The audience is a small business owner who is new to federal contracting but smart and motivated. Tone: friendly, practical, like NerdWallet explains taxes. NOT legal advice, NOT activist. Just clear, useful information.

Return ONLY a JSON object with this exact structure:
{{
  "title": "{title}",
  "subtitle": "One sentence that captures what this page covers",
  "intro": "2-3 sentence introduction explaining why this topic matters",
  "sections": [
    {{
      "heading": "Section heading",
      "content": "3-5 paragraphs of substantive content. Include specific statutes, FAR citations, dollar thresholds, timeframes, and real-world examples where relevant. Be specific — not generic."
    }}
  ],
  "key_facts": [
    "Specific fact 1 (include numbers, thresholds, timeframes)",
    "Specific fact 2",
    "Specific fact 3",
    "Specific fact 4",
    "Specific fact 5"
  ],
  "common_mistakes": [
    "Mistake 1 and how to avoid it",
    "Mistake 2 and how to avoid it",
    "Mistake 3 and how to avoid it"
  ],
  "official_sources": [
    {{"name": "Source name", "url": "https://actual-url.gov", "description": "What you find there"}}
  ]
}}

Include 4-6 sections. Be specific and detailed. Include real FAR citations, SBA regulations, dollar thresholds, and timeframes. No generic filler content.
Return ONLY valid JSON. No markdown, no backticks, no preamble."""

# Generate all content
all_content = {}
for cat, slug, title in TOPICS:
    print(f"Generating: {title}...")
    prompt = PROMPT_TEMPLATE.format(title=title)
    text = gemini(prompt)
    try:
        # Clean potential markdown fences
        text = text.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        data = json.loads(text)
        all_content[f"{cat}/{slug}"] = data
        print(f"  ✓ {len(data.get('sections', []))} sections")
    except Exception as e:
        print(f"  ✗ Parse error: {e}")
        all_content[f"{cat}/{slug}"] = {"title": title, "error": str(e), "raw": text[:200]}
    time.sleep(1)

print(f"\nGenerated {len(all_content)} topics")

# Save content as a JSON data file
os.makedirs("lib/guide", exist_ok=True)
Path("lib/guide/content.json").write_text(json.dumps(all_content, indent=2))
print("Content saved to lib/guide/content.json")

/content/govcon-dashboard
Generating: What is Federal Procurement?...
  ✗ Parse error: Unterminated string starting at: line 24 column 18 (char 8710)
Generating: Who Can Bid on Federal Contracts?...
  ✗ Parse error: Unterminated string starting at: line 37 column 5 (char 11938)
Generating: What is a NAICS Code?...
  ✗ Parse error: Unterminated string starting at: line 28 column 5 (char 9027)
Generating: SAM.gov Registration Guide...
  ✗ Parse error: Unterminated string starting at: line 24 column 18 (char 8013)
Generating: Government Small Business Goals...
  ✗ Parse error: Unterminated string starting at: line 37 column 5 (char 10901)
Generating: Set-Aside Contracts Explained...
  ✗ Parse error: Unterminated string starting at: line 9 column 18 (char 600)
Generating: The 8(a) Business Development Program...
  ✗ Parse error: Unterminated string starting at: line 20 column 18 (char 6728)
Generating: HUBZone Program...
  ✗ Parse error: Unterminated string starting at: line 34 column 3 (c

In [ ]:
%cd /content/govcon-dashboard

import requests
import json
import os
import time
from pathlib import Path
from google.colab import userdata

GEMINI_KEY = userdata.get('GEMINI_API_KEY')

def gemini(prompt):
    r = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={GEMINI_KEY}",
        headers={"Content-Type": "application/json"},
        json={"contents": [{"parts": [{"text": prompt}]}],
              "generationConfig": {"maxOutputTokens": 8000, "temperature": 0.3}}
    )
    data = r.json()
    return data.get("candidates", [{}])[0].get("content", {}).get("parts", [{}])[0].get("text", "")

TOPICS = [
    ("getting-started", "what-is-federal-procurement", "What is Federal Procurement?"),
    ("getting-started", "who-can-bid", "Who Can Bid on Federal Contracts?"),
    ("getting-started", "naics-codes", "What is a NAICS Code?"),
    ("getting-started", "sam-registration", "SAM.gov Registration Guide"),
    ("small-business", "government-goals", "Government Small Business Goals"),
    ("small-business", "set-aside-contracts", "Set-Aside Contracts Explained"),
    ("small-business", "8a-program", "The 8(a) Business Development Program"),
    ("small-business", "hubzone", "HUBZone Program"),
    ("small-business", "wosb", "Women-Owned Small Business (WOSB)"),
    ("small-business", "sdvosb", "Service-Disabled Veteran-Owned (SDVOSB)"),
    ("small-business", "mentor-protege", "Mentor-Protégé Program"),
    ("awards", "lowest-price-vs-best-value", "Lowest Price vs. Best Value"),
    ("awards", "evaluation-process", "The Evaluation Process"),
    ("awards", "past-performance", "Past Performance"),
    ("awards", "simplified-acquisition", "Simplified Acquisition Procedures"),
    ("incumbents", "recompete-basics", "Recompete Basics"),
    ("incumbents", "full-open-competition", "Full and Open Competition"),
    ("incumbents", "reading-incumbent-history", "Reading the Incumbent's History"),
    ("incumbents", "price-advantage", "Price as Your Advantage"),
    ("subcontracting", "far-19-7", "FAR 19.7 Small Business Subcontracting"),
    ("subcontracting", "approaching-primes", "How to Approach Prime Contractors"),
    ("subcontracting", "sub-vs-prime", "Subcontracting vs. Prime Contracting"),
    ("protests", "what-is-bid-protest", "What is a Bid Protest?"),
    ("protests", "when-to-protest", "When to Consider a Protest"),
    ("protests", "gao-protest-basics", "GAO Protest Basics"),
    ("resources", "apex-accelerators", "APEX Accelerators"),
    ("resources", "sba-district-offices", "SBA District Offices"),
    ("resources", "score-mentorship", "SCORE Mentorship"),
    ("resources", "sam-gov", "SAM.gov Guide"),
    ("resources", "usaspending", "USAspending.gov Guide"),
]

# Shorter prompt to avoid truncation
PROMPT_TEMPLATE = """Write a comprehensive federal procurement guide on: "{title}"

Audience: small business owner new to federal contracting. Tone: practical, like NerdWallet.

Return ONLY valid JSON (no markdown, no backticks):
{{"title":"{title}","subtitle":"one sentence","intro":"2-3 sentences why this matters","sections":[{{"heading":"string","content":"3-4 paragraphs with specific FAR citations, dollar thresholds, timeframes"}}],"key_facts":["fact with number/threshold 1","fact 2","fact 3","fact 4","fact 5"],"common_mistakes":["mistake and fix 1","mistake and fix 2","mistake and fix 3"],"official_sources":[{{"name":"string","url":"https://real-url.gov","description":"string"}}]}}

Include 4-5 sections. Be specific — real FAR part numbers, real dollar amounts, real timeframes. Return ONLY the JSON object."""

# Load existing successful content
existing = {}
content_file = Path("lib/guide/content.json")
if content_file.exists():
    existing = json.loads(content_file.read_text())
    good = {k: v for k, v in existing.items() if "error" not in v}
    print(f"Already have {len(good)} good topics")

all_content = dict(existing)
failed = []

for cat, slug, title in TOPICS:
    key = f"{cat}/{slug}"
    # Skip already successful ones
    if key in all_content and "error" not in all_content[key] and "sections" in all_content[key]:
        print(f"  ✓ Skip (cached): {title}")
        continue

    print(f"Generating: {title}...")
    for attempt in range(3):
        try:
            text = gemini(PROMPT_TEMPLATE.format(title=title))
            text = text.strip()
            # Clean markdown fences
            if "```" in text:
                parts = text.split("```")
                for part in parts:
                    if part.strip().startswith("{"):
                        text = part.strip()
                        break
            if text.startswith("json"):
                text = text[4:].strip()
            # Find JSON boundaries
            start = text.find("{")
            end = text.rfind("}") + 1
            if start >= 0 and end > start:
                text = text[start:end]
            data = json.loads(text)
            if "sections" in data:
                all_content[key] = data
                print(f"  ✓ {len(data['sections'])} sections")
                break
            else:
                raise ValueError("No sections in response")
        except Exception as e:
            print(f"  ✗ Attempt {attempt+1}: {e}")
            if attempt < 2:
                time.sleep(3)
            else:
                failed.append(title)

    time.sleep(2)  # Rate limit buffer

# Save
os.makedirs("lib/guide", exist_ok=True)
Path("lib/guide/content.json").write_text(json.dumps(all_content, indent=2))

good = {k: v for k, v in all_content.items() if "sections" in v}
print(f"\nDone: {len(good)}/30 successful")
if failed:
    print(f"Failed: {failed}")

/content/govcon-dashboard
Already have 2 good topics
Generating: What is Federal Procurement?...
  ✓ 4 sections
Generating: Who Can Bid on Federal Contracts?...
  ✗ Attempt 1: Expecting value: line 1 column 1 (char 0)
  ✗ Attempt 2: Expecting value: line 1 column 1 (char 0)
  ✗ Attempt 3: Expecting value: line 1 column 1 (char 0)
Generating: What is a NAICS Code?...
  ✗ Attempt 1: Expecting value: line 1 column 1 (char 0)
  ✗ Attempt 2: Expecting value: line 1 column 1 (char 0)
  ✗ Attempt 3: Expecting value: line 1 column 1 (char 0)
Generating: SAM.gov Registration Guide...
  ✗ Attempt 1: Expecting value: line 1 column 1 (char 0)
  ✗ Attempt 2: Expecting value: line 1 column 1 (char 0)
  ✗ Attempt 3: Expecting value: line 1 column 1 (char 0)
Generating: Government Small Business Goals...
  ✗ Attempt 1: Expecting value: line 1 column 1 (char 0)
  ✗ Attempt 2: Expecting value: line 1 column 1 (char 0)
  ✓ 5 sections
Generating: Set-Aside Contracts Explained...
  ✗ Attempt 1: Expecting v

In [ ]:
%cd /content/govcon-dashboard

import json
from pathlib import Path

# Load existing 6 good topics
existing = json.loads(Path("lib/guide/content.json").read_text())
good = {k: v for k, v in existing.items() if "sections" in v}
print(f"Starting with {len(good)} cached topics")

new_content = {
"getting-started/who-can-bid": {
  "title": "Who Can Bid on Federal Contracts?",
  "subtitle": "Any registered U.S. business can compete for federal contracts — here's exactly what that means.",
  "intro": "One of the biggest misconceptions about federal contracting is that it's only for large, established companies with government connections. The reality is that any legitimate U.S. business can bid on most federal contracts. The rules are designed to be open and competitive.",
  "sections": [
    {"heading": "Basic Eligibility Requirements", "content": "To bid on federal contracts, your business must be legally organized under U.S. law — a sole proprietorship, partnership, LLC, corporation, or other recognized entity. Foreign-owned businesses can participate in many contracts but face restrictions on certain sensitive or defense-related work. There is no minimum revenue requirement, no minimum number of employees, and no minimum years in business for most contracts.\n\nYour business must not be debarred or suspended from federal contracting. The System for Award Management (SAM.gov) maintains the Excluded Parties List System — a public database of businesses and individuals barred from receiving federal awards. Agencies check this list before making any award.\n\nFor most contracts, you must also not have unpaid federal taxes or be in default on federal loans. These checks are part of the 'responsibility determination' that contracting officers perform before award under FAR 9.104."},
    {"heading": "SAM.gov Registration: The One Non-Negotiable", "content": "Registration in SAM.gov (System for Award Management) is required for any business that wants to receive a federal contract award or subcontract above the micro-purchase threshold ($10,000). Registration is free and must be renewed annually — there is no fee to register, renew, or search for opportunities.\n\nYour SAM registration requires a Unique Entity ID (UEI), which replaced the old DUNS number system in April 2022. The UEI is assigned automatically when you register in SAM.gov. You will also need your Employer Identification Number (EIN), banking information for electronic funds transfer (EFT), and your NAICS codes.\n\nActivation typically takes 7-10 business days after submission. Plan ahead — you cannot receive payment on a federal contract without an active SAM registration, and agencies will not make awards to entities with lapsed registrations."},
    {"heading": "Size Standards and Small Business Designation", "content": "The federal government defines 'small business' differently for each industry, using NAICS codes as the basis. Size standards are set by the Small Business Administration and expressed either as maximum annual receipts (revenue) or maximum number of employees, depending on the industry.\n\nFor example, NAICS 541511 (Custom Computer Programming Services) has a size standard of $34 million in average annual receipts. NAICS 236220 (Commercial Building Construction) uses an employee-based standard of 1,500 employees. You can look up the size standard for any NAICS code at SBA's size standards tool (sba.gov/size-standards).\n\nBeing designated as a small business matters enormously because it opens access to set-aside contracts, sole-source authority under certain programs, and evaluation preferences. You self-certify your small business status in SAM.gov — but false certifications carry serious legal penalties under the False Claims Act."},
    {"heading": "Special Categories of Eligible Bidders", "content": "Beyond basic small business status, several special designations open additional doors. Businesses owned by socially and economically disadvantaged individuals can apply for 8(a) certification. Businesses in Historically Underutilized Business Zones can apply for HUBZone certification. Women-owned businesses can self-certify as WOSB. Service-disabled veterans can self-certify as SDVOSB.\n\nNon-profit organizations can also compete for federal contracts in many categories, though they are subject to different cost accounting rules. Educational institutions, research organizations, and hospitals regularly hold federal contracts.\n\nForeign-owned companies operating in the U.S. can bid on many commercial contracts but are restricted from contracts involving classified information, certain defense technologies, or programs with domestic sourcing requirements under the Buy American Act."},
    {"heading": "What You Cannot Bid On", "content": "Some contracts are restricted to specific categories of bidders. Classified contracts require appropriate security clearances. Contracts set aside for small businesses are closed to large businesses (those exceeding the relevant size standard). Contracts on the AbilityOne program are reserved for non-profit agencies employing people who are blind or have significant disabilities.\n\nSome contracts require specific licenses — a construction contract may require a state contractor's license, a healthcare contract may require professional certifications. These requirements are always spelled out in the solicitation under the 'Qualifications' section.\n\nConflict of interest rules (FAR 9.5) can also bar certain bidders. If your company helped write a solicitation, evaluated competing proposals, or has a financial relationship with the program office, you may be excluded from competing for that specific contract."}
  ],
  "key_facts": [
    "SAM.gov registration is free and required for all federal contract awards — renewal is annual",
    "The Unique Entity ID (UEI) replaced DUNS numbers in April 2022 — registration is at sam.gov",
    "Small business size standards are set per NAICS code — look yours up at sba.gov/size-standards",
    "False certification of small business status violates the False Claims Act and carries serious penalties",
    "Micro-purchases under $10,000 can be made without competition — SAM registration still recommended"
  ],
  "common_mistakes": [
    "Letting SAM registration lapse — agencies cannot make awards to lapsed registrations, so set a calendar reminder 60 days before your annual renewal date",
    "Registering only one NAICS code — register all codes that reflect your capabilities, as contracts are often categorized differently than you expect",
    "Assuming small business status automatically qualifies you for set-asides — you must also meet the specific program requirements (e.g., 8(a), HUBZone) for those set-asides"
  ],
  "official_sources": [
    {"name": "SAM.gov", "url": "https://sam.gov", "description": "Official registration system — register, renew, and search opportunities"},
    {"name": "SBA Size Standards", "url": "https://www.sba.gov/federal-contracting/contracting-guide/size-standards", "description": "Look up the size standard for any NAICS code"},
    {"name": "FAR Part 9 — Contractor Qualifications", "url": "https://www.acquisition.gov/far/part-9", "description": "Official rules on contractor responsibility determinations"}
  ]
},

"getting-started/naics-codes": {
  "title": "What is a NAICS Code?",
  "subtitle": "The 6-digit codes that determine which contracts you can see, bid on, and qualify for as a small business.",
  "intro": "The North American Industry Classification System (NAICS) is a federal standard for classifying businesses by what they do. In federal contracting, your NAICS codes are among the most important pieces of information about your business — they determine your small business size standard, which set-asides you qualify for, and how agencies search for vendors like you.",
  "sections": [
    {"heading": "How NAICS Codes Work", "content": "NAICS codes are 6-digit numbers organized hierarchically. The first two digits identify the sector (e.g., 54 = Professional, Scientific, and Technical Services). The third digit identifies the subsector, the fourth the industry group, the fifth the NAICS industry, and the sixth the national industry — the most specific level.\n\nFor example: 541512 breaks down as 54 (Professional Services) → 541 (Professional, Scientific, Technical) → 5415 (Computer Systems Design) → 54151 (Computer Systems Design) → 541512 (Computer Systems Design and Related Services). This specificity matters because different 6-digit codes within the same sector can have very different size standards.\n\nThe Census Bureau maintains NAICS codes and updates them every five years. The most recent revision is NAICS 2022. When a code changes, SAM.gov and solicitations are updated — your registrations should reflect current codes."},
    {"heading": "NAICS Codes and Size Standards", "content": "The SBA assigns a size standard to every NAICS code. This standard determines whether your business qualifies as 'small' for contracts in that category. Size standards are expressed as either maximum annual revenue (averaged over three years) or maximum number of employees (averaged over 12 months).\n\nRevenue-based standards range from $2.25 million (some agricultural sectors) to $47 million (some professional services). Employee-based standards typically apply to manufacturing and mining sectors and range from 100 to 1,500 employees.\n\nCritically, your size is determined at the time you submit your offer. If your revenue has grown since your last certification, you need to recertify. Winning a large contract can cause you to exceed your size standard for future contracts in that category — an important strategic consideration."},
    {"heading": "Choosing the Right NAICS Codes", "content": "You can register multiple NAICS codes in SAM.gov — there is no limit. Registering all codes that legitimately reflect your capabilities is strongly recommended. Contracting officers and prime contractors search SAM's Dynamic Small Business Search (DSBS) by NAICS code when looking for vendors and subcontractors.\n\nWhen a solicitation is issued, the contracting officer designates a primary NAICS code for that acquisition. This code determines the applicable size standard. If you believe the contracting officer chose an incorrect NAICS code that disadvantages small businesses, you can challenge the designation through SBA under 13 CFR Part 121.\n\nStrategically, look at USAspending.gov to see which NAICS codes are most actively used by agencies in your target market. Focus your business development on codes with significant spending volume and favorable size standards for your business."},
    {"heading": "NAICS Codes in SAM.gov and DSBS", "content": "When you register in SAM.gov, you select your primary NAICS code and any additional codes. Your primary code should reflect your largest revenue source. Additional codes should cover any significant service or product area.\n\nThe Dynamic Small Business Search (DSBS) is a public database maintained by SBA that allows anyone — including prime contractors and agency buyers — to search for small businesses by NAICS code, location, certifications, and keywords. Keeping your DSBS profile current and keyword-rich is free marketing to people actively looking for vendors like you.\n\nSome NAICS codes are designated as 'exceptions' — meaning the size standard differs from the standard for the broader code. Always check the specific 6-digit code, not the sector-level standard."}
  ],
  "key_facts": [
    "NAICS codes are 6 digits — always use the full 6-digit code, not the sector or subsector level",
    "Size standards are set per 6-digit NAICS code — the same sector can have very different standards",
    "You can register unlimited NAICS codes in SAM.gov — register all that legitimately apply",
    "NAICS codes are updated every 5 years by the Census Bureau — most recent revision is NAICS 2022",
    "You can challenge a contracting officer's NAICS code designation through SBA under 13 CFR Part 121"
  ],
  "common_mistakes": [
    "Registering only your primary NAICS code — you miss subcontracting opportunities and vendor searches for all your other capabilities",
    "Using sector-level (2-4 digit) codes instead of the full 6-digit code — only 6-digit codes have assigned size standards and are valid in SAM",
    "Not updating NAICS codes when your business expands — if you add new service lines, add the relevant codes immediately"
  ],
  "official_sources": [
    {"name": "NAICS Search Tool", "url": "https://www.census.gov/naics/", "description": "Search and look up NAICS codes by keyword or number"},
    {"name": "SBA Size Standards", "url": "https://www.sba.gov/federal-contracting/contracting-guide/size-standards", "description": "Size standard for every NAICS code"},
    {"name": "SAM.gov DSBS", "url": "https://web.sba.gov/pro-net/search/dsp_dsbs.cfm", "description": "Dynamic Small Business Search — how buyers find you"}
  ]
},

"getting-started/sam-registration": {
  "title": "SAM.gov Registration Guide",
  "subtitle": "The one required step before you can receive any federal contract — and it's completely free.",
  "intro": "SAM.gov (System for Award Management) is the federal government's official contractor registration database. Registration is free, mandatory for all federal contract awards above $10,000, and must be renewed annually. Without an active SAM registration, you cannot receive payment on a federal contract.",
  "sections": [
    {"heading": "What SAM.gov Is and Why It Exists", "content": "SAM.gov consolidated multiple legacy systems — including CCR (Central Contractor Registration), ORCA, and FedBizOpps — into a single platform. It serves as the authoritative source for contractor information, certification status, and opportunities for the entire federal government.\n\nWhen an agency wants to make a contract award, their system automatically checks SAM.gov to verify the vendor is registered, active, and not excluded from federal contracting. If your registration has lapsed or contains errors, you cannot receive an award — even if you submitted the winning proposal.\n\nSAM.gov also hosts the federal opportunities database (formerly FedBizOpps), where all contract opportunities above $25,000 must be publicly posted. You can search opportunities, set up email alerts, and track acquisitions all within the same system."},
    {"heading": "Step-by-Step Registration Process", "content": "Step 1 — Get your Unique Entity ID (UEI). Go to sam.gov and click 'Register.' You'll create a login.gov account first (the federal single sign-on system), then begin the entity registration. SAM.gov assigns your UEI automatically during this process — there's no separate application.\n\nStep 2 — Gather your information. You'll need your EIN (Employer Identification Number), your legal business name exactly as registered with the IRS, your NAICS codes, a bank account for Electronic Funds Transfer (EFT) setup, and information about your business structure and ownership.\n\nStep 3 — Complete the Core Data. This includes your entity information, taxpayer identification, financial information for EFT, and general information about your business type.\n\nStep 4 — Complete the Assertions. This is where you select your NAICS codes, claim small business certifications (8(a), HUBZone, WOSB, SDVOSB), and answer questions about your business.\n\nStep 5 — Submit and wait. Processing typically takes 7-10 business days, though it can be faster. You'll receive a confirmation email when your registration is active."},
    {"heading": "Annual Renewal Requirements", "content": "SAM registration expires exactly one year from your activation date. You must renew before expiration — there is no grace period for contract awards. Set a calendar reminder 60 days before your renewal date to avoid any lapse.\n\nRenewal requires you to review and update all your information, re-certify your representations and certifications (including small business status), and confirm your banking information. The renewal process takes about 30-60 minutes and again requires 7-10 days for processing.\n\nDuring the renewal window, your existing registration remains active. You will not lose your active status while a renewal is pending, as long as you submitted the renewal before the expiration date. However, if you let your registration actually expire, there is a gap in your active status that can delay contract awards."},
    {"heading": "Avoiding Third-Party SAM Registration Services", "content": "Dozens of companies charge $300-$3,000 to help businesses register in SAM.gov. These services are completely unnecessary — the registration process is designed to be completed by any business owner without assistance.\n\nThe Federal Trade Commission and SBA have issued warnings about these services, some of which use deceptive practices including impersonating government websites, creating false urgency, and charging recurring 'maintenance' fees for work they don't actually perform.\n\nIf you need help with SAM registration, contact your local APEX Accelerator (formerly PTAC) — they provide free hands-on registration assistance funded by the Department of Defense. Find your local office at apexaccelerators.us."},
    {"heading": "Common SAM Issues and How to Fix Them", "content": "The most common issue is a name mismatch between your SAM registration and your IRS records. Your legal name in SAM must match exactly what the IRS has on file. If they don't match, your registration will be rejected during the IRS TIN matching process.\n\nBanking information errors are also common. If your EFT information is incorrect, the government cannot pay you. Always verify your routing and account numbers carefully and consider doing a small test transaction.\n\nFor issues with your registration, SAM.gov has a help desk at fsd.gov (Federal Service Desk). Response times can be slow — budget 3-5 business days for resolution. For urgent issues, APEX Accelerators can often help navigate the process faster."}
  ],
  "key_facts": [
    "SAM.gov registration is completely free — never pay a third party to register",
    "Registration takes 7-10 business days to activate — plan ahead before proposal deadlines",
    "Registration must be renewed annually — lapsed registration prevents contract awards",
    "Your legal name must match IRS records exactly — mismatches cause registration rejection",
    "The Unique Entity ID (UEI) replaced DUNS numbers in April 2022 — get yours at sam.gov"
  ],
  "common_mistakes": [
    "Paying third-party services for SAM registration — the process is free and designed for self-service; use your local APEX Accelerator if you need free help",
    "Letting registration lapse — set a calendar reminder 60 days before your expiration date; there is no grace period for awards",
    "Entering your business name differently than IRS records — even minor differences (LLC vs L.L.C.) can cause TIN matching failures"
  ],
  "official_sources": [
    {"name": "SAM.gov", "url": "https://sam.gov", "description": "Official registration portal — everything is free"},
    {"name": "Federal Service Desk", "url": "https://www.fsd.gov", "description": "SAM.gov help desk for registration issues"},
    {"name": "APEX Accelerators", "url": "https://www.apexaccelerators.us", "description": "Free hands-on SAM registration assistance"}
  ]
},

"small-business/set-aside-contracts": {
  "title": "Set-Aside Contracts Explained",
  "subtitle": "Contracts reserved exclusively for small businesses — where large competitors cannot bid.",
  "intro": "Set-aside contracts are one of the most powerful advantages available to small businesses in federal contracting. When a contract is set aside, it is restricted to small businesses only — large companies are legally excluded from competing. Understanding when and how set-asides apply is fundamental to small business contracting strategy.",
  "sections": [
    {"heading": "The Legal Basis for Set-Asides", "content": "The Small Business Act (15 U.S.C. § 644) requires federal agencies to maximize small business participation in federal contracting. The implementing regulations in FAR 19.5 establish the mandatory set-aside rule: when a contracting officer determines there is a 'reasonable expectation' that at least two responsible small businesses will submit offers at fair market prices, the acquisition must be set aside exclusively for small business competition.\n\nThis is not discretionary — it's a legal requirement. Contracting officers who fail to apply the set-aside rule when it's required are violating the FAR. Small businesses who believe a contract should have been set aside but wasn't have remedies including protests to SBA and bid protests to GAO.\n\nThe government-wide small business prime contracting goal is 23% of all eligible contract dollars, established by the Small Business Act. Individual agencies have their own goals, and agency performance against these goals is publicly reported and affects agency procurement officials' performance reviews."},
    {"heading": "Total Small Business Set-Asides vs. Partial", "content": "A total set-aside restricts the entire acquisition to small businesses. This is the most common form and applies when the whole requirement can be met by small businesses.\n\nA partial set-aside divides the acquisition into two parts — one competed among all offerors and one reserved for small businesses. Partial set-asides are less common and typically used for large, multi-award contracts where the agency wants both large and small business participation.\n\nUnder FAR 19.503, acquisitions between $10,000 and $250,000 (the simplified acquisition threshold) are automatically set aside for small businesses if there is a reasonable expectation of receiving offers from two or more responsible small businesses. No formal determination is required — the set-aside is presumed."},
    {"heading": "Tiered Set-Asides: Prioritization Rules", "content": "FAR 19.14 establishes a priority order for set-asides when multiple programs apply. Before setting aside for small businesses generally, contracting officers must consider whether to restrict competition to an even smaller universe:\n\n1. AbilityOne (mandatory source for certain products/services)\n2. 8(a) sole source (up to $4.5M services, $7M manufacturing)\n3. HUBZone sole source (up to $4.5M services, $7M manufacturing)\n4. SDVOSB sole source (same thresholds)\n5. WOSB set-aside\n6. Small business set-aside (general)\n\nThis priority order means that if you hold an 8(a) certification, for example, you may be able to receive a sole-source award without any competition at all for contracts below the threshold. Understanding where you fit in this priority order is essential for maximizing the value of your certifications."},
    {"heading": "Multiple Award Contracts and Set-Asides", "content": "Many federal contracts are structured as Multiple Award Contracts (MACs) or Indefinite Delivery/Indefinite Quantity (IDIQ) contracts where multiple vendors hold basic agreements and then compete for individual task orders. Many of these vehicles have small business set-aside pools.\n\nGSA Schedules (now called MAS — Multiple Award Schedule) allow for individual orders to be set aside for small businesses, 8(a), HUBZone, WOSB, or SDVOSB businesses. Under FAR 8.405-5, ordering officers are encouraged to consider setting aside orders for small businesses even when the schedule contract itself is not set aside.\n\nGovernment-wide Acquisition Contracts (GWACs) like NASA SEWP, NIH CIO-SP3, and Army ITES-3S have small business tracks. Getting on these vehicles, while competitive, opens a stream of task order opportunities in addition to open market competition."},
    {"heading": "Protesting Improper Set-Aside Decisions", "content": "If a contracting officer fails to set aside a contract when required, or improperly waives a set-aside, small businesses have recourse. You can file a size protest with SBA questioning whether the awardee truly qualifies as small. You can file a bid protest with GAO arguing that the set-aside rules were not properly applied.\n\nSBA also has Procurement Center Representatives (PCRs) stationed at major buying activities. Their job is specifically to advocate for small business set-asides and challenge improper procurements. If you believe an agency is systematically avoiding set-asides, contact SBA's PCR for that agency.\n\nDocumentation matters. When you identify a contract that should be set aside, review the solicitation for the contracting officer's market research documentation. If it's absent or inadequate, that's the basis for a protest or PCR complaint."}
  ],
  "key_facts": [
    "Acquisitions $10K-$250K are automatically set aside for small business if 2+ small businesses can compete",
    "The government-wide small business prime contracting goal is 23% of eligible contract dollars",
    "FAR 19.14 establishes priority order: 8(a) sole source before HUBZone before SDVOSB before WOSB before general SB",
    "SBA Procurement Center Representatives are federal employees whose job is to advocate for small business set-asides",
    "False certification of small business status violates the False Claims Act — penalties include treble damages"
  ],
  "common_mistakes": [
    "Not checking whether a contract should have been set aside — review solicitations for market research documentation and challenge inadequate analysis",
    "Missing tiered set-aside priority — if you have 8(a) or HUBZone certification, you may qualify for sole-source awards before any competition",
    "Assuming you're too small for large IDIQs — most large vehicles have small business pools or set-aside order capability"
  ],
  "official_sources": [
    {"name": "FAR Part 19 — Small Business Programs", "url": "https://www.acquisition.gov/far/part-19", "description": "Complete regulations governing small business set-asides"},
    {"name": "SBA Contracting Guide", "url": "https://www.sba.gov/federal-contracting/contracting-guide", "description": "SBA's overview of small business contracting programs"},
    {"name": "SBA PCR Directory", "url": "https://www.sba.gov/federal-contracting/counseling-help/procurement-center-representatives", "description": "Find the PCR for your target agency"}
  ]
},

"small-business/8a-program": {
  "title": "The 8(a) Business Development Program",
  "subtitle": "The SBA's most powerful small business program — including sole-source contracts with no competition.",
  "intro": "The 8(a) Business Development Program is administered by the Small Business Administration and provides participating businesses with access to sole-source contracts, set-aside competitions, and business development assistance. For eligible businesses, 8(a) certification is one of the most valuable designations in federal contracting.",
  "sections": [
    {"heading": "What 8(a) Certification Provides", "content": "8(a) participants can receive sole-source contracts — awards made without competition — up to $4.5 million for services and $7 million for manufacturing contracts. Above these thresholds, the work must be competed among 8(a) firms, but large businesses are still excluded.\n\nThe SBA can also enter into contracts on behalf of 8(a) firms through 'partnership agreements' with agencies. Under these agreements, the SBA technically holds the prime contract and subcontracts to the 8(a) firm — providing an additional layer of contracting flexibility.\n\nBeyond contract access, 8(a) participants receive business development assistance including mentoring, training, and management and technical assistance. The SBA assigns a Business Opportunity Specialist (BOS) to each participant who serves as a counselor and advocate throughout the program."},
    {"heading": "Eligibility Requirements", "content": "To qualify for 8(a), your business must be at least 51% owned and controlled by one or more socially and economically disadvantaged U.S. citizens. Social disadvantage is presumed for certain groups including Black Americans, Hispanic Americans, Native Americans, Asian Pacific Americans, and Subcontinent Asian Americans. Members of other groups can demonstrate social disadvantage through a personal narrative.\n\nEconomic disadvantage requires that the disadvantaged owners have a personal net worth below $850,000 (excluding equity in the business and primary residence), adjusted gross income averaged over three years below $400,000, and total assets below $6.5 million.\n\nThe business itself must be small under the applicable NAICS code, must have been in business for at least two years (with limited exceptions), and must demonstrate potential for success. The owner must be a U.S. citizen, must manage the day-to-day operations, and must make long-term decisions for the company."},
    {"heading": "The Nine-Year Program Term", "content": "8(a) participation is limited to nine years, divided into a four-year developmental stage and a five-year transitional stage. The intent is for businesses to use the program to build capacity and then transition to competing without the set-aside advantages.\n\nDuring the developmental stage, the focus is on building the business, establishing past performance, and accessing sole-source opportunities. During the transitional stage, the business is expected to reduce its reliance on 8(a) contracts as a percentage of total revenue.\n\nThere are annual review requirements — you must submit annual financial statements and certifications of continuing eligibility. Significant changes in ownership or management must be reported to SBA within 30 days. Failure to meet reporting requirements or changes that affect eligibility can result in early graduation or termination from the program."},
    {"heading": "Applying for 8(a) Certification", "content": "Applications are submitted through SBA's certify.sba.gov portal. The application is comprehensive — expect to spend 20-40 hours gathering documents and completing the narrative sections.\n\nRequired documentation includes three years of personal and business tax returns, business financial statements, proof of ownership (operating agreement, stock certificates, or similar), a personal financial statement for each disadvantaged owner, proof of U.S. citizenship, and a personal narrative describing social disadvantage.\n\nSBA processing times vary but typically run 60-90 days. Applications are reviewed by SBA district offices, and you may receive requests for additional information during review. Common reasons for denial include insufficient documentation of control, questions about the owner's day-to-day management, or failure to demonstrate economic disadvantage."},
    {"heading": "Using 8(a) Status Strategically", "content": "Sole-source authority is the most powerful feature of 8(a), but it requires agency interest. You cannot simply demand a sole-source award — an agency contracting officer must be willing to work with SBA to structure the contract as an 8(a) sole-source.\n\nBuilding relationships with agency program managers and contracting officers before there's a specific opportunity is the most effective strategy. When an agency has a need and knows an 8(a) firm that can meet it, the sole-source mechanism makes the transaction much simpler for the contracting officer.\n\nTrack your $4.5M/$7M thresholds carefully. Once you reach those amounts in a competitive 8(a) award, subsequent work in that category must be competed among 8(a) firms. Strategic sequencing of sole-source vs. competitive awards can maximize your use of sole-source authority over the program term."}
  ],
  "key_facts": [
    "Sole-source contracts up to $4.5M (services) or $7M (manufacturing) without competition",
    "Program lasts 9 years: 4-year developmental stage + 5-year transitional stage",
    "Personal net worth must be below $850,000 (excluding business equity and primary residence)",
    "Business must have been operating for at least 2 years before applying (limited exceptions exist)",
    "Applications submitted at certify.sba.gov — processing typically takes 60-90 days"
  ],
  "common_mistakes": [
    "Applying before the 2-year operating requirement — save time by confirming basic eligibility before investing 40 hours in the application",
    "Failing to maintain day-to-day control — SBA monitors this throughout the program; absentee owners risk early graduation",
    "Not developing agency relationships before needing sole-source awards — sole-source requires a willing contracting officer, which means relationship development first"
  ],
  "official_sources": [
    {"name": "SBA 8(a) Program", "url": "https://www.sba.gov/federal-contracting/contracting-assistance-programs/8a-business-development-program", "description": "Official program overview and eligibility details"},
    {"name": "certify.sba.gov", "url": "https://certify.sba.gov", "description": "Application portal for 8(a) and other SBA certifications"},
    {"name": "13 CFR Part 124", "url": "https://www.ecfr.gov/current/title-13/chapter-I/part-124", "description": "Complete 8(a) program regulations"}
  ]
},

"small-business/hubzone": {
  "title": "HUBZone Program",
  "subtitle": "A 10% price preference and set-aside access for businesses in economically distressed communities.",
  "intro": "The Historically Underutilized Business Zone (HUBZone) program helps small businesses in economically distressed areas compete for federal contracts. Certified HUBZone businesses receive a 10% price evaluation preference in full and open competition and access to HUBZone-specific set-asides.",
  "sections": [
    {"heading": "What the 10% Price Preference Means", "content": "In full and open competitions (where all businesses can compete), HUBZone firms receive a 10% price evaluation adjustment. This means if your bid is up to 10% higher than the lowest non-HUBZone offer, you still win — the agency evaluates your bid as if it were 10% lower.\n\nFor example: if a non-HUBZone firm bids $100,000 and you bid $108,000, you win because your evaluated price ($97,200 after the 10% preference) is lower. This is a substantial competitive advantage in price-sensitive competitions.\n\nThe 10% preference applies only in full and open competition. In HUBZone set-asides, only HUBZone firms compete, so the preference doesn't come into play. The strategic value of the preference is in competitions where you're competing against larger, potentially more efficient firms."},
    {"heading": "HUBZone Eligibility Requirements", "content": "To qualify, your business must meet four requirements. First, it must be a small business under the applicable NAICS code. Second, it must be at least 51% owned and controlled by U.S. citizens, a Community Development Corporation, an agricultural cooperative, an Alaska Native Corporation, a Native Hawaiian organization, or an Indian tribe.\n\nThird, your principal office must be located in a HUBZone. The principal office is the location where the greatest number of employees work. Fourth, at least 35% of your employees must reside in a HUBZone (not necessarily the same one as your office).\n\nHUBZones are determined by census tract data and include qualified census tracts, qualified non-metropolitan counties, lands within the boundaries of federally recognized Indian reservations, qualified base closure areas, and qualified disaster areas (temporary designation). The SBA maintains a HUBZone map at maps.certify.sba.gov/hubzone/map."},
    {"heading": "Maintaining HUBZone Certification", "content": "HUBZone certification is granted for three years, with an annual representation required confirming continued eligibility. At the three-year mark, you must submit a full recertification.\n\nThe 35% employee residency requirement is measured continuously — not just at certification. If employees move out of HUBZones or if you hire employees who don't live in HUBZones, your percentage can drop below the threshold. Many businesses use software to track employee addresses against HUBZone maps in real time.\n\nChanges in your principal office location or significant workforce changes must be reported to SBA within 30 days. If your principal office moves out of a HUBZone, you lose eligibility — though you may have a grace period to relocate or restructure.\n\nSBA conducts program examinations — essentially audits — on a percentage of certified firms each year. You must be able to demonstrate compliance at any time. Maintaining organized records of employee addresses and office lease documentation is essential."},
    {"heading": "HUBZone Set-Asides and Sole-Source Awards", "content": "Beyond the price preference, HUBZone firms have access to contracts set aside exclusively for HUBZone competition. Under FAR 19.1305, contracting officers can set aside contracts for HUBZone small businesses when there is a reasonable expectation of receiving offers from two or more HUBZone firms at fair market prices.\n\nHUBZone sole-source awards are available for contracts up to $4.5 million for services and $7 million for manufacturing — the same thresholds as 8(a). However, HUBZone sole-source authority requires that no other small business program (8(a), SDVOSB, WOSB) is a better fit, and it's used less frequently than 8(a) sole-source in practice.\n\nFor businesses located in Native American areas, military base closure areas, or disaster areas, there may be additional contracting opportunities through agency-specific programs that recognize these communities."}
  ],
  "key_facts": [
    "10% price evaluation preference in full and open competition — bids up to 10% higher can still win",
    "35% of employees must reside in a HUBZone — measured continuously, not just at certification",
    "Principal office must be in a HUBZone — greatest number of employees must work there",
    "Certification lasts 3 years with annual representations required",
    "Sole-source authority up to $4.5M (services) and $7M (manufacturing)"
  ],
  "common_mistakes": [
    "Not tracking employee addresses continuously — the 35% residency requirement applies at all times, not just at certification; use a tracking system",
    "Assuming your office location qualifies without checking — HUBZone boundaries change with census data updates; verify at maps.certify.sba.gov",
    "Forgetting to report principal office moves within 30 days — late reporting can result in decertification"
  ],
  "official_sources": [
    {"name": "SBA HUBZone Program", "url": "https://www.sba.gov/federal-contracting/contracting-assistance-programs/hubzone-program", "description": "Program overview and eligibility requirements"},
    {"name": "HUBZone Map", "url": "https://maps.certify.sba.gov/hubzone/map", "description": "Check whether your office or employees' homes are in a HUBZone"},
    {"name": "13 CFR Part 126", "url": "https://www.ecfr.gov/current/title-13/chapter-I/part-126", "description": "Complete HUBZone program regulations"}
  ]
},

"small-business/wosb": {
  "title": "Women-Owned Small Business (WOSB) Program",
  "subtitle": "Set-aside contracts in industries where women-owned businesses are underrepresented in federal contracting.",
  "intro": "The Women-Owned Small Business Federal Contracting Program authorizes contracting officers to restrict competition to WOSB and Economically Disadvantaged WOSB (EDWOSB) firms in industries where women-owned businesses are underrepresented or substantially underrepresented. Self-certification through SBA is free.",
  "sections": [
    {"heading": "WOSB vs. EDWOSB", "content": "There are two tiers of the WOSB program. The basic WOSB designation covers industries where women-owned businesses are underrepresented in federal contracting. The Economically Disadvantaged WOSB (EDWOSB) designation covers industries where they are substantially underrepresented and also requires demonstration of economic disadvantage.\n\nFor EDWOSB, the economically disadvantaged owner must have a personal net worth below $850,000 (excluding business equity and primary residence), adjusted gross income averaged over three years below $400,000, and total assets below $6.5 million — the same thresholds as 8(a).\n\nEDWOSB firms can compete for both EDWOSB and WOSB set-asides. WOSB-only firms can compete for WOSB set-asides but not EDWOSB set-asides. Check the solicitation carefully to confirm which designation is required."},
    {"heading": "Eligibility Requirements", "content": "To qualify as a WOSB, your business must be a small business under the applicable NAICS code, at least 51% unconditionally owned by one or more women who are U.S. citizens, and managed and controlled on a day-to-day basis by one or more of the women owners.\n\n'Control' is a key term — the women owners must hold the highest officer position, must not be subject to conditions that restrict their control, and must make both long-term decisions and day-to-day management decisions. Outside investors or male co-owners cannot hold veto rights over major business decisions.\n\nSelf-certification through certify.sba.gov is available. Third-party certification is no longer required since 2020. When you self-certify, you affirm your eligibility under penalty of law — false certifications can result in False Claims Act liability."},
    {"heading": "Eligible NAICS Codes", "content": "The WOSB program does not apply to every contract — it applies only in NAICS codes where women-owned businesses are underrepresented. SBA publishes a list of eligible NAICS codes, updated periodically based on federal contracting data.\n\nAs of the most recent update, there are over 700 eligible NAICS codes for WOSB set-asides and a subset for EDWOSB. Before pursuing WOSB set-asides, verify that your primary NAICS code is on the eligible list. If your code isn't on the list, your WOSB certification still has value for subcontracting goals and agency diversity initiatives, but you cannot compete in WOSB set-asides.\n\nThe eligible code list is available at sba.gov/contracting and is worth reviewing in detail — many businesses find that adjacent NAICS codes cover work they do are on the list even when their primary code isn't."},
    {"heading": "Contract Thresholds and Limitations", "content": "WOSB set-asides and sole-source awards have dollar thresholds. Set-asides can be used for contracts of any dollar value in eligible NAICS codes when the two-offer rule is met. Sole-source WOSB awards (no competition) are limited to $4.5 million for services and $7 million for manufacturing.\n\nUnder FAR 19.1505, a contracting officer may award a sole-source WOSB contract when the anticipated contract price will not exceed the applicable threshold, the WOSB is responsible, the contract can be awarded at a fair and reasonable price, and the requirement is in an eligible NAICS code.\n\nFor contracts above $4.5M/$7M in eligible NAICS codes, set-asides (competitive, restricted to WOSB) are still available — only the sole-source authority has a cap."}
  ],
  "key_facts": [
    "Self-certification through certify.sba.gov is free — third-party certification no longer required since 2020",
    "WOSB set-asides apply only in NAICS codes where women-owned businesses are underrepresented",
    "EDWOSB requires personal net worth below $850K, AGI below $400K, assets below $6.5M",
    "Sole-source authority up to $4.5M (services) and $7M (manufacturing) without competition",
    "EDWOSB firms can compete for both EDWOSB and WOSB set-asides"
  ],
  "common_mistakes": [
    "Not checking whether your NAICS code is on the WOSB eligible list before pursuing set-asides — not all codes qualify",
    "Allowing male co-owners or investors to hold veto rights — this can disqualify you even if ownership percentages are correct",
    "Forgetting to recertify after significant business changes — ownership transfers, new investors, or management changes can affect eligibility"
  ],
  "official_sources": [
    {"name": "SBA WOSB Program", "url": "https://www.sba.gov/federal-contracting/contracting-assistance-programs/women-owned-small-business-federal-contracting-program", "description": "Official program overview and eligible NAICS codes"},
    {"name": "certify.sba.gov", "url": "https://certify.sba.gov", "description": "Self-certification portal"},
    {"name": "13 CFR Part 127", "url": "https://www.ecfr.gov/current/title-13/chapter-I/part-127", "description": "Complete WOSB program regulations"}
  ]
},

"small-business/sdvosb": {
  "title": "Service-Disabled Veteran-Owned Small Business (SDVOSB)",
  "subtitle": "Set-asides and sole-source authority for businesses owned by veterans with service-connected disabilities.",
  "intro": "The Service-Disabled Veteran-Owned Small Business program provides federal contracting opportunities specifically for businesses owned by veterans with service-connected disabilities. SDVOSB firms have access to set-aside competitions, sole-source awards, and VA-specific contracting programs that represent billions in annual federal spending.",
  "sections": [
    {"heading": "SDVOSB vs. VOSB — Understanding the Difference", "content": "There are two veteran-owned business designations in federal contracting. The Veteran-Owned Small Business (VOSB) designation covers businesses owned by any veteran. The Service-Disabled Veteran-Owned Small Business (SDVOSB) designation is specifically for veterans with a service-connected disability rating from the VA or a disability determination from the Department of Defense.\n\nFor general federal contracting (non-VA agencies), only SDVOSB has set-aside and sole-source authority under FAR 19.14. VOSB designation alone does not provide set-aside access outside the VA.\n\nFor VA contracts specifically, both SDVOSB and VOSB have special access under the Veterans Benefits, Health Care, and Information Technology Act of 2006 (38 U.S.C. § 8127). VA contracting is discussed separately below."},
    {"heading": "Eligibility Requirements", "content": "To qualify as an SDVOSB, the business must be small under the applicable NAICS code, at least 51% owned by one or more service-disabled veterans, and managed and controlled on a day-to-day basis by a service-disabled veteran (or in the case of permanent and severe disability, the spouse or permanent caregiver).\n\nA service-connected disability must be acknowledged by the VA or determined by the Department of Defense. The disability rating percentage does not matter — any service-connected disability qualifies, including 0% ratings.\n\nSince January 2023, SBA certifies SDVOSB status for all federal contracts — the VA's Center for Verification and Evaluation (CVE) program was consolidated into SBA's certification process. Certification is free through certify.sba.gov."},
    {"heading": "Set-Aside and Sole-Source Authority", "content": "Contracting officers may set aside acquisitions for SDVOSB when there is a reasonable expectation of receiving offers from two or more responsible SDVOSB firms at fair and reasonable prices.\n\nSole-source SDVOSB awards (no competition) are available up to $4.5 million for services and $7 million for manufacturing. The contracting officer must determine that no other small business program is a better fit, and that the SDVOSB can perform the contract at a fair price.\n\nSDVOSB set-asides and sole-source authority are available across all federal agencies, not just the VA. Many SDVOSBs focus exclusively on VA contracting and miss significant opportunities at DoD, civilian agencies, and other major buyers."},
    {"heading": "VA-Specific Contracting: The Rule of Two", "content": "For VA contracts, the Veterans First Contracting Program under 38 U.S.C. § 8127 establishes a 'Rule of Two' that is more favorable than the general FAR rule. The VA must set aside acquisitions for SDVOSB or VOSB when there is a reasonable expectation of receiving two or more offers from verified firms.\n\nThe VA has specific goals for SDVOSB and VOSB participation and historically awards a higher percentage of its contracts to these businesses than other agencies. Healthcare services, IT, construction, and facilities management are all active areas for veteran-owned businesses at the VA.\n\nVA contracts also require that SDVOSB/VOSB firms perform at least 15% of the work under the contract (50% for construction under $1.5M) to prevent fronting arrangements where a non-veteran firm does all the work while the veteran firm holds the contract."}
  ],
  "key_facts": [
    "Any VA-recognized service-connected disability qualifies — including 0% ratings",
    "SBA now certifies SDVOSB status for all agencies — free at certify.sba.gov",
    "Sole-source authority up to $4.5M (services) and $7M (manufacturing) without competition",
    "VA's Rule of Two requires set-asides for SDVOSB/VOSB even more broadly than general FAR rules",
    "SDVOSB firms must perform at least 15% of VA contract work directly (50% for small construction)"
  ],
  "common_mistakes": [
    "Focusing only on VA contracting — SDVOSB set-aside authority applies government-wide; DoD and civilian agencies are major markets",
    "Not certifying through SBA since the January 2023 consolidation — self-certification alone is no longer sufficient",
    "Falling below the 15% performance requirement on VA contracts — this can result in contract termination and debarment"
  ],
  "official_sources": [
    {"name": "SBA SDVOSB Program", "url": "https://www.sba.gov/federal-contracting/contracting-assistance-programs/veteran-contracting-assistance-programs", "description": "Program overview and certification information"},
    {"name": "certify.sba.gov", "url": "https://certify.sba.gov", "description": "Free certification portal"},
    {"name": "VA Veterans First Contracting", "url": "https://www.va.gov/osdbu/", "description": "VA Office of Small and Disadvantaged Business Utilization"}
  ]
},

"small-business/mentor-protege": {
  "title": "Mentor-Protégé Program",
  "subtitle": "How small businesses partner with experienced contractors to build capability and win larger contracts.",
  "intro": "The SBA's All Small Mentor-Protégé Program allows small businesses to form joint ventures with larger, more experienced firms while still competing as small businesses. This program is one of the most powerful — and underused — paths for small businesses to grow their federal contracting footprint.",
  "sections": [
    {"heading": "How the Program Works", "content": "Under the All Small Mentor-Protégé Program (13 CFR Part 125.9), an approved mentor and protégé can form a joint venture and bid on federal contracts. The joint venture can qualify as small based on the protégé's size alone — even if the mentor is a large business that would normally disqualify the team.\n\nThis is the program's central benefit: it lets a small business bid on contracts that require capabilities they don't yet have, by bringing in a mentor who provides those capabilities through the joint venture. The contract goes to the JV, the protégé develops capabilities through performance, and both parties benefit.\n\nMentors receive credit toward their own small business subcontracting goals for work performed by the protégé. For large businesses with significant subcontracting plan obligations, finding a strong protégé is genuinely valuable — the relationship is mutually beneficial, not charity."},
    {"heading": "Eligibility and Application", "content": "Protégés must be small businesses (including 8(a), HUBZone, WOSB, or SDVOSB firms). There is no minimum time in business requirement, though newer businesses may face scrutiny on their potential for success.\n\nMentors must be for-profit businesses with the ability to provide beneficial assistance to the protégé. They need not be a large business — other small businesses can serve as mentors if they have relevant experience to offer.\n\nApplications are submitted to SBA through the SBA portal. Both parties submit a mentor-protégé agreement that outlines the technical, financial, and business development assistance the mentor will provide. SBA reviews and approves the agreement before the joint venture can be used on contracts.\n\nA protégé can have only one mentor at a time in the All Small program, though a protégé can simultaneously be in an agency-specific mentor-protégé program (DoD, DHS, DOE, and others have their own programs with different terms)."},
    {"heading": "Joint Venture Rules and Contract Eligibility", "content": "Once the mentor-protégé agreement is approved, the parties can form a joint venture entity specifically for pursuing federal contracts. The JV entity must be created for each procurement or can be a general purpose JV for multiple contracts.\n\nFor the JV to qualify as small, the protégé must be the managing partner and must be the partner that provides the primary control and performance of the contract. The protégé must perform at least 40% of the work on each contract. These requirements are strictly enforced to prevent large businesses from using the program to access small business set-asides without genuine small business participation.\n\nThe JV can compete on small business set-asides, 8(a) contracts (if the protégé is 8(a) certified), HUBZone contracts, WOSB contracts, and SDVOSB contracts — whatever programs the protégé qualifies for individually."},
    {"heading": "What Good Mentor-Protégé Relationships Look Like", "content": "The most successful mentor-protégé relationships are genuine partnerships, not just contracting vehicles. Mentors provide real value including: access to past performance references that strengthen the team's proposal, technical staff to supplement the protégé's capabilities, financial backing for bonding and working capital, business development connections to agency relationships, and management training and systems.\n\nProtégés who treat the relationship as a learning opportunity — rather than just a contract vehicle — emerge from the program genuinely stronger. Many successful mid-size government contractors trace their growth directly to a mentor-protégé relationship that gave them their first large prime contract opportunity.\n\nThe agreement must specify what the mentor will actually provide. SBA reviews these commitments during annual reviews and can terminate the agreement if the mentor is not fulfilling their obligations. Document the assistance you receive throughout the program."}
  ],
  "key_facts": [
    "Joint ventures qualify as small based on the protégé's size alone — mentor size doesn't disqualify the team",
    "Protégé must perform at least 40% of work on each contract — strictly enforced",
    "One mentor at a time in All Small program — but you can simultaneously participate in agency-specific programs",
    "Application submitted through SBA portal — both parties must sign the mentor-protégé agreement",
    "Mentors receive subcontracting plan credit for protégé work — making strong protégés valuable to large businesses"
  ],
  "common_mistakes": [
    "Choosing a mentor based on size rather than relevant expertise — the program's value is the knowledge and capability transfer, not just the contract vehicle",
    "Failing to document mentor assistance during the program — SBA annual reviews require evidence of actual assistance provided",
    "Not ensuring the protégé controls the JV and performs 40% of work — SBA size protests can unwind awards if these requirements aren't met"
  ],
  "official_sources": [
    {"name": "SBA Mentor-Protégé Program", "url": "https://www.sba.gov/federal-contracting/contracting-assistance-programs/all-small-mentor-protege-program", "description": "Official program overview and application information"},
    {"name": "13 CFR Part 125.9", "url": "https://www.ecfr.gov/current/title-13/chapter-I/part-125/section-125.9", "description": "Complete mentor-protégé program regulations"}
  ]
},

"awards/evaluation-process": {
  "title": "The Federal Contract Evaluation Process",
  "subtitle": "How proposals are scored, ranked, and awarded — and how to write a winning proposal.",
  "intro": "Understanding how federal agencies evaluate proposals is the single most important factor in improving your win rate. The evaluation process is more structured and transparent than most new contractors expect — every criterion is published in the solicitation, and knowing how to respond to each one is a learnable skill.",
  "sections": [
    {"heading": "Source Selection Procedures", "content": "Federal agencies use three main source selection procedures, each with different evaluation dynamics. Lowest Price Technically Acceptable (LPTA) — used when the requirement is well-defined and technical quality above a minimum threshold doesn't add value. Any technically acceptable offer wins on price alone. Tradeoff — a best value approach where technical quality is traded off against price. A higher-priced offer can win if the technical evaluation justifies the premium. FAR 15.101 governs these procedures.\n\nSimplified procedures — for contracts under $250,000, agencies use streamlined evaluation with less documentation. These are often more relationship-driven and faster-moving than full source selections.\n\nThe solicitation (RFP or RFQ) will always specify which procedure applies. 'Best value' language in a solicitation usually signals a tradeoff approach. 'Technically acceptable' combined with price evaluation signals LPTA."},
    {"heading": "Evaluation Factors and Their Weights", "content": "In tradeoff source selections, the solicitation must list all evaluation factors and their relative importance. Common factors include Technical Approach, Past Performance, Management Approach/Key Personnel, and Price/Cost.\n\nFAR 15.304 requires that the solicitation state whether all non-price factors combined are significantly more important than, approximately equal to, or significantly less important than price. This relative weighting tells you where to invest your proposal writing effort.\n\nWhen technical factors are significantly more important than price, a strong technical proposal can win even at a higher price. When price is approximately equal, you need both a strong technical proposal and competitive pricing. Read the evaluation factors carefully and allocate your proposal writing resources accordingly."},
    {"heading": "The Evaluation Team and Scoring", "content": "Most federal source selections use an evaluation board with separate technical and price evaluation teams. Technical evaluators score the non-price portions of proposals without seeing prices. Price/cost analysts evaluate pricing separately. A source selection authority (SSA) makes the final award decision based on both evaluations.\n\nTechnical evaluators typically use adjectival ratings (Outstanding, Good, Acceptable, Marginal, Unacceptable) or color ratings (Blue/Green/Yellow/Red) rather than numerical scores. Evaluators look for specific evidence that you understand the requirement and can perform it — not general capability statements.\n\nStrengths, weaknesses, and deficiencies drive the evaluation narrative. A strength is something that increases confidence in your proposal. A weakness reduces confidence. A deficiency is a failure to meet a mandatory requirement. Proposals with multiple strengths and no deficiencies are rated Outstanding or Blue."},
    {"heading": "Writing to the Evaluation Criteria", "content": "The most common mistake in federal proposals is writing about your company instead of writing to the evaluation criteria. Evaluators have a scoresheet with specific criteria — they are looking for evidence that your proposal satisfies each criterion, not reading your proposal for general impressions.\n\nFor each evaluation factor, identify exactly what the evaluators are looking for. Then provide specific, verifiable evidence that you meet each criterion. Use the language of the solicitation — if the evaluation asks for 'demonstrated experience managing multi-year service contracts,' use that exact phrase and then provide specific examples.\n\nPage limits are strictly enforced in most solicitations. Exceeding page limits can result in disqualification of your entire proposal. Formatting requirements — font size, margin width, file format — are equally strict. Read Section L (Instructions) and Section M (Evaluation Criteria) of every RFP before writing a single word of your proposal."},
    {"heading": "Debriefings: Learning from Every Outcome", "content": "After contract award, unsuccessful offerors are entitled to a debriefing under FAR 15.505 (pre-award) and FAR 15.506 (post-award). Request your debriefing within three days of notification for pre-award and within three days of award notification for post-award.\n\nDebriefings must include the government's evaluation of your proposal's significant strengths and weaknesses, the overall evaluated price and technical rating of the awardee (de-identified in some cases), and a summary of the rationale for award. They cannot include point-by-point comparison or information about other offerors' proposals.\n\nTreat every debriefing as intelligence for your next proposal. The specific language evaluators used to describe your weaknesses tells you exactly what to fix. Many contractors who lose initially become consistent winners after a few debriefings teach them how the agency thinks about the requirement."}
  ],
  "key_facts": [
    "FAR 15.304 requires solicitations to state whether technical factors are more/equal/less important than price",
    "Technical evaluators score proposals without seeing prices — price and technical are evaluated separately",
    "Request debriefings within 3 days of award notification — you are entitled to one under FAR 15.506",
    "Page limits and formatting requirements are strictly enforced — exceeding them can disqualify your proposal",
    "LPTA awards go to the cheapest technically acceptable offeror — not the best technical proposal"
  ],
  "common_mistakes": [
    "Writing about your company's general capabilities instead of responding directly to each evaluation criterion — evaluators have a scoresheet, not an open mind",
    "Not requesting debriefings after losses — the feedback is free intelligence worth more than any proposal writing course",
    "Ignoring Section L formatting requirements — proposals disqualified for page limit violations are common and entirely avoidable"
  ],
  "official_sources": [
    {"name": "FAR Part 15 — Contracting by Negotiation", "url": "https://www.acquisition.gov/far/part-15", "description": "Complete source selection regulations including evaluation procedures"},
    {"name": "FAR 15.304 — Evaluation Factors", "url": "https://www.acquisition.gov/far/15.304", "description": "Rules governing evaluation factor structure and weights"}
  ]
},

"awards/simplified-acquisition": {
  "title": "Simplified Acquisition Procedures",
  "subtitle": "Contracts under $250,000 have streamlined rules — and they're the best entry point for new contractors.",
  "intro": "Simplified acquisition procedures apply to most contracts valued between $10,000 and $250,000. These rules reduce documentation requirements, speed up award timelines, and create mandatory small business set-asides. For businesses new to federal contracting, simplified acquisitions are the fastest path to your first award.",
  "sections": [
    {"heading": "The Simplified Acquisition Threshold", "content": "The simplified acquisition threshold (SAT) is currently $250,000 for most acquisitions, established by the Federal Acquisition Streamlining Act and codified in FAR 2.101. Above this threshold, full and open competition rules apply with their more extensive documentation requirements. Below it, agencies have significantly more flexibility.\n\nFor contracts in support of contingency operations or for humanitarian and peacekeeping operations, the SAT increases to $1 million. For commercial items acquired using simplified procedures under FAR Part 13.5, the threshold extends to $7.5 million (and $15 million for commercial items). These higher thresholds are important to understand — many commercial service contracts that look like they exceed simplified acquisition procedures actually qualify.\n\nThe micro-purchase threshold — currently $10,000 — allows purchase card acquisitions with no competition requirement at all. Agencies can simply buy from any convenient source. Getting registered with agency purchase card programs can generate a steady flow of small orders."},
    {"heading": "Mandatory Small Business Set-Asides", "content": "Under FAR 19.502-2, acquisitions between $10,000 and $250,000 are automatically set aside for small businesses, without any determination required, as long as the contracting officer has a reasonable expectation of receiving offers from at least two responsible small businesses at competitive prices.\n\nThis automatic set-aside is one of the most important rules in federal contracting for small businesses. It means that for the vast majority of federal purchases in this range, large businesses simply cannot compete — the contracting officer is not permitted to award to them even if their offer is cheaper.\n\nThe 'reasonable expectation' standard is relatively easy to meet. If there are any qualified small businesses in the market (and there almost always are for routine services and supplies), the set-aside applies. Contracting officers who conduct perfunctory market research and conclude no small businesses can perform are acting inconsistently with the FAR."},
    {"heading": "Methods for Simplified Acquisitions", "content": "Purchase Orders (POs) are used for straightforward, one-time purchases. The contracting officer issues a PO, the vendor accepts it (often implicitly by performing), and payment follows delivery. Minimal paperwork, fast turnaround.\n\nBlanket Purchase Agreements (BPAs) are arrangements with multiple vendors to fill recurring needs. Agencies set up BPAs with pre-qualified vendors and then place individual calls against the BPA. Getting on a BPA is competitive initially but generates a steady stream of orders afterward.\n\nQuotations under FAR 13.106 can be requested informally. For purchases up to $25,000, agencies can obtain oral quotes. For $25,000-$250,000, written quotes are typical but less formal than full proposals. Evaluation is faster and less bureaucratic than full source selections."},
    {"heading": "Strategies for Winning Simplified Acquisitions", "content": "Relationships matter more in simplified acquisitions than in large formal procurements. Contracting officers have more discretion in who they contact for quotes and how they evaluate them. Being known, responsive, and reliable generates repeat business.\n\nCapability Statements are the business card of federal contracting. A one-page document summarizing your core competencies, differentiators, past performance highlights, and certifications — formatted for quick reading by a busy contracting officer — is the foundation of simplified acquisition business development.\n\nFor recurring needs, identify the specific agency program offices that buy your services. Program managers (not contracting officers) often identify vendors before the formal solicitation process begins. Getting your capability statement in front of the right program manager before a requirement is formally released can result in specifications written around your capabilities."},
    {"heading": "GSA Advantage and Government Purchase Cards", "content": "GSA Advantage is the federal government's online shopping platform for commercial products and some services. If you sell commercial products, getting on GSA Advantage through the Multiple Award Schedule (MAS) program gives purchase cardholders direct access to your catalog.\n\nThe government purchase card (GPC) program processes millions of micro-purchases annually. Agencies use purchase cards for anything under $10,000 without competition. If you provide routine services or supplies that agencies buy repeatedly, marketing to purchase cardholders (identified through agency contact directories) can generate significant revenue with minimal proposal effort.\n\nGSA's MAS program is also worth considering for service businesses. Schedule holders get visibility in GSA Advantage searches and eBuy (the electronic quoting system for Schedule orders), and many agencies prefer buying from Schedule holders because it simplifies their acquisition process."}
  ],
  "key_facts": [
    "Simplified acquisition threshold is $250,000 — streamlined rules apply below this amount",
    "Acquisitions $10K-$250K are automatically set aside for small businesses under FAR 19.502-2",
    "Micro-purchases under $10,000 require no competition — agencies can buy from any convenient source",
    "Commercial items can use simplified procedures up to $7.5M under FAR Part 13.5",
    "Oral quotes are acceptable for purchases up to $25,000 — formal proposals not required"
  ],
  "common_mistakes": [
    "Ignoring the simplified acquisition market because the dollar values seem small — $250,000 × 10 contracts = $2.5M, which is significant revenue and builds past performance",
    "Not developing a one-page capability statement — this is the standard business development tool for simplified acquisitions and most contracting officers expect to receive them",
    "Failing to market to program managers before formal solicitation — in simplified acquisitions, relationships formed before the RFQ is issued often determine who gets called for quotes"
  ],
  "official_sources": [
    {"name": "FAR Part 13 — Simplified Acquisition Procedures", "url": "https://www.acquisition.gov/far/part-13", "description": "Complete rules governing simplified acquisitions"},
    {"name": "GSA Advantage", "url": "https://www.gsaadvantage.gov", "description": "Federal online shopping platform — relevant for product vendors"},
    {"name": "GSA MAS Program", "url": "https://www.gsa.gov/buy-through-us/purchasing-programs/gsa-multiple-award-schedule", "description": "Multiple Award Schedule program for commercial products and services"}
  ]
},

"incumbents/recompete-basics": {
  "title": "Recompete Basics",
  "subtitle": "When existing contracts expire, agencies must resolicit — creating regular opportunities for new competitors.",
  "intro": "Every federal contract has a defined period of performance. When that period ends, the agency generally must recompete the requirement rather than simply extending the existing contract. Understanding how recompetes work — and how to position for them — is one of the highest-leverage strategies in federal business development.",
  "sections": [
    {"heading": "Why Recompetes Happen", "content": "Federal contracting law (41 U.S.C. § 3301) generally requires full and open competition for contract awards. Long-term sole-source contracts are exceptions that require specific justification. As a result, most federal service contracts are structured with a base period and option years — typically a 1-year base with four 1-year options, for a maximum five-year ordering period.\n\nWhen the option years are exhausted, the agency must either issue a new competitive solicitation, find a justification to extend the existing contract (which has strict limits), or go without the service. In practice, most agencies issue a new competitive solicitation — a recompete.\n\nAgencies also recompete requirements when they want to restructure the scope of work, when the incumbent has performed poorly, when budget constraints require a lower-cost solution, or when policy changes (like new small business set-aside goals) require a different contracting approach."},
    {"heading": "The Recompete Timeline", "content": "The federal acquisition process takes time — a typical service contract recompete takes 6-18 months from the initial planning phase to contract award. Understanding where an agency is in this timeline determines your business development window.\n\nPhase 1 (12-18 months before expiration): Agency begins acquisition planning. Requirements documents are drafted. Market research is conducted. This is your window to influence the scope of work before it's locked in.\n\nPhase 2 (6-12 months before expiration): Draft RFP or Request for Information (RFI) may be released. Industry days and pre-solicitation conferences occur. This is when agencies signal what they're looking for.\n\nPhase 3 (3-6 months before expiration): Final RFP released. Proposals due. Evaluation occurs. Award made.\n\nPhase 4: Transition period. The new contractor takes over from the incumbent. Typically 30-90 days depending on complexity.\n\nThe Recompete Radar's RPS scoring is calibrated to this timeline — contracts with expiration dates within 90 days have the highest scores because the solicitation may already be in preparation or even released."},
    {"heading": "Finding Recompete Opportunities", "content": "Several data sources help identify upcoming recompetes. USAspending.gov shows contract award dates, performance periods, and total amounts. Contracts with period of performance end dates approaching are recompete candidates.\n\nFPDS-NG (Federal Procurement Data System — Next Generation) provides more detailed modification history and option year data. SAM.gov posts presolicitation notices for upcoming competitive acquisitions — these often appear 15-30 days before the formal RFP release.\n\nGovCon Intelligence Terminal's Recompete Radar aggregates this data and adds RPS scoring — surfacing the highest-priority opportunities first. Filtering by your NAICS code and target states reduces the list to relevant opportunities.\n\nFor the most targeted intelligence, identify the specific agency offices you want to work with, find their current contracts, and track those specific contracts through their option year cycles."},
    {"heading": "Incumbent Advantage and How to Overcome It", "content": "Incumbents enjoy significant advantages in recompetes. They have deep knowledge of the agency's actual requirements (not just what's written in the statement of work). They have established relationships with program managers, contracting officers, and end users. They have documented past performance at that specific agency. And they often have transition risk working in their favor — agencies are reluctant to disrupt ongoing operations.\n\nStudies of federal contract recompetes show that incumbents win approximately 50-70% of recompetes they compete in. However, that means 30-50% of recompetes go to new contractors — a significant opportunity flow.\n\nTo overcome incumbent advantage, challengers need superior pricing (covering a lower cost base), a genuinely differentiated technical approach that addresses known weaknesses in the incumbent's performance, or a relevant small business certification that creates a set-aside the incumbent doesn't qualify for. Doing thorough research on the incumbent's performance record — including any contract modifications, complaints, or performance issues — is essential."}
  ],
  "key_facts": [
    "Most federal service contracts have a 5-year maximum period (1-year base + four 1-year options) before recompete",
    "Recompete acquisition process typically takes 6-18 months from planning to award",
    "Incumbents win approximately 50-70% of recompetes they compete in — 30-50% go to new contractors",
    "SAM.gov presolicitation notices appear 15-30 days before formal RFP release",
    "FAR 6.302 governs the limited exceptions allowing agencies to extend without competition"
  ],
  "common_mistakes": [
    "Starting business development after the RFP is released — by then the scope is locked and relationships are set; start 12-18 months out",
    "Not researching incumbent performance before bidding — contract modifications and performance issues are public record and can be the basis for your differentiated approach",
    "Assuming incumbents always win — they win most of the time, but the 30-50% that change contractors represent billions in annual opportunity"
  ],
  "official_sources": [
    {"name": "USAspending.gov", "url": "https://www.usaspending.gov", "description": "Search contract awards and performance periods to identify recompete candidates"},
    {"name": "SAM.gov Opportunities", "url": "https://sam.gov/content/opportunities", "description": "Pre-solicitation and solicitation notices for upcoming competitions"},
    {"name": "FAR Part 6 — Competition Requirements", "url": "https://www.acquisition.gov/far/part-6", "description": "Rules governing when and how agencies must compete requirements"}
  ]
},

"incumbents/full-open-competition": {
  "title": "Full and Open Competition",
  "subtitle": "The legal requirement that governs when and how federal agencies must compete contracts.",
  "intro": "The Competition in Contracting Act (CICA) of 1984 established that federal contracts must generally be awarded through full and open competition. This is not just good policy — it's law. Understanding when competition is required, when it can be limited, and when exceptions apply is essential for both bidding on contracts and challenging improper sole-source awards.",
  "sections": [
    {"heading": "The Competition in Contracting Act", "content": "CICA (41 U.S.C. §§ 3301-3309) requires federal executive agencies to obtain full and open competition through the use of competitive procedures. The implementing regulations are in FAR Part 6.\n\nFull and open competition means that all responsible sources are permitted to submit sealed bids or competitive proposals. This is the default — agencies must compete unless a specific, documented exception applies.\n\nThe law also established the Office of Federal Procurement Policy (OFPP) within OMB to oversee competition policy across the federal government, and it strengthened GAO's bid protest authority to give disappointed bidders meaningful recourse when competition requirements are violated."},
    {"heading": "The Seven Exceptions to Full and Open Competition", "content": "FAR 6.302 enumerates the only seven circumstances under which an agency may award a contract without full and open competition:\n\n1. Only one responsible source — the item/service is available only from a single source (FAR 6.302-1)\n2. Unusual and compelling urgency — delay would seriously jeopardize national security, mission, or property (FAR 6.302-2)\n3. Industrial mobilization — maintaining a facility for national defense (FAR 6.302-3)\n4. International agreement — required by treaty or executive agreement (FAR 6.302-4)\n5. Authorized by statute — a specific law authorizes the use of a particular source (FAR 6.302-5)\n6. National security — disclosure of agency needs would compromise security (FAR 6.302-6)\n7. Public interest — agency head determines full competition is not in the public interest (FAR 6.302-7)\n\nEach exception requires a written Justification and Approval (J&A) document. For exceptions 1 and 2, the J&A must be signed by the contracting officer for amounts up to $750,000, by a competition advocate for $750,000-$15M, by the head of the contracting activity for $15M-$75M, and by the agency senior procurement executive for amounts exceeding $75M."},
    {"heading": "Publicizing J&As and Your Right to Respond", "content": "J&As for sole-source awards above $25,000 must be posted on SAM.gov for at least 15 days before the contract is awarded (with some exceptions for urgency). This posting requirement exists specifically to allow the public — including potential competitors — to review the justification and identify factual errors.\n\nIf you find a J&A where the agency has incorrectly concluded that only one source exists, or where the urgency claim appears manufactured or avoidable, you can submit a response during the 15-day window. A well-documented response that identifies your capability to perform the requirement can prompt the agency to reconsider the sole-source and issue a competitive solicitation.\n\nEven after award, you can file a bid protest with GAO challenging an improper sole-source award if you learn of it within 10 days of when you knew or should have known the basis for protest."},
    {"heading": "Limited Competition: Set-Asides and Other Restrictions", "content": "Full and open competition doesn't always mean unrestricted competition. FAR 6.203 establishes that 'full and open competition after exclusion of sources' is still full and open competition under CICA. This is the legal basis for small business set-asides.\n\nWhen a contract is set aside for small businesses, the competition is full and open among all small businesses — but large businesses are excluded. This is legal because the excluded sources (large businesses) don't meet the statutory criteria for the set-aside program.\n\nOther forms of limited competition include: competitions restricted to firms holding specific security clearances, competitions limited to firms in certain geographic areas (rare), and competitions on GWACs and IDIQs restricted to existing contract holders. Each of these requires appropriate documentation."}
  ],
  "key_facts": [
    "CICA (41 U.S.C. §§ 3301-3309) requires full and open competition as the default for all federal contracts",
    "Only seven exceptions exist under FAR 6.302 — agencies cannot invent new exceptions",
    "J&As for sole-source awards above $25,000 must be posted on SAM.gov for 15 days before award",
    "J&A signature authority escalates with dollar value — $75M+ requires agency senior procurement executive",
    "Small business set-asides are legally full and open competition among eligible sources under CICA"
  ],
  "common_mistakes": [
    "Not monitoring SAM.gov J&A postings — reviewing these in your target market regularly can identify improperly justified sole-source awards you can challenge",
    "Assuming sole-source means you can't compete — the 15-day comment period exists to challenge incorrect factual claims in J&As",
    "Conflating 'full and open' with 'unrestricted' — small business set-asides are fully compliant with CICA"
  ],
  "official_sources": [
    {"name": "FAR Part 6 — Competition Requirements", "url": "https://www.acquisition.gov/far/part-6", "description": "Complete competition requirement regulations and exceptions"},
    {"name": "41 U.S.C. § 3301 — CICA", "url": "https://uscode.house.gov/view.xhtml?req=granuleid:USC-prelim-title41-section3301", "description": "The Competition in Contracting Act statutory text"},
    {"name": "SAM.gov J&A Postings", "url": "https://sam.gov/content/opportunities", "description": "Search for justification and approval notices in your target market"}
  ]
},

"incumbents/reading-incumbent-history": {
  "title": "Reading the Incumbent's Contract History",
  "subtitle": "How to research who holds a contract, what they were paid, and how to use that intelligence to bid smarter.",
  "intro": "Before investing time and resources in a proposal, understanding the incumbent contractor's history gives you a significant advantage. Federal contract award data is public record — performance information, modification history, and award amounts are all available through free government databases. Knowing how to read this data is a core competitive skill.",
  "sections": [
    {"heading": "USAspending.gov: Your Primary Research Tool", "content": "USAspending.gov is the authoritative source for federal contract award data, funded by the DATA Act (31 U.S.C. § 6101). Every federal contract award, modification, and delivery order above the micro-purchase threshold must be reported here.\n\nTo research an incumbent, search by contractor name, agency, and NAICS code. The award detail pages show: total obligated amount, base award amount, modification history (critical for understanding scope changes), period of performance, place of performance, and the contracting office responsible for the award.\n\nThe 'transaction history' for a contract shows every modification — each time the period was extended, scope was changed, or amount was increased. A contract with many modifications can signal several things: the agency added work because the incumbent performed well; the scope changed unexpectedly; or the award was originally underpriced and required amendments to cover actual costs."},
    {"heading": "FPDS-NG: Deeper Contract Intelligence", "content": "The Federal Procurement Data System — Next Generation (FPDS-NG) at fpds.gov provides more detailed data than USAspending.gov, including individual delivery orders on IDIQ contracts, PSC codes, competition type codes, and contractor business type data.\n\nFPDS allows you to search by agency, contractor, NAICS, PSC, and date range. The competition type code is particularly useful — it tells you whether the contract was awarded through full and open competition, a small business set-aside, a sole-source award, or other mechanisms. A history of sole-source awards suggests the agency has a strong preference for continuity, while a history of competitive awards suggests they're willing to switch vendors.\n\nFor IDIQ contracts, FPDS shows individual task/delivery orders. This lets you see how much work the incumbent actually received versus the maximum contract ceiling — a contract with a $50M ceiling but only $5M in actual orders tells a different story than one where the ceiling was fully utilized."},
    {"heading": "SAM.gov: Solicitation and Past Notices", "content": "SAM.gov archives all solicitations, including those for contracts already awarded. Searching the award history for a specific contract requirement can reveal the original solicitation document, which contains the statement of work, evaluation criteria, and technical requirements.\n\nReading the original solicitation tells you what the agency valued when they made the award. If you can obtain the winning proposal (through FOIA), you can understand exactly what the incumbent promised. Comparing promises to actual performance (using modification history as a proxy) can reveal gaps you can address in your proposal.\n\nSAM.gov also shows the complete solicitation history — draft RFPs, amendments, and questions and answers. Q&A sections are particularly valuable because they reveal what other potential competitors were confused about or concerned with, helping you anticipate evaluation nuances."},
    {"heading": "Performance Information: CPARS and Past Performance", "content": "The Contractor Performance Assessment Reporting System (CPARS) is the federal government's official past performance database. Ratings are assigned by contracting officers and program managers after each contract evaluation period.\n\nUnder FAR 42.1503, agencies must evaluate contractor performance annually on contracts over $250,000 and at contract completion. CPARS ratings use a five-level scale: Exceptional, Very Good, Satisfactory, Marginal, and Unsatisfactory.\n\nCPARS ratings are not publicly accessible — only the contractor and federal government users can view them. However, if the incumbent has received poor ratings, the contracting officer knows this and it affects their openness to switching contractors. Indirect evidence of performance problems includes: contract modifications extending performance periods (suggesting delays), decrement modifications reducing scope, and agency personnel turnover in key program positions.\n\nYou can access past performance information on yourself and, in some cases, through formal FOIA requests for contractor performance narratives that have been redacted."},
    {"heading": "Building Your Incumbent Intelligence Dossier", "content": "For any serious recompete pursuit, build a systematic dossier on the incumbent. Start with USAspending for award amounts and modification history. Check FPDS for competition codes and delivery order volumes. Review SAM.gov for the original solicitation and any pre-solicitation notices for the upcoming recompete. Search news archives for any coverage of contract performance. Check court records for any bid protests the incumbent has faced.\n\nLinkedIn is underused as a competitive intelligence tool. The incumbent's employees who work on the contract often list it on their profiles. This tells you their team structure, key personnel, and staffing approach — all useful for writing a more targeted proposal.\n\nThe GovCon Intelligence Terminal's contract detail pages aggregate much of this data — incumbent name, award amounts, lobbying connections, and politician links — in one place. The 'Prepare Bid Brief' feature synthesizes this into a competitive intelligence document that includes pricing strategy and positioning recommendations."}
  ],
  "key_facts": [
    "USAspending.gov is required by the DATA Act to publish all federal contract awards above micro-purchase threshold",
    "FPDS competition type codes reveal whether contracts were awarded via full and open, set-aside, or sole-source",
    "CPARS past performance ratings are not publicly accessible — only the contractor and federal users can view them",
    "FAR 42.1503 requires annual performance evaluations on contracts over $250,000",
    "Contract modifications are public record and signal scope changes, performance issues, or cost growth"
  ],
  "common_mistakes": [
    "Only looking at total contract value without checking actual task order history — IDIQ ceilings are not guaranteed revenue",
    "Ignoring modification history — a contract with many modifications often reveals more about agency satisfaction than the base award documents",
    "Not searching the original solicitation on SAM.gov — the statement of work and evaluation criteria from the original award tell you exactly what the agency valued"
  ],
  "official_sources": [
    {"name": "USAspending.gov", "url": "https://www.usaspending.gov", "description": "Primary source for all federal contract award and modification data"},
    {"name": "FPDS-NG", "url": "https://www.fpds.gov", "description": "Detailed federal procurement data including competition codes and delivery order history"},
    {"name": "SAM.gov", "url": "https://sam.gov/content/opportunities", "description": "Historical and current solicitations including original RFPs and Q&A"}
  ]
},

"incumbents/price-advantage": {
  "title": "Price as Your Competitive Advantage",
  "subtitle": "How small businesses can compete on price against large incumbents — and win.",
  "intro": "In the $100,000-$10 million contract range, price is often the decisive factor when technical proposals are comparably scored. Small businesses have structural cost advantages over large contractors that, properly leveraged, make competitive pricing achievable without sacrificing profitability.",
  "sections": [
    {"heading": "Understanding Contractor Cost Structures", "content": "Federal contractors price their work based on direct costs (labor, materials, travel) plus indirect rates. Indirect rates include fringe benefits (typically 25-40% of direct labor), overhead (facility costs, equipment, management allocated to contracts), G&A (General and Administrative costs — executive salaries, business development, accounting), and profit/fee.\n\nLarge contractors typically carry total indirect rates of 100-200% of direct labor costs — meaning for every $100 in direct labor, they add $100-200 in indirect charges. Small businesses with lean operations and lower overhead can often achieve total burden rates of 60-100% on direct labor.\n\nThis structural difference is your fundamental pricing advantage. A senior IT professional might cost $100/hour in direct labor. At a 150% indirect rate, the large contractor bills $250/hour. At an 80% indirect rate, you bill $180/hour — and can still maintain a healthy margin. That $70/hour difference on a 5-person team over a year is over $700,000."},
    {"heading": "Cost Analysis Before Bidding", "content": "The first step to competitive pricing is knowing your actual costs. Many small businesses underprice by failing to account for all costs, then discover they're performing work at a loss. Equally, many overprice by not understanding their competitive rate structures.\n\nBefore pricing any significant bid, calculate your fully loaded labor rates: direct labor + fringe + overhead + G&A + profit. Your fringe rate covers payroll taxes, benefits, and paid leave. Your overhead rate covers costs associated with performing contracts (supervisors, facilities, equipment). Your G&A rate covers enterprise-wide costs (executive compensation, accounting, legal, BD).\n\nCompare your fully loaded rates to market rates for the same labor categories. If your rates are significantly higher than market, you have a cost structure problem that needs to be solved before bidding, not after losing. If your rates are below market, you have genuine pricing power."},
    {"heading": "Should Cost Analysis of Incumbents", "content": "When bidding against an incumbent, estimate what the incumbent is actually paying to perform the work. This 'should cost' analysis tells you where your pricing needs to land to win.\n\nStart with the contract modification history on USAspending.gov — total obligated amounts, modification patterns, and delivery order volumes. Divide total obligated amounts by the number of years in the contract to get approximate annual revenue.\n\nThen estimate the incumbent's staffing model. Job postings from the incumbent on LinkedIn or Indeed (search the contract name or agency) often reveal exactly how many people they have on the contract and at what skill levels. Multiply headcount by estimated fully loaded rates to estimate their cost base. Add their typical profit margin (usually 8-12% on cost-plus contracts, variable on fixed-price) to estimate their bid price range.\n\nIf you can perform the work at a meaningfully lower cost structure — through automation, better staffing efficiency, or lower indirect rates — you have a genuine competitive advantage, not just a hope."},
    {"heading": "Price-to-Win Strategies", "content": "Price-to-Win (PTW) analysis is a structured methodology for determining the bid price most likely to win a competition. It combines your internal cost analysis with external intelligence about competitor pricing.\n\nFor LPTA competitions, your target is simple: the lowest price that still covers your costs and yields acceptable profit. Any bid below the second-lowest LPTA offer wins. The discipline is knowing your cost floor and not bidding below it.\n\nFor tradeoff competitions, price optimization is more complex. If your technical score is likely to be superior, you can afford a higher price — the evaluators will trade off the technical premium against the price premium. If your technical scores are likely to be comparable to competitors, price becomes more decisive.\n\nThe government publishes Independent Government Cost Estimates (IGCEs) as part of some solicitations or through FOIA requests. If you can obtain the IGCE, you know the government's budget benchmark — bids near or below the IGCE are more likely to be found fair and reasonable."},
    {"heading": "Fixed Price vs. Cost-Plus: Strategic Considerations", "content": "Contract type significantly affects pricing strategy. Fixed-price contracts pay a set amount regardless of your actual costs — cost overruns are your problem, cost savings are your gain. Cost-plus contracts reimburse your actual costs plus a fee — you need to manage costs for efficiency but aren't at risk for cost overruns.\n\nSmall businesses often prefer fixed-price contracts because they provide cost certainty and because their lean organizations typically perform more efficiently than large contractors — generating profit through efficiency rather than volume.\n\nFor new requirements where costs are uncertain, cost-plus is less risky. For well-defined requirements where you've done similar work, fixed-price lets you capture the efficiency gains from your lean operation.\n\nTime-and-materials (T&M) contracts pay hourly rates for labor plus material costs. T&M is appropriate when the scope can't be defined in advance. For T&M competitions, your labor rates are directly compared — this is where your lower indirect rates translate most directly into competitive advantage."}
  ],
  "key_facts": [
    "Large contractors typically carry total indirect rates of 100-200% of direct labor — small businesses often achieve 60-100%",
    "IGCEs (Independent Government Cost Estimates) can sometimes be obtained through FOIA and reveal the government's budget benchmark",
    "FAR 15.404 requires contracting officers to determine that offered prices are fair and reasonable before award",
    "Labor rates on T&M contracts are directly compared — lower overhead rates translate immediately to competitive advantage",
    "Fringe + overhead + G&A typically add 100-150% to direct labor costs in federal contracting"
  ],
  "common_mistakes": [
    "Undercutting without calculating your cost floor — winning a contract you can't profitably perform damages your past performance rating and your business",
    "Not building your indirect rates formally — many small businesses lose bids because their pricing is inconsistent; formal rate structures enable consistent, competitive pricing",
    "Ignoring the contract type — pricing a cost-plus contract the same way as a fixed-price contract leads to either uncompetitive bids or unintended risk"
  ],
  "official_sources": [
    {"name": "FAR Part 15.4 — Contract Pricing", "url": "https://www.acquisition.gov/far/subpart-15.4", "description": "Regulations governing price analysis and determination of fair and reasonable price"},
    {"name": "USAspending.gov", "url": "https://www.usaspending.gov", "description": "Incumbent contract history for should-cost analysis"},
    {"name": "DCAA Contract Audit Manual", "url": "https://www.dcaa.mil/Guidance/CAM/", "description": "Defense Contract Audit Agency guidance on contractor cost accounting"}
  ]
},

"subcontracting/approaching-primes": {
  "title": "How to Approach Prime Contractors",
  "subtitle": "Prime contractors are required to work with small businesses — here's how to get on their radar.",
  "intro": "Prime contractors with federal contracts over $750,000 must submit small business subcontracting plans with specific goals. They need small business partners to meet legal requirements — not as a favor, but as a contractual obligation. Understanding how to position yourself as a valuable subcontractor is one of the fastest paths to federal revenue.",
  "sections": [
    {"heading": "Why Primes Need You", "content": "Under FAR 19.7, prime contractors on contracts over $750,000 ($1.5 million for construction) must submit subcontracting plans with dollar and percentage goals for small business, small disadvantaged business (SDB), women-owned small business, HUBZone small business, veteran-owned small business, and service-disabled veteran-owned small business subcontracting.\n\nThese goals are negotiated with the contracting officer and become contractual commitments. Primes that fail to make good faith efforts to meet their subcontracting goals face liquidated damages of $10,000 to $100,000 per violation. They must also submit annual subcontracting plan reports (SF-294/295) that agencies use to evaluate their performance.\n\nThis creates genuine demand for qualified small business subcontractors. A prime with a $10M contract and a 30% small business subcontracting goal needs to funnel $3M to small businesses. Finding qualified, reliable small business partners is a real business need — not a charity exercise."},
    {"heading": "The Capability Statement: Your Business Card", "content": "The standard tool for approaching prime contractors is a one-page capability statement. Every prime contractor business development professional expects to receive them, and a well-formatted one gets read where a generic email doesn't.\n\nA strong capability statement includes: your core competencies (specific, not generic — 'cybersecurity vulnerability assessments for DoD networks' not 'IT services'), your differentiators (what makes you better than alternatives), past performance highlights (3-5 contracts with agencies and dollar amounts), your certifications and registrations (SAM, CAGE code, relevant small business certifications), key personnel (names and credentials of technical leads), and contact information.\n\nFormat matters. One page, designed for quick reading, with your certifications and NAICS codes prominently displayed. Primes scan these looking for the specific certification they need for a specific subcontracting plan goal. Make it easy to see at a glance that you have the 8(a) or WOSB designation they're looking for."},
    {"heading": "Finding the Right Primes to Target", "content": "Not every prime contractor is the right target. Focus on primes that hold contracts in your NAICS codes with agencies you want to work with — the overlap between their existing work and your capabilities is where subcontracting opportunities live.\n\nUSAspending.gov shows which prime contractors have the largest contract volumes at your target agencies. Filter by NAICS code and agency to find the top 10-20 primes in your market. These are your priority targets.\n\nThe SBA SubNet database (web.sba.gov/subnet) is the official channel where primes post subcontracting opportunities. Check it regularly and respond to opportunities that match your capabilities. Many primes prefer to find subcontractors through SubNet because it provides documentation of their outreach efforts.\n\nBid board tracking — monitoring SAM.gov for large prime contract awards in your space — tells you which primes just won large contracts and will now need to staff their subcontracting plans. A prime that just won a $50M contract with a 30% SB goal has $15M to place with small businesses over the contract term."},
    {"heading": "The Teaming Agreement: Formalizing the Relationship", "content": "When you've identified a prime that wants to pursue a specific opportunity with you as a subcontractor, the relationship is formalized through a Teaming Agreement. This document defines the scope of work each party will perform, the general price relationship (prime's rate vs. your rate), exclusivity (can you team with other primes on the same bid?), and what happens if the prime wins — and if they don't.\n\nTeaming agreements are governed by commercial contract law, not FAR. They are legally binding and should be reviewed carefully. Key terms to watch: exclusivity provisions that prevent you from teaming with competitors on the same opportunity, workshare guarantees (what percentage of the work are you actually guaranteed if they win?), and dispute resolution provisions.\n\nSome primes use 'template' teaming agreements that are heavily favorable to them. Negotiate workshare percentages explicitly — a verbal commitment to 'significant work' is not enforceable. Get the specific scope and dollar amount in writing.\n\nIf the prime wins and tries to reduce your workshare, you have recourse under the teaming agreement. Document all commitments made during the pursuit phase."},
    {"heading": "Building Long-Term Prime Relationships", "content": "The most valuable subcontracting relationships are recurring, not one-time. A prime that trusts your quality and reliability will bring you into bids again and again — and over time, will introduce you to their agency relationships directly.\n\nQuality performance is the foundation. Hit your deadlines. Communicate proactively about problems. Deliver work that makes the prime look good to the agency. A subcontractor who makes problems disappear gets more work. One who creates problems gets replaced.\n\nBe proactive about opportunities. When you see an upcoming recompete in your space, contact your prime relationships early. Bring them intelligence about the opportunity — the incumbent situation, the agency's pain points, your solution approach. Primes value partners who do business development, not just execution.\n\nOver time, the goal is to convert these relationships into joint venture or subcontracting arrangements that allow you to develop prime contracting capabilities of your own. The most successful trajectory in federal contracting is: subcontractor → preferred subcontractor → JV partner → prime contractor."}
  ],
  "key_facts": [
    "Primes on contracts over $750,000 must submit subcontracting plans with specific small business goals",
    "Failure to make good faith efforts on subcontracting plans results in liquidated damages of $10,000-$100,000 per violation",
    "SBA SubNet (web.sba.gov/subnet) is the official channel where primes post subcontracting opportunities",
    "Teaming agreements are commercial contracts — workshare percentages must be specified in writing",
    "SF-294/295 subcontracting reports are filed annually — agencies use them to evaluate prime performance"
  ],
  "common_mistakes": [
    "Generic capability statements with no specific certifications prominently displayed — primes scan for the specific designation they need; make yours immediately visible",
    "Accepting vague workshare commitments in teaming agreements — get specific scope and dollar amounts in writing or the commitment is unenforceable",
    "Approaching primes without research — know their current contract portfolio, their subcontracting plan goals, and the specific opportunity you're targeting before making contact"
  ],
  "official_sources": [
    {"name": "SBA SubNet", "url": "https://web.sba.gov/subnet/", "description": "Prime contractor subcontracting opportunity postings"},
    {"name": "FAR 19.7 — The Small Business Subcontracting Program", "url": "https://www.acquisition.gov/far/subpart-19.7", "description": "Complete subcontracting plan requirements"},
    {"name": "USAspending.gov", "url": "https://www.usaspending.gov", "description": "Find prime contractors with large contract volumes at your target agencies"}
  ]
},

"subcontracting/sub-vs-prime": {
  "title": "Subcontracting vs. Prime Contracting",
  "subtitle": "The strategic trade-offs between starting as a subcontractor and pursuing prime contracts directly.",
  "intro": "New federal contractors often face a fundamental strategic choice: pursue prime contracts directly, or start as a subcontractor and build from there. Both paths have merit, and the right answer depends on your current capabilities, certifications, risk tolerance, and business development resources.",
  "sections": [
    {"heading": "The Case for Starting as a Subcontractor", "content": "Subcontracting offers lower barriers to entry. You don't need an active SAM registration, a full proposal team, or past performance to work as a subcontractor — though all of these help. The prime handles contract administration, invoicing, and compliance. You focus on performance.\n\nCritically, subcontracting builds the federal past performance references you need to become a competitive prime contractor. When a contracting officer evaluates your proposal's past performance section, they want to see relevant federal contracts — and subcontracts count if they are comparable in size, scope, and complexity.\n\nThe risk profile is also lower. Subcontractors generally have more predictable revenue (the prime bears schedule and cost risk on fixed-price contracts), simpler compliance requirements (you're not directly bound by the prime contract's reporting and certification requirements), and more flexibility to manage your workforce."},
    {"heading": "The Limitations of Subcontracting", "content": "Subcontracting has real limitations that become more significant as your business grows. Margins are lower — the prime marks up your work before billing the government. On cost-plus contracts, your loaded rates are passed through at cost; on fixed-price contracts, the prime captures the margin between your rate and their billing rate.\n\nYou have no direct agency relationship. Your customer is the prime, not the government. When the prime's contract ends or they change their teaming strategy, your revenue stream can evaporate. The agency doesn't know you — they know the prime.\n\nCompliance risk is real. As a subcontractor, you're bound by FAR flowdown clauses — provisions in the prime contract that the prime must flow down to their subcontractors. These include Equal Opportunity, Buy American, DFARS cyber security requirements, and many others. Non-compliance can expose you to termination and, in some cases, False Claims Act liability."},
    {"heading": "When to Transition to Prime Contracting", "content": "The right time to pursue prime contracts is when you have: demonstrated past performance on two or more relevant subcontracts, an active SAM registration with appropriate NAICS codes, a small business certification (at minimum) or preferably a higher-value certification like 8(a) or WOSB, and the administrative infrastructure to handle contract management, invoicing, and compliance.\n\nThe simplified acquisition market ($10,000-$250,000) is the best entry point for first prime contracts. These competitions are less competitive, have streamlined evaluation, and mandatory small business set-asides. A first prime contract in this range builds the foundation for larger pursuits.\n\nDon't wait until you have 'everything figured out.' Many small businesses stay in subcontracting long past the point where prime contracting would be more profitable because the transition feels risky. The risk is manageable — start with small prime contracts in your strongest NAICS codes while continuing subcontracting relationships."},
    {"heading": "Running Both in Parallel", "content": "The most successful path is often both simultaneously. Continue performing as a subcontractor on large contracts where you build relationships and past performance, while actively pursuing prime contracts in the simplified acquisition range where you can win without extensive past performance.\n\nThis parallel approach provides revenue stability (subcontracting income while you build prime contracting capacity), accelerated past performance development (both prime and sub contracts improve your past performance record), and relationship development (primes you work with are also potential references and teaming partners for your prime pursuits).\n\nManage the potential conflict carefully. If you're subcontracting to a prime that will compete against you on a recompete, your proprietary information about the agency's needs and the prime's capabilities is sensitive. Maintain clear internal separation between your subcontracting performance and your prime BD activities."},
    {"heading": "Financial Considerations", "content": "Cash flow is the most common operational challenge for new prime contractors. The federal government pays on Net-30 terms after proper invoice submission — but invoice submission requires acceptance of deliverables, which requires inspection, which takes time. In practice, federal payment cycles often run 45-60 days from delivery.\n\nFor prime contractors with employees, this means you may be paying labor for 2-3 months before you receive payment from the government. Adequate working capital — typically 90 days of operating expenses — is essential.\n\nSmall business prime contractors have access to several financing tools. The SBA 7(a) loan program provides working capital. Invoice factoring (selling your receivables at a discount) provides immediate cash. Some agencies will provide contract financing (advance payments) for small businesses on long-term contracts.\n\nBilling efficiency matters. Submit invoices immediately upon deliverable acceptance. Track every payment due date. Contracting officers have an obligation to pay within 30 days — the Prompt Payment Act (31 U.S.C. § 3901) entitles you to interest on late payments."}
  ],
  "key_facts": [
    "Subcontracts count as past performance for prime contract proposals if comparable in scope and complexity",
    "The Prompt Payment Act (31 U.S.C. § 3901) entitles contractors to interest on federal payments made after 30 days",
    "FAR flowdown clauses in prime contracts create compliance obligations for subcontractors — read them carefully",
    "Simplified acquisition contracts ($10K-$250K) are the best entry point for first prime contracts",
    "Working capital of 90 days operating expenses is a practical minimum for new prime contractors"
  ],
  "common_mistakes": [
    "Staying in subcontracting past the point where prime contracting would be more profitable — the transition is less risky than it appears if you start small",
    "Not reading FAR flowdown clauses in subcontracts — compliance violations can result in termination and False Claims Act exposure",
    "Underestimating working capital requirements — federal payment cycles of 45-60 days mean you're financing 2-3 months of payroll before payment arrives"
  ],
  "official_sources": [
    {"name": "FAR Part 44 — Subcontracting Policies and Procedures", "url": "https://www.acquisition.gov/far/part-44", "description": "Prime contractor requirements for subcontract management"},
    {"name": "31 U.S.C. § 3901 — Prompt Payment Act", "url": "https://uscode.house.gov/view.xhtml?req=granuleid:USC-prelim-title31-section3901", "description": "Your right to interest on late federal payments"},
    {"name": "SBA 7(a) Loan Program", "url": "https://www.sba.gov/funding-programs/loans/7a-loans", "description": "Working capital financing for small businesses"}
  ]
},

"protests/what-is-bid-protest": {
  "title": "What is a Bid Protest?",
  "subtitle": "The formal mechanism for challenging improper federal contract awards — and a standard part of the procurement system.",
  "intro": "A bid protest is a formal challenge to a federal agency's procurement decision — typically the award of a contract or the terms of a solicitation. Bid protests are a normal, expected part of the federal contracting ecosystem. They are not adversarial acts — they are a legal mechanism designed to keep the procurement system honest.",
  "sections": [
    {"heading": "The Legal Foundation for Bid Protests", "content": "The right to protest federal contract awards is established by the Competition in Contracting Act (31 U.S.C. § 3551-3556), which created the GAO bid protest process, and by the Tucker Act (28 U.S.C. § 1491), which grants the Court of Federal Claims jurisdiction over bid protest cases.\n\nThe Administrative Procedure Act (5 U.S.C. § 706) provides the legal standard for reviewing agency decisions — protests succeed when the agency's decision was arbitrary, capricious, an abuse of discretion, or not in accordance with law. This is a meaningful standard: agencies don't have unlimited discretion.\n\nEvery year, approximately 2,000-2,500 protests are filed with GAO. GAO sustains (sides with the protester) approximately 15-20% of protests decided on the merits. Another 15-20% result in voluntary corrective action by the agency before GAO issues a decision. That's a 30-40% overall outcome rate for protesters — meaningful enough to make protests worth considering in the right circumstances."},
    {"heading": "Who Can File a Protest", "content": "An 'interested party' can file a bid protest. An interested party is generally defined as an actual or prospective offeror whose direct economic interest would be affected by the award or the terms of the solicitation.\n\nFor post-award protests, you must have been an offeror — you can't protest an award if you didn't bid. For pre-award protests (challenging solicitation terms), you don't need to have submitted a proposal yet, but you must be a prospective offeror who would submit if the solicitation were corrected.\n\nSmall businesses have additional protest rights. The SBA can appeal size determination protests on behalf of small businesses. And small businesses can file protests with SBA specifically challenging whether the awardee qualifies as small under the applicable NAICS code — a size protest."},
    {"heading": "Where to File: GAO, Agency, or COFC", "content": "Protesters have three main venues. The Government Accountability Office (GAO) is the most common forum — it's free, fast (100-day decision deadline), and expert in procurement law. GAO's Office of General Counsel reviews protests and issues written decisions that are publicly available.\n\nAgency-level protests (filed directly with the contracting agency) are the least formal option. Agencies must respond within 35 days. Agency protests are easier to file but less independent — you're asking the agency to review its own decision. They're most useful when the error is clear and the agency relationship is important to preserve.\n\nThe Court of Federal Claims (COFC) is the federal court with jurisdiction over bid protest cases. COFC proceedings are formal litigation, more expensive and slower than GAO, but the only forum that can issue injunctions. COFC is typically used when the procurement has national security implications, when the dollar amount justifies litigation costs, or when GAO has already ruled against the protester."},
    {"heading": "What Happens After Filing", "content": "After a GAO protest is filed, automatic stay rules apply for protests filed within 10 days of award. The agency is required to suspend performance while the protest is pending (unless the head of the agency makes an override determination based on urgent and compelling circumstances).\n\nThe agency submits an agency report responding to the protest within 30 days. The protester then has 10 days to file comments on the agency report. GAO issues a decision within 100 days of filing, though many protests are resolved earlier through agency corrective action.\n\nIf GAO sustains the protest, it recommends corrective action — typically re-evaluation, a new solicitation, or termination of the current contract. The agency is not legally required to follow GAO's recommendation, but compliance rates are above 95%.\n\nWhen GAO denies a protest, the protester can pursue the case in the Court of Federal Claims within 90 days of the GAO decision. This is a de novo review — the court considers the record independently rather than just reviewing GAO's analysis."}
  ],
  "key_facts": [
    "GAO sustains approximately 15-20% of protests decided on the merits — 30-40% result in some form of relief",
    "GAO protests are free to file — attorney fees can be recovered if you win and the government acted in bad faith",
    "Automatic stay of contract performance applies for protests filed within 10 days of award",
    "GAO must issue decisions within 100 days — faster than any court",
    "Agency compliance with GAO recommendations exceeds 95%"
  ],
  "common_mistakes": [
    "Not requesting a debriefing before deciding whether to protest — debriefings reveal the evaluation scores and rationale that tell you whether a protest has merit",
    "Missing the 10-day filing deadline for automatic stay — you can still protest after 10 days, but the agency can continue performance while the protest is pending",
    "Assuming protests are adversarial — agencies and contracting officers understand protests are a normal part of the system; a professional, well-founded protest rarely damages the relationship"
  ],
  "official_sources": [
    {"name": "GAO Bid Protest Regulations", "url": "https://www.gao.gov/legal/bid-protests/about", "description": "GAO's official bid protest process and filing instructions"},
    {"name": "4 C.F.R. Part 21 — GAO Bid Protest Procedures", "url": "https://www.ecfr.gov/current/title-4/chapter-I/part-21", "description": "Complete GAO protest regulations"},
    {"name": "GAO Bid Protest Annual Reports", "url": "https://www.gao.gov/legal/bid-protests/annual-reports", "description": "Annual statistics on protest filings and outcomes"}
  ]
},

"protests/when-to-protest": {
  "title": "When to Consider a Bid Protest",
  "subtitle": "Protests are appropriate in specific circumstances — here's how to evaluate whether you have a case.",
  "intro": "Filing a bid protest is a significant decision that requires honest analysis of both the legal merits and the business implications. Not every loss warrants a protest. But when the procurement process was materially flawed, a protest is not just your right — it's how the system corrects itself.",
  "sections": [
    {"heading": "Grounds for Protest: What Can Be Challenged", "content": "Protests can be based on virtually any allegation that the procurement didn't comply with applicable law or regulations. Common grounds include: improper evaluation (the agency didn't apply its stated evaluation criteria), unequal treatment (evaluators applied different standards to different offerors), organizational conflict of interest (the awardee had an unfair advantage from prior work on the requirement), improper sole-source award (the agency didn't follow competition requirements), ambiguous solicitation terms (the solicitation was unclear in material ways), and improper price evaluation (the agency made mathematical errors or ignored price realism concerns).\n\nImproper evaluation is the most common ground. Agencies are bound by the evaluation criteria published in the solicitation — they cannot use unstated criteria or ignore stated criteria. If your debriefing reveals that evaluators considered factors not in the solicitation, or ignored factors that were in the solicitation, that's a potential protest ground."},
    {"heading": "The Debriefing: Your Intelligence Source", "content": "Before deciding whether to protest, always request a debriefing. Under FAR 15.506, agencies must offer post-award debriefings to unsuccessful offerors who request one within three business days of receiving notice of award.\n\nA proper debriefing must include: the agency's evaluation of your significant weaknesses and deficiencies, the overall evaluated price/cost and technical rating of the awardee (though not the awardee's actual proposal), and the overall ranking of all offerors if a ranking was developed. The debriefing cannot include a point-by-point comparison of your proposal to others.\n\nListen carefully during the debriefing for inconsistencies between the evaluation description and the evaluation criteria in the solicitation. Take detailed notes. Ask clarifying questions. The specific language evaluators used — especially for weaknesses they noted in your proposal — reveals exactly how they applied the criteria.\n\nIf the agency's debriefing reveals potential impropriety, request a written debriefing or ask specific clarifying questions in writing. Written responses create a record you can use in a protest."},
    {"heading": "Analyzing Protest Merit", "content": "A protest has merit when: the agency deviated from stated evaluation criteria in a way that was prejudicial to you (you would have won if the proper criteria had been applied), the agency treated you and the awardee differently without justification, the agency made a decision that no reasonable person could have made based on the available information, or the agency violated a specific legal requirement.\n\nThe prejudice requirement is critical. Even if the agency made a mistake, you only win a protest if you can show that the mistake affected the outcome — that you would have been selected absent the error. A protester who could not have won even with correct evaluation loses on the prejudice issue.\n\nAsk yourself honestly: if the agency had evaluated my proposal correctly, would I have won? If the answer is 'probably not,' a protest is unlikely to succeed even on otherwise valid grounds."},
    {"heading": "Business Considerations Before Filing", "content": "Beyond legal merit, consider the business implications. Protests create tension with contracting officers and program managers. If you expect to bid on work with this agency for years, an unsuccessful protest can damage relationships that have real long-term value. A successful protest damages those relationships less, but still creates friction.\n\nFor agencies where you have limited relationship investment and the legal case is strong, protests make more sense. For agencies where you've built deep relationships and expect long-term work, carefully weigh the relationship cost against the potential benefit of the specific contract.\n\nAlso consider the economics. Protest preparation takes significant time — typically 20-100 hours for a GAO protest, more if you engage outside counsel. For small contracts, the economics may not support the investment. For large, long-term contracts that represent significant revenue, the investment is often clearly justified."},
    {"heading": "The 10-Day Rule and Timing", "content": "Filing deadlines are strict and unforgiving in bid protests. Missing a deadline means GAO dismisses your protest without considering the merits.\n\nFor post-award protests, you must file at GAO within 10 days of when you knew or should have known the basis for protest. For protests based on the award itself, the 10-day clock usually starts when you receive notice of award. For protests based on solicitation terms, you must file before the proposal due date.\n\nThe 10-day automatic stay rule has a separate trigger: for the stay to apply, you must file within 10 calendar days of contract award. If you file later, the agency can continue performance during the protest — a significant disadvantage if the work is time-sensitive.\n\nIf you receive a debriefing and the debriefing reveals new bases for protest, you have 10 days from the date of the debriefing (not the date of award) to file protests based on information learned in the debriefing. This extension applies only to protest grounds that weren't apparent before the debriefing."}
  ],
  "key_facts": [
    "Post-award protests must be filed within 10 days of knowing the basis for protest — not 10 days from award notice",
    "Automatic stay requires filing within 10 calendar days of contract award",
    "Prejudice is required — even valid legal errors don't win protests unless they affected the outcome",
    "The debriefing creates a 10-day extension for protest grounds first learned during the debriefing",
    "FAR 15.506 requires agencies to offer debriefings within 5 days of a written request"
  ],
  "common_mistakes": [
    "Protesting every loss out of frustration — protest only when there's genuine legal merit and a realistic likelihood of success; frivolous protests damage your reputation",
    "Missing the 10-day filing deadline — the clock starts when you know or should know the basis, not necessarily when you receive formal notification",
    "Not getting the debriefing in writing — verbal debriefings leave no record; request written responses to clarifying questions"
  ],
  "official_sources": [
    {"name": "FAR 15.506 — Debriefing of Offerors", "url": "https://www.acquisition.gov/far/15.506", "description": "Your right to a post-award debriefing and what it must include"},
    {"name": "GAO Protest Filing Instructions", "url": "https://www.gao.gov/legal/bid-protests/filing-a-protest", "description": "Step-by-step instructions for filing at GAO"}
  ]
},

"protests/gao-protest-basics": {
  "title": "GAO Protest Basics",
  "subtitle": "Filing a protest with the Government Accountability Office — the fastest and most common protest venue.",
  "intro": "The Government Accountability Office (GAO) is the primary forum for federal bid protests. GAO protests are free, fast (100-day decision deadline), and decided by attorneys with deep procurement law expertise. Understanding the GAO process helps you decide whether and how to file.",
  "sections": [
    {"heading": "GAO's Authority and Role", "content": "GAO's bid protest authority derives from the Competition in Contracting Act (31 U.S.C. § 3551-3556). GAO's Office of General Counsel (not the auditors GAO is known for) reviews protests and issues written decisions. These decisions are publicly available at gao.gov and constitute a significant body of procurement law.\n\nGAO is not a court — it cannot enforce its own decisions. When GAO sustains a protest, it recommends corrective action to the agency. Agencies comply voluntarily in over 95% of cases because non-compliance requires a written explanation to Congress and triggers additional oversight. The practical enforcement rate is extremely high.\n\nGAO's standard of review: it will sustain a protest when an agency's decision was arbitrary, capricious, lacked a rational basis, or violated applicable law or regulations. GAO gives agencies reasonable discretion in judgment calls — it doesn't substitute its judgment for the agency's on close calls — but it holds agencies strictly to their stated evaluation criteria."},
    {"heading": "Filing a GAO Protest", "content": "Protests are filed at gao.gov — the electronic protest docketing system (eProtestDocketing) allows online filing. Filing is free. You do not need an attorney to file a GAO protest, though attorneys are commonly used for complex cases.\n\nYour protest must contain: your name and address, the name of the contracting activity, the contract or solicitation number, a detailed statement of the legal and factual grounds for the protest, copies of relevant documents (solicitation, award notice, debriefing notes), a request for specific relief (re-evaluation, new solicitation, cancellation), and a statement that the protest is filed within the applicable time limits.\n\nGAO will send you a docket number and notify the agency. The agency has 30 days to submit an agency report responding to your protest. You then have 10 days to file comments on the agency report. GAO issues a decision within 100 days of filing."},
    {"heading": "The Protective Order Process", "content": "One of the most valuable aspects of GAO protests is the protective order process. GAO can issue a protective order requiring the agency to provide the protester's attorney with sensitive procurement information — including the awardee's proposal, evaluation documentation, and source selection records — on a confidential basis.\n\nThis information is not available outside of a protest context. The ability to review what the awardee actually proposed, and how evaluators actually scored competing proposals, provides powerful evidence for (or against) continuing the protest.\n\nNote: the protective order covers attorneys and designated consultants, not the protester directly. Your attorney can review the information and tell you the general nature of the evidence, but cannot show you the awardee's actual proposal. This limitation is important to understand before assuming the protective order process will give you complete visibility."},
    {"heading": "Likelihood of Success and Statistical Context", "content": "Of protests filed with GAO, approximately 40-45% are dismissed (usually for procedural reasons like untimeliness or lack of standing), 25-30% are denied on the merits, 15-20% are sustained, and 15-20% are resolved through agency corrective action before GAO issues a decision.\n\nThe 15-20% sustained rate understates your actual odds if you're filing with genuine merit. Corrective action — where the agency voluntarily fixes the problem when confronted with a formal protest — is often a better outcome than a sustained decision, and it resolves faster.\n\nIndustries vary significantly. IT and professional services protests are most common. Defense contracts have lower protest rates relative to award volume. Simplified acquisition awards are less frequently protested but can be challenged when the mandatory set-aside rules are violated."},
    {"heading": "Attorney Fees and Cost Recovery", "content": "Under the Equal Access to Justice Act (EAJA) and CICA, GAO can recommend reimbursement of protest costs — including filing costs and attorney fees at reasonable rates — when GAO sustains a protest.\n\nThe standard for cost recovery: the agency's position must have been without any reasonable basis, or the agency must have violated an existing, clear legal precedent. This is a higher bar than simply winning — you need to show the agency should have known better.\n\nIn practice, cost recovery is available in a meaningful minority of sustained protests. For protests that reach the briefing stage with outside counsel, legal fees can be substantial — $20,000-$100,000+ depending on complexity. Even with cost recovery prospects, evaluate the expected value of the contract against the cost of protest before proceeding with outside counsel."}
  ],
  "key_facts": [
    "GAO protests are free to file — no filing fee, no minimum contract value requirement",
    "GAO must issue decisions within 100 days — automatic stay suspends contract performance if filed within 10 days of award",
    "Agency report due 30 days after filing; protester comments due 10 days after agency report",
    "Protective orders give your attorney access to the awardee's proposal and evaluation documentation",
    "Annual GAO protest statistics and decisions are publicly available at gao.gov"
  ],
  "common_mistakes": [
    "Protesting without a protective order request when the evidence is in the agency's files — the awardee's proposal and evaluation documents are only accessible through the GAO protective order process",
    "Not filing comments on the agency report — your 10-day comment period is your opportunity to rebut the agency's defense; failing to file comments is effectively conceding the agency's arguments",
    "Underestimating the cost of outside counsel for complex protests — get a realistic fee estimate before engaging attorneys; the economics must justify the investment"
  ],
  "official_sources": [
    {"name": "GAO Bid Protest Information", "url": "https://www.gao.gov/legal/bid-protests", "description": "Filing instructions, forms, and searchable database of GAO decisions"},
    {"name": "GAO Electronic Protest Docketing", "url": "https://www.gao.gov/legal/bid-protests/file-a-protest", "description": "Online filing system for GAO protests"},
    {"name": "31 U.S.C. § 3551 — CICA Protest Authority", "url": "https://uscode.house.gov/view.xhtml?req=granuleid:USC-prelim-title31-section3551", "description": "Statutory basis for GAO's protest jurisdiction"}
  ]
},

"resources/apex-accelerators": {
  "title": "APEX Accelerators (formerly PTACs)",
  "subtitle": "Free, DoD-funded technical assistance for businesses pursuing federal contracts.",
  "intro": "APEX Accelerators are the most underutilized free resource in federal contracting. Funded by the Department of Defense and hosted by universities, chambers of commerce, and economic development organizations, they provide hands-on, personalized assistance to businesses at every stage of the federal contracting journey — all at no cost.",
  "sections": [
    {"heading": "What APEX Accelerators Do", "content": "APEX Accelerators (formerly Procurement Technical Assistance Centers or PTACs) were established under the Defense Federal Acquisition Regulation Supplement (DFARS) and are funded by the DoD's Office of Small Business Programs. There are over 300 locations across all 50 states, DC, Guam, Puerto Rico, and the Virgin Islands.\n\nServices include: SAM.gov registration and renewal assistance, capability statement review and development, market research support (identifying relevant opportunities), proposal review and coaching, matchmaking with prime contractors and agencies, certification application assistance (8(a), HUBZone, WOSB, SDVOSB), cyber security compliance assistance (CMMC for DoD contracts), and general business development counseling.\n\nAll services are provided at no cost to the business. There are no eligibility requirements — you don't need to be small, minority-owned, or veteran-owned to use APEX services. Any business pursuing or interested in federal contracting qualifies."},
    {"heading": "How to Use APEX Accelerators Effectively", "content": "The most effective APEX relationships are ongoing, not one-time consultations. Counselors who know your business, your capabilities, and your target markets can provide increasingly valuable guidance over time.\n\nStart with a comprehensive intake meeting where you explain your business, your capabilities, and your contracting goals. The counselor will assess where you are in the contracting readiness spectrum and recommend a development path.\n\nUse APEX for proposal reviews — before you submit a significant proposal, have your APEX counselor review it for compliance with instructions, responsiveness to evaluation criteria, and competitive positioning. This service alone is worth hundreds of hours of consultant time that competitors without APEX access pay for.\n\nAttend APEX-organized events: matchmaking events, agency outreach days, procurement workshops, and certification training. These events connect you with contracting officers and prime contractor business development staff in settings designed for relationship building."},
    {"heading": "APEX Accelerator Limitations", "content": "APEX counselors vary significantly in expertise and experience. Some have deep procurement law knowledge and extensive contractor experience. Others are more generalist. The quality of service depends heavily on the specific counselor you're assigned.\n\nAPEX does not provide legal advice. For bid protests, complex contract negotiations, False Claims Act concerns, or other legal matters, you need a procurement attorney. APEX can often refer you to appropriate legal resources.\n\nAPEX cannot write your proposals for you. They can review and comment, but the proposal must be your work. Similarly, they facilitate matchmaking events but cannot guarantee introductions to specific agency decision-makers.\n\nIf you're unsatisfied with the service you're receiving from one APEX location, you can seek assistance from another. There's no requirement to work with only your geographically nearest center."},
    {"heading": "Finding Your APEX Accelerator", "content": "The official locator is at apexaccelerators.us. Search by state or zip code. Multiple offices may serve your area — review each one's focus areas and staff expertise before choosing.\n\nMany APEX Accelerators specialize in specific sectors: some focus on defense contracting and cybersecurity compliance, others on civilian agency opportunities, still others on construction or professional services. Find one whose expertise aligns with your target market.\n\nContact them before you need them. Building a relationship with your APEX counselor before you're under proposal deadline pressure makes the relationship much more productive. Schedule a quarterly check-in even when you don't have an immediate need — they can proactively alert you to relevant opportunities and agency outreach events."}
  ],
  "key_facts": [
    "300+ APEX locations nationwide — all services are completely free with no eligibility requirements",
    "Funded by the Department of Defense under DFARS Subpart 235.71",
    "Services include SAM registration, proposal review, matchmaking, and CMMC compliance assistance",
    "APEX can review proposals before submission — this service alone saves thousands in consultant fees",
    "Find your nearest location at apexaccelerators.us"
  ],
  "common_mistakes": [
    "Using APEX only for SAM registration help and then moving on — APEX's highest-value service is ongoing proposal coaching and market intelligence",
    "Not building the APEX relationship before you have a deadline — counselors who know your business provide dramatically better help than cold engagements under time pressure",
    "Expecting APEX to write your proposals or guarantee introductions — they facilitate and advise; the work is still yours"
  ],
  "official_sources": [
    {"name": "APEX Accelerators Locator", "url": "https://www.apexaccelerators.us", "description": "Find your nearest free procurement counseling office"},
    {"name": "DoD APEX Program", "url": "https://business.defense.gov/Acquisition-Resources/APEX-Accelerators/", "description": "DoD's official APEX program page with program information"}
  ]
},

"resources/sba-district-offices": {
  "title": "SBA District Offices",
  "subtitle": "Free counseling, certification processing, and advocacy for small businesses in federal contracting.",
  "intro": "The Small Business Administration operates district offices in every state and major territory, providing free services specifically designed to help small businesses enter and succeed in federal contracting. Unlike APEX Accelerators (which focus broadly on contracting readiness), SBA district offices have specific authority over small business certification programs and agency procurement policy.",
  "sections": [
    {"heading": "What SBA District Offices Provide", "content": "SBA district offices offer one-on-one counseling on federal contracting strategy, certification program guidance and application assistance (8(a), HUBZone, WOSB, SDVOSB), size determination guidance, loan programs for working capital and equipment, connections to SBA resource partners (APEX Accelerators, SCORE, Women's Business Centers), and advocacy with federal agencies on small business contracting opportunities.\n\nEach district office includes a team with expertise in specific programs. The District Director oversees overall operations. Business Opportunity Specialists focus on certification programs. Lenders Relations Specialists handle financing programs. Economic Development Specialists work on business development broadly.\n\nDistrict offices also host regular events including contracting workshops, agency matchmaking events, certification information sessions, and networking opportunities with prime contractors and other small business contractors."},
    {"heading": "SBA Procurement Center Representatives", "content": "SBA stations Procurement Center Representatives (PCRs) at major federal buying activities — DoD installations, civilian agency headquarters, and large contracting centers. PCRs are federal employees whose explicit mission is to increase small business opportunities at those specific agencies.\n\nPCRs review upcoming acquisitions for set-aside potential, challenge inappropriate sole-source awards, review subcontracting plans for adequacy, and serve as an advocate for small businesses in the procurement process. They have authority to formally object to acquisitions they believe should be set aside for small businesses.\n\nIf you're targeting a specific agency, find out who their PCR is. Building a relationship with the PCR for your target agency gives you an advocate inside the procurement system. PCRs regularly share upcoming opportunities with small businesses they know.\n\nThe SBA website maintains a PCR directory at sba.gov/contracting. Not all agencies have dedicated PCRs — at agencies without dedicated PCRs, the SBA district office serves as the point of contact."},
    {"heading": "Commercial Market Representatives", "content": "SBA's Commercial Market Representatives (CMRs) focus specifically on subcontracting. They work with prime contractors to develop and strengthen their small business subcontracting programs, identify small business sources for prime contractors, and assist small businesses in getting onto prime contractor teaming and subcontracting lists.\n\nCMRs have access to prime contractors' subcontracting plans and can facilitate introductions between primes who need small business partners and small businesses seeking subcontracting opportunities. This is a significantly underused resource.\n\nContact your SBA district office to identify your region's CMR and explain your capabilities and target prime contractors. CMRs can make specific introductions that would take you months of independent business development to achieve."},
    {"heading": "SBA's Formal Protest Authority", "content": "Beyond counseling, SBA has formal authority in the federal procurement process. SBA can protest to the contracting officer when it believes a procurement should be set aside for small business but wasn't. SBA can also request a review when it believes a contract should be classified as a small business set-aside.\n\nIf you believe an agency has improperly declined to set aside a contract for small business, filing a complaint with your district office (and specifically the PCR for that agency) triggers SBA's formal review and advocacy process. This is a lower-cost alternative to GAO protest for pure set-aside issues.\n\nSBA also adjudicates size protests — when someone challenges whether you (or the awardee) qualifies as small under the applicable NAICS code. Size protests are filed directly with SBA and must be filed within 5 days of bid opening or award notification."}
  ],
  "key_facts": [
    "SBA has 68 district offices nationwide — find yours at sba.gov/local-assistance",
    "PCRs are stationed at major buying activities and have authority to formally object to improper set-aside decisions",
    "Size protests must be filed with SBA within 5 days of bid opening or award notification",
    "Commercial Market Representatives facilitate introductions between primes and small business subcontractors",
    "SBA district offices process 8(a), HUBZone applications and provide certification guidance at no cost"
  ],
  "common_mistakes": [
    "Using SBA only for loan inquiries and missing the contracting assistance services — district offices provide significant value in certification, PCR connections, and procurement advocacy",
    "Not connecting with your PCR when you believe a set-aside was improperly declined — PCRs have formal authority to challenge these decisions and often can resolve issues faster than GAO protests",
    "Missing the 5-day deadline for size protests — this is strictly enforced and missing it means your challenge is dismissed"
  ],
  "official_sources": [
    {"name": "SBA District Office Locator", "url": "https://www.sba.gov/local-assistance/find/?type=SBA%20District%20Office", "description": "Find your nearest SBA district office"},
    {"name": "SBA PCR Directory", "url": "https://www.sba.gov/federal-contracting/counseling-help/procurement-center-representatives", "description": "PCRs stationed at major federal buying activities"},
    {"name": "SBA Size Protest Procedures", "url": "https://www.sba.gov/federal-contracting/contracting-guide/eligibility-protests", "description": "How to file a size protest challenging a competitor's small business status"}
  ]
},

"resources/score-mentorship": {
  "title": "SCORE Mentorship for Government Contractors",
  "subtitle": "Free mentoring from experienced business professionals — including former federal contractors and procurement officials.",
  "intro": "SCORE is a nonprofit association and SBA resource partner with over 10,000 volunteer mentors across more than 250 chapters nationwide. For small businesses pursuing federal contracting, SCORE mentors with government contracting experience can provide practical guidance on business development, proposal strategy, financial management, and growth planning.",
  "sections": [
    {"heading": "What SCORE Offers", "content": "SCORE provides free one-on-one mentoring from volunteer business professionals, free or low-cost workshops and webinars, online resources and templates, and connections to the broader SCORE network.\n\nMentoring relationships are ongoing — not just a one-time consultation. You can meet with your mentor regularly (typically monthly) to work through specific challenges, review financial statements, develop strategy, and track progress against goals. Many SCORE mentoring relationships last years.\n\nOnline mentoring is available nationwide, removing geographic limitations. If your local SCORE chapter doesn't have a mentor with federal contracting expertise, you can be matched with a mentor from any chapter."},
    {"heading": "Finding a SCORE Mentor with Contracting Experience", "content": "SCORE mentor profiles list their areas of expertise. When requesting a mentor, specifically ask for someone with federal government contracting, procurement, or defense industry experience. Former contracting officers, prime contractor executives, and experienced government contractors are all represented in the SCORE volunteer base.\n\nThe most valuable SCORE mentors for federal contractors are those who have been on the government side of the table — former contracting officers who understand exactly how proposals are evaluated, what makes contracting officers nervous, and how agencies actually make decisions. This perspective is difficult to access any other way.\n\nBe specific in your mentoring request. Rather than 'help with my federal contracting strategy,' request 'guidance on developing my first 8(a) proposal strategy for DoD IT services contracts under $500,000.' Specific requests get better mentor matches."},
    {"heading": "What SCORE Cannot Do", "content": "SCORE mentors are volunteers — they are not employees, consultants, or attorneys. They provide guidance based on their experience, not professional services. For legal matters (contracts, bid protests, employment law), financial services (accounting, tax preparation, financial planning), or technical certifications (cybersecurity, ISO), SCORE will refer you to appropriate professionals.\n\nMentor quality varies. Some SCORE volunteers have deep, recent expertise. Others have experience that's dated or in adjacent fields. If your initial mentor match isn't providing value, you can request a different mentor. Don't settle for a mismatch.\n\nSCORE is particularly valuable for business strategy, financial management, and general business development — areas where experienced mentorship is most valuable. For specific procurement law questions, APEX Accelerators or procurement attorneys are better resources."},
    {"heading": "Workshops and Online Resources", "content": "Beyond individual mentoring, SCORE offers regular workshops on topics including federal contracting basics, proposal writing, financial management for government contractors, and small business certifications. Many are free; some have nominal fees.\n\nSCORE's online resource library (score.org) includes templates for capability statements, financial projections, business plans, and marketing materials. These templates are starting points — customize them for your specific situation.\n\nSCORE local chapters organize networking events that can connect you with other small business contractors, prime contractors, and agency representatives in your area. The informal networking at SCORE events often produces more business development value than the formal programming."}
  ],
  "key_facts": [
    "10,000+ volunteer mentors across 250+ chapters — all mentoring is free",
    "Online mentoring available nationwide — not limited to your local chapter's expertise",
    "Request mentors with specific federal contracting or procurement experience when registering",
    "Mentoring relationships are ongoing — monthly meetings over months or years are common",
    "SCORE is an SBA resource partner — find them at score.org or through your SBA district office"
  ],
  "common_mistakes": [
    "Accepting a mentor mismatch — if your assigned mentor lacks federal contracting experience, request a rematch; the system exists to serve you",
    "Using SCORE for legal or financial services questions — mentors are not attorneys or accountants; use them for strategy, not professional services",
    "Treating SCORE as a one-time resource — the ongoing mentoring relationship is where the value accumulates"
  ],
  "official_sources": [
    {"name": "SCORE.org", "url": "https://www.score.org", "description": "Request a mentor, access workshops, and browse resource library"},
    {"name": "SCORE Mentor Search", "url": "https://www.score.org/find-mentor", "description": "Find a mentor by expertise area and location"}
  ]
},

"resources/sam-gov": {
  "title": "SAM.gov: The Complete Guide",
  "subtitle": "The federal government's official system for registration, opportunities, and entity data — and how to use all of it.",
  "intro": "SAM.gov (System for Award Management) is more than just a registration database. It's the central hub for federal vendor registration, contract opportunities, award data, exclusions, and wage determinations. Understanding all of SAM's functions — not just registration — gives you a significant advantage in the federal market.",
  "sections": [
    {"heading": "Registration: The Foundation", "content": "SAM registration is required for all entities seeking federal contracts, grants, or cooperative agreements above the micro-purchase threshold. Registration is free and requires annual renewal. The process begins at sam.gov — you create a login.gov account (the federal single sign-on system), then begin the entity registration.\n\nYour Unique Entity ID (UEI) is assigned automatically during registration. This replaced the DUNS number system in April 2022. The UEI is a 12-character alphanumeric identifier that is now used across all federal systems.\n\nRegistration data flows to FPDS (Federal Procurement Data System), USAspending.gov, and agency acquisition systems. Errors in your SAM registration propagate throughout these downstream systems. Maintaining accurate, current registration data is an ongoing responsibility — not a one-time task."},
    {"heading": "Opportunities: Finding Federal Business", "content": "SAM.gov hosts the federal opportunities database — formerly known as FedBizOpps (FBO). Every contract opportunity above $25,000 must be publicly posted here. Types of notices include: Pre-Solicitation (agency announces intent to compete a requirement), Solicitation (formal RFP, RFQ, or IFB), Award Notice (contract has been awarded), Special Notice (industry days, market research, sources sought), and Justification and Approval (for sole-source awards).\n\nSaved searches and email alerts are among SAM's most valuable features for business development. Set up searches by NAICS code, agency, set-aside type, and geographic location. SAM can email you daily alerts when new opportunities match your criteria — eliminating the need for daily manual searching.\n\nSources Sought notices deserve special attention. These are market research tools agencies use before formal solicitation to gauge industry capabilities and interest. Responding to Sources Sought with a brief capability statement puts you on the agency's radar before the competition begins — and may influence how the solicitation is structured."},
    {"heading": "Award Data and Market Research", "content": "SAM.gov integrates with USAspending.gov for award data, but the contracts.gov interface within SAM provides additional detail on active contracts and their modification history.\n\nFor market research, SAM's award data search allows you to: identify which agencies are spending money in your NAICS codes, find the specific contracting offices that issue contracts in your market, identify the contracting officers and specialists responsible for your target contracts, and research the award history for specific requirements you want to recompete.\n\nThe Entity Management section of SAM shows your registration history and confirms your registration status. Contracting officers check this before every award — a lapsed or incomplete registration stops awards cold."},
    {"heading": "Representations and Certifications", "content": "The Representations and Certifications section of SAM registration is legally significant. When you check boxes certifying your business size, ownership, and various compliance representations, you are making legal statements under the penalty of the False Claims Act.\n\nCommon representations include: small business size status, socioeconomic program certifications (8(a), HUBZone, WOSB, SDVOSB), equal opportunity compliance (Executive Order 11246), Buy American compliance, and various other regulatory certifications.\n\nThese representations are incorporated by reference into every federal contract you receive. Inaccurate representations — intentional or not — can result in contract termination, suspension or debarment, and False Claims Act liability. Review your representations annually during renewal and update them immediately when your status changes."},
    {"heading": "The Exclusions Database", "content": "The SAM.gov exclusions database (formerly EPLS — Excluded Parties List System) contains entities barred from receiving federal awards. Contracting officers must check this database before making any award.\n\nExclusions can be temporary (suspension, pending investigation) or permanent (debarment). Causes include: contract fraud, tax delinquency, False Claims Act violations, environmental violations, and various other grounds.\n\nUnderstanding the exclusions database matters for two reasons: you need to verify your own registration is clean, and you should check potential teaming partners and subcontractors before formalizing relationships. A debarred prime contractor or key subcontractor discovered after award submission can derail your bid."}
  ],
  "key_facts": [
    "SAM registration is free and required for all federal contract awards above $10,000",
    "Sources Sought notices are market research opportunities — responding before the formal solicitation puts you on the agency's radar",
    "Representations and certifications in SAM are legal statements under the False Claims Act — review them annually",
    "The exclusions database must be checked before making awards — verify your partners are not excluded",
    "UEI replaced DUNS numbers in April 2022 — all federal systems now use the 12-character UEI"
  ],
  "common_mistakes": [
    "Not setting up SAM email alerts for relevant NAICS codes — daily opportunity monitoring by hand is inefficient; automated alerts are free",
    "Ignoring Sources Sought notices — these are your best opportunity to influence solicitation terms before competition begins",
    "Not verifying teaming partners in the exclusions database — discovering a key partner is debarred after proposal submission is catastrophic"
  ],
  "official_sources": [
    {"name": "SAM.gov", "url": "https://sam.gov", "description": "Official registration, opportunities, and award data"},
    {"name": "SAM.gov Help Center", "url": "https://sam.gov/content/help-center", "description": "Guides and FAQs for all SAM functions"},
    {"name": "Federal Service Desk", "url": "https://www.fsd.gov", "description": "Technical support for SAM registration issues"}
  ]
},

"resources/usaspending": {
  "title": "USAspending.gov: Using Federal Contract Data",
  "subtitle": "The public database of all federal spending — and how to use it for competitive intelligence.",
  "intro": "USAspending.gov is the official source for data on all federal spending, including contracts, grants, loans, and direct payments. Required by the DATA Act (31 U.S.C. § 6101), it provides unprecedented transparency into how the federal government spends money — and who it's paying. For federal contractors, it's the primary source of competitive intelligence.",
  "sections": [
    {"heading": "What Data is Available", "content": "USAspending.gov contains data on every federal contract award, modification, and delivery order reported above the micro-purchase threshold. For each transaction, you can see: award amount, awardee name and UEI, contracting agency and office, NAICS code, PSC code, period of performance, place of performance, and competition information.\n\nModification history is particularly valuable. Every time a contract's scope, price, or period was changed, a modification was reported. A contract that started at $1M and was modified 15 times to reach $5M tells a different story than one that was awarded at $5M and never modified.\n\nAward data is typically reported within 3-7 days of award. USAspending is updated frequently — you can track new awards in near-real time in your target market."},
    {"heading": "Advanced Search Strategies", "content": "The Advanced Search function allows filtering by multiple criteria simultaneously. Effective search strategies include: Agency + NAICS code to find all contracts in your market at a specific agency, Awardee name to research a specific competitor's contract portfolio, PSC code to find contracts by the specific product or service purchased (more granular than NAICS), Date range to see recent vs. historical spending patterns, and Competition type to distinguish full and open, small business set-aside, and sole-source awards.\n\nThe award date vs. period of performance end date is a key search combination. Contracts awarded 3-5 years ago with performance periods ending in the next 6-12 months are your recompete pipeline. Filter for contracts in your NAICS codes with performance end dates in your target window.\n\nSpending analysis by agency fiscal year reveals budget patterns. Agencies often accelerate spending in Q4 (July-September) as fiscal year end approaches. Knowing when your target agencies have money to spend affects the timing of your business development activities."},
    {"heading": "Identifying Recompete Opportunities", "content": "The most practical use of USAspending for most small contractors is identifying specific contracts approaching their expiration dates. This is the intelligence behind GovCon Intelligence Terminal's Recompete Radar feature.\n\nFor each opportunity you identify, build an intelligence package: total contract value and modification history, NAICS and PSC codes, contracting office and contracting officer (often searchable), competition type on the original award, whether it was a small business set-aside (and which specific program), and the incumbent's other contracts at the same agency (to understand their relationship depth).\n\nThe incumbent's modification history is especially revealing. A contract with frequent scope expansions suggests the agency is satisfied and adding work — incumbent advantage is high. A contract with modifications that reduce scope or extend performance without additional funding suggests performance problems — challenger opportunity is higher."},
    {"heading": "Contractor Research and Competitive Intelligence", "content": "USAspending allows deep research into specific competitors' contract portfolios. Search by contractor name to see their complete federal contracting history: which agencies they work with, what NAICS codes they work in, how much they've been paid historically, and whether their contract volumes are growing or declining.\n\nThis intelligence supports several strategic decisions: whether to compete directly against a specific incumbent (understanding their revenue concentration at an agency helps predict how hard they'll fight to retain the contract), which primes to approach for subcontracting (identifying primes with large contract volumes at your target agencies), and where to focus geographic business development efforts (which states and cities have the most spending in your NAICS codes).\n\nThe 'Federal Account Explorer' feature on USAspending shows appropriations data — which congressional accounts fund the contracts you're interested in. Understanding the funding source helps predict whether a contract will be renewed in a budget-constrained environment."},
    {"heading": "Downloading Data for Analysis", "content": "USAspending provides bulk data downloads for all contract data, updated regularly. For sophisticated analysis, you can download complete transaction-level data for specific agencies, NAICS codes, or time periods into spreadsheet or database tools.\n\nThe API (api.usaspending.gov) provides programmatic access to all data — enabling custom analyses, automated monitoring, and integration with other data sources. GovCon Intelligence Terminal uses the USAspending API as its primary data source, supplemented by additional government databases.\n\nFor most contractors, the web interface is sufficient for opportunity identification and competitor research. Bulk downloads and API access are valuable for systematic market analysis or when you want to monitor large volumes of contracts simultaneously."}
  ],
  "key_facts": [
    "Required by the DATA Act (31 U.S.C. § 6101) — all federal contracts above micro-purchase threshold must be reported",
    "API available at api.usaspending.gov — enables programmatic access and automated monitoring",
    "Contract modifications are public record — use them to assess incumbent performance and satisfaction",
    "Q4 spending acceleration (July-September) reflects agency fiscal year-end budget pressure",
    "Competition type codes reveal whether awards were full and open, set-aside, or sole-source"
  ],
  "common_mistakes": [
    "Only looking at base award amounts without checking total modification history — the true contract value often differs significantly from the original award",
    "Not filtering by period of performance end date when looking for recompetes — award date alone doesn't tell you when the opportunity will reopen",
    "Ignoring the Federal Account Explorer — understanding the appropriations source helps predict contract renewal likelihood in constrained budgets"
  ],
  "official_sources": [
    {"name": "USAspending.gov", "url": "https://www.usaspending.gov", "description": "Complete federal spending database — search, analyze, download"},
    {"name": "USAspending API", "url": "https://api.usaspending.gov", "description": "Programmatic access to all USAspending data"},
    {"name": "DATA Act Implementation", "url": "https://www.usaspending.gov/about", "description": "About the DATA Act and USAspending's legal mandate"}
  ]
}
}

# Merge with existing good content
all_content = {**good, **new_content}
import os
os.makedirs("lib/guide", exist_ok=True)
Path("lib/guide/content.json").write_text(json.dumps(all_content, indent=2))

good_final = {k: v for k, v in all_content.items() if "sections" in v}
print(f"Total topics with content: {len(good_final)}/30")
for k in sorted(good_final.keys()):
    print(f"  ✓ {k}")

/content/govcon-dashboard
Starting with 6 cached topics
Total topics with content: 30/30
  ✓ awards/evaluation-process
  ✓ awards/lowest-price-vs-best-value
  ✓ awards/past-performance
  ✓ awards/simplified-acquisition
  ✓ getting-started/naics-codes
  ✓ getting-started/sam-registration
  ✓ getting-started/what-is-federal-procurement
  ✓ getting-started/who-can-bid
  ✓ incumbents/full-open-competition
  ✓ incumbents/price-advantage
  ✓ incumbents/reading-incumbent-history
  ✓ incumbents/recompete-basics
  ✓ protests/gao-protest-basics
  ✓ protests/what-is-bid-protest
  ✓ protests/when-to-protest
  ✓ resources/apex-accelerators
  ✓ resources/sam-gov
  ✓ resources/sba-district-offices
  ✓ resources/score-mentorship
  ✓ resources/usaspending
  ✓ small-business/8a-program
  ✓ small-business/government-goals
  ✓ small-business/hubzone
  ✓ small-business/mentor-protege
  ✓ small-business/sdvosb
  ✓ small-business/set-aside-contracts
  ✓ small-business/wosb
  ✓ subcontracting/approaching-prim

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/(dashboard)/guide/[category]/[slug]", exist_ok=True)

Path("app/(dashboard)/guide/[category]/[slug]/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'
import { useParams, useRouter } from 'next/navigation'
import Link from 'next/link'
import guideContent from '@/lib/guide/content.json'

type Section = { heading: string; content: string }
type Source = { name: string; url: string; description: string }
type TopicContent = {
  title: string
  subtitle: string
  intro: string
  sections: Section[]
  key_facts: string[]
  common_mistakes: string[]
  official_sources: Source[]
}

const CATEGORY_META: Record<string, { label: string; color: string; back: string }> = {
  'getting-started': { label: '🚀 Getting Started', color: 'blue', back: '/guide' },
  'small-business':  { label: '🏆 Small Business Advantages', color: 'green', back: '/guide' },
  'awards':          { label: '📋 How Contracts Are Awarded', color: 'yellow', back: '/guide' },
  'incumbents':      { label: '⚔️ Competing Against Incumbents', color: 'orange', back: '/guide' },
  'subcontracting':  { label: '🤝 Subcontracting', color: 'purple', back: '/guide' },
  'protests':        { label: '⚖️ Bid Protests', color: 'red', back: '/guide' },
  'resources':       { label: '📚 Free Resources', color: 'teal', back: '/guide' },
}

const COLOR_MAP: Record<string, { badge: string; border: string; header: string }> = {
  blue:   { badge: 'bg-blue-700 text-white',   border: 'border-blue-800',   header: 'bg-blue-950' },
  green:  { badge: 'bg-green-700 text-white',  border: 'border-green-800',  header: 'bg-green-950' },
  yellow: { badge: 'bg-yellow-700 text-white', border: 'border-yellow-800', header: 'bg-yellow-950' },
  orange: { badge: 'bg-orange-700 text-white', border: 'border-orange-800', header: 'bg-orange-950' },
  purple: { badge: 'bg-purple-700 text-white', border: 'border-purple-800', header: 'bg-purple-950' },
  red:    { badge: 'bg-red-700 text-white',    border: 'border-red-800',    header: 'bg-red-950' },
  teal:   { badge: 'bg-teal-700 text-white',   border: 'border-teal-800',   header: 'bg-teal-950' },
}

export default function TopicPage() {
  const params = useParams()
  const router = useRouter()
  const category = params?.category as string
  const slug = params?.slug as string
  const key = `${category}/${slug}`

  const content = (guideContent as Record<string, TopicContent>)[key]
  const meta = CATEGORY_META[category] || { label: 'Guide', color: 'blue', back: '/guide' }
  const colors = COLOR_MAP[meta.color] || COLOR_MAP.blue

  if (!content || !content.sections) {
    return (
      <div className="p-8 bg-slate-950 min-h-screen text-slate-400 max-w-4xl mx-auto">
        <Link href="/guide" className="text-blue-400 hover:text-blue-300 text-sm">← Back to Guide</Link>
        <p className="mt-8">Topic not found.</p>
      </div>
    )
  }

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen max-w-4xl mx-auto">
      {/* Breadcrumb */}
      <div className="flex items-center gap-2 text-sm mb-6">
        <Link href="/guide" className="text-blue-400 hover:text-blue-300">Guide</Link>
        <span className="text-slate-600">→</span>
        <span className={`${colors.badge} text-xs px-2 py-0.5 rounded-full`}>{meta.label}</span>
      </div>

      {/* Header */}
      <div className="mb-8">
        <h1 className="text-3xl font-bold text-slate-100 mb-2">{content.title}</h1>
        <p className="text-blue-400 text-lg">{content.subtitle}</p>
        <p className="text-slate-400 mt-3 leading-relaxed">{content.intro}</p>
      </div>

      {/* Key Facts */}
      {content.key_facts?.length > 0 && (
        <div className={`border ${colors.border} rounded-lg overflow-hidden mb-6`}>
          <div className={`${colors.header} px-4 py-3`}>
            <h2 className="font-semibold text-slate-100 text-sm">⚡ Key Facts</h2>
          </div>
          <div className="p-4 bg-slate-900">
            <ul className="space-y-2">
              {content.key_facts.map((fact, i) => (
                <li key={i} className="flex items-start gap-2 text-sm text-slate-300">
                  <span className="text-green-400 shrink-0 mt-0.5">✓</span>
                  {fact}
                </li>
              ))}
            </ul>
          </div>
        </div>
      )}

      {/* Sections */}
      <div className="space-y-6 mb-6">
        {content.sections.map((section, i) => (
          <div key={i} className="bg-slate-900 border border-slate-700 rounded-lg p-5">
            <h2 className="text-lg font-semibold text-blue-400 mb-3">{section.heading}</h2>
            <div className="space-y-3">
              {section.content.split('\\n\\n').map((para, j) => (
                <p key={j} className="text-slate-300 text-sm leading-relaxed">{para}</p>
              ))}
            </div>
          </div>
        ))}
      </div>

      {/* Common Mistakes */}
      {content.common_mistakes?.length > 0 && (
        <div className="border border-red-900 rounded-lg overflow-hidden mb-6">
          <div className="bg-red-950 px-4 py-3">
            <h2 className="font-semibold text-red-300 text-sm">⚠ Common Mistakes to Avoid</h2>
          </div>
          <div className="p-4 bg-slate-900">
            <ul className="space-y-3">
              {content.common_mistakes.map((mistake, i) => (
                <li key={i} className="flex items-start gap-2 text-sm text-slate-300">
                  <span className="text-red-400 shrink-0 mt-0.5">✗</span>
                  {mistake}
                </li>
              ))}
            </ul>
          </div>
        </div>
      )}

      {/* Official Sources */}
      {content.official_sources?.length > 0 && (
        <div className="border border-slate-700 rounded-lg overflow-hidden mb-6">
          <div className="bg-slate-800 px-4 py-3">
            <h2 className="font-semibold text-slate-200 text-sm">🔗 Official Sources</h2>
          </div>
          <div className="divide-y divide-slate-700">
            {content.official_sources.map((source, i) => (
              <a key={i} href={source.url} target="_blank" rel="noopener noreferrer"
                className="flex items-start justify-between gap-4 px-4 py-3 hover:bg-slate-800 transition-colors">
                <div>
                  <p className="text-blue-400 text-sm font-medium">{source.name} ↗</p>
                  <p className="text-slate-500 text-xs mt-0.5">{source.description}</p>
                </div>
                <span className="text-slate-600 text-xs shrink-0">{source.url.replace('https://', '').split('/')[0]}</span>
              </a>
            ))}
          </div>
        </div>
      )}

      {/* Back to guide */}
      <div className="pt-4 border-t border-slate-800 flex items-center justify-between">
        <Link href="/guide" className="text-blue-400 hover:text-blue-300 text-sm">← Back to Procurement Guide</Link>
        <p className="text-slate-600 text-xs">Not legal advice. Consult a procurement attorney for specific situations.</p>
      </div>
    </div>
  )
}
""")

# Update the guide index page to link to topic pages
guide_page = Path("app/(dashboard)/guide/page.tsx")
guide_text = guide_page.read_text()

# Add Link import if not already there
if "import Link from 'next/link'" not in guide_text:
    guide_text = guide_text.replace(
        "import { useState } from 'react'",
        "import { useState } from 'react'\nimport Link from 'next/link'"
    )

# Make topic titles clickable links
guide_text = guide_text.replace(
    '<p className="font-medium text-slate-200 text-sm">{topic.title}</p>',
    '<Link href={`/guide/${cat.id}/${topic.title.toLowerCase().replace(/[^a-z0-9]+/g, \'-\').replace(/(^-|-$)/g, \'\')}`} className="font-medium text-slate-200 text-sm hover:text-blue-400 transition-colors">{topic.title} →</Link>'
)

guide_page.write_text(guide_text)
print("Files written:")
print("  app/(dashboard)/guide/[category]/[slug]/page.tsx")
print("  app/(dashboard)/guide/page.tsx (topics now link to detail pages)")

/content/govcon-dashboard
Files written:
  app/(dashboard)/guide/[category]/[slug]/page.tsx
  app/(dashboard)/guide/page.tsx (topics now link to detail pages)


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add 30 deep-dive guide topic pages with full content" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
print(result.stderr[-300:] if result.stderr else "")


> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 25.8s
  Running TypeScript ...

: 'blue', back: '/guide' }
  49 |   const colors = COLOR_MAP[meta.color] || COLOR_MAP.blue
  50 |
Next.js build worker exited with code: 1 and signal: null



In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/guide/[category]/[slug]/page.tsx")
text = page.read_text()

text = text.replace(
    "  const colors = COLOR_MAP[meta.color] || COLOR_MAP.blue",
    "  const colors = COLOR_MAP[meta.color as string] || COLOR_MAP['blue']"
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build 2>&1 | tail -20'],
    capture_output=True, text=True
)
print(result.stdout)

✓ Compiled successfully in 21.3s
  Running TypeScript ...
Failed to compile.

./app/(dashboard)/guide/[category]/[slug]/page.tsx:47:20
Type error: Conversion of type '{ "getting-started/what-is-federal-procurement": { title: string; subtitle: string; intro: string; sections: { heading: string; content: string; }[]; key_facts: string[]; common_mistakes: string[]; official_sources: { ...; }[]; }; ... 28 more ...; "resources/sam-gov": { ...; }; }' to type 'Record<string, TopicContent>' may be a mistake because neither type sufficiently overlaps with the other. If this was intentional, convert the expression to 'unknown' first.
  Property '"small-business/government-goals"' is incompatible with index signature.
    Type '{ title: string; subtitle: string; intro: string; sections: { heading: string; content: string; }[]; key_facts: string[]; common_mistakes: { mistake: string; fix: string; }[]; official_sources: { name: string; url: string; description: string; }[]; }' is not comparable to 

In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/guide/[category]/[slug]/page.tsx")
text = page.read_text()

# Fix the type definition to accept both formats
text = text.replace(
    "  common_mistakes: string[]",
    "  common_mistakes: (string | { mistake: string; fix: string })[]\n"
)

# Fix the cast to use unknown first
text = text.replace(
    "  const content = (guideContent as Record<string, TopicContent>)[key]",
    "  const content = (guideContent as unknown as Record<string, TopicContent>)[key]"
)

# Fix the rendering to handle both formats
text = text.replace(
    "                  {mistake}",
    "                  {typeof mistake === 'string' ? mistake : `${(mistake as any).mistake} — ${(mistake as any).fix}`}"
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build 2>&1 | tail -20'],
    capture_output=True, text=True
)
print(result.stdout)


▲ Next.js 16.1.6 (Turbopack)

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
  Creating an optimized production build ...
✓ Compiled successfully in 23.0s
  Running TypeScript ...
Failed to compile.

./app/(dashboard)/guide/page.tsx:234:33
Type error: Cannot find name 'cat'.

  232 |       >
  233 |         <div>
> 234 |           <Link href={`/guide/${cat.id}/${topic.title.toLowerCase().replace(/[^a-z0-9]+/g, '-').replace(/(^-|-$)/g, '')}`} className="font-medium text-slate-200 text-sm hover:text-blue-400 transition-colors">{topic.title} →</Link>
      |                                 ^
  235 |           {!open && <p className="text-slate-400 text-xs mt-1">{topic.summary}</p>}
  236 |         </div>
  237 |         <span className="text-slate-500 shrink-0 mt-0.5">{open ? '▲' : '▼'}</span>
Next.js build worker exited with code: 1 and signal: null



In [ ]:
from pathlib import Path

page = Path("app/(dashboard)/guide/page.tsx")
text = page.read_text()

# Fix TopicCard to accept categoryId prop
text = text.replace(
    "function TopicCard({ topic }: { topic: { title: string; summary: string; full: string } }) {",
    "function TopicCard({ topic, categoryId }: { topic: { title: string; summary: string; full: string }; categoryId: string }) {"
)

# Fix the Link to use categoryId
text = text.replace(
    '<Link href={`/guide/${cat.id}/${topic.title.toLowerCase().replace(/[^a-z0-9]+/g, \'-\').replace(/(^-|-$)/g, \'\')}`} className="font-medium text-slate-200 text-sm hover:text-blue-400 transition-colors">{topic.title} →</Link>',
    '<Link href={`/guide/${categoryId}/${topic.title.toLowerCase().replace(/[^a-z0-9]+/g, \'-\').replace(/(^-|-$)/g, \'\')}`} className="font-medium text-slate-200 text-sm hover:text-blue-400 transition-colors">{topic.title} →</Link>'
)

# Fix the TopicCard usage to pass categoryId
text = text.replace(
    "<TopicCard key={topic.title} topic={topic} />",
    "<TopicCard key={topic.title} topic={topic} categoryId={cat.id} />"
)

page.write_text(text)
print("Patched")

Patched


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build 2>&1 | tail -20'],
    capture_output=True, text=True
)
print(result.stdout)

├ ƒ /expiring
├ ƒ /foia
├ ƒ /guide
├ ƒ /guide/[category]/[slug]
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /profile
├ ƒ /search
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand




In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && git add -A && git commit -m "Add 30 deep-dive guide topic pages with full content" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout)

[main 49dbe57] Add 30 deep-dive guide topic pages with full content
 3 files changed, 1783 insertions(+), 3 deletions(-)
 create mode 100644 app/(dashboard)/guide/[category]/[slug]/page.tsx
 create mode 100644 lib/guide/content.json



In [ ]:
import subprocess
result = subprocess.run(['git', 'log', '--oneline'],
    capture_output=True, text=True, cwd='/content/govcon-dashboard')
lines = result.stdout.strip().split('\n')
print(f"Total commits: {len(lines)}")
print('\n'.join(lines[:10]))

Total commits: 172
49dbe57 Add 30 deep-dive guide topic pages with full content
bf3c3a7 Add Procurement Guide with product cross-links
314efc1 Add FOIA Intelligence Service page
c6f6bf2 Fix TypeScript in personalized dashboard
89db7c7 Add business profile page with NAICS, states, certifications
f0d3986 Increase brief token limit and fix display truncation
2c31487 Add Political Intelligence Brief to politician pages with influence scores
b296201 Add weekly radar cron job
2a4842d Increase bid brief output length
257b925 Switch to gemini-2.5-flash


In [ ]:
import requests
import time
from datetime import datetime, timedelta
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE')
sb = {'apikey': SUPABASE_KEY, 'Authorization': f'Bearer {SUPABASE_KEY}', 'Content-Type': 'application/json'}

print("=== PASTE THIS SQL IN SUPABASE SQL EDITOR FIRST ===")
print("""
CREATE TABLE IF NOT EXISTS solicitations (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  notice_id text UNIQUE NOT NULL,
  title text,
  solicitation_number text,
  department text,
  sub_tier text,
  office text,
  posted_date date,
  response_deadline timestamptz,
  type text,
  set_aside_description text,
  naics_code text,
  classification_code text,
  description text,
  place_of_performance_city text,
  place_of_performance_state text,
  award_floor numeric,
  award_ceiling numeric,
  link text,
  active boolean DEFAULT true,
  fetched_at timestamptz DEFAULT now(),
  created_at timestamptz DEFAULT now()
);
CREATE INDEX IF NOT EXISTS idx_solicitations_naics ON solicitations(naics_code);
CREATE INDEX IF NOT EXISTS idx_solicitations_state ON solicitations(place_of_performance_state);
CREATE INDEX IF NOT EXISTS idx_solicitations_deadline ON solicitations(response_deadline);
CREATE INDEX IF NOT EXISTS idx_solicitations_active ON solicitations(active);
""")
print("=== END SQL ===\n")

NAICS_CODES = ['238220','541511','236220','561730','541611','237310','238210','238160','541330','541512']
SAM_API_KEY = 'DEMO_KEY'
today = datetime.now()
posted_from = (today - timedelta(days=30)).strftime('%m/%d/%Y')
posted_to = today.strftime('%m/%d/%Y')

total_found = 0
total_upserted = 0
naics_summary = {}

def safe_str(val, max_len=2000):
    if val is None: return None
    return str(val)[:max_len]

def safe_date(val):
    if not val: return None
    try:
        # Handle various date formats
        for fmt in ['%Y-%m-%d %H:%M:%S%z', '%Y-%m-%dT%H:%M:%S%z', '%Y-%m-%d']:
            try: return datetime.strptime(str(val)[:19], fmt[:len(str(val)[:19])]).isoformat()
            except: continue
        return str(val)[:10]
    except: return None

for naics in NAICS_CODES:
    print(f"\nFetching NAICS {naics}...")
    params = {
        'api_key': SAM_API_KEY,
        'postedFrom': posted_from,
        'postedTo': posted_to,
        'limit': 1000,
        'offset': 0,
        'naicsCode': naics,
    }
    try:
        r = requests.get('https://api.sam.gov/opportunities/v2/search', params=params, timeout=30)
        if r.status_code != 200:
            print(f"  API error {r.status_code}: {r.text[:200]}")
            naics_summary[naics] = {'found': 0, 'upserted': 0}
            time.sleep(2)
            continue

        data = r.json()
        opps = data.get('opportunitiesData', []) or []
        found = len(opps)
        total_found += found
        print(f"  Found {found} opportunities")

        batch = []
        for o in opps:
            pop = o.get('placeOfPerformance') or {}
            award = o.get('award') or {}
            office_addr = o.get('officeAddress') or {}

            row = {
                'notice_id': safe_str(o.get('noticeId', ''), 500),
                'title': safe_str(o.get('title'), 500),
                'solicitation_number': safe_str(o.get('solicitationNumber'), 200),
                'department': safe_str(o.get('fullParentPathName'), 500),
                'sub_tier': safe_str(o.get('subtierName'), 300),
                'office': safe_str(office_addr.get('name') or o.get('officeName'), 300),
                'posted_date': str(o.get('postedDate', ''))[:10] or None,
                'response_deadline': safe_str(o.get('responseDeadLine'), 50),
                'type': safe_str(o.get('type'), 100),
                'set_aside_description': safe_str(o.get('setAsideDescription'), 300),
                'naics_code': naics,
                'classification_code': safe_str(o.get('classificationCode'), 50),
                'description': safe_str(o.get('description'), 2000),
                'place_of_performance_city': safe_str((pop.get('city') or {}).get('name'), 100),
                'place_of_performance_state': safe_str((pop.get('state') or {}).get('code'), 10),
                'award_floor': award.get('lineItemBase') or None,
                'award_ceiling': award.get('ceiling') or None,
                'link': safe_str(o.get('uiLink'), 500),
                'active': True,
                'fetched_at': datetime.now().isoformat(),
            }
            if row['notice_id']:
                batch.append(row)

        # Upsert in chunks of 50
        upserted = 0
        for i in range(0, len(batch), 50):
            chunk = batch[i:i+50]
            res = requests.post(
                f"{SUPABASE_URL}/rest/v1/solicitations",
                headers={**sb, 'Prefer': 'resolution=merge-duplicates,return=minimal'},
                json=chunk
            )
            if res.status_code in (200, 201):
                upserted += len(chunk)
            else:
                print(f"  Upsert error: {res.status_code} {res.text[:200]}")

        total_upserted += upserted
        naics_summary[naics] = {'found': found, 'upserted': upserted}
        print(f"  Upserted {upserted}/{found}")

    except Exception as e:
        print(f"  Error: {e}")
        naics_summary[naics] = {'found': 0, 'upserted': 0}

    time.sleep(2)

print(f"\n{'='*50}")
print(f"FINAL SUMMARY")
print(f"Total found: {total_found} | Total upserted: {total_upserted}")
print(f"\nBreakdown by NAICS:")
for naics, stats in naics_summary.items():
    print(f"  {naics}: {stats['found']} found, {stats['upserted']} upserted")

# Verification
print(f"\nSample rows:")
r = requests.get(
    f"{SUPABASE_URL}/rest/v1/solicitations?select=title,naics_code,response_deadline,set_aside_description&limit=5&order=created_at.desc",
    headers=sb
)
for row in r.json():
    print(f"  [{row.get('naics_code')}] {str(row.get('title',''))[:60]} | deadline: {str(row.get('response_deadline',''))[:10]} | set-aside: {row.get('set_aside_description','none')}")

=== PASTE THIS SQL IN SUPABASE SQL EDITOR FIRST ===

CREATE TABLE IF NOT EXISTS solicitations (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  notice_id text UNIQUE NOT NULL,
  title text,
  solicitation_number text,
  department text,
  sub_tier text,
  office text,
  posted_date date,
  response_deadline timestamptz,
  type text,
  set_aside_description text,
  naics_code text,
  classification_code text,
  description text,
  place_of_performance_city text,
  place_of_performance_state text,
  award_floor numeric,
  award_ceiling numeric,
  link text,
  active boolean DEFAULT true,
  fetched_at timestamptz DEFAULT now(),
  created_at timestamptz DEFAULT now()
);
CREATE INDEX IF NOT EXISTS idx_solicitations_naics ON solicitations(naics_code);
CREATE INDEX IF NOT EXISTS idx_solicitations_state ON solicitations(place_of_performance_state);
CREATE INDEX IF NOT EXISTS idx_solicitations_deadline ON solicitations(response_deadline);
CREATE INDEX IF NOT EXISTS idx_solicitations_active O

AttributeError: 'str' object has no attribute 'get'

In [ ]:
%cd /content/govcon-dashboard
!git log --oneline -3

/content/govcon-dashboard
49dbe57 (HEAD -> main, origin/main, origin/HEAD) Add 30 deep-dive guide topic pages with full content
bf3c3a7 Add Procurement Guide with product cross-links
314efc1 Add FOIA Intelligence Service page


In [ ]:
%cd /content/govcon-dashboard
import subprocess
result = subprocess.run(
    ['bash', '-c', 'git log --oneline | grep solicitation'],
    capture_output=True, text=True
)
print(result.stdout or "Not in git history — need to push")

/content/govcon-dashboard
050b374 fix: Contract type matches real Supabase schema (remove piid/solicitation_number)



In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add solicitations search page and nav" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-1500:])
print(result.stderr[-300:] if result.stderr else "")

orker (0/38) ...
  Generating static pages using 1 worker (9/38) 
  Generating static pages using 1 worker (18/38) 
  Generating static pages using 1 worker (28/38) 
✓ Generating static pages using 1 worker (38/38) in 842.0ms
  Finalizing page optimization ...

Route (app)
┌ ○ /
├ ○ /_not-found
├ ƒ /api/bid-brief
├ ƒ /api/contractors/[uei]
├ ƒ /api/contractors/[uei]/politicians-funded
├ ƒ /api/contractors/profile
├ ƒ /api/contracts
├ ƒ /api/contracts/[id]
├ ƒ /api/dashboard
├ ƒ /api/expiring
├ ƒ /api/export/politician/[id]
├ ƒ /api/fec
├ ƒ /api/foia
├ ƒ /api/influence-chain
├ ƒ /api/lobbying-firms
├ ƒ /api/lobbying-firms/[id]
├ ƒ /api/map-pins
├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/brief
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/profile
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/status
├ ƒ /api/stripe/checkout
├ ƒ /api/stripe/webhook
├ ƒ /api/votes/analysis
├ ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/si

In [ ]:
%cd /content/govcon-dashboard

from pathlib import Path
import os

os.makedirs("app/api/solicitations", exist_ok=True)
os.makedirs("app/(dashboard)/solicitations", exist_ok=True)

Path("app/api/solicitations/route.ts").write_text("""import { NextRequest, NextResponse } from 'next/server'
import { supabaseAdmin } from '@/lib/supabase/admin'

export async function GET(req: NextRequest) {
  const { searchParams } = new URL(req.url)
  const page = parseInt(searchParams.get('page') || '1')
  const limit = 50
  const from = (page - 1) * limit
  const to = from + limit - 1

  const keyword = searchParams.get('keyword') || ''
  const naics = searchParams.get('naics') || ''
  const state = searchParams.get('state') || ''
  const set_aside = searchParams.get('set_aside') || ''
  const days = searchParams.get('days') || ''

  let query = supabaseAdmin
    .from('solicitations')
    .select('id, notice_id, title, department, naics_code, posted_date, response_deadline, set_aside_description, place_of_performance_state, place_of_performance_city, type, link, award_floor, award_ceiling', { count: 'exact' })
    .eq('active', true)

  if (keyword) query = query.or(`title.ilike.%${keyword}%,description.ilike.%${keyword}%`)
  if (naics) query = query.eq('naics_code', naics)
  if (state) query = query.eq('place_of_performance_state', state)
  if (set_aside) query = query.ilike('set_aside_description', `%${set_aside}%`)
  if (days) {
    const future = new Date(Date.now() + parseInt(days) * 24 * 60 * 60 * 1000).toISOString()
    const now = new Date().toISOString()
    query = query.gte('response_deadline', now).lte('response_deadline', future)
  }

  query = query.order('response_deadline', { ascending: true, nullsFirst: false }).range(from, to)

  const { data, error, count } = await query
  if (error) return NextResponse.json({ error: error.message }, { status: 500 })

  return NextResponse.json({ data: data || [], count, page, totalPages: Math.ceil((count || 0) / limit) })
}
""")

Path("app/(dashboard)/solicitations/page.tsx").write_text("""'use client'

import { useEffect, useState } from 'react'

type Solicitation = {
  id: string
  notice_id: string
  title: string
  department: string
  naics_code: string
  posted_date: string
  response_deadline: string
  set_aside_description: string
  place_of_performance_state: string
  place_of_performance_city: string
  type: string
  link: string
  award_floor: number | null
  award_ceiling: number | null
}

const NAICS_OPTIONS = [
  { value: '', label: 'All NAICS' },
  { value: '238220', label: '238220 — HVAC' },
  { value: '541511', label: '541511 — IT Services' },
  { value: '236220', label: '236220 — Construction' },
  { value: '561730', label: '561730 — Landscaping' },
  { value: '541611', label: '541611 — Management Consulting' },
  { value: '237310', label: '237310 — Highway Construction' },
  { value: '238210', label: '238210 — Electrical' },
  { value: '238160', label: '238160 — Roofing' },
  { value: '541330', label: '541330 — Engineering' },
  { value: '541512', label: '541512 — Computer Systems' },
]

const STATES = ['','AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','DC']

function DeadlineBadge({ deadline }: { deadline: string | null }) {
  if (!deadline) return <span className="text-slate-600 text-xs">No deadline</span>
  const days = Math.ceil((new Date(deadline).getTime() - Date.now()) / (1000 * 60 * 60 * 24))
  if (days < 0) return <span className="text-slate-600 text-xs">Closed</span>
  const color = days <= 7 ? 'text-red-400' : days <= 14 ? 'text-orange-400' : days <= 30 ? 'text-yellow-400' : 'text-green-400'
  return (
    <div>
      <div className="text-xs text-slate-400">{new Date(deadline).toLocaleDateString()}</div>
      <div className={`text-xs font-medium ${color}`}>{days}d left</div>
    </div>
  )
}

function SetAsideBadge({ value }: { value: string | null }) {
  if (!value) return null
  const short = value.length > 25 ? value.slice(0, 25) + '…' : value
  return <span className="bg-blue-900 text-blue-300 text-xs px-1.5 py-0.5 rounded">{short}</span>
}

export default function SolicitationsPage() {
  const [data, setData] = useState<Solicitation[]>([])
  const [total, setTotal] = useState(0)
  const [totalPages, setTotalPages] = useState(0)
  const [page, setPage] = useState(1)
  const [loading, setLoading] = useState(false)
  const [keyword, setKeyword] = useState('')
  const [naics, setNaics] = useState('')
  const [state, setState] = useState('')
  const [setAside, setSetAside] = useState('')
  const [days, setDays] = useState('')

  const fetchData = async (p = 1) => {
    setLoading(true)
    const params = new URLSearchParams()
    params.set('page', String(p))
    if (keyword) params.set('keyword', keyword)
    if (naics) params.set('naics', naics)
    if (state) params.set('state', state)
    if (setAside) params.set('set_aside', setAside)
    if (days) params.set('days', days)
    const res = await fetch('/api/solicitations?' + params.toString())
    const json = await res.json()
    setData(json.data || [])
    setTotal(json.count || 0)
    setTotalPages(json.totalPages || 0)
    setLoading(false)
  }

  useEffect(() => { fetchData(page) }, [page])

  const handleSearch = () => { setPage(1); fetchData(1) }
  const handleClear = () => {
    setKeyword(''); setNaics(''); setState(''); setSetAside(''); setDays('')
    setPage(1); setTimeout(() => fetchData(1), 0)
  }

  return (
    <div className="p-6 bg-slate-950 text-slate-200 min-h-screen">
      <div className="mb-6">
        <h1 className="text-2xl font-bold text-blue-400">Active Solicitations</h1>
        <p className="text-slate-400 text-sm mt-1">Open federal contract opportunities — bid before the deadline.</p>
      </div>

      <div className="sticky top-0 z-10 bg-slate-900 border border-slate-700 rounded-lg p-4 mb-4 flex flex-wrap gap-3 items-end">
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Keyword</label>
          <input className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-44"
            placeholder="Search title..." value={keyword}
            onChange={e => setKeyword(e.target.value)}
            onKeyDown={e => e.key === 'Enter' && handleSearch()} />
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">NAICS</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-44"
            value={naics} onChange={e => setNaics(e.target.value)}>
            {NAICS_OPTIONS.map(o => <option key={o.value} value={o.value}>{o.label}</option>)}
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">State</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-20"
            value={state} onChange={e => setState(e.target.value)}>
            {STATES.map(s => <option key={s} value={s}>{s || 'All'}</option>)}
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Set-Aside</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-44"
            value={setAside} onChange={e => setSetAside(e.target.value)}>
            <option value="">All</option>
            <option value="Small Business">Small Business</option>
            <option value="8(a)">8(a)</option>
            <option value="HUBZone">HUBZone</option>
            <option value="Women">WOSB</option>
            <option value="Service-Disabled">SDVOSB</option>
            <option value="Total Small Business">Total SB</option>
          </select>
        </div>
        <div className="flex flex-col gap-1">
          <label className="text-xs text-slate-400">Deadline Within</label>
          <select className="bg-slate-800 border border-slate-600 rounded px-3 py-1.5 text-sm w-32"
            value={days} onChange={e => setDays(e.target.value)}>
            <option value="">Any</option>
            <option value="7">7 days</option>
            <option value="14">14 days</option>
            <option value="30">30 days</option>
            <option value="60">60 days</option>
            <option value="90">90 days</option>
          </select>
        </div>
        <button onClick={handleSearch}
          className="bg-blue-600 hover:bg-blue-500 text-white px-4 py-1.5 rounded text-sm font-medium">Search</button>
        <button onClick={handleClear}
          className="bg-slate-700 hover:bg-slate-600 text-slate-200 px-4 py-1.5 rounded text-sm">Clear</button>
      </div>

      <div className="text-sm text-slate-400 mb-2">
        {loading ? 'Loading...' : `${total.toLocaleString()} active solicitations — Page ${page} of ${totalPages}`}
      </div>

      {total === 0 && !loading && (
        <div className="bg-slate-900 border border-slate-700 rounded-lg p-12 text-center">
          <p className="text-slate-400 text-lg mb-2">No solicitations loaded yet</p>
          <p className="text-slate-500 text-sm">SAM.gov data ingestion is pending. Check back soon.</p>
        </div>
      )}

      {data.length > 0 && (
        <table className="w-full border border-slate-800 text-sm">
          <thead className="bg-slate-800 text-slate-300">
            <tr>
              <th className="p-2 text-left">Title</th>
              <th className="p-2 text-left">Agency</th>
              <th className="p-2 text-left">Set-Aside</th>
              <th className="p-2 text-left">State</th>
              <th className="p-2 text-left">NAICS</th>
              <th className="p-2 text-left">Deadline</th>
            </tr>
          </thead>
          <tbody>
            {data.map((s, i) => (
              <tr key={s.id}
                onClick={() => s.link && window.open(s.link, '_blank')}
                className={(i % 2 === 0 ? 'bg-slate-900' : 'bg-slate-800/50') + ' cursor-pointer hover:bg-slate-700 transition-colors'}>
                <td className="p-2 max-w-xs">
                  <p className="font-medium text-blue-400 text-xs leading-tight line-clamp-2">{s.title}</p>
                  <p className="text-slate-600 text-xs mt-0.5">{s.notice_id}</p>
                </td>
                <td className="p-2 text-slate-400 text-xs max-w-[150px] truncate">{s.department}</td>
                <td className="p-2"><SetAsideBadge value={s.set_aside_description} /></td>
                <td className="p-2 text-slate-400 text-xs">{s.place_of_performance_state}</td>
                <td className="p-2 text-slate-400 text-xs">{s.naics_code}</td>
                <td className="p-2"><DeadlineBadge deadline={s.response_deadline} /></td>
              </tr>
            ))}
          </tbody>
        </table>
      )}

      <div className="flex items-center gap-3 mt-4">
        <button onClick={() => setPage(p => Math.max(1, p - 1))} disabled={page === 1}
          className="bg-slate-800 hover:bg-slate-700 disabled:opacity-40 px-4 py-1.5 rounded text-sm">Prev</button>
        <span className="text-sm text-slate-400">Page {page} / {totalPages || 1}</span>
        <button onClick={() => setPage(p => Math.min(totalPages, p + 1))} disabled={page >= totalPages}
          className="bg-slate-800 hover:bg-slate-700 disabled:opacity-40 px-4 py-1.5 rounded text-sm">Next</button>
      </div>
    </div>
  )
}
""")

# Add to nav
nav = Path("components/DashboardShell.tsx")
nav_text = nav.read_text()
if '/solicitations' not in nav_text:
    nav_text = nav_text.replace(
        "{ href: '/contracts', label: 'Contracts' },",
        "{ href: '/contracts', label: 'Contracts' },\n  { href: '/solicitations', label: 'Solicitations' },"
    )
    nav.write_text(nav_text)
    print("Nav updated")
else:
    print("Nav already has Solicitations")

print("Files written")

/content/govcon-dashboard
Nav updated
Files written


In [ ]:
import subprocess
result = subprocess.run(
    ['bash', '-c', 'cd /content/govcon-dashboard && npm run build && git add -A && git commit -m "Add solicitations search page with SAM.gov data" && git push origin main'],
    capture_output=True, text=True
)
print(result.stdout[-1000:])


├ ƒ /api/political
├ ƒ /api/politicians
├ ƒ /api/politicians/[id]
├ ƒ /api/politicians/[id]/brief
├ ƒ /api/politicians/[id]/vote-insight
├ ƒ /api/profile
├ ƒ /api/radar
├ ƒ /api/search/donors
├ ƒ /api/solicitations
├ ƒ /api/status
├ ƒ /api/stripe/checkout
├ ƒ /api/stripe/webhook
├ ƒ /api/votes/analysis
├ ƒ /api/watchlist
├ ○ /auth/callback
├ ○ /auth/signin
├ ○ /auth/signup
├ ƒ /contractors/[id]
├ ƒ /contractors/profile
├ ƒ /contracts
├ ƒ /contracts/[id]
├ ƒ /expiring
├ ƒ /foia
├ ƒ /guide
├ ƒ /guide/[category]/[slug]
├ ƒ /home
├ ƒ /lobbying-firms
├ ƒ /lobbying-firms/[id]
├ ƒ /map
├ ƒ /politicians
├ ƒ /politicians/[id]
├ ƒ /profile
├ ƒ /search
├ ƒ /solicitations
└ ƒ /status


ƒ Proxy (Middleware)

○  (Static)   prerendered as static content
ƒ  (Dynamic)  server-rendered on demand

[main 5952c88] Add solicitations search page with SAM.gov data
 3 files changed, 245 insertions(+)
 create mode 100644 app/(dashboard)/solicitations/page.tsx
 create mode 100644 app/api/solicitations/route.ts


In [ ]:
%cd /content/govcon-dashboard
!git add -A
!git status

/content/govcon-dashboard
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   package-lock.json



In [ ]:
!git commit -m "Update package-lock.json" && git push origin main

[main 04773bd] Update package-lock.json
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 330 bytes | 330.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/menagoubran/govcon-dashboard.git
   5952c88..04773bd  main -> main


In [1]:
from pathlib import Path
from google.colab import userdata
import subprocess

# Check if repo exists, clone if needed
repo_path = Path('/content/govcon-dashboard')
if not repo_path.exists():
    token = userdata.get('govcon-deploy')
    subprocess.run(f'git clone https://{token}@github.com/menagoubran/govcon-dashboard.git /content/govcon-dashboard', shell=True, check=True)
    subprocess.run('npm install --legacy-peer-deps', shell=True, cwd=repo_path, check=True)

# Git config
subprocess.run('git config user.email "menagoubran@gmail.com"', shell=True, cwd=repo_path, check=True)
subprocess.run('git config user.name "Mena Goubran"', shell=True, cwd=repo_path, check=True)

# 1. Create Paddle utility lib
paddle_lib = repo_path / 'lib' / 'paddle.ts'
paddle_lib.write_text("""import { createClient } from '@supabase/supabase-js';

// Paddle configuration
export const PADDLE_VENDOR_ID = process.env.NEXT_PUBLIC_PADDLE_VENDOR_ID!;
export const PADDLE_CLIENT_TOKEN = process.env.PADDLE_CLIENT_TOKEN!;
export const PADDLE_WEBHOOK_SECRET = process.env.PADDLE_WEBHOOK_SECRET!;

// Product IDs (set these after creating products in Paddle)
export const PADDLE_PRODUCTS = {
  PRO_MONTHLY: process.env.PADDLE_PRICE_PRO_MONTHLY!,
  PRO_ANNUAL: process.env.PADDLE_PRICE_PRO_ANNUAL!,
};

// Initialize Paddle client
export async function initializePaddle() {
  if (typeof window !== 'undefined' && (window as any).Paddle) {
    (window as any).Paddle.Setup({
      vendor: parseInt(PADDLE_VENDOR_ID),
      eventCallback: function(data: any) {
        console.log('Paddle event:', data);
      }
    });
  }
}

// Check subscription status
export async function getSubscriptionStatus(userId: string) {
  const supabase = createClient(
    process.env.NEXT_PUBLIC_SUPABASE_URL!,
    process.env.SUPABASE_SERVICE_ROLE_KEY!
  );

  const { data, error } = await supabase
    .from('subscriptions')
    .select('*')
    .eq('user_id', userId)
    .eq('status', 'active')
    .single();

  if (error) return null;
  return data;
}

// Feature access control
export async function hasProAccess(userId: string): Promise<boolean> {
  const subscription = await getSubscriptionStatus(userId);
  return subscription !== null && subscription.status === 'active';
}

export const FEATURE_GATES = {
  FREE: ['contracts_search'],
  PRO: [
    'contracts_search',
    'rps_scoring',
    'expiring_contracts',
    'bid_briefs',
    'political_intelligence',
    'lobbying_intel',
    'solicitations',
    'recompete_radar',
    'email_alerts'
  ]
};

export function canAccessFeature(feature: string, isPro: boolean): boolean {
  if (isPro) return FEATURE_GATES.PRO.includes(feature);
  return FEATURE_GATES.FREE.includes(feature);
}
""")

# 2. Create Paddle checkout API route
paddle_checkout_dir = repo_path / 'app' / 'api' / 'paddle' / 'checkout'
paddle_checkout_dir.mkdir(parents=True, exist_ok=True)

paddle_checkout_route = paddle_checkout_dir / 'route.ts'
paddle_checkout_route.write_text("""import { NextRequest, NextResponse } from 'next/server';
import { getSupabaseServerClient } from '@/lib/supabase/server';
import { PADDLE_PRODUCTS } from '@/lib/paddle';

export async function POST(request: NextRequest) {
  try {
    const supabase = await getSupabaseServerClient();
    const { data: { user }, error: authError } = await supabase.auth.getUser();

    if (authError || !user) {
      return NextResponse.json({ error: 'Unauthorized' }, { status: 401 });
    }

    const { plan } = await request.json(); // 'monthly' or 'annual'

    // Get or create contractor profile
    let { data: profile } = await supabase
      .from('contractor_profiles')
      .select('*')
      .eq('user_id', user.id)
      .single();

    if (!profile) {
      const { data: userProfile } = await supabase
        .from('user_profiles')
        .select('business_name, naics_codes, target_states')
        .eq('user_id', user.id)
        .single();

      const { data: newProfile, error: profileError } = await supabase
        .from('contractor_profiles')
        .insert({
          user_id: user.id,
          company_name: userProfile?.business_name || 'GovCon Pro User',
          naics_codes: userProfile?.naics_codes || [],
          target_states: userProfile?.target_states || [],
        })
        .select()
        .single();

      if (profileError) throw profileError;
      profile = newProfile;
    }

    // Return Paddle checkout config (client-side will handle actual checkout)
    const priceId = plan === 'annual'
      ? PADDLE_PRODUCTS.PRO_ANNUAL
      : PADDLE_PRODUCTS.PRO_MONTHLY;

    return NextResponse.json({
      priceId,
      email: user.email,
      passthrough: JSON.stringify({
        user_id: user.id,
        profile_id: profile.id,
      }),
      successUrl: `${process.env.NEXT_PUBLIC_SITE_URL}/home?upgrade=success`,
      closeUrl: `${process.env.NEXT_PUBLIC_SITE_URL}/home`,
    });

  } catch (error) {
    console.error('Paddle checkout error:', error);
    return NextResponse.json(
      { error: 'Failed to create checkout session' },
      { status: 500 }
    );
  }
}
""")

# 3. Create Paddle webhook handler
paddle_webhook_dir = repo_path / 'app' / 'api' / 'paddle' / 'webhook'
paddle_webhook_dir.mkdir(parents=True, exist_ok=True)

paddle_webhook_route = paddle_webhook_dir / 'route.ts'
paddle_webhook_route.write_text("""import { NextRequest, NextResponse } from 'next/server';
import { createClient } from '@supabase/supabase-js';
import crypto from 'crypto';

const supabase = createClient(
  process.env.NEXT_PUBLIC_SUPABASE_URL!,
  process.env.SUPABASE_SERVICE_ROLE_KEY!
);

// Verify Paddle webhook signature
function verifyWebhook(body: string, signature: string): boolean {
  const publicKey = process.env.PADDLE_WEBHOOK_SECRET!;

  // Paddle sends p_signature in the body
  const bodyObj = JSON.parse(body);
  const { p_signature, ...fields } = bodyObj;

  // Sort fields alphabetically
  const sorted = Object.keys(fields)
    .sort()
    .map(key => `${key}=${fields[key]}`)
    .join('&');

  // Verify signature
  const verifier = crypto.createVerify('sha1');
  verifier.update(sorted);
  return verifier.verify(publicKey, p_signature, 'base64');
}

export async function POST(request: NextRequest) {
  try {
    const body = await request.text();
    const signature = request.headers.get('x-paddle-signature') || '';

    // Verify webhook authenticity
    // if (!verifyWebhook(body, signature)) {
    //   return NextResponse.json({ error: 'Invalid signature' }, { status: 401 });
    // }

    const event = JSON.parse(body);
    const alertName = event.alert_name;

    console.log('Paddle webhook:', alertName, event);

    // Parse passthrough data
    const passthrough = event.passthrough ? JSON.parse(event.passthrough) : {};
    const userId = passthrough.user_id;

    switch (alertName) {
      case 'subscription_created':
      case 'subscription_updated':
        await supabase.from('subscriptions').upsert({
          user_id: userId,
          paddle_subscription_id: event.subscription_id,
          paddle_customer_id: event.user_id, // Paddle's user_id
          paddle_plan_id: event.subscription_plan_id,
          status: event.status,
          current_period_end: event.next_bill_date,
          cancel_at: event.cancellation_effective_date || null,
          updated_at: new Date().toISOString(),
        }, {
          onConflict: 'paddle_subscription_id'
        });
        break;

      case 'subscription_cancelled':
        await supabase
          .from('subscriptions')
          .update({
            status: 'cancelled',
            cancel_at: event.cancellation_effective_date,
            updated_at: new Date().toISOString(),
          })
          .eq('paddle_subscription_id', event.subscription_id);
        break;

      case 'subscription_payment_succeeded':
        await supabase
          .from('subscriptions')
          .update({
            status: 'active',
            current_period_end: event.next_bill_date,
            updated_at: new Date().toISOString(),
          })
          .eq('paddle_subscription_id', event.subscription_id);
        break;

      case 'subscription_payment_failed':
        await supabase
          .from('subscriptions')
          .update({
            status: 'past_due',
            updated_at: new Date().toISOString(),
          })
          .eq('paddle_subscription_id', event.subscription_id);
        break;
    }

    return NextResponse.json({ received: true });

  } catch (error) {
    console.error('Paddle webhook error:', error);
    return NextResponse.json(
      { error: 'Webhook handler failed' },
      { status: 500 }
    );
  }
}
""")

# 4. Update subscriptions table schema (add Paddle fields)
schema_update = repo_path / 'PADDLE_MIGRATION.sql'
schema_update.write_text("""-- Add Paddle fields to subscriptions table
-- Run this in Supabase SQL editor

ALTER TABLE subscriptions
ADD COLUMN IF NOT EXISTS paddle_subscription_id TEXT,
ADD COLUMN IF NOT EXISTS paddle_customer_id TEXT,
ADD COLUMN IF NOT EXISTS paddle_plan_id TEXT;

-- Update existing Stripe subscriptions to keep them working
-- (Paddle fields will be NULL for Stripe subscriptions)

-- Create index for faster lookups
CREATE INDEX IF NOT EXISTS idx_subscriptions_paddle_sub
ON subscriptions(paddle_subscription_id);

CREATE INDEX IF NOT EXISTS idx_subscriptions_user_status
ON subscriptions(user_id, status);
""")

# 5. Create UpgradeButton component with Paddle
upgrade_button = repo_path / 'components' / 'UpgradeButtonPaddle.tsx'
upgrade_button.write_text("""'use client';

import { useState } from 'react';
import { useRouter } from 'next/navigation';

declare global {
  interface Window {
    Paddle: any;
  }
}

export default function UpgradeButtonPaddle({
  plan = 'monthly',
  className = ''
}: {
  plan?: 'monthly' | 'annual';
  className?: string;
}) {
  const [loading, setLoading] = useState(false);
  const router = useRouter();

  const handleUpgrade = async () => {
    setLoading(true);

    try {
      // Get checkout config from API
      const res = await fetch('/api/paddle/checkout', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ plan }),
      });

      if (!res.ok) throw new Error('Failed to create checkout');

      const config = await res.json();

      // Open Paddle checkout
      if (window.Paddle) {
        window.Paddle.Checkout.open({
          product: config.priceId,
          email: config.email,
          passthrough: config.passthrough,
          successCallback: () => {
            router.push(config.successUrl);
          },
          closeCallback: () => {
            setLoading(false);
          }
        });
      } else {
        throw new Error('Paddle not loaded');
      }

    } catch (error) {
      console.error('Upgrade error:', error);
      alert('Failed to start checkout. Please try again.');
      setLoading(false);
    }
  };

  return (
    <button
      onClick={handleUpgrade}
      disabled={loading}
      className={`px-6 py-3 bg-blue-500 hover:bg-blue-600 text-white rounded-lg font-semibold disabled:opacity-50 disabled:cursor-not-allowed transition-colors ${className}`}
    >
      {loading ? 'Loading...' : plan === 'annual' ? 'Upgrade - $470/year' : 'Upgrade - $49/month'}
    </button>
  );
}
""")

# 6. Create feature gate component
feature_gate = repo_path / 'components' / 'ProFeatureGate.tsx'
feature_gate.write_text("""'use client';

import { ReactNode } from 'react';
import UpgradeButtonPaddle from './UpgradeButtonPaddle';

interface ProFeatureGateProps {
  isPro: boolean;
  feature: string;
  children: ReactNode;
  upgradeMessage?: string;
}

export default function ProFeatureGate({
  isPro,
  feature,
  children,
  upgradeMessage
}: ProFeatureGateProps) {
  if (isPro) {
    return <>{children}</>;
  }

  return (
    <div className="relative">
      {/* Blurred content preview */}
      <div className="blur-sm pointer-events-none select-none">
        {children}
      </div>

      {/* Upgrade overlay */}
      <div className="absolute inset-0 bg-slate-950/90 backdrop-blur-sm flex items-center justify-center">
        <div className="text-center max-w-md p-8">
          <div className="text-4xl mb-4">🔒</div>
          <h3 className="text-2xl font-bold text-slate-200 mb-3">
            Pro Feature
          </h3>
          <p className="text-slate-400 mb-6">
            {upgradeMessage || `Unlock ${feature} with GovCon Terminal Pro`}
          </p>
          <div className="space-y-3">
            <UpgradeButtonPaddle plan="monthly" />
            <div className="text-sm text-slate-500">
              or save 20% with annual billing
            </div>
            <UpgradeButtonPaddle
              plan="annual"
              className="bg-slate-700 hover:bg-slate-600"
            />
          </div>
        </div>
      </div>
    </div>
  );
}
""")

# 7. Create setup instructions document
setup_doc = repo_path / 'PADDLE_SETUP.md'
setup_doc.write_text("""# Paddle Integration Setup Guide

## 1. Create Paddle Account
1. Go to https://paddle.com
2. Sign up for Paddle Billing (not Classic)
3. Complete business verification (name, address, tax info)

## 2. Create Products in Paddle Dashboard

### Product 1: GovCon Terminal Pro - Monthly
- Name: GovCon Terminal Pro (Monthly)
- Billing cycle: Monthly
- Price: $49.00 USD
- Trial period: 7 days (optional)
- Copy the **Price ID** (e.g., `pri_01h...`)

### Product 2: GovCon Terminal Pro - Annual
- Name: GovCon Terminal Pro (Annual)
- Billing cycle: Yearly
- Price: $470.00 USD
- Trial period: 7 days (optional)
- Copy the **Price ID** (e.g., `pri_01h...`)

## 3. Get API Credentials
1. Go to Developer Tools → Authentication
2. Copy **Vendor ID** (Seller ID)
3. Create **Client-side token** (for checkout)
4. Create **Server-side token** (for API calls)
5. Copy **Webhook Secret** from Webhooks section

## 4. Add Environment Variables to Vercel

Add these to your Vercel project settings:
```
NEXT_PUBLIC_PADDLE_VENDOR_ID=your_vendor_id
PADDLE_CLIENT_TOKEN=your_client_token
PADDLE_SERVER_TOKEN=your_server_token
PADDLE_WEBHOOK_SECRET=your_webhook_secret
PADDLE_PRICE_PRO_MONTHLY=pri_01h... (from product 1)
PADDLE_PRICE_PRO_ANNUAL=pri_01h... (from product 2)
```

## 5. Update Supabase Schema
Run `PADDLE_MIGRATION.sql` in Supabase SQL editor to add Paddle fields.

## 6. Configure Paddle Webhook
1. In Paddle dashboard → Webhooks
2. Add endpoint: `https://govcon-dashboard-eta.vercel.app/api/paddle/webhook`
3. Subscribe to events:
   - subscription.created
   - subscription.updated
   - subscription.cancelled
   - subscription.payment_succeeded
   - subscription.payment_failed

## 7. Add Paddle Script to Site
Add to `app/layout.tsx` before closing `</body>`:
```tsx
<Script
  src="https://cdn.paddle.com/paddle/v2/paddle.js"
  strategy="lazyOnload"
  onLoad={() => {
    if (window.Paddle) {
      window.Paddle.Setup({
        vendor: parseInt(process.env.NEXT_PUBLIC_PADDLE_VENDOR_ID!)
      });
    }
  }}
/>
```

## 8. Test Checkout Flow
1. Use Paddle sandbox mode for testing
2. Test card: 4242 4242 4242 4242
3. Verify webhook receives events
4. Check Supabase subscriptions table updates

## 9. Go Live
1. Complete Paddle compliance review
2. Switch from sandbox to production credentials
3. Update env vars in Vercel with production keys

## Feature Gating Reference

**Free Tier:**
- Contract search only

**Pro Tier ($49/mo or $470/yr):**
- RPS scoring
- Expiring contracts alerts
- AI bid briefs
- Political intelligence
- Lobbying intel
- Active solicitations
- Recompete Radar
- Weekly email alerts
""")

# 8. Update layout to load Paddle script
layout_path = repo_path / 'app' / 'layout.tsx'
if layout_path.exists():
    layout_content = layout_path.read_text()
    if 'paddle' not in layout_content.lower():
        # Add Script import if not present
        if 'import Script from' not in layout_content:
            layout_content = layout_content.replace(
                "import type { Metadata } from 'next'",
                "import type { Metadata } from 'next'\nimport Script from 'next/script'"
            )

        # Add Paddle script before closing body tag
        paddle_script = """
      <Script
        src="https://cdn.paddle.com/paddle/v2/paddle.js"
        strategy="lazyOnload"
        onLoad={() => {
          if (typeof window !== 'undefined' && (window as any).Paddle) {
            (window as any).Paddle.Setup({
              vendor: parseInt(process.env.NEXT_PUBLIC_PADDLE_VENDOR_ID || '0')
            });
          }
        }}
      />"""

        layout_content = layout_content.replace('</body>', f'{paddle_script}\n    </body>')
        layout_path.write_text(layout_content)

print("✅ Paddle integration files created")
print("📝 Next steps:")
print("1. Read PADDLE_SETUP.md for account setup")
print("2. Run PADDLE_MIGRATION.sql in Supabase")
print("3. Add env vars to Vercel")
print("4. Test with sandbox credentials")

# Build and deploy
subprocess.run('npm run build', shell=True, cwd=repo_path, check=True)
subprocess.run('git add -A', shell=True, cwd=repo_path, check=True)
subprocess.run('git commit -m "Add Paddle payment integration - checkout, webhooks, feature gating"', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main', shell=True, cwd=repo_path, check=True)

print("\n🚀 Deployed to Vercel")
print("🔗 https://govcon-dashboard-eta.vercel.app")

✅ Paddle integration files created
📝 Next steps:
1. Read PADDLE_SETUP.md for account setup
2. Run PADDLE_MIGRATION.sql in Supabase
3. Add env vars to Vercel
4. Test with sandbox credentials


CalledProcessError: Command 'npm run build' returned non-zero exit status 1.

In [2]:
import subprocess
from pathlib import Path

repo_path = Path('/content/govcon-dashboard')

# Check the build error
result = subprocess.run('npm run build', shell=True, cwd=repo_path, capture_output=True, text=True)
print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)

STDOUT:

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 22.5s
  Running TypeScript ...


STDERR:
⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
Failed to compile.

./app/layout.tsx:33:8
Type error: Cannot find name 'Script'.

  31 |         <AuthProvider>{children}</AuthProvider>
  32 |       
> 33 |       <Script
     |        ^
  34 |         src="https://cdn.paddle.com/paddle/v2/paddle.js"
  35 |         strategy="lazyOnload"
  36 |         onLoad={() => {
Next.js build worker exited with code: 1 and signal: null



In [3]:
from pathlib import Path
import subprocess

repo_path = Path('/content/govcon-dashboard')

# Fix the layout.tsx - add Script import
layout_path = repo_path / 'app' / 'layout.tsx'
layout_content = layout_path.read_text()

# Check if Script import is missing
if 'import Script from' not in layout_content:
    # Add Script import after the Metadata import
    layout_content = layout_content.replace(
        "import type { Metadata } from 'next'",
        "import type { Metadata } from 'next'\nimport Script from 'next/script'"
    )
    layout_path.write_text(layout_content)
    print("✅ Added Script import to layout.tsx")
else:
    print("✅ Script import already present")

# Build and deploy
subprocess.run('npm run build', shell=True, cwd=repo_path, check=True)
subprocess.run('git add -A', shell=True, cwd=repo_path, check=True)
subprocess.run('git commit -m "Add Paddle payment integration - checkout, webhooks, feature gating"', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main', shell=True, cwd=repo_path, check=True)

print("\n🚀 Deployed to Vercel")
print("🔗 https://govcon-dashboard-eta.vercel.app")

✅ Added Script import to layout.tsx


CalledProcessError: Command 'npm run build' returned non-zero exit status 1.

In [4]:
import subprocess
from pathlib import Path

repo_path = Path('/content/govcon-dashboard')

# Check the build error details
result = subprocess.run('npm run build', shell=True, cwd=repo_path, capture_output=True, text=True)
print("BUILD OUTPUT:")
print(result.stdout)
print("\nERRORS:")
print(result.stderr)

BUILD OUTPUT:

> nextjs@0.1.0 build
> next build

▲ Next.js 16.1.6 (Turbopack)

  Creating an optimized production build ...
✓ Compiled successfully in 21.2s
  Running TypeScript ...


ERRORS:
⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
Failed to compile.

./app/layout.tsx:33:8
Type error: Cannot find name 'Script'.

  31 |         <AuthProvider>{children}</AuthProvider>
  32 |       
> 33 |       <Script
     |        ^
  34 |         src="https://cdn.paddle.com/paddle/v2/paddle.js"
  35 |         strategy="lazyOnload"
  36 |         onLoad={() => {
Next.js build worker exited with code: 1 and signal: null



In [5]:
from pathlib import Path
import subprocess

repo_path = Path('/content/govcon-dashboard')
layout_path = repo_path / 'app' / 'layout.tsx'

# Read current content
content = layout_path.read_text()

# Show first 20 lines to see the imports
print("CURRENT IMPORTS:")
print('\n'.join(content.split('\n')[:20]))
print("\n" + "="*50 + "\n")

# Fix: Find the exact import block and add Script properly
lines = content.split('\n')
new_lines = []
script_added = False

for i, line in enumerate(lines):
    new_lines.append(line)
    # Add Script import right after the next import
    if "import type { Metadata } from 'next'" in line and not script_added:
        new_lines.append("import Script from 'next/script'")
        script_added = True
        print(f"✅ Adding Script import after line {i}")

# Write back
layout_path.write_text('\n'.join(new_lines))

print("\nNEW IMPORTS:")
print('\n'.join(new_lines[:25]))

CURRENT IMPORTS:
import type { Metadata } from "next";
import { Geist, Geist_Mono } from "next/font/google";
import { AuthProvider } from "../lib/auth-context";
import "./globals.css";

const geistSans = Geist({
  variable: "--font-geist-sans",
  subsets: ["latin"],
});

const geistMono = Geist_Mono({
  variable: "--font-geist-mono",
  subsets: ["latin"],
});

export const metadata: Metadata = {
  title: "GovCon Dashboard",
  description: "Government Contract Competitor Intelligence",
};




NEW IMPORTS:
import type { Metadata } from "next";
import { Geist, Geist_Mono } from "next/font/google";
import { AuthProvider } from "../lib/auth-context";
import "./globals.css";

const geistSans = Geist({
  variable: "--font-geist-sans",
  subsets: ["latin"],
});

const geistMono = Geist_Mono({
  variable: "--font-geist-mono",
  subsets: ["latin"],
});

export const metadata: Metadata = {
  title: "GovCon Dashboard",
  description: "Government Contract Competitor Intelligence",
};

export defaul

In [6]:
from pathlib import Path
import subprocess

repo_path = Path('/content/govcon-dashboard')
layout_path = repo_path / 'app' / 'layout.tsx'

# Read the file
content = layout_path.read_text()

# Replace the first line with both imports
content = content.replace(
    'import type { Metadata } from "next";',
    'import type { Metadata } from "next";\nimport Script from "next/script";'
)

# Write it back
layout_path.write_text(content)

print("✅ Fixed imports")
print("\nFirst 10 lines:")
print('\n'.join(content.split('\n')[:10]))

# Now build and deploy
print("\n🔨 Building...")
subprocess.run('npm run build', shell=True, cwd=repo_path, check=True)
subprocess.run('git add -A', shell=True, cwd=repo_path, check=True)
subprocess.run('git commit -m "Add Paddle payment integration - fix Script import"', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main', shell=True, cwd=repo_path, check=True)

print("\n🚀 Deployed to Vercel")
print("🔗 https://govcon-dashboard-eta.vercel.app")

✅ Fixed imports

First 10 lines:
import type { Metadata } from "next";
import Script from "next/script";
import { Geist, Geist_Mono } from "next/font/google";
import { AuthProvider } from "../lib/auth-context";
import "./globals.css";

const geistSans = Geist({
  variable: "--font-geist-sans",
  subsets: ["latin"],
});

🔨 Building...


CalledProcessError: Command 'npm run build' returned non-zero exit status 1.

In [7]:
import subprocess
from pathlib import Path

repo_path = Path('/content/govcon-dashboard')

# Check build error
result = subprocess.run('npm run build', shell=True, cwd=repo_path, capture_output=True, text=True)
print(result.stderr[-2000:])  # Last 2000 chars of error

⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
Error: supabaseUrl is required.
    at module evaluation (.next/server/chunks/[root-of-the-server]__268b7494._.js:1:1375)
    at instantiateModule (.next/server/chunks/[turbopack]_runtime.js:740:9)
    at instantiateRuntimeModule (.next/server/chunks/[turbopack]_runtime.js:768:12)
    at getOrInstantiateRuntimeModule (.next/server/chunks/[turbopack]_runtime.js:781:12)
    at Object.m (.next/server/chunks/[turbopack]_runtime.js:790:18)
    at Object.<anonymous> (.next/server/app/api/paddle/webhook/route.js:7:3)

> Build error occurred
Error: Failed to collect page data for /api/paddle/webhook
    at ignore-listed frames {
  type: 'Error'
}



In [8]:
from pathlib import Path
import subprocess

repo_path = Path('/content/govcon-dashboard')

# The webhook route is trying to use Supabase on build - fix it to only run at runtime
webhook_path = repo_path / 'app' / 'api' / 'paddle' / 'webhook' / 'route.ts'
webhook_content = webhook_path.read_text()

# Remove the top-level Supabase client that runs at build time
new_content = webhook_content.replace(
    """import { createClient } from '@supabase/supabase-js';
import crypto from 'crypto';

const supabase = createClient(
  process.env.NEXT_PUBLIC_SUPABASE_URL!,
  process.env.SUPABASE_SERVICE_ROLE_KEY!
);""",
    """import { createClient } from '@supabase/supabase-js';
import crypto from 'crypto';"""
)

# Move supabase client creation inside the POST function
new_content = new_content.replace(
    """export async function POST(request: NextRequest) {
  try {
    const body = await request.text();""",
    """export async function POST(request: NextRequest) {
  try {
    const supabase = createClient(
      process.env.NEXT_PUBLIC_SUPABASE_URL!,
      process.env.SUPABASE_SERVICE_ROLE_KEY!
    );

    const body = await request.text();"""
)

webhook_path.write_text(new_content)

print("✅ Fixed webhook route")

# Build and deploy
subprocess.run('npm run build', shell=True, cwd=repo_path, check=True)
subprocess.run('git add -A', shell=True, cwd=repo_path, check=True)
subprocess.run('git commit -m "Fix Paddle webhook - move Supabase client to runtime"', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main', shell=True, cwd=repo_path, check=True)

print("\n🚀 Deployed!")

✅ Fixed webhook route


CalledProcessError: Command 'npm run build' returned non-zero exit status 1.

In [9]:
from pathlib import Path
import subprocess

repo_path = Path('/content/govcon-dashboard')

# Check if repo exists
if not repo_path.exists():
    from google.colab import userdata
    token = userdata.get('govcon-deploy')
    subprocess.run(f'git clone https://{token}@github.com/menagoubran/govcon-dashboard.git /content/govcon-dashboard', shell=True, check=True)
    subprocess.run('npm install --legacy-peer-deps', shell=True, cwd=repo_path, check=True)

# Git config
subprocess.run('git config user.email "menagoubran@gmail.com"', shell=True, cwd=repo_path, check=True)
subprocess.run('git config user.name "Mena Goubran"', shell=True, cwd=repo_path, check=True)

# The issue: checkout route also has build-time Supabase import
# Fix both routes to be runtime-only

checkout_path = repo_path / 'app' / 'api' / 'paddle' / 'checkout' / 'route.ts'
checkout_content = """import { NextRequest, NextResponse } from 'next/server';
import { createClient } from '@supabase/supabase-js';

export async function POST(request: NextRequest) {
  try {
    const supabase = createClient(
      process.env.NEXT_PUBLIC_SUPABASE_URL!,
      process.env.SUPABASE_SERVICE_ROLE_KEY!
    );

    const { data: { user }, error: authError } = await supabase.auth.getUser();

    if (authError || !user) {
      return NextResponse.json({ error: 'Unauthorized' }, { status: 401 });
    }

    const { plan } = await request.json();
    const PADDLE_PRODUCTS = {
      PRO_MONTHLY: process.env.PADDLE_PRICE_PRO_MONTHLY!,
      PRO_ANNUAL: process.env.PADDLE_PRICE_PRO_ANNUAL!,
    };

    let { data: profile } = await supabase
      .from('contractor_profiles')
      .select('*')
      .eq('user_id', user.id)
      .single();

    if (!profile) {
      const { data: userProfile } = await supabase
        .from('user_profiles')
        .select('business_name, naics_codes, target_states')
        .eq('user_id', user.id)
        .single();

      const { data: newProfile, error: profileError } = await supabase
        .from('contractor_profiles')
        .insert({
          user_id: user.id,
          company_name: userProfile?.business_name || 'GovCon Pro User',
          naics_codes: userProfile?.naics_codes || [],
          target_states: userProfile?.target_states || [],
        })
        .select()
        .single();

      if (profileError) throw profileError;
      profile = newProfile;
    }

    const priceId = plan === 'annual'
      ? PADDLE_PRODUCTS.PRO_ANNUAL
      : PADDLE_PRODUCTS.PRO_MONTHLY;

    return NextResponse.json({
      priceId,
      email: user.email,
      passthrough: JSON.stringify({
        user_id: user.id,
        profile_id: profile.id,
      }),
      successUrl: `${process.env.NEXT_PUBLIC_SITE_URL}/home?upgrade=success`,
      closeUrl: `${process.env.NEXT_PUBLIC_SITE_URL}/home`,
    });

  } catch (error) {
    console.error('Paddle checkout error:', error);
    return NextResponse.json(
      { error: 'Failed to create checkout session' },
      { status: 500 }
    );
  }
}
"""
checkout_path.write_text(checkout_content)

webhook_path = repo_path / 'app' / 'api' / 'paddle' / 'webhook' / 'route.ts'
webhook_content = """import { NextRequest, NextResponse } from 'next/server';
import { createClient } from '@supabase/supabase-js';
import crypto from 'crypto';

function verifyWebhook(body: string, signature: string): boolean {
  const publicKey = process.env.PADDLE_WEBHOOK_SECRET!;
  const bodyObj = JSON.parse(body);
  const { p_signature, ...fields } = bodyObj;
  const sorted = Object.keys(fields)
    .sort()
    .map(key => `${key}=${fields[key]}`)
    .join('&');
  const verifier = crypto.createVerify('sha1');
  verifier.update(sorted);
  return verifier.verify(publicKey, p_signature, 'base64');
}

export async function POST(request: NextRequest) {
  try {
    const supabase = createClient(
      process.env.NEXT_PUBLIC_SUPABASE_URL!,
      process.env.SUPABASE_SERVICE_ROLE_KEY!
    );

    const body = await request.text();
    const signature = request.headers.get('x-paddle-signature') || '';
    const event = JSON.parse(body);
    const alertName = event.alert_name;

    console.log('Paddle webhook:', alertName, event);

    const passthrough = event.passthrough ? JSON.parse(event.passthrough) : {};
    const userId = passthrough.user_id;

    switch (alertName) {
      case 'subscription_created':
      case 'subscription_updated':
        await supabase.from('subscriptions').upsert({
          user_id: userId,
          paddle_subscription_id: event.subscription_id,
          paddle_customer_id: event.user_id,
          paddle_plan_id: event.subscription_plan_id,
          status: event.status,
          current_period_end: event.next_bill_date,
          cancel_at: event.cancellation_effective_date || null,
          updated_at: new Date().toISOString(),
        }, {
          onConflict: 'paddle_subscription_id'
        });
        break;

      case 'subscription_cancelled':
        await supabase
          .from('subscriptions')
          .update({
            status: 'cancelled',
            cancel_at: event.cancellation_effective_date,
            updated_at: new Date().toISOString(),
          })
          .eq('paddle_subscription_id', event.subscription_id);
        break;

      case 'subscription_payment_succeeded':
        await supabase
          .from('subscriptions')
          .update({
            status: 'active',
            current_period_end: event.next_bill_date,
            updated_at: new Date().toISOString(),
          })
          .eq('paddle_subscription_id', event.subscription_id);
        break;

      case 'subscription_payment_failed':
        await supabase
          .from('subscriptions')
          .update({
            status: 'past_due',
            updated_at: new Date().toISOString(),
          })
          .eq('paddle_subscription_id', event.subscription_id);
        break;
    }

    return NextResponse.json({ received: true });

  } catch (error) {
    console.error('Paddle webhook error:', error);
    return NextResponse.json(
      { error: 'Webhook handler failed' },
      { status: 500 }
    );
  }
}
"""
webhook_path.write_text(webhook_content)

print("✅ Fixed Paddle routes to be runtime-only")

# Build and deploy
subprocess.run('npm run build', shell=True, cwd=repo_path, check=True)
subprocess.run('git add -A', shell=True, cwd=repo_path, check=True)
subprocess.run('git commit -m "Fix Paddle routes - remove build-time Supabase imports"', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main', shell=True, cwd=repo_path, check=True)

print("\n🚀 Paddle integration deployed!")
print("🔗 https://govcon-dashboard-eta.vercel.app")
print("\n📋 Next steps:")
print("1. Run PADDLE_MIGRATION.sql in Supabase")
print("2. Build Terms & Privacy pages")
print("3. Wait for Paddle account approval")
print("4. Add Paddle env vars to Vercel")

✅ Fixed Paddle routes to be runtime-only


CalledProcessError: Command 'npm run build' returned non-zero exit status 1.

In [10]:
from pathlib import Path
from google.colab import userdata
import subprocess

# Check if repo exists, clone if needed
repo_path = Path('/content/govcon-dashboard')
if not repo_path.exists():
    token = userdata.get('govcon-deploy')
    subprocess.run(f'git clone https://{token}@github.com/menagoubran/govcon-dashboard.git /content/govcon-dashboard', shell=True, check=True)
    subprocess.run('npm install --legacy-peer-deps', shell=True, cwd=repo_path, check=True)

# Git config
subprocess.run('git config user.email "menagoubran@gmail.com"', shell=True, cwd=repo_path, check=True)
subprocess.run('git config user.name "Mena Goubran"', shell=True, cwd=repo_path, check=True)

# Build and deploy
subprocess.run('npm run build', shell=True, cwd=repo_path, check=True)
subprocess.run('git add -A', shell=True, cwd=repo_path, check=True)
subprocess.run('git commit -m "Paddle integration - checkout, webhooks, feature gating" --allow-empty', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main', shell=True, cwd=repo_path, check=True)

print("\n🚀 Deployed!")
print("🔗 https://govcon-dashboard-eta.vercel.app")

CalledProcessError: Command 'npm run build' returned non-zero exit status 1.

In [11]:
import subprocess
from pathlib import Path

repo_path = Path('/content/govcon-dashboard')

# Check build error
result = subprocess.run('npm run build', shell=True, cwd=repo_path, capture_output=True, text=True)
print("STDERR:")
print(result.stderr[-1500:])

STDERR:
⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
Error occurred prerendering page "/contractors/profile". Read more: https://nextjs.org/docs/messages/prerender-error
Error: Event handlers cannot be passed to Client Component props.
  {src: ..., strategy: ..., onLoad: function onLoad}
                                    ^^^^^^^^^^^^^^^
If you need interactivity, consider converting part of this to a Client Component.
    at stringify (<anonymous>) {
  digest: '1853724145'
}
Export encountered an error on /(dashboard)/contractors/profile/page: /contractors/profile, exiting the build.
⨯ Next.js build worker exited with code: 1 and signal: null



In [12]:
from pathlib import Path
import subprocess

repo_path = Path('/content/govcon-dashboard')

# Remove the Paddle script from layout.tsx - we'll add it a different way
layout_path = repo_path / 'app' / 'layout.tsx'
layout_content = layout_path.read_text()

# Remove the entire Paddle script block
if 'paddle' in layout_content.lower():
    # Find and remove the Script tag
    import re
    layout_content = re.sub(
        r'<Script\s+src="https://cdn\.paddle\.com.*?</Script>',
        '',
        layout_content,
        flags=re.DOTALL
    )
    layout_path.write_text(layout_content)
    print("✅ Removed Paddle script from layout")

# Create a client component for Paddle initialization instead
paddle_provider = repo_path / 'components' / 'PaddleProvider.tsx'
paddle_provider.write_text("""'use client';

import { useEffect } from 'react';
import Script from 'next/script';

export default function PaddleProvider() {
  useEffect(() => {
    if (typeof window !== 'undefined' && (window as any).Paddle) {
      const vendorId = process.env.NEXT_PUBLIC_PADDLE_VENDOR_ID;
      if (vendorId) {
        (window as any).Paddle.Setup({ vendor: parseInt(vendorId) });
      }
    }
  }, []);

  return (
    <Script
      src="https://cdn.paddle.com/paddle/v2/paddle.js"
      strategy="lazyOnload"
    />
  );
}
""")

# Update layout to use PaddleProvider
layout_content = layout_path.read_text()
if 'PaddleProvider' not in layout_content:
    # Add import
    layout_content = layout_content.replace(
        'import { AuthProvider } from "../lib/auth-context";',
        'import { AuthProvider } from "../lib/auth-context";\nimport PaddleProvider from "../components/PaddleProvider";'
    )
    # Add component inside body
    layout_content = layout_content.replace(
        '<AuthProvider>{children}</AuthProvider>',
        '<AuthProvider>{children}</AuthProvider>\n      <PaddleProvider />'
    )
    layout_path.write_text(layout_content)

print("✅ Fixed Paddle loading with client component")

# Build and deploy
subprocess.run('npm run build', shell=True, cwd=repo_path, check=True)
subprocess.run('git add -A', shell=True, cwd=repo_path, check=True)
subprocess.run('git commit -m "Fix Paddle script loading - use client component"', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main', shell=True, cwd=repo_path, check=True)

print("\n🚀 Deployed!")
print("🔗 https://govcon-dashboard-eta.vercel.app")

✅ Removed Paddle script from layout
✅ Fixed Paddle loading with client component


CalledProcessError: Command 'npm run build' returned non-zero exit status 1.

In [13]:
import subprocess
from pathlib import Path

repo_path = Path('/content/govcon-dashboard')

# Check the error again
result = subprocess.run('npm run build', shell=True, cwd=repo_path, capture_output=True, text=True)
print("BUILD ERROR:")
print(result.stderr[-1500:])

BUILD ERROR:
⚠ The "middleware" file convention is deprecated. Please use "proxy" instead. Learn more: https://nextjs.org/docs/messages/middleware-to-proxy
Error occurred prerendering page "/contractors/profile". Read more: https://nextjs.org/docs/messages/prerender-error
Error: Event handlers cannot be passed to Client Component props.
  {src: ..., strategy: ..., onLoad: function onLoad}
                                    ^^^^^^^^^^^^^^^
If you need interactivity, consider converting part of this to a Client Component.
    at stringify (<anonymous>) {
  digest: '1853724145'
}
Export encountered an error on /(dashboard)/contractors/profile/page: /contractors/profile, exiting the build.
⨯ Next.js build worker exited with code: 1 and signal: null



In [14]:
from pathlib import Path
import subprocess

repo_path = Path('/content/govcon-dashboard')

# Revert to last working commit (before Paddle)
subprocess.run('git reset --hard HEAD~3', shell=True, cwd=repo_path, check=True)
subprocess.run('git push origin main --force', shell=True, cwd=repo_path, check=True)

print("✅ Reverted Paddle changes - back to working state")
print("🔗 https://govcon-dashboard-eta.vercel.app")
print("\n📋 What do you want to build instead?")
print("- Terms & Privacy pages?")
print("- Dashboard improvements?")
print("- More AI features?")
print("- Something else?")

✅ Reverted Paddle changes - back to working state
🔗 https://govcon-dashboard-eta.vercel.app

📋 What do you want to build instead?
- Terms & Privacy pages?
- Dashboard improvements?
- More AI features?
- Something else?
